# nb_inventory_example_orchestrate_gold

Inputs are immutable run, context and prepared-attempt references.

Requires schema-enabled Fabric PySpark, Delta and notebookutils.

No active source contract or credentials are embedded. Missing dispatch proof fails closed.

Generate locally with deployment/build.py --metadata; deployment remains gated.


In [ ]:
run_id = ""
version_id = ""
manifest_path = ""
snapshot_sha256 = ""
dispatch_path = ""
dispatch_sha256 = ""
context_path = ""
context_sha256 = ""
task_id = ""
worker_id = "gold-worker"
concurrency = 2
operation = "prepare"
reconcile_observed_utc = ""
callback_path = ""
callback_sha256 = ""
file_arrival_path = ""
file_arrival_sha256 = ""
source_workspace_id = ""
source_lakehouse_id = ""
gold_preparation_json = ""
gold_reconciliation_path = ""
gold_reconciliation_sha256 = ""


In [ ]:
import hashlib, json, sys, types
RUNTIME_SOURCES = {'source_rules': '"""Generated generic-only case adapter; DQ specifications are pinned metadata."""\n\ndef source_business_checks(data, source):\n    return []\n\ndef row_rules(table):\n    return []\n', 'platform': '"""Microsoft Fabric REST client used by the reusable accelerator deployer."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom typing import Any\n\nFABRIC_API = "https://api.fabric.microsoft.com/v1"\n\n\nclass FabricApiError(RuntimeError):\n    """Raised when a Fabric REST operation fails."""\n\n\n@dataclass(frozen=True)\nclass FabricItem:\n    """Minimal Fabric item identity."""\n\n    id: str\n    display_name: str\n    type: str\n    folder_id: str | None = None\n\n\n@dataclass(frozen=True)\nclass ApiResponse:\n    """HTTP response data used by the client."""\n\n    status_code: int\n    headers: Any\n    content: bytes\n\n    def json(self) -> dict[str, Any]:\n        """Decode the JSON response body."""\n        return json.loads(self.content.decode("utf-8")) if self.content else {}\n\n\nclass FabricClient:\n    """Small Fabric REST client with pagination and LRO handling."""\n\n    def __init__(\n        self,\n        token_provider: Callable[[], str],\n        *,\n        timeout_seconds: int = 120,\n        lro_timeout_seconds: int = 1800,\n        operation_observer: Callable[[dict[str, str]], None] | None = None,\n    ) -> None:\n        self.token_provider = token_provider\n        self.timeout_seconds = timeout_seconds\n        self.lro_timeout_seconds = lro_timeout_seconds\n        self.operation_observer = operation_observer\n\n    def _headers(self) -> dict[str, str]:\n        return {\n            "Authorization": f"Bearer {self.token_provider()}",\n            "Content-Type": "application/json",\n            "x-ms-fabric-skill": "spark-cli",\n        }\n\n    def request(\n        self,\n        method: str,\n        path_or_url: str,\n        *,\n        body: dict[str, Any] | None = None,\n        expected: tuple[int, ...] = (200,),\n    ) -> ApiResponse:\n        """Send one request and raise an actionable error on failure."""\n        url = (\n            path_or_url\n            if path_or_url.startswith("https://")\n            else f"{FABRIC_API}{path_or_url}"\n        )\n        data = json.dumps(body).encode("utf-8") if body is not None else None\n        for attempt in range(5):\n            request = urllib.request.Request(\n                url,\n                data=data,\n                headers=self._headers(),\n                method=method,\n            )\n            try:\n                with urllib.request.urlopen(\n                    request, timeout=self.timeout_seconds\n                ) as response:\n                    result = ApiResponse(\n                        status_code=response.status,\n                        headers=response.headers,\n                        content=response.read(),\n                    )\n                break\n            except urllib.error.HTTPError as error:\n                retryable = method.upper() == "GET" and (\n                    error.code == 429 or error.code >= 500\n                )\n                if retryable and attempt < 4:\n                    delay = int(error.headers.get("Retry-After", "5").split(",")[0])\n                    time.sleep(max(delay, 1))\n                    continue\n                detail = error.read().decode("utf-8", errors="replace").strip()\n                raise FabricApiError(\n                    f"{method} {url} failed with HTTP {error.code}: {detail}"\n                ) from error\n        else:\n            raise FabricApiError(f"{method} {url} exhausted retry attempts")\n        if result.status_code not in expected:\n            detail = result.content.decode("utf-8", errors="replace").strip()\n            raise FabricApiError(\n                f"{method} {url} failed with HTTP {result.status_code}: {detail}"\n            )\n        return result\n\n    def request_lro(\n        self,\n        method: str,\n        path: str,\n        *,\n        body: dict[str, Any] | None = None,\n        synchronous: tuple[int, ...] = (200, 201),\n    ) -> dict[str, Any]:\n        """Send a Fabric operation and wait for an accepted LRO."""\n        response = self.request(\n            method,\n            path,\n            body=body,\n            expected=(*synchronous, 202),\n        )\n        if response.status_code in synchronous:\n            return response.json()\n\n        location = response.headers.get("Location")\n        if not location:\n            raise FabricApiError(f"{method} {path} returned 202 without Location")\n        if self.operation_observer is not None:\n            self.operation_observer(\n                {"method": method, "request_path": path, "operation_location": location}\n            )\n        return self._wait_for_operation(location, f"{method} {path}")\n\n    def _wait_for_operation(self, location: str, operation_name: str) -> dict[str, Any]:\n        """Wait for one Fabric long-running operation."""\n        parsed = urllib.parse.urlparse(location)\n        if (\n            parsed.hostname\n            and parsed.hostname.startswith("wabi-")\n            and parsed.hostname.endswith(".analysis.windows.net")\n            and parsed.path.startswith("/v1/operations/")\n        ):\n            location = "https://api.fabric.microsoft.com" + parsed.path\n        delay = 5\n        deadline = time.monotonic() + self.lro_timeout_seconds\n        while time.monotonic() < deadline:\n            time.sleep(max(delay, 1))\n            poll = self.request("GET", location, expected=(200,))\n            payload = poll.json()\n            status = payload.get("status")\n            if status == "Succeeded":\n                return payload\n            if status == "Failed":\n                raise FabricApiError(\n                    f"{operation_name} failed: {payload.get(\'error\', payload)}"\n                )\n            delay = int(poll.headers.get("Retry-After", str(delay)).split(",")[0])\n        raise FabricApiError(f"{operation_name} did not complete before timeout")\n\n    def paged(self, path: str) -> list[dict[str, Any]]:\n        """Return every value from a paginated Fabric collection."""\n        results: list[dict[str, Any]] = []\n        next_url: str | None = path\n        while next_url:\n            payload = self.request("GET", next_url, expected=(200,)).json()\n            results.extend(payload.get("value", []))\n            next_url = payload.get("continuationUri")\n        return results\n\n    def resolve_workspace(self, name_or_id: str) -> dict[str, Any]:\n        """Resolve exactly one existing workspace."""\n        if len(name_or_id) == 36 and name_or_id.count("-") == 4:\n            return self.request(\n                "GET", f"/workspaces/{name_or_id}", expected=(200,)\n            ).json()\n        matches = [\n            workspace\n            for workspace in self.paged("/workspaces")\n            if workspace.get("displayName") == name_or_id\n        ]\n        if len(matches) != 1:\n            raise FabricApiError(\n                f"Expected one workspace named {name_or_id!r}; found {len(matches)}"\n            )\n        return self.request(\n            "GET", f"/workspaces/{matches[0][\'id\']}", expected=(200,)\n        ).json()\n\n    def resolve_oauth2_connection(\n        self, name_or_id: str, connection_type: str\n    ) -> dict[str, Any]:\n        """Resolve one OAuth2 cloud connection by name and type."""\n        return self.resolve_managed_connection(\n            name_or_id,\n            connection_type,\n            credential_type="OAuth2",\n            connectivity_types={"ShareableCloud"},\n        )\n\n    def resolve_managed_connection(\n        self,\n        name_or_id: str,\n        connection_type: str,\n        *,\n        credential_type: str,\n        connectivity_types: set[str],\n        gateway_id: str | None = None,\n    ) -> dict[str, Any]:\n        """Resolve managed references without requesting or handling credentials."""\n        try:\n            connection_id = str(uuid.UUID(name_or_id))\n        except ValueError:\n            matches = [\n                connection\n                for connection in self.paged("/connections")\n                if connection.get("displayName") == name_or_id\n                and connection.get("connectionDetails", {}).get("type")\n                == connection_type\n            ]\n            if len(matches) != 1:\n                raise FabricApiError(\n                    f"Expected one {connection_type} {credential_type} connection named "\n                    f"{name_or_id!r}; found {len(matches)}. Create and authorize "\n                    "the connection in Fabric, or pass its ID with the "\n                    "corresponding --<source>-sql-connection option."\n                )\n            connection_id = matches[0]["id"]\n        connection = self.request(\n            "GET", f"/connections/{connection_id}", expected=(200,)\n        ).json()\n        actual_type = connection.get("connectionDetails", {}).get("type")\n        if actual_type != connection_type:\n            raise FabricApiError(\n                f"Connection {name_or_id!r} must have type {connection_type}; "\n                f"found {actual_type or \'no connection type\'}"\n            )\n        if connection.get("connectivityType") not in connectivity_types:\n            raise FabricApiError(\n                f"Connection {name_or_id!r} must use {sorted(connectivity_types)} "\n                f"connection; found {connection.get(\'connectivityType\')}"\n            )\n        actual_credential = connection.get("credentialDetails", {}).get(\n            "credentialType"\n        )\n        if actual_credential != credential_type:\n            raise FabricApiError(\n                f"Connection {name_or_id!r} must use {credential_type} credentials; "\n                f"found {actual_credential or \'no credential type\'}"\n            )\n        actual_gateway = connection.get("gatewayId")\n        if connection.get("connectivityType") != "ShareableCloud":\n            try:\n                if (\n                    not isinstance(actual_gateway, str)\n                    or str(uuid.UUID(actual_gateway)) != actual_gateway\n                ):\n                    raise ValueError\n            except ValueError:\n                raise FabricApiError(\n                    f"Connection {name_or_id!r} requires a resolved gateway reference"\n                ) from None\n        if gateway_id is not None and actual_gateway != gateway_id:\n            raise FabricApiError(\n                f"Connection {name_or_id!r} differs from its declared gateway reference"\n            )\n        return connection\n\n    def validate_connection_online(self, connection: dict[str, Any]) -> None:\n        """Fail before deployment when a connection is offline or expired."""\n        connection_id = connection["id"]\n        path = f"/connections/{connection_id}/testConnection"\n        response = self.request("POST", path, expected=(200, 202))\n        if response.status_code == 200:\n            result = response.json()\n        else:\n            location = response.headers.get("Location")\n            if not location:\n                raise FabricApiError(f"POST {path} returned 202 without Location")\n            self._wait_for_operation(location, f"Test connection {connection_id}")\n            result = self.request("GET", f"{location}/result", expected=(200,)).json()\n        if result.get("status") != "Online":\n            errors = result.get("errors") or []\n            details = "; ".join(error.get("message", str(error)) for error in errors)\n            if not details:\n                details = (\n                    "Reauthorize or repair it in Manage connections and "\n                    "gateways before deployment."\n                )\n            raise FabricApiError(\n                f"Connection {connection.get(\'displayName\', connection_id)!r} "\n                f"is {result.get(\'status\', \'not online\')}. {details}"\n            )\n\n    def list_items(self, workspace_id: str) -> list[FabricItem]:\n        """List generic workspace items."""\n        return [\n            FabricItem(\n                id=item["id"],\n                display_name=item["displayName"],\n                type=item["type"],\n                folder_id=item.get("folderId"),\n            )\n            for item in self.paged(f"/workspaces/{workspace_id}/items")\n        ]\n\n    def find_item(\n        self, workspace_id: str, display_name: str, item_type: str\n    ) -> FabricItem | None:\n        """Find a unique item by display name and type."""\n        matches = [\n            item\n            for item in self.list_items(workspace_id)\n            if item.display_name == display_name\n            and item.type.casefold() == item_type.casefold()\n        ]\n        if len(matches) > 1:\n            raise FabricApiError(\n                f"Multiple {item_type} items are named {display_name!r}"\n            )\n        return matches[0] if matches else None\n\n    def create_item(\n        self,\n        workspace_id: str,\n        route: str,\n        display_name: str,\n        *,\n        description: str = "",\n        definition: dict[str, Any] | None = None,\n        creation_payload: dict[str, Any] | None = None,\n        folder_id: str | None = None,\n    ) -> FabricItem:\n        """Create an item and resolve its identity after provisioning."""\n        body: dict[str, Any] = {\n            "displayName": display_name,\n            "description": description,\n        }\n        if definition is not None:\n            body["definition"] = definition\n        if creation_payload is not None:\n            body["creationPayload"] = creation_payload\n        if folder_id is not None:\n            body["folderId"] = folder_id\n        self.request_lro(\n            "POST",\n            f"/workspaces/{workspace_id}/{route}",\n            body=body,\n            synchronous=(201,),\n        )\n        item_type = {\n            "lakehouses": "Lakehouse",\n            "sqlDatabases": "SQLDatabase",\n            "notebooks": "Notebook",\n            "dataPipelines": "DataPipeline",\n        }[route]\n        for _ in range(12):\n            item = self.find_item(workspace_id, display_name, item_type)\n            if item is not None:\n                return item\n            time.sleep(5)\n        raise FabricApiError(f"Created {item_type} {display_name!r} was not found")\n\n    def update_definition(\n        self,\n        workspace_id: str,\n        item_id: str,\n        definition: dict[str, Any],\n    ) -> None:\n        """Update one item definition."""\n        self.request_lro(\n            "POST",\n            f"/workspaces/{workspace_id}/items/{item_id}/updateDefinition",\n            body={"definition": definition},\n            synchronous=(200,),\n        )\n\n    def list_folders(self, workspace_id: str) -> list[dict[str, Any]]:\n        """List every workspace folder."""\n        return self.paged(f"/workspaces/{workspace_id}/folders")\n\n    def create_folder(self, workspace_id: str, display_name: str) -> str:\n        """Create one root workspace folder and return its ID."""\n        payload = self.request(\n            "POST",\n            f"/workspaces/{workspace_id}/folders",\n            body={"displayName": display_name},\n            expected=(201,),\n        ).json()\n        return payload["id"]\n\n    def move_items(\n        self, workspace_id: str, item_ids: list[str], folder_id: str\n    ) -> None:\n        """Move up to 50 items into a workspace folder."""\n        self.request(\n            "POST",\n            f"/workspaces/{workspace_id}/items/bulkMove",\n            body={"items": item_ids, "targetFolderId": folder_id},\n            expected=(200,),\n        )\n\n    @staticmethod\n    def _mlv_path(workspace_id: str, lakehouse_id: str) -> str:\n        """Use the Lakehouse-owned MLV scheduler, not generic RunNotebook."""\n        workspace_id = str(uuid.UUID(workspace_id))\n        lakehouse_id = str(uuid.UUID(lakehouse_id))\n        return f"/workspaces/{workspace_id}/lakehouses/{lakehouse_id}"\n\n    def list_mlv_refresh_schedules(\n        self, workspace_id: str, lakehouse_id: str\n    ) -> list[dict[str, Any]]:\n        """List full/selected-lineage schedules, including disabled schedules."""\n        path = self._mlv_path(workspace_id, lakehouse_id)\n        return self._mlv_collection(\n            f"{path}/jobs/refreshMaterializedLakeViews/schedules"\n        )\n\n    def list_mlv_refresh_jobs(\n        self, workspace_id: str, lakehouse_id: str\n    ) -> list[dict[str, Any]]:\n        """Read MLV history without unsupported OData filters."""\n        path = self._mlv_path(workspace_id, lakehouse_id)\n        return self._mlv_collection(\n            f"{path}/jobs/refreshMaterializedLakeViews/instances"\n        )\n\n    def _mlv_collection(self, path: str) -> list[dict[str, Any]]:\n        """Read all pages, including token-only MLV history responses."""\n        results, seen = [], set()\n        next_path: str | None = path\n        while next_path:\n            if next_path in seen:\n                raise FabricApiError("Repeated MLV continuation page")\n            seen.add(next_path)\n            payload = self.request("GET", next_path).json()\n            if not isinstance(payload.get("value"), list):\n                raise FabricApiError("Malformed MLV collection response")\n            results.extend(payload["value"])\n            next_path = payload.get("continuationUri")\n            token = payload.get("continuationToken")\n            if next_path:\n                parsed = urllib.parse.urlsplit(next_path)\n                if (\n                    parsed.scheme != "https"\n                    or parsed.netloc != "api.fabric.microsoft.com"\n                    or parsed.path != f"/v1{path}"\n                    or parsed.fragment\n                ):\n                    raise FabricApiError("Untrusted MLV continuation URI")\n            elif token:\n                next_path = (\n                    path + "?" + urllib.parse.urlencode({"continuationToken": token})\n                )\n        return results\n\n    def get_mlv_execution_definition(\n        self, workspace_id: str, lakehouse_id: str, definition_id: str\n    ) -> dict[str, Any]:\n        """Read a saved selected-lineage configuration; not a visual graph API."""\n        path = self._mlv_path(workspace_id, lakehouse_id)\n        return self.request(\n            "GET",\n            f"{path}/mlvexecutiondefinitions/{uuid.UUID(definition_id)}",\n        ).json()\n\n    def submit_mlv_refresh(\n        self,\n        workspace_id: str,\n        lakehouse_id: str,\n        *,\n        execution_definition_id: str | None = None,\n    ) -> dict[str, Any]:\n        """Submit ONCE; caller must persist intent before calling this method.\n\n        See the Lakehouse Run On Demand Refresh Materialized Lake Views REST\n        reference. A missing/malformed receipt is uncertain, never permission to\n        retry. No polling, POST retries or fabricated per-view mode are added.\n        """\n        path = self._mlv_path(workspace_id, lakehouse_id)\n        body = (\n            {\n                "executionData": {\n                    "mlvExecutionDefinitionId": str(uuid.UUID(execution_definition_id))\n                }\n            }\n            if execution_definition_id is not None\n            else None\n        )\n        response = self.request(\n            "POST",\n            f"{path}/jobs/refreshMaterializedLakeViews/instances",\n            body=body,\n            expected=(202,),\n        )\n        location = response.headers.get("Location", "")\n        parsed = urllib.parse.urlsplit(location)\n        # Both Location shapes are documented. Do not send a bearer token to\n        # arbitrary response-provided hosts, paths, or another Lakehouse.\n        allowed = (\n            (\n                f"/v1/workspaces/{uuid.UUID(workspace_id)}/items/"\n                f"{uuid.UUID(lakehouse_id)}/jobs/instances/"\n            ),\n            f"/v1{path}/jobs/instances/",\n        )\n        prefix = next((p for p in allowed if parsed.path.startswith(p)), None)\n        if (\n            parsed.scheme != "https"\n            or parsed.netloc != "api.fabric.microsoft.com"\n            or parsed.query\n            or parsed.fragment\n            or prefix is None\n        ):\n            raise FabricApiError("MLV submission receipt has no trusted job Location")\n        try:\n            job_id = str(uuid.UUID(parsed.path[len(prefix) :]))\n            delay = int(response.headers["Retry-After"])\n            if delay < 0:\n                raise ValueError("negative delay")\n        except (ValueError, KeyError) as error:\n            raise FabricApiError("MLV submission receipt is incomplete") from error\n        return {"job_id": job_id, "location": location, "retry_after": delay}\n\n    def get_mlv_refresh_job(\n        self, workspace_id: str, lakehouse_id: str, job_id: str\n    ) -> dict[str, Any]:\n        """Read the documented item-scoped status; status is lineage-level."""\n        self._mlv_path(workspace_id, lakehouse_id)\n        return self.request(\n            "GET",\n            f"/workspaces/{uuid.UUID(workspace_id)}/items/"\n            f"{uuid.UUID(lakehouse_id)}/jobs/instances/{uuid.UUID(job_id)}",\n        ).json()\n\n    def cancel_mlv_refresh_job(\n        self, workspace_id: str, lakehouse_id: str, job_id: str\n    ) -> None:\n        """Request cancellation once; cancellation is not completion evidence."""\n        self._mlv_path(workspace_id, lakehouse_id)\n        self.request(\n            "POST",\n            f"/workspaces/{uuid.UUID(workspace_id)}/items/"\n            f"{uuid.UUID(lakehouse_id)}/jobs/instances/{uuid.UUID(job_id)}/cancel",\n            expected=(202,),\n        )\n', 'contracts': '"""Pure, immutable metadata contracts shared by deployment and notebook code.\n\nLoading a definition validates its structure; it does not authorise a\nconnection, approve a configuration or prove a Fabric capability.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport heapq\nimport json\nimport re\nimport uuid\nfrom collections.abc import Mapping\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom typing import cast\n\n\nclass MetadataError(ValueError):\n    """A configuration is unsafe, ambiguous or structurally incompatible."""\n\n\nIDENTIFIER = re.compile(r"[A-Za-z_][A-Za-z0-9_]{0,127}\\Z")\nTOKEN = re.compile(r"[a-z][a-z0-9_]{0,62}\\Z")\nIDENTITY = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.:-]{0,127}\\Z")\nSHA256 = re.compile(r"[0-9a-f]{64}\\Z")\nLAYERS = ("bronze", "silver", "gold")\nMATERIALIZATIONS = {\n    "bronze": {"files"},\n    "silver": {"append_versions"},\n    "gold": {"scd2_dimension", "atomic_fact", "snapshot_fact", "calendar", "mlv"},\n}\nCONNECTORS = {"sql": {"FabricSql", "SqlServer"}, "ftp": {"Ftp"}, "sftp": {"Sftp"}}\nSQL_AUTHENTICATION = {"FabricSql": {"OAuth2"}, "SqlServer": {"Basic"}}\nDATA_TYPES = {\n    "int",\n    "bigint",\n    "money",\n    "decimal",\n    "float",\n    "bit",\n    "boolean",\n    "datetime",\n    "uuid",\n    "varchar",\n    "string",\n    "long",\n    "double",\n    "date",\n    "timestamp",\n    "timestamp_ntz",\n}\nCOLUMN_FIELDS = {\n    "name",\n    "data_type",\n    "nullable",\n    "precision",\n    "scale",\n    "max_length",\n    "min_length",\n    "fixed_length",\n    "minimum",\n    "maximum",\n    "format",\n    "pattern",\n    "classification",\n    "source_type",\n    "unit",\n    "code_list",\n    "constraint_origin",\n    "default",\n    "extraction_repair",\n    "generation_policy",\n    "group_id",\n    "pdf_pages",\n    "source_field",\n    "source_group",\n    "source_rules",\n    "specification_gap",\n}\nSECRET_FIELDS = {\n    "password",\n    "accesstoken",\n    "access_token",\n    "refreshtoken",\n    "refresh_token",\n    "clientsecret",\n    "client_secret",\n    "privatekey",\n    "private_key",\n    "connectionstring",\n    "connection_string",\n}\n\n\ndef _reject_credentials(\n    value: object, location: str = "configuration", depth: int = 0\n) -> None:\n    if depth > 100:\n        raise MetadataError("Metadata nesting exceeds the bounded contract depth")\n    if isinstance(value, dict):\n        for key, child in value.items():\n            if not isinstance(key, str):\n                raise MetadataError(f"{location}: metadata keys must be strings")\n            if key.casefold() in SECRET_FIELDS:\n                raise MetadataError(f"{location}: credentials are prohibited")\n            _reject_credentials(child, location, depth + 1)\n    elif isinstance(value, (list, tuple)):\n        for child in value:\n            _reject_credentials(child, location, depth + 1)\n\n\ndef canonical_json(value: object) -> str:\n    _reject_credentials(value)\n    try:\n        return json.dumps(\n            value,\n            sort_keys=True,\n            ensure_ascii=True,\n            separators=(",", ":"),\n            allow_nan=False,\n        )\n    except (TypeError, ValueError):\n        raise MetadataError(\n            "Metadata must contain JSON values and finite numbers"\n        ) from None\n\n\ndef metadata_digest(value: object) -> str:\n    return hashlib.sha256(canonical_json(value).encode("ascii")).hexdigest()\n\n\ndef _record(value: object, location: str, fields: set[str]) -> dict[str, object]:\n    if not isinstance(value, dict) or not all(isinstance(k, str) for k in value):\n        raise MetadataError(f"{location}: expected an object")\n    unknown = set(value) - fields\n    if unknown:\n        raise MetadataError(f"{location}: unknown fields")\n    return cast(dict[str, object], value)\n\n\ndef _text(value: object, location: str, pattern: re.Pattern[str] = IDENTITY) -> str:\n    if not isinstance(value, str) or not pattern.fullmatch(value):\n        raise MetadataError(f"{location}: invalid identifier")\n    return value\n\n\ndef _description(value: object, location: str) -> str:\n    if not isinstance(value, str) or not 1 <= len(value) <= 2000:\n        raise MetadataError(f"{location}: invalid description")\n    return value\n\n\ndef _boolean(value: object, location: str) -> bool:\n    if type(value) is not bool:\n        raise MetadataError(f"{location}: expected a Boolean flag")\n    return value\n\n\ndef _list(value: object, location: str) -> list[object]:\n    if not isinstance(value, list):\n        raise MetadataError(f"{location}: expected an ordered array")\n    return value\n\n\ndef _names(value: object, location: str) -> tuple[str, ...]:\n    result = tuple(_text(v, location, IDENTIFIER) for v in _list(value, location))\n    if len({v.casefold() for v in result}) != len(result):\n        raise MetadataError(f"{location}: duplicate column")\n    return result\n\n\ndef _identities(value: object, location: str) -> tuple[str, ...]:\n    result = tuple(_text(v, location) for v in _list(value, location))\n    if len(set(result)) != len(result):\n        raise MetadataError(f"{location}: duplicate identity")\n    return result\n\n\ndef _hash(value: object, location: str) -> str:\n    if not isinstance(value, str) or not SHA256.fullmatch(value):\n        raise MetadataError(f"{location}: expected a lowercase SHA-256 hash")\n    return value\n\n\ndef _uuid(value: object, location: str) -> str | None:\n    if value is None:\n        return None\n    if not isinstance(value, str):\n        raise MetadataError(f"{location}: invalid UUID")\n    try:\n        parsed = str(uuid.UUID(value))\n    except ValueError:\n        raise MetadataError(f"{location}: invalid UUID") from None\n    if parsed != value:\n        raise MetadataError(f"{location}: UUID must be canonical")\n    return parsed\n\n\n@dataclass(frozen=True)\nclass ConnectionSpec:\n    id: str\n    name: str\n    connector_type: str\n    authentication_type: str\n    fabric_connection_id: str | None\n    gateway_id: str | None\n\n\n@dataclass(frozen=True)\nclass SqlSourceGeometry:\n    workspace_id: str\n    artifact_id: str\n\n\n@dataclass(frozen=True)\nclass FeedSpec:\n    id: str\n    source_code: str\n    source_instance: str\n    source_type: str\n    description_slug: str\n    description: str\n    connection_id: str\n    silver_schema: str\n    quarantine_schema: str\n    authority: str\n    enabled: bool\n    legacy_contract_hash: str | None\n    legacy_bronze_root: str | None\n    geometry: SqlSourceGeometry | None\n\n    @property\n    def bronze_prefix(self) -> str:\n        return (\n            f"Files/bronze/{self.source_code}_"\n            f"{self.description_slug}_{self.source_type}"\n        )\n\n\n@dataclass(frozen=True)\nclass ColumnSpec:\n    name: str\n    ordinal: int\n    data_type: str\n    nullable: bool\n    definition_json: str\n\n\n@dataclass(frozen=True)\nclass LayoutSpec:\n    strategy: str\n    partition_columns: tuple[str, ...]\n    cluster_by_columns: tuple[str, ...]\n\n\n@dataclass(frozen=True)\nclass ObjectSpec:\n    id: str\n    feed_id: str | None\n    layer: str\n    name: str\n    target_schema: str | None\n    lakehouse: str\n    materialization: str\n    enabled: bool\n    execution_priority: int\n    business_keys: tuple[str, ...]\n    columns: tuple[ColumnSpec, ...]\n    contract_json: str\n    worker_id: str | None\n    layout: LayoutSpec | None\n    required_capabilities: tuple[str, ...]\n    existing_target: bool\n\n    @property\n    def target(self) -> tuple[str, str | None, str]:\n        return self.lakehouse, self.target_schema, self.name\n\n\n@dataclass(frozen=True)\nclass DependencySpec:\n    upstream_id: str\n    downstream_id: str\n    condition: str\n\n\n@dataclass(frozen=True)\nclass RuleSpec:\n    id: str\n    object_id: str\n    handler: str\n    version: str\n    severity: str\n    action: str\n    parameters_json: str\n\n\n@dataclass(frozen=True)\nclass TransformationSpec:\n    id: str\n    object_id: str\n    kind: str\n    version: str\n    definition_json: str\n    definition_sha256: str\n\n\n@dataclass(frozen=True)\nclass ConstraintSpec:\n    id: str\n    object_id: str\n    kind: str\n    enforcement: str\n    definition_json: str\n\n\n@dataclass(frozen=True)\nclass WorkerSpec:\n    id: str\n    notebook_name: str\n    lakehouse: str\n    code_sha256: str\n\n\n@dataclass(frozen=True)\nclass CapabilitySpec:\n    id: str\n    verified: bool\n    evidence_sha256: str | None\n\n\n@dataclass(frozen=True)\nclass Configuration:\n    version_id: str\n    owner: str\n    connections: tuple[ConnectionSpec, ...]\n    feeds: tuple[FeedSpec, ...]\n    objects: tuple[ObjectSpec, ...]\n    dependencies: tuple[DependencySpec, ...]\n    rules: tuple[RuleSpec, ...]\n    constraints: tuple[ConstraintSpec, ...]\n    transformations: tuple[TransformationSpec, ...]\n    workers: tuple[WorkerSpec, ...]\n    capabilities: tuple[CapabilitySpec, ...]\n    definition_json: str\n\n    @property\n    def definition_sha256(self) -> str:\n        return hashlib.sha256(self.definition_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.definition_json))\n\n\ndef sql_copy_projection(column: Mapping[str, object]) -> str:\n    """Compile the allowlisted SQL-to-text encodings shared by legacy and metadata Copy."""\n    name = "[" + _text(column.get("name"), "SQL projection column", IDENTIFIER) + "]"\n    kind = column.get("data_type")\n    if kind == "boolean":\n        return (\n            f"CASE WHEN {name} IS NULL THEN NULL WHEN {name}=1 "\n            f"THEN N\'true\' ELSE N\'false\' END AS {name}"\n        )\n    if kind == "bit":\n        return f"CONVERT(char(1),CONVERT(int,{name})) AS {name}"\n    if kind == "datetime" and column.get("format") != "CSD_TIMESTAMP":\n        return (\n            f"CONVERT(char(19),{name},126)+\'.\'+"\n            f"RIGHT(\'00\'+CONVERT(varchar(3),DATEPART(millisecond,{name})),3) AS {name}"\n        )\n    if kind in {"timestamp", "timestamp_ntz"}:\n        return (\n            f"CONVERT(char(19),{name},126)+\'.\'+"\n            f"RIGHT(\'00000\'+CONVERT(varchar(6),DATEPART(microsecond,{name})),6) AS {name}"\n        )\n    if kind in {"money", "decimal"}:\n        precision, scale = (\n            (19, 4)\n            if kind == "money"\n            else (column.get("precision"), column.get("scale"))\n        )\n        if (\n            type(precision) is not int\n            or type(scale) is not int\n            or not 0 <= scale <= precision <= 38\n            or precision == 0\n        ):\n            raise MetadataError(\n                "SQL decimal projection needs explicit precision and scale"\n            )\n        return f"CONVERT(varchar(50),CAST({name} AS decimal({precision},{scale}))) AS {name}"\n    if column.get("format") == "HHMMSS":\n        return f"RIGHT(\'000000\'+CONVERT(varchar(6),{name}),6) AS {name}"\n    if kind == "uuid":\n        return f"LOWER(CONVERT(char(36),{name})) AS {name}"\n    if kind == "date":\n        return f"CONVERT(char(10),{name},23) AS {name}"\n    if not isinstance(kind, str) or kind not in DATA_TYPES:\n        raise MetadataError("Unsupported SQL Copy column type")\n    return name\n\n\ndef _column(value: object, ordinal: int, location: str) -> ColumnSpec:\n    row = _record(value, location, COLUMN_FIELDS)\n    name = _text(row.get("name"), location + ".name", IDENTIFIER)\n    kind = row.get("data_type")\n    if not isinstance(kind, str) or kind not in DATA_TYPES:\n        raise MetadataError(f"{location}: unsupported column type")\n    nullable = _boolean(row.get("nullable"), location + ".nullable")\n    if kind == "decimal":\n        precision, scale = row.get("precision"), row.get("scale")\n        if (\n            type(precision) is not int\n            or type(scale) is not int\n            or not 0 <= scale <= precision <= 38\n            or precision == 0\n        ):\n            raise MetadataError(f"{location}: invalid precision or scale")\n    for field in ("max_length", "min_length", "fixed_length"):\n        n = row.get(field)\n        if n is not None and (type(n) is not int or n < 0):\n            raise MetadataError(f"{location}: invalid length constraint")\n    return ColumnSpec(name, ordinal, kind, nullable, canonical_json(row))\n\n\ndef _layout(\n    value: object, columns: tuple[ColumnSpec, ...], location: str\n) -> LayoutSpec:\n    row = _record(\n        value, location, {"layout_strategy", "partition_columns", "cluster_by_columns"}\n    )\n    strategy = row.get("layout_strategy")\n    if not isinstance(strategy, str) or strategy not in {"partitioned", "liquid"}:\n        raise MetadataError(f"{location}: select one supported layout strategy")\n    partitions = _names(row.get("partition_columns"), location + ".partition_columns")\n    clusters = _names(row.get("cluster_by_columns"), location + ".cluster_by_columns")\n    if strategy == "partitioned" and (not partitions or clusters):\n        raise MetadataError(\n            f"{location}: missing selected or conflicting inactive columns"\n        )\n    if strategy == "liquid" and (not clusters or partitions):\n        raise MetadataError(\n            f"{location}: missing selected or conflicting inactive columns"\n        )\n    if not set(partitions + clusters) <= {c.name for c in columns}:\n        raise MetadataError(f"{location}: unknown layout column")\n    return LayoutSpec(cast(str, strategy), partitions, clusters)\n\n\ndef _object(value: object) -> ObjectSpec:\n    row = _record(\n        value,\n        "object",\n        {\n            "id",\n            "feed_id",\n            "layer",\n            "name",\n            "target_schema",\n            "lakehouse",\n            "materialization",\n            "enabled",\n            "execution_priority",\n            "business_keys",\n            "columns",\n            "contract",\n            "worker_id",\n            "layout",\n            "required_capabilities",\n            "existing_target",\n        },\n    )\n    identity = _text(row.get("id"), "object.id")\n    layer = row.get("layer")\n    if not isinstance(layer, str) or layer not in LAYERS:\n        raise MetadataError(f"{identity}: unknown layer")\n    materialization = row.get("materialization")\n    if (\n        not isinstance(materialization, str)\n        or materialization not in MATERIALIZATIONS[layer]\n    ):\n        raise MetadataError(f"{identity}: unsupported materialization for layer")\n    columns = tuple(\n        _column(c, ordinal, f"{identity}.columns")\n        for ordinal, c in enumerate(_list(row.get("columns"), identity), 1)\n    )\n    names = [c.name for c in columns]\n    if len({n.casefold() for n in names}) != len(names):\n        raise MetadataError(f"{identity}: duplicate column")\n    if not columns and materialization not in {"calendar", "mlv"}:\n        raise MetadataError(f"{identity}: an explicit column contract is required")\n    keys = _names(row.get("business_keys"), identity + ".business_keys")\n    if not set(keys) <= set(names):\n        raise MetadataError(f"{identity}: unknown business key column")\n    if (\n        materialization\n        in {"append_versions", "scd2_dimension", "atomic_fact", "snapshot_fact"}\n        and not keys\n    ):\n        raise MetadataError(f"{identity}: business keys are required")\n    priority = row.get("execution_priority")\n    if type(priority) is not int or not 0 <= priority <= 2_147_483_647:\n        raise MetadataError(f"{identity}: invalid execution priority")\n    schema = row.get("target_schema")\n    if layer == "bronze":\n        if schema is not None or row.get("layout") is not None:\n            raise MetadataError(\n                f"{identity}: raw files cannot have a Delta layout/schema"\n            )\n    else:\n        _text(schema, identity + ".target_schema", TOKEN)\n    existing = _boolean(row.get("existing_target"), identity + ".existing_target")\n    layout = (\n        _layout(row["layout"], columns, identity)\n        if row.get("layout") is not None\n        else None\n    )\n    if (\n        layer != "bronze"\n        and materialization != "mlv"\n        and not existing\n        and layout is None\n    ):\n        raise MetadataError(\n            f"{identity}: new managed Delta targets need an explicit layout"\n        )\n    if materialization == "mlv" and layout is not None and layout.strategy == "liquid":\n        raise MetadataError(\n            f"{identity}: liquid layout is not a verified MLV capability"\n        )\n    feed = row.get("feed_id")\n    worker = row.get("worker_id")\n    contract = row.get("contract")\n    if not isinstance(contract, dict):\n        raise MetadataError(f"{identity}: object contract must be structured")\n    return ObjectSpec(\n        identity,\n        _text(feed, identity + ".feed_id") if feed is not None else None,\n        layer,\n        _text(row.get("name"), identity + ".name", TOKEN),\n        cast(str | None, schema),\n        _text(row.get("lakehouse"), identity + ".lakehouse", TOKEN),\n        cast(str, materialization),\n        _boolean(row.get("enabled"), identity + ".enabled"),\n        priority,\n        keys,\n        columns,\n        canonical_json(contract),\n        _text(worker, identity + ".worker_id") if worker is not None else None,\n        layout,\n        _identities(\n            row.get("required_capabilities"), identity + ".required_capabilities"\n        ),\n        existing,\n    )\n\n\ndef _unique(identities: tuple[str, ...], location: str) -> None:\n    if len(set(identities)) != len(identities):\n        raise MetadataError(f"Duplicate {location} identity")\n\n\ndef load_configuration(document: object) -> Configuration:\n    """Validate a definition and detach it completely from caller-owned objects."""\n    snapshot = canonical_json(document)\n    row = _record(\n        json.loads(snapshot),\n        "configuration",\n        {\n            "schema_version",\n            "version_id",\n            "owner",\n            "connections",\n            "feeds",\n            "objects",\n            "dependencies",\n            "rules",\n            "transformations",\n            "workers",\n            "capabilities",\n            "constraints",\n            "extensions",\n        },\n    )\n    if type(row.get("schema_version")) is not int or row["schema_version"] != 1:\n        raise MetadataError("Unsupported metadata schema version")\n    version = _text(row.get("version_id"), "configuration.version_id")\n    owner = _text(row.get("owner"), "configuration.owner")\n    if not isinstance(row.get("extensions"), dict):\n        raise MetadataError("configuration.extensions: expected an object")\n    connections = []\n    for value in _list(row.get("connections"), "connections"):\n        item = _record(\n            value,\n            "connection",\n            {\n                "id",\n                "name",\n                "connector_type",\n                "authentication_type",\n                "fabric_connection_id",\n                "gateway_id",\n            },\n        )\n        connector = item.get("connector_type")\n        if not isinstance(connector, str) or connector not in {\n            name for names in CONNECTORS.values() for name in names\n        }:\n            raise MetadataError("connection: unsupported connector type")\n        auth = _text(item.get("authentication_type"), "connection.authentication_type")\n        if (\n            connector in SQL_AUTHENTICATION\n            and auth not in SQL_AUTHENTICATION[connector]\n        ):\n            raise MetadataError(\n                f"connection: {connector} requires supported authentication "\n                f"{sorted(SQL_AUTHENTICATION[connector])}"\n            )\n        connections.append(\n            ConnectionSpec(\n                _text(item.get("id"), "connection.id"),\n                _text(item.get("name"), "connection.name", TOKEN),\n                cast(str, connector),\n                auth,\n                _uuid(\n                    item.get("fabric_connection_id"), "connection.fabric_connection_id"\n                ),\n                _uuid(item.get("gateway_id"), "connection.gateway_id"),\n            )\n        )\n    feeds = []\n    for value in _list(row.get("feeds"), "feeds"):\n        item = _record(\n            value,\n            "feed",\n            {\n                "id",\n                "source_code",\n                "source_instance",\n                "source_type",\n                "description_slug",\n                "description",\n                "connection_id",\n                "silver_schema",\n                "quarantine_schema",\n                "authority",\n                "enabled",\n                "legacy_contract_hash",\n                "legacy_bronze_root",\n                "geometry",\n            },\n        )\n        transport = item.get("source_type")\n        if not isinstance(transport, str) or transport not in CONNECTORS:\n            raise MetadataError("feed: unsupported source type")\n        authority = item.get("authority")\n        if not isinstance(authority, str) or authority not in {\n            "authoritative",\n            "supplemental",\n        }:\n            raise MetadataError("feed: explicit authority policy is required")\n        source = _text(item.get("source_code"), "feed.source_code", TOKEN)\n        legacy_root = item.get("legacy_bronze_root")\n        if legacy_root is not None and legacy_root != f"Files/bronze/{source}":\n            raise MetadataError("feed: invalid legacy Bronze scope")\n        legacy_hash = item.get("legacy_contract_hash")\n        geometry = None\n        if item.get("geometry") is not None:\n            if transport != "sql":\n                raise MetadataError("Fabric SQL geometry belongs only to SQL feeds")\n            value = _record(\n                item["geometry"], "feed.geometry", {"workspace_id", "artifact_id"}\n            )\n            workspace_id = _uuid(\n                value.get("workspace_id"), "feed.geometry.workspace_id"\n            )\n            artifact_id = _uuid(value.get("artifact_id"), "feed.geometry.artifact_id")\n            if workspace_id is None or artifact_id is None:\n                raise MetadataError(\n                    "SQL geometry requires explicit workspace and artifact UUIDs"\n                )\n            geometry = SqlSourceGeometry(workspace_id, artifact_id)\n        feeds.append(\n            FeedSpec(\n                _text(item.get("id"), "feed.id"),\n                source,\n                _text(item.get("source_instance"), "feed.source_instance"),\n                cast(str, transport),\n                _text(item.get("description_slug"), "feed.description_slug", TOKEN),\n                _description(item.get("description"), "feed.description"),\n                _text(item.get("connection_id"), "feed.connection_id"),\n                _text(item.get("silver_schema"), "feed.silver_schema", TOKEN),\n                _text(item.get("quarantine_schema"), "feed.quarantine_schema", TOKEN),\n                cast(str, authority),\n                _boolean(item.get("enabled"), "feed.enabled"),\n                _hash(legacy_hash, "feed.legacy_contract_hash")\n                if legacy_hash is not None\n                else None,\n                cast(str | None, legacy_root),\n                geometry,\n            )\n        )\n    workers = []\n    for value in _list(row.get("workers"), "workers"):\n        item = _record(\n            value, "worker", {"id", "notebook_name", "lakehouse", "code_sha256"}\n        )\n        workers.append(\n            WorkerSpec(\n                _text(item.get("id"), "worker.id"),\n                _text(item.get("notebook_name"), "worker.notebook_name", TOKEN),\n                _text(item.get("lakehouse"), "worker.lakehouse", TOKEN),\n                _hash(item.get("code_sha256"), "worker.code_sha256"),\n            )\n        )\n    capabilities = []\n    for value in _list(row.get("capabilities"), "capabilities"):\n        item = _record(value, "capability", {"id", "verified", "evidence_sha256"})\n        verified = _boolean(item.get("verified"), "capability.verified")\n        evidence = item.get("evidence_sha256")\n        if verified and evidence is None:\n            raise MetadataError("capability: verified status requires evidence")\n        capabilities.append(\n            CapabilitySpec(\n                _text(item.get("id"), "capability.id"),\n                verified,\n                _hash(evidence, "capability.evidence_sha256")\n                if evidence is not None\n                else None,\n            )\n        )\n    objects = tuple(_object(v) for v in _list(row.get("objects"), "objects"))\n    dependencies = []\n    for value in _list(row.get("dependencies"), "dependencies"):\n        item = _record(\n            value, "dependency", {"upstream_id", "downstream_id", "condition"}\n        )\n        condition = item.get("condition")\n        if not isinstance(condition, str) or condition not in {\n            "succeeded",\n            "source_committed",\n            "facts_ready",\n        }:\n            raise MetadataError("dependency: unsupported readiness condition")\n        dependencies.append(\n            DependencySpec(\n                _text(item.get("upstream_id"), "dependency.upstream_id"),\n                _text(item.get("downstream_id"), "dependency.downstream_id"),\n                cast(str, condition),\n            )\n        )\n    rules = []\n    for value in _list(row.get("rules"), "rules"):\n        item = _record(\n            value,\n            "rule",\n            {\n                "id",\n                "object_id",\n                "handler",\n                "version",\n                "severity",\n                "action",\n                "parameters",\n            },\n        )\n        if not isinstance(item.get("severity"), str) or item["severity"] not in {\n            "error",\n            "warning",\n        }:\n            raise MetadataError("rule: invalid severity")\n        if not isinstance(item.get("action"), str) or item["action"] not in {\n            "fail",\n            "quarantine",\n            "warn",\n        }:\n            raise MetadataError("rule: unsupported action")\n        if not isinstance(item.get("parameters"), dict):\n            raise MetadataError("rule: parameters must be structured")\n        rules.append(\n            RuleSpec(\n                _text(item.get("id"), "rule.id"),\n                _text(item.get("object_id"), "rule.object_id"),\n                _text(item.get("handler"), "rule.handler"),\n                _text(item.get("version"), "rule.version"),\n                cast(str, item["severity"]),\n                cast(str, item["action"]),\n                canonical_json(item["parameters"]),\n            )\n        )\n    transformations = []\n    for value in _list(row.get("transformations"), "transformations"):\n        item = _record(\n            value,\n            "transformation",\n            {\n                "id",\n                "object_id",\n                "kind",\n                "version",\n                "definition",\n                "definition_sha256",\n            },\n        )\n        if not isinstance(item.get("kind"), str) or item["kind"] not in {\n            "pattern",\n            "sql",\n            "notebook",\n        }:\n            raise MetadataError("transformation: unsupported kind")\n        if not isinstance(item.get("definition"), dict):\n            raise MetadataError("transformation: definition must be structured")\n        fingerprint = _hash(\n            item.get("definition_sha256"), "transformation.definition_sha256"\n        )\n        if metadata_digest(item["definition"]) != fingerprint:\n            raise MetadataError("transformation: definition hash mismatch")\n        transformations.append(\n            TransformationSpec(\n                _text(item.get("id"), "transformation.id"),\n                _text(item.get("object_id"), "transformation.object_id"),\n                cast(str, item["kind"]),\n                _text(item.get("version"), "transformation.version"),\n                canonical_json(item["definition"]),\n                fingerprint,\n            )\n        )\n    constraints = []\n    for value in _list(row.get("constraints", []), "constraints"):\n        item = _record(\n            value,\n            "constraint",\n            {\n                "id",\n                "object_id",\n                "kind",\n                "enforcement",\n                "definition",\n            },\n        )\n        if not isinstance(item.get("kind"), str) or item["kind"] not in {\n            "check",\n            "unique",\n            "relationship",\n            "reconciliation",\n        }:\n            raise MetadataError("constraint: unsupported kind")\n        if not isinstance(item.get("enforcement"), str) or item["enforcement"] not in {\n            "runtime",\n            "engine",\n        }:\n            raise MetadataError("constraint: declare the enforcement mechanism")\n        if not isinstance(item.get("definition"), dict):\n            raise MetadataError("constraint: definition must be structured")\n        constraints.append(\n            ConstraintSpec(\n                _text(item.get("id"), "constraint.id"),\n                _text(item.get("object_id"), "constraint.object_id"),\n                cast(str, item["kind"]),\n                cast(str, item["enforcement"]),\n                canonical_json(item["definition"]),\n            )\n        )\n    config = Configuration(\n        version,\n        owner,\n        tuple(connections),\n        tuple(feeds),\n        objects,\n        tuple(dependencies),\n        tuple(rules),\n        tuple(constraints),\n        tuple(transformations),\n        tuple(workers),\n        tuple(capabilities),\n        snapshot,\n    )\n    _validate_references(config)\n    _validate_hash_policies(config)\n    _validate_sql_sources(config)\n    _validate_file_sources(config)\n    return config\n\n\ndef file_source_contract(value: object) -> dict[str, object]:\n    row = _record(\n        value,\n        "file source contract",\n        {\n            "source_directory",\n            "file_pattern",\n            "recursive",\n            "file_format",\n            "arrival_policy",\n        },\n    )\n    directory = row.get("source_directory")\n    if (\n        not isinstance(directory, str)\n        or re.fullmatch(r"/?[A-Za-z0-9_./ -]{1,1999}", directory) is None\n    ):\n        raise MetadataError(\n            "File source directory must be an explicit relative remote path"\n        )\n    if any(part in {"", ".", ".."} for part in directory.strip("/").split("/")):\n        raise MetadataError("File source directory cannot traverse another root")\n    pattern = row.get("file_pattern")\n    if (\n        not isinstance(pattern, str)\n        or re.fullmatch(r"[A-Za-z0-9_*?][A-Za-z0-9_. *?-]{0,255}", pattern) is None\n    ):\n        raise MetadataError("File selection requires an explicit basename pattern")\n    recursive = _boolean(row.get("recursive"), "file source recursive flag")\n    if row.get("file_format") != "csv":\n        raise MetadataError(\n            "Only explicitly contracted CSV file ingestion is implemented"\n        )\n    policy = row.get("arrival_policy")\n    if not isinstance(policy, dict):\n        raise MetadataError(\n            "File ingestion requires an explicit reviewed arrival policy"\n        )\n    if policy.get("kind") == "marker":\n        _record(policy, "arrival marker", {"kind", "marker"})\n        if (\n            not isinstance(policy.get("marker"), str)\n            or re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]{0,255}", policy["marker"])\n            is None\n        ):\n            raise MetadataError("File arrival marker must be a safe explicit basename")\n    elif policy.get("kind") == "stable":\n        _record(\n            policy,\n            "stable arrival",\n            {"kind", "minimum_age_seconds", "stable_for_seconds"},\n        )\n        if any(\n            type(policy.get(name)) is not int or not 1 <= policy[name] <= 604800\n            for name in ("minimum_age_seconds", "stable_for_seconds")\n        ):\n            raise MetadataError("Stable arrival intervals must be positive and bounded")\n    else:\n        raise MetadataError("File existence alone is not an arrival policy")\n    return {\n        "source_directory": directory,\n        "file_pattern": pattern,\n        "recursive": recursive,\n        "file_format": "csv",\n        "arrival_policy": json.loads(canonical_json(policy)),\n    }\n\n\ndef _validate_file_sources(config: Configuration) -> None:\n    feeds = {feed.id: feed for feed in config.feeds}\n    connections = {connection.id: connection for connection in config.connections}\n    for obj in config.objects:\n        if obj.layer != "bronze" or obj.feed_id is None:\n            continue\n        feed = feeds[obj.feed_id]\n        if feed.source_type == "sql":\n            continue\n        auth = connections[feed.connection_id].authentication_type\n        if auth not in (\n            {"Basic", "Anonymous"} if feed.source_type == "ftp" else {"Basic"}\n        ):\n            raise MetadataError(\n                "File connector authentication has no supported implementation"\n            )\n        file_source_contract(json.loads(obj.contract_json))\n\n\ndef _validate_sql_sources(config: Configuration) -> None:\n    feeds = {feed.id: feed for feed in config.feeds}\n    for obj in config.objects:\n        if (\n            obj.layer != "bronze"\n            or obj.feed_id is None\n            or feeds[obj.feed_id].source_type != "sql"\n        ):\n            continue\n        contract = json.loads(obj.contract_json)\n        _text(contract.get("source_schema"), "SQL source schema", IDENTIFIER)\n        _text(contract.get("source_object"), "SQL source object", IDENTIFIER)\n        projection = ",".join(\n            sql_copy_projection(json.loads(c.definition_json)) for c in obj.columns\n        )\n        if "projection_sql" in contract and contract["projection_sql"] != projection:\n            raise MetadataError(\n                "SQL projection differs from the allowlisted typed column contract"\n            )\n\n\ndef _validate_hash_policies(config: Configuration) -> None:\n    extensions = cast(dict[str, object], config.to_document()["extensions"])\n    policies = extensions.get("hash_policies", {})\n    if not isinstance(policies, dict):\n        raise MetadataError("Hash policies must be structured object contracts")\n    objects = {o.id: o for o in config.objects}\n    for identity, value in policies.items():\n        obj = objects.get(identity)\n        if obj is None or obj.layer != "silver":\n            raise MetadataError("Hash policy must reference a Silver object")\n        policy = _record(\n            value, "hash_policy", {"key_columns", "value_columns", "encoding"}\n        )\n        if policy.get("encoding") != "spark_json_struct_sha256_v1":\n            raise MetadataError("Unsupported hash encoding requires explicit migration")\n        if (\n            _names(policy.get("key_columns"), "hash_policy.key_columns")\n            != obj.business_keys\n        ):\n            raise MetadataError("Hash key order differs from the business-key contract")\n        fields = _names(policy.get("value_columns"), "hash_policy.value_columns")\n        contract = cast(dict[str, object], json.loads(obj.contract_json))\n        exclusions = _names(\n            contract.get("change_hash_exclusions", []), "hash exclusions"\n        )\n        if not set(exclusions) <= {c.name for c in obj.columns}:\n            raise MetadataError("Unknown observation-only hash exclusion")\n        if set(fields) != {c.name for c in obj.columns} - set(exclusions):\n            raise MetadataError("Hash fields differ from the tracked business columns")\n\n\ndef _validate_references(config: Configuration) -> None:\n    for name, values in (\n        ("connections", config.connections),\n        ("feeds", config.feeds),\n        ("objects", config.objects),\n        ("rules", config.rules),\n        ("constraints", config.constraints),\n        ("transformations", config.transformations),\n        ("workers", config.workers),\n        ("capabilities", config.capabilities),\n    ):\n        _unique(tuple(item.id for item in values), name)\n    connections = {c.id: c for c in config.connections}\n    feeds = {f.id: f for f in config.feeds}\n    workers = {w.id: w for w in config.workers}\n    objects = {o.id: o for o in config.objects}\n    capabilities = {c.id for c in config.capabilities}\n    authority = set()\n    paths = set()\n    for feed in config.feeds:\n        connection = connections.get(feed.connection_id)\n        if (\n            connection is None\n            or connection.connector_type not in CONNECTORS[feed.source_type]\n        ):\n            raise MetadataError(\n                f"{feed.id}: missing or incompatible connector reference"\n            )\n        if connection.connector_type == "SqlServer" and feed.geometry is not None:\n            raise MetadataError(\n                f"{feed.id}: SQL Server uses its managed connection, not Fabric item geometry"\n            )\n        if feed.enabled and feed.authority == "authoritative":\n            key = feed.source_code, feed.source_instance\n            if key in authority:\n                raise MetadataError(f"{feed.id}: ambiguous source authority")\n            authority.add(key)\n        if feed.bronze_prefix in paths:\n            raise MetadataError(f"{feed.id}: duplicate Bronze target prefix")\n        paths.add(feed.bronze_prefix)\n    targets = set()\n    for obj in config.objects:\n        if obj.feed_id is not None and obj.feed_id not in feeds:\n            raise MetadataError(f"{obj.id}: missing feed reference")\n        if obj.layer in {"bronze", "silver"} and obj.feed_id is None:\n            raise MetadataError(f"{obj.id}: source-aligned objects require a feed")\n        if (\n            obj.layer == "silver"\n            and obj.target_schema != feeds[cast(str, obj.feed_id)].silver_schema\n        ):\n            raise MetadataError(f"{obj.id}: Silver schema differs from feed contract")\n        if not set(obj.required_capabilities) <= capabilities:\n            raise MetadataError(f"{obj.id}: missing capability declaration")\n        if obj.worker_id is not None:\n            worker = workers.get(obj.worker_id)\n            if worker is None or worker.lakehouse != obj.lakehouse:\n                raise MetadataError(f"{obj.id}: incompatible worker/Lakehouse binding")\n        if obj.layer == "silver" and obj.worker_id is None:\n            raise MetadataError(f"{obj.id}: Silver worker reference is required")\n        target = (\n            feeds[cast(str, obj.feed_id)].bronze_prefix\n            if obj.layer == "bronze"\n            else obj.target_schema\n        )\n        key = obj.lakehouse, target, obj.name\n        if key in targets:\n            raise MetadataError(f"{obj.id}: duplicate target writer")\n        targets.add(key)\n    for item in (*config.rules, *config.constraints, *config.transformations):\n        if item.object_id not in objects:\n            raise MetadataError(f"{item.id}: missing object reference")\n    edges = set()\n    parents: dict[str, set[str]] = {o.id: set() for o in config.objects}\n    for dependency in config.dependencies:\n        upstream = objects.get(dependency.upstream_id)\n        downstream = objects.get(dependency.downstream_id)\n        if upstream is None or downstream is None:\n            raise MetadataError("Missing dependency object")\n        if LAYERS.index(upstream.layer) > LAYERS.index(downstream.layer):\n            raise MetadataError("A dependency cannot move backwards through a layer")\n        edge = dependency.upstream_id, dependency.downstream_id\n        if edge in edges:\n            raise MetadataError("Duplicate dependency edge")\n        edges.add(edge)\n        parents[dependency.downstream_id].add(dependency.upstream_id)\n    remaining = set(parents)\n    while remaining:\n        ready = {\n            identity for identity in remaining if not parents[identity] & remaining\n        }\n        if not ready:\n            raise MetadataError("Dependency cycle detected")\n        remaining -= ready\n\n\n@dataclass(frozen=True)\nclass RunPlan:\n    """Pinned configuration readiness, not proof of live permissions or service health."""\n\n    run_id: str\n    version_id: str\n    definition_sha256: str\n    object_ids: tuple[str, ...]\n    blockers: tuple[str, ...]\n    snapshot_json: str\n\n    @property\n    def can_execute(self) -> bool:\n        return not self.blockers\n\n    @property\n    def snapshot_sha256(self) -> str:\n        return hashlib.sha256(self.snapshot_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.snapshot_json))\n\n    def pages(\n        self, page_size: int = 1000, max_bytes: int = 4 * 1024 * 1024\n    ) -> tuple[tuple[dict[str, object], ...], ...]:\n        if type(page_size) is not int or not 1 <= page_size <= 5000:\n            raise MetadataError("Lookup page size must be between 1 and 5000")\n        if type(max_bytes) is not int or not 2 <= max_bytes <= 4 * 1024 * 1024:\n            raise MetadataError("Lookup byte limit must not exceed 4 MiB")\n        tasks = cast(list[dict[str, object]], self.to_document()["tasks"])\n        pages = []\n        current: list[dict[str, object]] = []\n        current_bytes = 2\n        for task in tasks:\n            size = len(canonical_json(task).encode("ascii"))\n            if size + 2 > max_bytes:\n                raise MetadataError("One task exceeds the Lookup byte limit")\n            added = size + (1 if current else 0)\n            if current and (\n                len(current) == page_size or current_bytes + added > max_bytes\n            ):\n                pages.append(tuple(current))\n                current = []\n                current_bytes = 2\n                added = size\n            current.append(task)\n            current_bytes += added\n        if current:\n            pages.append(tuple(current))\n        return tuple(pages)\n\n\ndef compile_run_plan(\n    configuration: Configuration,\n    run_id: str,\n    observed_utc: str,\n    *,\n    object_ids: list[str] | tuple[str, ...] | None = None,\n    lifecycle_state: str = "DRAFT",\n    approved_definition_sha256: str | None = None,\n    require_ready: bool = False,\n) -> RunPlan:\n    """Compile a read-only preview; callers must separately validate live bindings."""\n    if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", run_id):\n        raise MetadataError("Run identity must be a bounded path-safe token")\n    if not isinstance(observed_utc, str) or not re.fullmatch(\n        r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,6})?Z", observed_utc\n    ):\n        raise MetadataError("Observation timestamp must be an explicit UTC instant")\n    try:\n        observed = datetime.fromisoformat(observed_utc)\n    except ValueError:\n        raise MetadataError("Invalid observation timestamp") from None\n    if not isinstance(lifecycle_state, str) or lifecycle_state not in {\n        "DRAFT",\n        "VALIDATED",\n        "APPROVED",\n        "RETIRED",\n    }:\n        raise MetadataError("Unknown metadata approval state")\n    _boolean(require_ready, "require_ready")\n    objects = {o.id: o for o in configuration.objects}\n    feeds = {f.id: f for f in configuration.feeds}\n    connections = {c.id: c for c in configuration.connections}\n    if object_ids is not None and not isinstance(object_ids, (list, tuple)):\n        raise MetadataError("Requested object identities must be an array")\n    roots = (\n        list(object_ids)\n        if object_ids is not None\n        else [o.id for o in configuration.objects if o.enabled]\n    )\n    if not roots or any(not isinstance(identity, str) for identity in roots):\n        raise MetadataError("Select at least one valid object identity")\n    if len(set(roots)) != len(roots) or not set(roots) <= objects.keys():\n        raise MetadataError("Unknown or duplicate object selection")\n    parents: dict[str, set[str]] = {o.id: set() for o in configuration.objects}\n    conditions = {}\n    for dependency in configuration.dependencies:\n        parents[dependency.downstream_id].add(dependency.upstream_id)\n        conditions[dependency.upstream_id, dependency.downstream_id] = (\n            dependency.condition\n        )\n    selected = set(roots)\n    pending = list(roots)\n    while pending:\n        identity = pending.pop()\n        obj = objects[identity]\n        if not obj.enabled:\n            raise MetadataError(f"{identity}: required object is disabled")\n        if obj.feed_id is not None:\n            feed = feeds[obj.feed_id]\n            if not feed.enabled:\n                raise MetadataError(f"{identity}: required feed is disabled")\n            if feed.authority != "authoritative":\n                raise MetadataError(\n                    f"{identity}: supplemental authority needs an approved merge policy"\n                )\n        for upstream in parents[identity]:\n            if upstream not in selected:\n                selected.add(upstream)\n                pending.append(upstream)\n\n    children: dict[str, list[str]] = {identity: [] for identity in selected}\n    indegree = {identity: len(parents[identity] & selected) for identity in selected}\n    for child in selected:\n        for parent in parents[child]:\n            children[parent].append(child)\n\n    def priority(identity: str) -> tuple[int, int, str]:\n        obj = objects[identity]\n        return LAYERS.index(obj.layer), obj.execution_priority, identity\n\n    ready = [priority(identity) for identity, degree in indegree.items() if not degree]\n    heapq.heapify(ready)\n    order = []\n    while ready:\n        _, _, identity = heapq.heappop(ready)\n        order.append(identity)\n        for child in children[identity]:\n            indegree[child] -= 1\n            if not indegree[child]:\n                heapq.heappush(ready, priority(child))\n    if len(order) != len(selected):\n        raise MetadataError("Selected dependency graph contains a cycle")\n    blockers = []\n    if (\n        lifecycle_state != "APPROVED"\n        or approved_definition_sha256 != configuration.definition_sha256\n    ):\n        blockers.append("Metadata approval does not match the pinned definition")\n    needed = {\n        name for identity in order for name in objects[identity].required_capabilities\n    }\n    for identity in order:\n        obj = objects[identity]\n        if obj.layer != "bronze" or obj.feed_id is None:\n            continue\n        feed = feeds[obj.feed_id]\n        if connections[feed.connection_id].connector_type == "SqlServer":\n            needed.add("sql-server-copy-basic")\n        if feed.source_type in ("ftp", "sftp"):\n            authentication = connections[feed.connection_id].authentication_type\n            needed.update(\n                (\n                    f"{feed.source_type}-copy-{authentication.lower()}",\n                    f"{feed.source_type}-arrival-observer",\n                )\n            )\n    capabilities = {c.id: c for c in configuration.capabilities}\n    for name in sorted(needed):\n        if name not in capabilities or not capabilities[name].verified:\n            blockers.append(f"Unverified capability: {name}")\n    extensions = cast(dict[str, object], configuration.to_document()["extensions"])\n    policies = cast(dict[str, object], extensions.get("hash_policies", {}))\n    for identity in order:\n        if objects[identity].layer == "silver" and identity not in policies:\n            blockers.append(f"Missing explicit business hash policy: {identity}")\n        obj = objects[identity]\n        if (\n            obj.layer == "bronze"\n            and obj.feed_id is not None\n            and feeds[obj.feed_id].source_type == "sql"\n            and "projection_sql" not in json.loads(obj.contract_json)\n        ):\n            blockers.append(f"Missing typed SQL Copy projection: {identity}")\n    used_feeds = {objects[identity].feed_id for identity in order} - {None}\n    for feed_id in sorted(cast(set[str], used_feeds)):\n        connection = connections[feeds[feed_id].connection_id]\n        if connection.fabric_connection_id is None:\n            blockers.append(f"Unbound connection: {connection.name}")\n        if (\n            feeds[feed_id].source_type == "sql"\n            and connection.connector_type == "FabricSql"\n            and feeds[feed_id].geometry is None\n            and any(\n                objects[identity].feed_id == feed_id\n                and objects[identity].layer == "bronze"\n                for identity in order\n            )\n        ):\n            blockers.append(f"Missing explicit SQL source geometry: {feed_id}")\n    if require_ready and blockers:\n        raise MetadataError("; ".join(blockers))\n    task_ids = {\n        identity: metadata_digest({"run_id": run_id, "object_id": identity})\n        for identity in order\n    }\n    tasks = []\n    for ordinal, identity in enumerate(order, 1):\n        obj = objects[identity]\n        feed = feeds[obj.feed_id] if obj.feed_id is not None else None\n        source_batch_id = (\n            metadata_digest({"run_id": run_id, "feed_id": feed.id}) if feed else None\n        )\n        if obj.layer == "bronze":\n            if feed is None:\n                raise MetadataError("Bronze task requires a pinned feed")\n            folder = f"{feed.bronze_prefix}/{observed:%Y%m%d}/{source_batch_id}"\n            target = (\n                f"{folder}/{obj.name}.csv"\n                if feed.source_type == "sql"\n                else f"{folder}/{obj.name}"\n            )\n        else:\n            target = f"{obj.lakehouse}.{obj.target_schema}.{obj.name}"\n        tasks.append(\n            {\n                "task_id": task_ids[identity],\n                "object_id": identity,\n                "ordinal": ordinal,\n                "layer": obj.layer,\n                "materialization": obj.materialization,\n                "feed_id": obj.feed_id,\n                "source_batch_id": source_batch_id,\n                "target_path": target,\n                "target_lock_key": metadata_digest(\n                    {\n                        "lakehouse": obj.lakehouse,\n                        "scope": feed.bronze_prefix\n                        if obj.layer == "bronze" and feed\n                        else obj.target_schema,\n                        "name": obj.name,\n                    }\n                ),\n                "source_lock_key": metadata_digest(\n                    {\n                        "lakehouse": obj.lakehouse,\n                        "scope": "source",\n                        "silver_schema": feed.silver_schema,\n                        "source_code": feed.source_code,\n                    }\n                )\n                if feed is not None\n                else None,\n                "worker_id": obj.worker_id,\n                "dependencies": [\n                    {\n                        "task_id": task_ids[parent],\n                        "condition": conditions[parent, identity],\n                    }\n                    for parent in sorted(parents[identity])\n                ],\n            }\n        )\n    snapshot = {\n        "format_version": 1,\n        "run_id": run_id,\n        "version_id": configuration.version_id,\n        "definition_sha256": configuration.definition_sha256,\n        "observed_utc": observed.isoformat(timespec="microseconds").replace(\n            "+00:00", "Z"\n        ),\n        "requested_objects": sorted(roots),\n        "tasks": tasks,\n        "can_execute": not blockers,\n        "blockers": blockers,\n        "configuration": configuration.to_document(),\n    }\n    return RunPlan(\n        run_id,\n        configuration.version_id,\n        configuration.definition_sha256,\n        tuple(order),\n        tuple(blockers),\n        canonical_json(snapshot),\n    )\n\n\ndef copy_task_descriptors(plan: RunPlan) -> tuple[dict[str, object], ...]:\n    """Resolve Copy inputs from one detached snapshot, never an active pointer."""\n    if not plan.can_execute:\n        raise MetadataError("A blocked preview cannot dispatch source Copy")\n    document = plan.to_document()\n    configuration = load_configuration(document["configuration"])\n    objects = {obj.id: obj for obj in configuration.objects}\n    feeds = {feed.id: feed for feed in configuration.feeds}\n    connections = {\n        connection.id: connection for connection in configuration.connections\n    }\n    rows = []\n    for task in cast(list[dict[str, object]], document["tasks"]):\n        if task["layer"] != "bronze":\n            continue\n        obj = objects[cast(str, task["object_id"])]\n        feed = feeds[cast(str, obj.feed_id)]\n        if (\n            connections[feed.connection_id].connector_type == "FabricSql"\n            and feed.geometry is None\n        ):\n            raise MetadataError("SQL Copy requires pinned source geometry")\n        contract = json.loads(obj.contract_json)\n        projection = ",".join(\n            sql_copy_projection(json.loads(c.definition_json)) for c in obj.columns\n        )\n        path = str(task["target_path"]).removeprefix("Files/")\n        folder, _, filename = path.rpartition("/")\n        row = {\n            "task_id": task["task_id"],\n            "task_ordinal": task["ordinal"],\n            "feed_id": task["feed_id"],\n            "source_type": feed.source_type,\n            "connector_type": connections[feed.connection_id].connector_type,\n            "source_batch_id": task["source_batch_id"],\n            "target_lock_key": task["target_lock_key"],\n            "source_lock_key": task["source_lock_key"],\n            "connection_id": connections[feed.connection_id].fabric_connection_id,\n            "authentication_type": connections[feed.connection_id].authentication_type,\n            "source_workspace_id": feed.geometry.workspace_id\n            if feed.geometry is not None\n            else None,\n            "source_artifact_id": feed.geometry.artifact_id\n            if feed.geometry is not None\n            else None,\n            "sql_query": f"SELECT {projection} FROM [{contract[\'source_schema\']}].[{contract[\'source_object\']}]"\n            if feed.source_type == "sql"\n            else None,\n            "folder_path": folder,\n            "file_name": filename,\n        }\n        if feed.source_type != "sql":\n            geometry = file_source_contract(contract)\n            row.update(\n                {\n                    "folder_path": path,\n                    "file_name": None,\n                    "source_directory": geometry["source_directory"],\n                    "file_pattern": geometry["file_pattern"],\n                    "recursive": geometry["recursive"],\n                    "file_format": geometry["file_format"],\n                    "arrival_policy_json": canonical_json(geometry["arrival_policy"]),\n                }\n            )\n        rows.append(row)\n    return tuple(rows)\n\n\ndef copy_page_descriptors(\n    plan: RunPlan, *, page_size: int = 1000, max_bytes: int = 4 * 1024 * 1024 - 1024\n) -> tuple[dict[str, int], ...]:\n    """Bound the escaped SQL Lookup response without passing source arrays to pipelines."""\n    if type(page_size) is not int or not 1 <= page_size <= 5000:\n        raise MetadataError("Lookup page size must be between 1 and 5000")\n    if type(max_bytes) is not int or not 2 <= max_bytes <= 4 * 1024 * 1024 - 1024:\n        raise MetadataError("Invalid Lookup response byte bound")\n    pages = []\n    after = count = size = 0\n    for row in copy_task_descriptors(plan):\n        required = len(canonical_json(row).encode("ascii")) * 2 + 256\n        if required > max_bytes:\n            raise MetadataError("One Copy task exceeds the Lookup response bound")\n        if count and (count == page_size or size + required > max_bytes):\n            pages.append({"after_ordinal": after, "page_size": count})\n            after += count\n            count = size = 0\n        if row["task_ordinal"] != after + count + 1:\n            raise MetadataError(\n                "Bronze task ordinals must be contiguous for bounded dispatch"\n            )\n        count += 1\n        size += required\n    if count:\n        pages.append({"after_ordinal": after, "page_size": count})\n    return tuple(pages)\n', 'rules': '"""Reusable versioned rule engine; business rules are injected case-owned code."""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport re\nfrom functools import reduce\nfrom typing import Any\n\n\ndef _column(name: str):\n    from pyspark.sql import functions as F\n\n    return F.col(f"`{name}`")\n\n\ndef _case_rules():\n    adapter = globals().get("_CASE_RULES")\n    if adapter is None and "__file__" in globals():\n        # Local compatibility wrapper follows the catalog, never a business default.\n        from pathlib import Path\n\n        from fabric_medallion_deployer.use_cases import (\n            load_case_extension,\n            load_use_case,\n        )\n\n        adapter = load_case_extension(\n            load_use_case(Path(__file__).resolve().parents[2]), "source_rules"\n        )\n        globals()["_CASE_RULES"] = adapter\n    if adapter is None:\n        raise RuntimeError("The reviewed selected case source rules must be injected")\n    return adapter\n\n\ndef source_business_checks(data: dict[str, Any], source: str):\n    return _case_rules().source_business_checks(data, source)\n\n\ndef row_rules(table: dict[str, Any]):\n    return _case_rules().row_rules(table)\n\n\ndef legacy_context_bindings(source: str) -> dict[str, str]:\n    function = getattr(_case_rules(), "legacy_context_bindings", None)\n    if function is None:\n        raise ValueError(\n            "Selected case has no legacy raw-path contract; use pinned manifests"\n        )\n    return function(source)\n\n\ndef validate_rule_specs(specs: object, *, scope: str) -> tuple[dict[str, Any], ...]:\n    if (\n        scope not in {"object", "source", "gold"}\n        or not isinstance(specs, list)\n        or not specs\n    ):\n        raise ValueError("Explicit versioned rule specifications are required")\n    required = (\n        {"legacy.prepare_rows", "legacy.relationship_checks", "legacy.silver_checks"}\n        if scope == "object"\n        else {"legacy.source_business_checks"}\n        if scope == "source"\n        else set()\n    )\n    generic = {\n        "value_range",\n        "required_columns",\n        "equals_columns",\n        "business_key_unique",\n        "allowed_values",\n    }\n    allowed = required | (generic if scope != "source" else set())\n    identities, handlers, result = set(), [], []\n    for spec in specs:\n        if not isinstance(spec, dict) or set(spec) != {\n            "id",\n            "handler",\n            "version",\n            "severity",\n            "action",\n            "parameters",\n        }:\n            raise ValueError("Rule fields differ from the supported runtime contract")\n        if (\n            not isinstance(spec["id"], str)\n            or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.:-]{0,127}", spec["id"])\n            or spec["id"] in identities\n            or spec["handler"] not in allowed\n            or spec["version"] != "1"\n        ):\n            raise ValueError("Unknown, duplicate or unsupported rule version")\n        identities.add(spec["id"])\n        handlers.append(spec["handler"])\n        if spec["severity"] not in {"error", "warning"} or spec["action"] not in {\n            "fail",\n            "quarantine",\n            "warn",\n        }:\n            raise ValueError("Unsupported rule severity or action")\n        if (spec["severity"] == "warning") != (spec["action"] == "warn"):\n            raise ValueError("Rule warning severity and action must agree")\n        if spec["handler"].startswith("legacy.") and (\n            spec["severity"] != "error" or spec["action"] != "fail"\n        ):\n            raise ValueError(\n                "Required baseline checks must retain their blocking behavior"\n            )\n        parameters = spec["parameters"]\n        if not isinstance(parameters, dict):\n            raise TypeError("Rule parameters must be structured")\n        kind = spec["handler"]\n        if kind.startswith("legacy."):\n            if not set(parameters) <= {\n                "scope",\n                "legacy_contract_hash",\n                "runtime_sha256",\n                "source_code",\n                "objects",\n            }:\n                raise ValueError("Unknown baseline rule parameter")\n        else:\n            fields = {\n                "value_range": {"column", "minimum", "maximum", "allow_null"},\n                "required_columns": {"columns"},\n                "equals_columns": {"left", "right"},\n                "business_key_unique": {"columns"},\n                "allowed_values": {"column", "values", "allow_null"},\n            }[kind]\n            if not set(parameters) <= fields:\n                raise ValueError(\n                    "Unknown generic rule parameter; executable expressions are prohibited"\n                )\n            names = (\n                parameters.get("columns")\n                if kind in {"required_columns", "business_key_unique"}\n                else [parameters.get("left"), parameters.get("right")]\n                if kind == "equals_columns"\n                else [parameters.get("column")]\n            )\n            if (\n                not isinstance(names, list)\n                or not names\n                or len(set(map(str, names))) != len(names)\n                or any(\n                    not isinstance(name, str)\n                    or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", name)\n                    for name in names\n                )\n            ):\n                raise ValueError(\n                    "Generic rules require distinct validated column names"\n                )\n            if (\n                "allow_null" in parameters\n                and type(parameters["allow_null"]) is not bool\n            ):\n                raise ValueError("Rule null behavior must be explicitly Boolean")\n            if kind == "value_range":\n                low, high = parameters.get("minimum"), parameters.get("maximum")\n                if (\n                    (low is None and high is None)\n                    or any(\n                        value is not None\n                        and (\n                            type(value) not in (int, float) or not math.isfinite(value)\n                        )\n                        for value in (low, high)\n                    )\n                    or (low is not None and high is not None and low > high)\n                ):\n                    raise ValueError("Range bounds must be finite ordered numbers")\n            if kind == "allowed_values":\n                values = parameters.get("values")\n                if (\n                    not isinstance(values, list)\n                    or not 1 <= len(values) <= 1000\n                    or any(\n                        type(value) not in (str, int, float, bool) for value in values\n                    )\n                ):\n                    raise ValueError(\n                        "Allowed values must be a bounded explicit scalar list"\n                    )\n        result.append(json.loads(json.dumps(spec, allow_nan=False)))\n    if not required <= set(handlers) or any(\n        handlers.count(handler) != 1 for handler in required\n    ):\n        raise ValueError("Required baseline rule handlers are missing or duplicated")\n    return tuple(result)\n\n\ndef apply_rules(frame, table: dict[str, Any], *, scope: str = "object"):\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    profile = validate_rule_specs(table["rule_specs"], scope=scope)\n    output = frame.withColumn("_dq_warnings", F.array().cast("array<string>"))\n    for spec in profile:\n        kind, parameters = spec["handler"], spec["parameters"]\n        if kind.startswith("legacy."):\n            continue\n        columns = (\n            parameters.get("columns")\n            if kind in {"required_columns", "business_key_unique"}\n            else [parameters["left"], parameters["right"]]\n            if kind == "equals_columns"\n            else [parameters["column"]]\n        )\n        if not set(columns) <= set(frame.columns):\n            raise ValueError("Generic rule references an absent contracted column")\n        if kind == "value_range":\n            value = _column(parameters["column"])\n            invalid = (\n                value.isNull()\n                if not parameters.get("allow_null", False)\n                else F.lit(False)\n            )\n            if parameters.get("minimum") is not None:\n                invalid = invalid | (value < F.lit(parameters["minimum"]))\n            if parameters.get("maximum") is not None:\n                invalid = invalid | (value > F.lit(parameters["maximum"]))\n        elif kind == "required_columns":\n            invalid = reduce(\n                lambda left, right: left | right,\n                [_column(name).isNull() for name in columns],\n            )\n        elif kind == "equals_columns":\n            invalid = ~_column(parameters["left"]).eqNullSafe(\n                _column(parameters["right"])\n            )\n        elif kind == "business_key_unique":\n            invalid = (\n                F.count(F.lit(1)).over(\n                    Window.partitionBy(*[_column(name) for name in columns])\n                )\n                > 1\n            )\n        else:\n            value = _column(parameters["column"])\n            invalid = ~value.isin(parameters["values"])\n            invalid = (\n                invalid | value.isNull()\n                if not parameters.get("allow_null", False)\n                else invalid\n            )\n        label = f"{spec[\'id\']}@{spec[\'version\']}"\n        target = "_dq_warnings" if spec["action"] == "warn" else "_dq_reasons"\n        output = output.withColumn(\n            target,\n            F.when(\n                F.coalesce(invalid, F.lit(False)),\n                F.array_union(_column(target), F.array(F.lit(label))),\n            ).otherwise(_column(target)),\n        )\n    return output\n\n\ndef blocking_reasons(table: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    quarantined = []\n    if "rule_specs" in table:\n        quarantined = [\n            f"{spec[\'id\']}@{spec[\'version\']}"\n            for spec in validate_rule_specs(table["rule_specs"], scope="object")\n            if spec["action"] == "quarantine"\n        ]\n    reasons = _column("_dq_reasons")\n    if not quarantined:\n        return reasons\n    return F.filter(reasons, lambda reason: ~reason.isin(quarantined))\n\n\n_MEDALLION_RULE_HANDLERS = {\n    "source_business_checks": source_business_checks,\n    "row_rules": row_rules,\n    "validate_rule_specs": validate_rule_specs,\n    "apply_rules": apply_rules,\n    "blocking_reasons": blocking_reasons,\n    "legacy_context_bindings": legacy_context_bindings,\n}\n', 'ingestion': '"""Shared Fabric runtime, embedded into each generated source-specific notebook.\n\nRun only against the selected attached Lakehouse. Source values are never\nprinted; rejected raw fields remain in source-isolated quarantine tables.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport re\nimport uuid\nfrom datetime import UTC, date, datetime\nfrom decimal import Decimal\nfrom functools import reduce\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any\n\nAUDIT_SCHEMAS = {\n    "bronze_batch_manifest": (\n        "source_path string, snapshot_hash string, observed_utc timestamp, source_rows long"\n    ),\n    "bronze_inventory": (\n        "source_path string, row_count long, column_count int, status string, message string"\n    ),\n    "silver_load_log": (\n        "source_path string, target_table string, source_rows long, silver_rows long, "\n        "quarantine_rows long, status string, message string"\n    ),\n    "silver_file_manifest": (\n        "source_path string, source_file_name string, source_rows long, silver_rows long, "\n        "quarantine_rows long, status string"\n    ),\n    "silver_dq_rule_summary": "dq_rule string, failure_count long",\n    "bronze_silver_validation": "check_name string, failed_count long, status string",\n    "bronze_cleanup": (\n        "source_path string, file_date date, size_bytes long, action string, status string, message string"\n    ),\n}\nAUDIT_COMMON = (\n    "source_code string, table_name string, pipeline_run_id string, processing_date date, "\n    "contract_hash string, event_utc timestamp"\n)\nLINEAGE = [\n    "ingest_batch_run_id",\n    "ingest_processing_date",\n    "ingest_extract_date",\n    "ingest_loaded_utc",\n    "ingest_source_system",\n    "ingest_source_file_name",\n    "ingest_contract_hash",\n]\n\n\ndef _business_rule_handlers() -> dict[str, Any]:\n    handlers = globals().get("_MEDALLION_RULE_HANDLERS")\n    if handlers is None:\n        location = globals().get("__file__")\n        if location is None:\n            raise RuntimeError("The packaged shared rule module is missing")\n        path = Path(location).with_name("rules.py")\n        spec = importlib.util.spec_from_file_location("medallion_packaged_rules", path)\n        if spec is None or spec.loader is None:\n            raise RuntimeError("The packaged shared rule module cannot be resolved")\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        handlers = module._MEDALLION_RULE_HANDLERS\n        globals()["_MEDALLION_RULE_HANDLERS"] = handlers\n    return handlers\n\n\ndef processing_context(\n    config: dict[str, Any],\n    processing_date: str,\n    pipeline_run_id: str,\n    batch_started_utc: str = "",\n) -> dict[str, Any]:\n    """Validate bounded source/date scope before accessing any files or tables."""\n    source = config["source_code"]\n    bindings = _business_rule_handlers()["legacy_context_bindings"](source)\n    for field in ("lakehouse", "bronze_root", "silver_schema", "quarantine_schema"):\n        if config[field] != bindings[field]:\n            raise ValueError(f"Unexpected historical source {field}")\n    if "history_schema" in config:\n        raise ValueError("Legacy dual-store contract; regenerate all source notebooks")\n    value = processing_date.strip() or datetime.now(UTC).date().isoformat()\n    if not re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", value):\n        raise ValueError("processing_date must use yyyy-MM-dd")\n    result = {\n        **config,\n        "processing_date": date.fromisoformat(value),\n        "pipeline_run_id": pipeline_run_id.strip() or str(uuid.uuid4()),\n        "batch_started_utc": None,\n    }\n    if batch_started_utc.strip():\n        if not re.fullmatch(\n            r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,6})?Z", batch_started_utc\n        ):\n            raise ValueError(\n                "batch_started_utc must be an explicit UTC ISO timestamp ending Z"\n            )\n        result["batch_started_utc"] = datetime.fromisoformat(batch_started_utc)\n    if not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", result["pipeline_run_id"]):\n        raise ValueError("Invalid pipeline_run_id")\n    for table in result["tables"]:\n        if not re.fullmatch(r"[a-z][a-z0-9_]*", table["name"]):\n            raise ValueError("Invalid source table name")\n    return result\n\n\ndef manifest_processing_context(\n    context_json: str, expected_context_sha256: str, *, expected_lakehouse: str\n) -> dict[str, Any]:\n    """Validate a detached manifest/receipt context without relaxing legacy guards."""\n    if (\n        not isinstance(context_json, str)\n        or len(context_json.encode("utf-8")) > 16 * 1024 * 1024\n    ):\n        raise ValueError("Manifest source context exceeds its supported bound")\n    try:\n        config = json.loads(context_json)\n        canonical = json.dumps(\n            config,\n            sort_keys=True,\n            ensure_ascii=True,\n            separators=(",", ":"),\n            allow_nan=False,\n        )\n    except (TypeError, ValueError):\n        raise ValueError("Manifest source context must contain finite JSON") from None\n    if (\n        not isinstance(config, dict)\n        or hashlib.sha256(canonical.encode("ascii")).hexdigest()\n        != expected_context_sha256\n    ):\n        raise ValueError(\n            "Manifest source context hash differs from its pinned reference"\n        )\n    fields = {\n        "source_code",\n        "lakehouse",\n        "silver_schema",\n        "quarantine_schema",\n        "bronze_root",\n        "contract_hash",\n        "seed_snapshot_date",\n        "tables",\n        "metadata_version",\n        "metadata_run_id",\n        "source_feed_id",\n        "source_batch_id",\n        "source_type",\n        "description_slug",\n        "observed_utc",\n        "processing_date",\n        "pipeline_run_id",\n        "raw_source_files",\n        "raw_source_sizes",\n        "raw_targets",\n        "source_selection_complete",\n        "required_source_tables",\n        "selected_tables",\n        "bronze_receipt_sha256",\n        "source_rule_specs",\n    }\n    if set(config) != fields:\n        raise ValueError(\n            "Manifest source context fields differ from the supported contract"\n        )\n    for name in (\n        "source_code",\n        "lakehouse",\n        "silver_schema",\n        "quarantine_schema",\n        "description_slug",\n    ):\n        if not isinstance(config[name], str) or not re.fullmatch(\n            r"[a-z][a-z0-9_]{0,127}", config[name]\n        ):\n            raise ValueError("Invalid source namespace in manifest context")\n    if config["lakehouse"] != expected_lakehouse:\n        raise ValueError("Manifest source context differs from the worker Lakehouse")\n    if config["source_type"] not in {"sql", "ftp", "sftp"}:\n        raise ValueError("Invalid source transport in manifest context")\n    source = config["source_code"]\n    prefix = (\n        f"Files/bronze/{source}_{config[\'description_slug\']}_{config[\'source_type\']}"\n    )\n    if config["bronze_root"] != prefix:\n        raise ValueError("Manifest source Bronze prefix differs from its feed")\n    if not isinstance(config["metadata_run_id"], str) or not re.fullmatch(\n        r"[A-Za-z0-9_-]{1,128}", config["metadata_run_id"]\n    ):\n        raise ValueError("Invalid metadata run identity")\n    for name in ("contract_hash", "source_batch_id", "bronze_receipt_sha256"):\n        if not isinstance(config[name], str) or not re.fullmatch(\n            r"[0-9a-f]{64}", config[name]\n        ):\n            raise ValueError("An explicit source/receipt SHA-256 is required")\n    expected_batch = hashlib.sha256(\n        json.dumps(\n            {"feed_id": config["source_feed_id"], "run_id": config["metadata_run_id"]},\n            ensure_ascii=True,\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("ascii")\n    ).hexdigest()\n    if (\n        config["source_batch_id"] != expected_batch\n        or config["pipeline_run_id"] != expected_batch\n    ):\n        raise ValueError(\n            "Source batch lineage cannot be replaced by a run or attempt identity"\n        )\n    if not isinstance(config["observed_utc"], str) or not re.fullmatch(\n        r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,6})?Z", config["observed_utc"]\n    ):\n        raise ValueError("Manifest observation time must be an explicit UTC instant")\n    observed = datetime.fromisoformat(config["observed_utc"])\n    if config["processing_date"] != observed.date().isoformat():\n        raise ValueError(\n            "A manifest batch cannot be assigned a different processing date"\n        )\n    if type(config["source_selection_complete"]) is not bool:\n        raise ValueError("Source completeness must be explicitly Boolean")\n    tables = config["tables"]\n    if not isinstance(tables, list) or not tables:\n        raise ValueError("Manifest source requires explicit table contracts")\n    names = []\n    for table in tables:\n        if not isinstance(table, dict) or not re.fullmatch(\n            r"[a-z][a-z0-9_]*", str(table.get("name", ""))\n        ):\n            raise ValueError("Invalid source table contract")\n        names.append(table["name"])\n        hash_column_order(table)\n    _business_rule_handlers()["validate_rule_specs"](\n        config["source_rule_specs"], scope="source"\n    )\n    if len(set(names)) != len(names):\n        raise ValueError("Duplicate source table identity")\n    required, selected = config["required_source_tables"], config["selected_tables"]\n    if (\n        not isinstance(required, list)\n        or not isinstance(selected, list)\n        or any(not isinstance(name, str) for name in [*required, *selected])\n        or set(required) != set(names)\n        or len(required) != len(set(required))\n        or not selected\n        or not set(selected) <= set(required)\n        or len(selected) != len(set(selected))\n        or config["source_selection_complete"] != (set(selected) == set(required))\n    ):\n        raise ValueError(\n            "Manifest source selection/completeness differs from its contracts"\n        )\n    for table in tables:\n        if table["name"] in selected:\n            _business_rule_handlers()["validate_rule_specs"](\n                table.get("rule_specs"), scope="object"\n            )\n    mappings = [\n        config[name] for name in ("raw_source_files", "raw_source_sizes", "raw_targets")\n    ]\n    if any(not isinstance(mapping, dict) for mapping in mappings):\n        raise ValueError("Manifest raw membership must be structured")\n    files, sizes, targets = mappings\n    if (\n        set(files) != set(sizes)\n        or set(files) != set(targets)\n        or not set(files) <= set(names)\n        or not set(selected) <= set(files)\n    ):\n        raise ValueError("Manifest raw membership omits or invents source tables")\n    batch_prefix = f"{prefix}/{observed:%Y%m%d}/{expected_batch}"\n    all_paths = set()\n    for name, paths in files.items():\n        if (\n            not isinstance(paths, list)\n            or not 1 <= len(paths) <= 5000\n            or not isinstance(sizes[name], list)\n            or len(sizes[name]) != len(paths)\n        ):\n            raise ValueError("Invalid bounded raw-file membership")\n        target = targets[name]\n        if not isinstance(target, str) or not target.startswith(batch_prefix + "/"):\n            raise ValueError("Raw target escapes its pinned source batch")\n        for path, size in zip(paths, sizes[name], strict=True):\n            if (\n                not isinstance(path, str)\n                or not re.fullmatch(r"Files/bronze/[A-Za-z0-9_./ -]{1,1950}", path)\n                or not path.startswith(batch_prefix + "/")\n                or any(part in {"", ".", ".."} for part in path.split("/"))\n                or type(size) is not int\n                or not 0 <= size <= 2**63 - 1\n                or path in all_paths\n            ):\n                raise ValueError("Invalid or duplicate raw-file membership")\n            if config["source_type"] == "sql" and (\n                path != target or len(paths) != 1 or size == 0\n            ):\n                raise ValueError("SQL source requires one complete contracted CSV")\n            if config["source_type"] != "sql" and not path.startswith(target + "/"):\n                raise ValueError("File membership escapes its configured object folder")\n            all_paths.add(path)\n    return {\n        **config,\n        "processing_date": observed.date(),\n        "batch_started_utc": observed,\n        "metadata_mode": "manifest_v1",\n    }\n\n\ndef source_path(context: dict[str, Any], table: dict[str, Any]) -> str:\n    if context.get("metadata_mode") == "manifest_v1":\n        target = context["raw_targets"].get(table["name"])\n        if target is None:\n            raise ValueError(\n                "No selected immutable Bronze target exists for this table"\n            )\n        return target\n    key = context["processing_date"].strftime("%Y%m%d")\n    return f"{context[\'bronze_root\']}/{key}/{context[\'pipeline_run_id\']}/{table[\'name\']}.csv"\n\n\ndef source_files(context: dict[str, Any], table: dict[str, Any]) -> tuple[str, ...]:\n    if context.get("metadata_mode") != "manifest_v1":\n        return (source_path(context, table),)\n    paths = context["raw_source_files"].get(table["name"])\n    if not paths:\n        raise ValueError("No selected immutable Bronze files exist for this table")\n    return tuple(paths)\n\n\ndef table_path(\n    context: dict[str, Any], table: dict[str, Any], *, quarantine: bool = False\n) -> str:\n    schema = context["quarantine_schema" if quarantine else "silver_schema"]\n    return f"{context[\'lakehouse\']}.{schema}.{table[\'name\']}"\n\n\ndef _column(name: str):\n    from pyspark.sql import functions as F\n\n    return F.col(f"`{name}`")\n\n\ndef _operation_errors():\n    """Only expected engine/IO failures are aggregated; programming errors propagate."""\n    from py4j.protocol import Py4JError\n    from pyspark.errors import PySparkException\n\n    return (OSError, ValueError, RuntimeError, csv.Error, Py4JError, PySparkException)\n\n\ndef _cast(column: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    name = f"`{column[\'name\']}`"\n    kind = column["data_type"]\n    if kind in {"varchar", "uuid"}:\n        return _column(column["name"])\n    if kind == "datetime":\n        if column.get("format") == "CSD_TIMESTAMP":\n            expression = f"regexp_replace({name}, \':([0-9]{{3}})$\', \'.$1\')"\n        else:\n            expression = name\n        return F.expr(f"try_cast({expression} AS timestamp_ntz)")\n    target = {\n        "int": "int",\n        "bigint": "bigint",\n        "money": "decimal(19,4)",\n        "float": "double",\n        "bit": "boolean",\n        "boolean": "boolean",\n    }.get(kind)\n    if kind == "decimal":\n        target = f"decimal({column[\'precision\']},{column[\'scale\']})"\n    return F.expr(f"try_cast({name} AS {target})")\n\n\ndef _invalid_value(column: dict[str, Any], parsed):\n    from pyspark.sql import functions as F\n\n    raw = _column(column["name"])\n    kind = column["data_type"]\n    invalid = F.lit(False)\n    if kind not in {"varchar", "uuid"}:\n        invalid = parsed.isNull()\n    if kind in {"int", "bigint"}:\n        invalid = invalid | ~raw.rlike(r"^-?[0-9]+$")\n    elif kind in {"money", "decimal"}:\n        scale = column.get("scale", 4 if kind == "money" else 2)\n        invalid = invalid | ~raw.rlike(rf"^-?[0-9]+\\.[0-9]{{{scale}}}$")\n        if kind == "money":\n            invalid = invalid | ~parsed.between(\n                F.lit(Decimal("-922337203685477.5808")),\n                F.lit(Decimal("922337203685477.5807")),\n            )\n    elif kind == "float":\n        invalid = invalid | F.isnan(parsed) | parsed.isin(float("inf"), float("-inf"))\n        if "perc" in column["name"]:\n            invalid = invalid | ~parsed.between(0, 100)\n    elif kind == "bit":\n        invalid = invalid | ~raw.isin("0", "1")\n    elif kind == "boolean":\n        invalid = invalid | ~raw.isin("true", "false")\n    elif kind == "uuid":\n        invalid = ~raw.rlike(\n            r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"\n        )\n    elif kind == "datetime":\n        if column.get("format") == "CSD_TIMESTAMP":\n            valid_shape = raw.rlike(r"^\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d:\\d{3}$") | (\n                raw == "1900-01-01T00:00:00"\n            )\n        else:\n            valid_shape = raw.rlike(r"^\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d\\.\\d{3}$")\n        invalid = invalid | ~valid_shape | raw.startswith("0000")\n    for key, check in (\n        ("max_length", lambda n: F.length(raw) > n),\n        ("min_length", lambda n: F.length(raw) < n),\n        ("fixed_length", lambda n: F.length(raw) != n),\n        ("minimum", lambda n: parsed < F.lit(n).cast(parsed_type(column))),\n        ("maximum", lambda n: parsed > F.lit(n).cast(parsed_type(column))),\n    ):\n        if column.get(key) is not None:\n            invalid = invalid | check(column[key])\n    if column.get("pattern"):\n        invalid = invalid | ~raw.rlike("^(?:" + column["pattern"] + ")$")\n    date_format = column.get("format")\n    if date_format in {"YYYYMMDD", "YYYYMM"}:\n        invalid = invalid | raw.startswith("0000")\n    if date_format in {"YYYYMMDD", "YYYYMM", "HHMMSS"}:\n        name = f"`{column[\'name\']}`"\n        if date_format == "YYYYMMDD":\n            normalized = f"concat(substr({name},1,4),\'-\',substr({name},5,2),\'-\',substr({name},7,2))"\n            parsed_date = F.expr(f"try_cast({normalized} AS date)")\n            invalid = invalid | ~raw.rlike(r"^\\d{8}$") | parsed_date.isNull()\n        elif date_format == "YYYYMM":\n            normalized = f"concat(substr({name},1,4),\'-\',substr({name},5,2),\'-01\')"\n            invalid = (\n                invalid\n                | ~raw.rlike(r"^\\d{6}$")\n                | F.expr(f"try_cast({normalized} AS date)").isNull()\n            )\n        else:\n            invalid = invalid | ~raw.rlike(r"^(?:[01]\\d|2[0-3])[0-5]\\d[0-5]\\d$")\n    return raw.isNotNull() & F.coalesce(invalid, F.lit(True))\n\n\ndef parsed_type(column: dict[str, Any]) -> str:\n    if column["data_type"] == "decimal":\n        return f"decimal({column[\'precision\']},{column[\'scale\']})"\n    return {\n        "money": "decimal(19,4)",\n        "float": "double",\n        "int": "int",\n        "bigint": "bigint",\n    }.get(column["data_type"], "string")\n\n\ndef prepare_rows(raw, table: dict[str, Any]):\n    """Parse with native Spark expressions and retain raw rejected field values."""\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    columns = table["columns"]\n    names = [c["name"] for c in columns]\n    if "_corrupt_record" not in raw.columns:\n        raw = raw.withColumn("_corrupt_record", F.lit(None).cast("string"))\n    raw = raw.select(\n        *[\n            F.when(_column(n) == "", F.lit(None)).otherwise(_column(n)).alias(n)\n            for n in names\n        ],\n        "_corrupt_record",\n        *([F.col("_source_file_path")] if "_source_file_path" in raw.columns else []),\n    )\n    reasons = [\n        F.when(_column("_corrupt_record").isNotNull(), F.lit("MALFORMED_CSV_RECORD"))\n    ]\n    expressions = []\n    for column in columns:\n        name = column["name"]\n        parsed = _cast(column)\n        expressions.append(\n            parsed.alias(\n                name,\n                metadata={\n                    key: column[key]\n                    for key in ("classification", "source_type", "unit")\n                    if column.get(key) is not None\n                },\n            )\n        )\n        if not column["nullable"] or name in table["keys"]:\n            reasons.append(\n                F.when(_column(name).isNull(), F.lit(f"REQUIRED_FIELD_NULL:{name}"))\n            )\n        reasons.append(\n            F.when(_invalid_value(column, parsed), F.lit(f"INVALID_VALUE:{name}"))\n        )\n    typed = raw.select(\n        *expressions,\n        F.struct(*[_column(n) for n in names]).alias("_raw_record"),\n        F.col("_corrupt_record").alias("ingest_corrupt_record"),\n        F.filter(F.array(*reasons), lambda reason: reason.isNotNull()).alias(\n            "_dq_reasons"\n        ),\n        *([F.col("_source_file_path")] if "_source_file_path" in raw.columns else []),\n    )\n    duplicates = Window.partitionBy(*[_column(k) for k in table["keys"]])\n    typed = typed.withColumn(\n        "_dq_reasons",\n        F.when(\n            F.count(F.lit(1)).over(duplicates) > 1,\n            F.array_union("_dq_reasons", F.array(F.lit("DUPLICATE_BUSINESS_KEY"))),\n        ).otherwise(F.col("_dq_reasons")),\n    )\n    row_rules = _business_rule_handlers()["row_rules"](table)\n    if row_rules:\n        typed = typed.withColumn(\n            "_dq_reasons",\n            F.array_union(\n                "_dq_reasons",\n                F.filter(\n                    F.array(\n                        *[\n                            F.when(condition, F.lit(rule))\n                            for condition, rule in row_rules\n                        ]\n                    ),\n                    lambda reason: reason.isNotNull(),\n                ),\n            ),\n        )\n    if "rule_specs" in table:\n        typed = _business_rule_handlers()["apply_rules"](typed, table)\n    return typed\n\n\ndef blocking_dq_reasons(table: dict[str, Any]):\n    return _business_rule_handlers()["blocking_reasons"](table)\n\n\ndef _audit(\n    spark, context: dict[str, Any], name: str, rows: list[dict[str, Any]]\n) -> None:\n    if not rows:\n        return\n    common = {\n        "source_code": context["source_code"],\n        "pipeline_run_id": context["pipeline_run_id"],\n        "processing_date": context["processing_date"],\n        "contract_hash": context["contract_hash"],\n        "event_utc": datetime.now(UTC),\n    }\n    schema = AUDIT_COMMON + ", " + AUDIT_SCHEMAS[name]\n    frame = spark.createDataFrame([{**common, **row} for row in rows], schema=schema)\n    frame.write.format("delta").mode("append").saveAsTable(\n        f"{context[\'lakehouse\']}.audit.{name}"\n    )\n\n\ndef setup_namespaces(spark, context: dict[str, Any]) -> None:\n    """Create explicit audit schemas before independent sources can append."""\n    lakehouse = context["lakehouse"]\n    for schema in (\n        context["silver_schema"],\n        context["quarantine_schema"],\n        "audit",\n    ):\n        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {lakehouse}.{schema}")\n    for name, fields in AUDIT_SCHEMAS.items():\n        target = f"{lakehouse}.audit.{name}"\n        ddl = AUDIT_COMMON + ", " + fields\n        spark.sql(f"CREATE TABLE IF NOT EXISTS {target} ({ddl}) USING DELTA")\n        expected = spark.createDataFrame([], ddl).schema\n        actual = spark.table(target).schema\n        if [(f.name, f.dataType) for f in expected] != [\n            (f.name, f.dataType) for f in actual\n        ]:\n            raise RuntimeError(\n                f"Incompatible audit schema: {target}; do not overwrite it"\n            )\n\n\ndef read_raw(spark, fs, context: dict[str, Any], table: dict[str, Any]):\n    from pyspark.sql import functions as F\n    from pyspark.sql import types as T\n\n    paths = source_files(context, table)\n    names = [c["name"] for c in table["columns"]]\n    for index, path in enumerate(paths):\n        if not fs.exists(path):\n            raise FileNotFoundError(f"Missing immutable extraction batch: {path}")\n        if context.get("metadata_mode") == "manifest_v1":\n            entries = [\n                entry\n                for entry in fs.ls(str(PurePosixPath(path).parent))\n                if entry.name == PurePosixPath(path).name\n            ]\n            if (\n                len(entries) != 1\n                or entries[0].size != context["raw_source_sizes"][table["name"]][index]\n            ):\n                raise RuntimeError(\n                    "Bronze file membership changed after receipt sealing"\n                )\n        head = fs.head(path, 65536).removeprefix("\\ufeff")\n        first_line = head.splitlines()[0] if head else ""\n        if next(csv.reader([first_line]), []) != names:\n            raise ValueError(\n                f"{table[\'name\']}: source header/order differs from the contract"\n            )\n    schema = T.StructType(\n        [T.StructField(n, T.StringType(), True) for n in names]\n        + [T.StructField("_corrupt_record", T.StringType(), True)]\n    )\n    frame = (\n        spark.read.schema(schema)\n        .option("header", True)\n        .option("inferSchema", False)\n        .option("mode", "PERMISSIVE")\n        .option("multiLine", True)\n        .option("columnNameOfCorruptRecord", "_corrupt_record")\n        .option("quote", \'"\')\n        .option("escape", "\\\\")\n        .option("nullValue", "")\n        .csv(list(paths) if context.get("metadata_mode") == "manifest_v1" else paths[0])\n    )\n    if context.get("metadata_mode") == "manifest_v1":\n        frame = frame.withColumn("_source_file_path", F.input_file_name())\n    return frame\n\n\ndef snapshot_fingerprint(frame) -> tuple[str, int]:\n    """Hash a logical record multiset with bounded, distributed hash-word sums."""\n    from pyspark.sql import functions as F\n\n    columns = [name for name in frame.columns if name != "_source_file_path"]\n    hashed = frame.select(\n        F.sha2(\n            F.to_json(\n                F.struct(*[_column(n) for n in columns]),\n                {"ignoreNullFields": "false"},\n            ),\n            256,\n        ).alias("_record_hash")\n    )\n    summary = hashed.agg(\n        F.count("*").alias("rows"),\n        *[\n            F.sum(\n                F.conv(F.substring("_record_hash", start, 16), 16, 10).cast(\n                    "decimal(38,0)"\n                )\n            ).alias(f"word_{start}")\n            for start in (1, 17, 33, 49)\n        ],\n    ).first()\n    if summary["rows"] and any(summary[f"word_{n}"] is None for n in (1, 17, 33, 49)):\n        raise RuntimeError("Extraction fingerprint aggregation overflow")\n    payload = [\n        columns,\n        summary["rows"],\n        *[str(summary[f"word_{n}"]) for n in (1, 17, 33, 49)],\n    ]\n    return hashlib.sha256(\n        json.dumps(payload, separators=(",", ":")).encode()\n    ).hexdigest(), summary["rows"]\n\n\ndef batch_manifest(spark, context: dict[str, Any], table: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    rows = (\n        spark.table(f"{context[\'lakehouse\']}.audit.bronze_batch_manifest")\n        .filter(\n            (F.col("source_code") == context["source_code"])\n            & (F.col("pipeline_run_id") == context["pipeline_run_id"])\n            & (F.col("table_name") == table["name"])\n        )\n        .limit(2)\n        .collect()\n    )\n    if len(rows) > 1:\n        raise RuntimeError("Duplicate immutable batch manifest")\n    if not rows:\n        return None\n    result = rows[0].asDict()\n    if (\n        result["processing_date"] != context["processing_date"]\n        or result["contract_hash"] != context["contract_hash"]\n        or result["source_path"] != source_path(context, table)\n    ):\n        raise RuntimeError(\n            "A sealed extraction batch cannot be relabelled or re-contracted"\n        )\n    observed = result["observed_utc"].astimezone(UTC)\n    if (\n        context["batch_started_utc"] is not None\n        and context["batch_started_utc"] != observed\n    ):\n        raise RuntimeError("A sealed extraction batch timestamp cannot change")\n    context["batch_started_utc"] = observed\n    return result\n\n\ndef verify_batch(spark, fs, context: dict[str, Any], table: dict[str, Any]):\n    manifest = batch_manifest(spark, context, table)\n    if manifest is None:\n        raise RuntimeError("Inventory must seal the extraction batch before processing")\n    raw = read_raw(spark, fs, context, table)\n    fingerprint, count = snapshot_fingerprint(raw)\n    if fingerprint != manifest["snapshot_hash"] or count != manifest["source_rows"]:\n        raise RuntimeError(\n            "Sealed Bronze batch content changed; use a new extraction run"\n        )\n    return raw, manifest\n\n\ndef inventory(\n    spark, fs, context: dict[str, Any], *, table_names: list[str] | None = None\n) -> None:\n    selected = (\n        table_names\n        if table_names is not None\n        else (\n            list(context["raw_source_files"])\n            if context.get("metadata_mode") == "manifest_v1"\n            else [table["name"] for table in context["tables"]]\n        )\n    )\n    if (\n        not selected\n        or len(set(selected)) != len(selected)\n        or not set(selected) <= {table["name"] for table in context["tables"]}\n    ):\n        raise ValueError("Inventory requires an exact selected table population")\n    rows = []\n    for table in context["tables"]:\n        if table["name"] not in selected:\n            continue\n        row = {\n            "table_name": table["name"],\n            "source_path": source_path(context, table),\n            "row_count": None,\n            "column_count": None,\n            "status": "SUCCESS",\n            "message": "",\n        }\n        try:\n            fingerprint, row["row_count"] = snapshot_fingerprint(\n                read_raw(spark, fs, context, table)\n            )\n            row["column_count"] = len(table["columns"])\n            if not row["row_count"] and not table["allow_empty"]:\n                raise ValueError("Empty source snapshot is not allowed")\n            manifest = batch_manifest(spark, context, table)\n            if manifest is None:\n                context["batch_started_utc"] = context[\n                    "batch_started_utc"\n                ] or datetime.now(UTC)\n                _audit(\n                    spark,\n                    context,\n                    "bronze_batch_manifest",\n                    [\n                        {\n                            "table_name": table["name"],\n                            "source_path": row["source_path"],\n                            "snapshot_hash": fingerprint,\n                            "source_rows": row["row_count"],\n                            "observed_utc": context["batch_started_utc"],\n                        }\n                    ],\n                )\n            elif (\n                manifest["snapshot_hash"] != fingerprint\n                or manifest["source_rows"] != row["row_count"]\n            ):\n                raise RuntimeError(\n                    "Sealed Bronze batch content changed; use a new extraction run"\n                )\n        except _operation_errors() as error:\n            row.update(\n                status="ERROR", message=f"Inventory failed: {type(error).__name__}"\n            )\n        rows.append(row)\n    _audit(spark, context, "bronze_inventory", rows)\n    failed = [row["table_name"] for row in rows if row["status"] != "SUCCESS"]\n    if failed:\n        raise RuntimeError(\n            f"Bronze inventory failed for {failed}; inspect source-scoped audit"\n        )\n\n\ndef add_foreign_key_rules(\n    spark, fs, context: dict[str, Any], table: dict[str, Any], frame\n):\n    from pyspark.sql import functions as F\n\n    specs = {t["name"]: t for t in context["tables"]}\n    for index, relation in enumerate(table["foreign_keys"]):\n        parent = specs[relation["table"]]\n        parent_specs = {c["name"]: c for c in parent["columns"]}\n        keys = (\n            read_raw(spark, fs, context, parent)\n            .select(\n                *[\n                    _cast(parent_specs[name]).alias(f"_parent_key_{position}")\n                    for position, name in enumerate(relation["target_columns"])\n                ]\n            )\n            .distinct()\n            .withColumn("_parent_found", F.lit(True))\n        )\n        condition = reduce(\n            lambda left, right: left & right,\n            [\n                _column(name).eqNullSafe(F.col(f"_parent_key_{position}"))\n                for position, name in enumerate(relation["columns"])\n            ],\n        )\n        frame = (\n            frame.join(keys, condition, "left")\n            .withColumn(\n                "_dq_reasons",\n                F.when(\n                    F.col("_parent_found").isNull(),\n                    F.array_union(\n                        "_dq_reasons",\n                        F.array(F.lit(f"ORPHAN_FOREIGN_KEY:{index}:{parent[\'name\']}")),\n                    ),\n                ).otherwise(F.col("_dq_reasons")),\n            )\n            .drop(\n                "_parent_found",\n                *[f"_parent_key_{p}" for p in range(len(relation["columns"]))],\n            )\n        )\n    return frame\n\n\ndef _with_lineage(frame, context: dict[str, Any], table: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    return frame.select(\n        "*",\n        F.lit(context["pipeline_run_id"]).alias("ingest_batch_run_id"),\n        F.lit(context["processing_date"]).alias("ingest_processing_date"),\n        F.lit(context["processing_date"]).alias("ingest_extract_date"),\n        F.current_timestamp().alias("ingest_loaded_utc"),\n        F.lit(context["source_code"]).alias("ingest_source_system"),\n        (\n            F.regexp_extract("_source_file_path", r"([^/\\\\]+)$", 1)\n            if "_source_file_path" in frame.columns\n            else F.lit(PurePosixPath(source_path(context, table)).name)\n        ).alias("ingest_source_file_name"),\n        F.lit(context["contract_hash"]).alias("ingest_contract_hash"),\n        (F.size("_dq_reasons") == 0).alias("dq_passed"),\n        F.when(F.size("_dq_reasons") > 0, F.concat_ws(";", "_dq_reasons")).alias(\n            "dq_failure_reason"\n        ),\n        (\n            F.when(F.size("_dq_warnings") > 0, F.concat_ws(";", "_dq_warnings"))\n            if "_dq_warnings" in frame.columns\n            else F.lit(None).cast("string")\n        ).alias("dq_warning"),\n    )\n\n\ndef hash_column_order(table: dict[str, Any]) -> tuple[list[str], list[str]]:\n    observations = set(table.get("change_hash_exclusions", []))\n    tracked = [c["name"] for c in table["columns"] if c["name"] not in observations]\n    keys = table["keys"]\n    policy = table.get("hash_policy")\n    if policy is not None:\n        if (\n            not isinstance(policy, dict)\n            or set(policy) != {"key_columns", "value_columns", "encoding"}\n            or policy["encoding"] != "spark_json_struct_sha256_v1"\n            or policy["key_columns"] != keys\n            or not isinstance(policy["value_columns"], list)\n            or any(not isinstance(name, str) for name in policy["value_columns"])\n            or len(policy["value_columns"]) != len(set(policy["value_columns"]))\n            or set(policy["value_columns"]) != set(tracked)\n        ):\n            raise ValueError(\n                "Stable business hash policy differs from the table contract"\n            )\n        tracked = policy["value_columns"]\n    if (\n        not keys\n        or len(keys) != len(set(keys))\n        or not set(keys) <= {column["name"] for column in table["columns"]}\n    ):\n        raise ValueError("Business hash keys differ from the column contract")\n    return list(keys), list(tracked)\n\n\ndef with_record_hashes(frame, table: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    keys, tracked = hash_column_order(table)\n    return frame.select(\n        "*",\n        F.sha2(\n            F.to_json(\n                F.struct(*[_column(k) for k in keys]),\n                {"ignoreNullFields": "false"},\n            ),\n            256,\n        ).alias("_record_key_hash"),\n        F.sha2(\n            F.to_json(\n                F.struct(*[_column(n) for n in tracked]), {"ignoreNullFields": "false"}\n            ),\n            256,\n        ).alias("_record_hash"),\n    )\n\n\ndef _record_join(table: dict[str, Any], prefix: str):\n    from pyspark.sql import functions as F\n\n    return reduce(\n        lambda a, b: a & b,\n        [\n            _column(key).eqNullSafe(F.col(f"{prefix}{i}"))\n            for i, key in enumerate(table["keys"])\n        ],\n    )\n\n\ndef eligible_silver(\n    spark,\n    context: dict[str, Any],\n    table: dict[str, Any],\n    *,\n    include_current: bool = False,\n    exclude_current: bool = False,\n):\n    """Return only committed versions, optionally including this staged batch."""\n    from pyspark.sql import functions as F\n\n    commits = (\n        spark.table(f"{context[\'lakehouse\']}.audit.bronze_silver_validation")\n        .filter(\n            (F.col("source_code") == context["source_code"])\n            & (F.col("check_name") == "source_batch_committed")\n            & (F.col("status") == "SUCCESS")\n        )\n        .select(\n            F.col("pipeline_run_id").alias("_commit_run"),\n            F.col("processing_date").alias("_commit_date"),\n            F.col("contract_hash").alias("_commit_contract"),\n        )\n        .distinct()\n        .withColumn("_committed", F.lit(True))\n    )\n    records = spark.table(table_path(context, table))\n    current = (\n        (F.col("ingest_batch_run_id") == context["pipeline_run_id"])\n        & (F.col("ingest_processing_date") == F.lit(context["processing_date"]))\n        & (F.col("ingest_contract_hash") == context["contract_hash"])\n    )\n    condition = (\n        (F.col("ingest_batch_run_id") == F.col("_commit_run"))\n        & (F.col("ingest_processing_date") == F.col("_commit_date"))\n        & (F.col("ingest_contract_hash") == F.col("_commit_contract"))\n    )\n    eligible = F.coalesce(F.col("_committed"), F.lit(False))\n    if include_current:\n        eligible = eligible | current\n    if exclude_current:\n        eligible = eligible & ~current\n    return (\n        records.join(commits, condition, "left")\n        .filter(eligible)\n        .drop("_commit_run", "_commit_date", "_commit_contract", "_committed")\n    )\n\n\ndef latest_silver(\n    spark,\n    context: dict[str, Any],\n    table: dict[str, Any],\n    *,\n    include_current: bool = False,\n    exclude_current: bool = False,\n):\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    return (\n        eligible_silver(\n            spark,\n            context,\n            table,\n            include_current=include_current,\n            exclude_current=exclude_current,\n        )\n        .withColumn(\n            "_record_latest",\n            F.row_number().over(\n                Window.partitionBy(*table["keys"]).orderBy(\n                    F.col("record_version").desc()\n                )\n            ),\n        )\n        .filter("_record_latest = 1")\n        .drop("_record_latest")\n    )\n\n\ndef silver_versions(\n    frame, context: dict[str, Any], table: dict[str, Any], columns: list[str]\n):\n    from pyspark.sql import functions as F\n\n    change_type = F.when(F.col("_previous_hash").isNull(), "INSERT").otherwise("UPDATE")\n    event_id = F.sha2(\n        F.to_json(\n            F.struct(\n                F.lit(context["source_code"]).alias("source"),\n                F.lit(table["name"]).alias("table"),\n                F.col("ingest_batch_run_id"),\n                F.col("_record_key_hash"),\n            )\n        ),\n        256,\n    )\n    return frame.select(\n        *[_column(n) for n in columns],\n        event_id.alias("record_id"),\n        F.col("_record_key_hash").alias("record_key_hash"),\n        F.col("_record_hash").alias("record_hash"),\n        F.col("_previous_hash").alias("record_previous_hash"),\n        (F.coalesce(F.col("_max_version"), F.lit(0)) + 1)\n        .cast("long")\n        .alias("record_version"),\n        change_type.alias("record_change_type"),\n        F.lit(context["batch_started_utc"]).alias("record_observed_utc"),\n        F.lit(source_path(context, table)).alias("record_source_path"),\n        F.lit(context["snapshot_hash"]).alias("record_batch_hash"),\n    )\n\n\ndef delta_table(spark, target: str):\n    """Resolve multipart catalog names through Spark rather than Delta\'s two-part parser."""\n    from delta.tables import DeltaTable\n\n    location = (\n        spark.sql(f"DESCRIBE DETAIL {target}").select("location").first()["location"]\n    )\n    return DeltaTable.forPath(spark, location)\n\n\ndef delta_version(spark, target: str) -> int:\n    return int(\n        delta_table(spark, target).history(1).select("version").first()["version"]\n    )\n\n\ndef append_silver(spark, context: dict[str, Any], table: dict[str, Any], valid) -> None:\n    """Append changed business versions directly to the sole source Silver table."""\n    from pyspark.sql import functions as F\n\n    target = table_path(context, table)\n    if spark.catalog.tableExists(target):\n        physical = [\n            field.name\n            for field in spark.table(target).schema\n            if field.name in valid.columns\n        ]\n        if set(physical) != set(valid.columns):\n            raise RuntimeError(\n                "Silver source fields changed; an explicit compatible migration is required"\n            )\n        valid = valid.select(*[_column(name) for name in physical])\n    hashed = with_record_hashes(valid, table)\n    empty = silver_versions(\n        hashed.limit(0)\n        .withColumn("_previous_hash", F.lit(None).cast("string"))\n        .withColumn("_max_version", F.lit(0).cast("long")),\n        context,\n        table,\n        valid.columns,\n    )\n    if context.get("metadata_mode") == "manifest_v1":\n        layout_runtime = globals().get("_MEDALLION_LAYOUT")\n        if layout_runtime is None:\n            raise RuntimeError(\n                "Generic Silver requires the packaged physical layout adapter"\n            )\n        settings = table.get("runtime_layout", {})\n        capability = settings.get("capabilities")\n        capability = (\n            layout_runtime.RuntimeCapabilities(**capability)\n            if capability is not None\n            else None\n        )\n        existed = spark.catalog.tableExists(target)\n        layout_runtime.ensure_target_layout(\n            spark,\n            target,\n            empty,\n            settings.get("layout"),\n            existing_target=existed,\n            capabilities=capability,\n        )\n        if not existed:\n            spark.sql(\n                f"ALTER TABLE {layout_runtime.quote_table(target)} SET TBLPROPERTIES "\n                "(\'delta.appendOnly\'=\'true\',\'delta.enableChangeDataFeed\'=\'true\')"\n            )\n    if not spark.catalog.tableExists(target):\n        empty.write.format("delta").mode("errorifexists").option(\n            "delta.appendOnly", "true"\n        ).option("delta.enableChangeDataFeed", "true").saveAsTable(target)\n    if [(f.name, f.dataType) for f in spark.table(target).schema] != [\n        (f.name, f.dataType) for f in empty.schema\n    ]:\n        raise RuntimeError(\n            f"Silver version schema mismatch: {target}; explicit non-destructive migration required"\n        )\n    silver = delta_table(spark, target)\n    if (\n        silver.detail()\n        .select("properties")\n        .first()["properties"]\n        .get("delta.appendOnly")\n        != "true"\n    ):\n        raise RuntimeError(\n            "Silver must be an append-only Delta table; explicit migration required"\n        )\n    previous = latest_silver(spark, context, table, exclude_current=True).select(\n        *[_column(k).alias(f"_previous_key_{i}") for i, k in enumerate(table["keys"])],\n        F.col("record_hash").alias("_previous_hash"),\n    )\n    versions = (\n        spark.table(target)\n        .groupBy(*table["keys"])\n        .agg(F.max("record_version").alias("_max_version"))\n        .select(\n            *[\n                _column(k).alias(f"_version_key_{i}")\n                for i, k in enumerate(table["keys"])\n            ],\n            "_max_version",\n        )\n    )\n    changes = (\n        hashed.join(previous, _record_join(table, "_previous_key_"), "left")\n        .join(versions, _record_join(table, "_version_key_"), "left")\n        .filter(~F.col("_record_hash").eqNullSafe(F.col("_previous_hash")))\n    )\n    events = silver_versions(changes, context, table, valid.columns)\n    if events.limit(1).count():\n        silver.alias("tgt").merge(\n            events.alias("src"), "tgt.record_id = src.record_id"\n        ).whenNotMatchedInsertAll().execute()\n\n\ndef guard_batch_order(spark, context: dict[str, Any], table: dict[str, Any]) -> None:\n    from pyspark.sql import functions as F\n\n    newer = spark.table(f"{context[\'lakehouse\']}.audit.bronze_batch_manifest").filter(\n        (F.col("source_code") == context["source_code"])\n        & (F.col("table_name") == table["name"])\n        & (F.col("pipeline_run_id") != context["pipeline_run_id"])\n        & (F.col("observed_utc") >= F.lit(context["batch_started_utc"]))\n    )\n    if newer.limit(1).count():\n        raise RuntimeError(\n            "An older or concurrent batch cannot overwrite newer source state"\n        )\n\n\ndef reconcile_quarantine(\n    spark,\n    context: dict[str, Any],\n    table: dict[str, Any],\n    rejected,\n    *,\n    write_missing: bool,\n) -> None:\n    """Reconcile the exact rejected-row multiset for this source batch."""\n    from pyspark.sql import functions as F\n\n    raw_rejects = rejected.select(\n        *[\n            F.col(f"_raw_record.`{column[\'name\']}`").alias(column["name"])\n            for column in table["columns"]\n        ],\n        *LINEAGE,\n        "ingest_corrupt_record",\n        "dq_passed",\n        "dq_failure_reason",\n        "dq_warning",\n    )\n    target = table_path(context, table, quarantine=True)\n    exists = spark.catalog.tableExists(target)\n    if not exists:\n        if write_missing:\n            if raw_rejects.limit(1).count():\n                raw_rejects.write.format("delta").mode("errorifexists").saveAsTable(\n                    target\n                )\n        elif raw_rejects.limit(1).count():\n            raise RuntimeError("Quarantine evidence is missing for the source batch")\n        return\n\n    existing = spark.table(target)\n    expected_schema = {field.name: field.dataType for field in raw_rejects.schema}\n    actual_schema = {field.name: field.dataType for field in existing.schema}\n    if expected_schema != actual_schema:\n        raise RuntimeError(\n            "Quarantine schema differs from the source contract; migration is required"\n        )\n    comparable = [name for name in raw_rejects.columns if name != "ingest_loaded_utc"]\n    expected = raw_rejects.select(*comparable)\n    retained = existing.filter(\n        F.col("ingest_batch_run_id") == context["pipeline_run_id"]\n    ).select(*comparable)\n    if retained.exceptAll(expected).limit(1).count():\n        raise RuntimeError("Quarantine evidence differs from the source batch")\n    missing = expected.exceptAll(retained)\n    if missing.limit(1).count():\n        if not write_missing:\n            raise RuntimeError("Quarantine evidence is missing for the source batch")\n        (\n            missing.withColumn("ingest_loaded_utc", F.current_timestamp())\n            .select(*existing.columns)\n            .write.format("delta")\n            .mode("append")\n            .saveAsTable(target)\n        )\n\n\ndef process_table(\n    spark, fs, context: dict[str, Any], table: dict[str, Any], progress: dict[str, Any]\n) -> dict[str, Any]:\n    from pyspark.sql import functions as F\n\n    raw, manifest = verify_batch(spark, fs, context, table)\n    guard_batch_order(spark, context, table)\n    context["snapshot_hash"] = manifest["snapshot_hash"]\n    frame = prepare_rows(raw, table)\n    frame = add_foreign_key_rules(spark, fs, context, table, frame)\n    frame = _with_lineage(frame, context, table).cache()\n    try:\n        source_rows = frame.count()\n        if not source_rows and not table["allow_empty"]:\n            raise ValueError("Unexpected empty snapshot")\n        rejected = frame.filter(~F.col("dq_passed"))\n        quarantine_rows = rejected.count()\n        blocking_rows = frame.filter(F.size(blocking_dq_reasons(table)) > 0).count()\n        progress.update(\n            source_rows=source_rows,\n            silver_rows=source_rows - quarantine_rows,\n            quarantine_rows=quarantine_rows,\n        )\n        valid = frame.filter("dq_passed").drop(\n            "_raw_record",\n            "_dq_reasons",\n            "ingest_corrupt_record",\n            "_source_file_path",\n            "_dq_warnings",\n        )\n        append_silver(spark, context, table, valid)\n        reconcile_quarantine(spark, context, table, rejected, write_missing=True)\n        has_warning_rules = any(\n            rule["action"] == "warn" for rule in table.get("rule_specs", [])\n        )\n        if quarantine_rows or has_warning_rules:\n            # Escape reserved prefixes only in summaries, not retained rejection reasons.\n            reasons = F.transform(\n                F.col("_dq_reasons"),\n                lambda reason: F.when(\n                    reason.startswith("warning:") | reason.startswith("failure:"),\n                    F.concat(F.lit("failure:"), reason),\n                ).otherwise(reason),\n            )\n            if has_warning_rules:\n                warnings = F.transform(\n                    F.col("_dq_warnings"),\n                    lambda warning: F.concat(F.lit("warning:"), warning),\n                )\n                reasons = F.concat(reasons, warnings)\n            counts = (\n                frame.select(F.explode(reasons).alias("dq_rule"))\n                .groupBy("dq_rule")\n                .count()\n            )\n            _audit(\n                spark,\n                context,\n                "silver_dq_rule_summary",\n                [\n                    {\n                        "table_name": table["name"],\n                        "dq_rule": row["dq_rule"],\n                        "failure_count": row["count"],\n                    }\n                    for row in counts.toLocalIterator()\n                ],\n            )\n        progress.update(\n            status="SUCCESS" if blocking_rows == 0 else "QUARANTINED", message=""\n        )\n        return progress\n    finally:\n        frame.unpersist()\n\n\ndef load_silver(\n    spark, fs, context: dict[str, Any], *, table_names: list[str] | None = None\n) -> list[dict[str, Any]]:\n    selected = (\n        table_names\n        if table_names is not None\n        else (\n            context["selected_tables"]\n            if context.get("metadata_mode") == "manifest_v1"\n            else [table["name"] for table in context["tables"]]\n        )\n    )\n    if (\n        not isinstance(selected, list)\n        or not selected\n        or any(not isinstance(name, str) for name in selected)\n        or len(selected) != len(set(selected))\n        or not set(selected) <= {table["name"] for table in context["tables"]}\n    ):\n        raise ValueError("Object-scoped Silver selection is missing or ambiguous")\n    results = []\n    for table in context["tables"]:\n        if table["name"] not in selected:\n            continue\n        result = {\n            "table_name": table["name"],\n            "source_path": source_path(context, table),\n            "target_table": table_path(context, table),\n            "source_rows": None,\n            "silver_rows": None,\n            "quarantine_rows": None,\n            "status": "ERROR",\n            "message": "",\n        }\n        try:\n            result = process_table(spark, fs, context, table, result)\n        except _operation_errors() as error:\n            result.update(\n                status="ERROR",\n                message=f"Silver processing failed: {type(error).__name__}",\n            )\n        results.append(result)\n        _audit(spark, context, "silver_load_log", [result])\n        _audit(\n            spark,\n            context,\n            "silver_file_manifest",\n            [\n                {\n                    key: result[key]\n                    for key in (\n                        "table_name",\n                        "source_path",\n                        "source_rows",\n                        "silver_rows",\n                        "quarantine_rows",\n                        "status",\n                    )\n                }\n                | {"source_file_name": PurePosixPath(result["source_path"]).name}\n            ],\n        )\n    failed = [r["table_name"] for r in results if r["status"] != "SUCCESS"]\n    if failed:\n        raise RuntimeError(\n            f"Silver failed for {failed}; Bronze retained; inspect audit/quarantine"\n        )\n    print(\n        json.dumps(\n            {\n                "source": context["source_code"],\n                "tables": len(results),\n                "accepted_rows": sum(r["silver_rows"] for r in results),\n            }\n        )\n    )\n    return results\n\n\ndef _run_audit(spark, context: dict[str, Any], name: str):\n    from pyspark.sql import functions as F\n\n    return spark.table(f"{context[\'lakehouse\']}.audit.{name}").filter(\n        (F.col("source_code") == context["source_code"])\n        & (F.col("pipeline_run_id") == context["pipeline_run_id"])\n        & (F.col("processing_date") == F.lit(context["processing_date"]))\n        & (F.col("contract_hash") == context["contract_hash"])\n    )\n\n\ndef source_commit_evidence(spark, context: dict[str, Any]) -> dict[str, Any]:\n    """Pin one authoritative source commit to the audit Delta version read."""\n    from pyspark.sql import functions as F\n\n    target = f"{context[\'lakehouse\']}.audit.bronze_silver_validation"\n    version = delta_version(spark, target)\n    frame = spark.read.format("delta").option("versionAsOf", version).table(target)\n    markers = (\n        frame.filter(\n            (F.col("source_code") == context["source_code"])\n            & (F.col("pipeline_run_id") == context["pipeline_run_id"])\n            & (F.col("processing_date") == F.lit(context["processing_date"]))\n            & (F.col("contract_hash") == context["contract_hash"])\n            & (F.col("check_name") == "source_batch_committed")\n            & (F.col("status") == "SUCCESS")\n            & (F.col("failed_count") == 0)\n        )\n        .limit(2)\n        .collect()\n    )\n    if len(markers) != 1:\n        raise RuntimeError("Exactly one authoritative source commit is required")\n    proof = {\n        "location": target,\n        "version": version,\n        "source_code": context["source_code"],\n        "source_batch_id": context["pipeline_run_id"],\n        "contract_hash": context["contract_hash"],\n        "processing_date": context["processing_date"].isoformat(),\n    }\n    return {\n        "kind": "source_commit",\n        "location": target,\n        "version": version,\n        "sha256": hashlib.sha256(\n            json.dumps(\n                proof, ensure_ascii=True, sort_keys=True, separators=(",", ":")\n            ).encode("ascii")\n        ).hexdigest(),\n    }\n\n\ndef source_business_checks(data: dict[str, Any], source: str):\n    return _business_rule_handlers()["source_business_checks"](data, source)\n\n\ndef relationship_checks(spark, context: dict[str, Any], table: dict[str, Any], frame):\n    """Validate retained current-state relationships as well as incoming rows."""\n    from pyspark.sql import functions as F\n\n    checks = []\n    for index, relation in enumerate(table["foreign_keys"]):\n        parent = next(t for t in context["tables"] if t["name"] == relation["table"])\n        parent_frame = latest_silver(spark, context, parent, include_current=True)\n        duplicates = (\n            parent_frame.groupBy(*relation["target_columns"])\n            .count()\n            .filter("count > 1")\n            .count()\n        )\n        parent_keys = parent_frame.select(\n            *[\n                _column(key).alias(f"_parent_key_{position}")\n                for position, key in enumerate(relation["target_columns"])\n            ]\n        ).distinct()\n        condition = reduce(\n            lambda left, right: left & right,\n            [\n                _column(key).eqNullSafe(F.col(f"_parent_key_{position}"))\n                for position, key in enumerate(relation["columns"])\n            ],\n        )\n        checks.append((f"foreign_key_{index}_parent_unique", duplicates))\n        checks.append(\n            (\n                f"foreign_key_{index}_resolved",\n                frame.join(parent_keys, condition, "left_anti").count(),\n            )\n        )\n    return checks\n\n\ndef silver_checks(spark, context: dict[str, Any], table: dict[str, Any]):\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    records = spark.table(table_path(context, table))\n    window = Window.partitionBy(*table["keys"]).orderBy("record_version")\n    chain = eligible_silver(spark, context, table, include_current=True).withColumn(\n        "_prior_hash", F.lag("record_hash").over(window)\n    )\n    valid_chain = F.when(\n        F.col("_prior_hash").isNull(),\n        F.col("record_previous_hash").isNull()\n        & (F.col("record_change_type") == "INSERT"),\n    ).otherwise(\n        F.col("record_previous_hash").eqNullSafe(F.col("_prior_hash"))\n        & (F.col("record_change_type") == "UPDATE")\n        & ~F.col("record_hash").eqNullSafe(F.col("_prior_hash"))\n    )\n    hashed = with_record_hashes(records, table)\n    required = [\n        *LINEAGE,\n        *table["keys"],\n        "record_id",\n        "record_key_hash",\n        "record_hash",\n        "record_version",\n        "record_change_type",\n        "record_observed_utc",\n        "record_source_path",\n        "record_batch_hash",\n    ]\n    invalid_hash = (\n        ~F.col("record_key_hash").eqNullSafe(F.col("_record_key_hash"))\n        | ~F.col("record_hash").eqNullSafe(F.col("_record_hash"))\n        | reduce(lambda a, b: a | b, [_column(n).isNull() for n in required])\n        | (F.col("record_version") <= 0)\n    )\n    return [\n        (\n            "record_id_uniqueness",\n            records.groupBy("record_id").count().filter("count > 1").count(),\n        ),\n        (\n            "record_version_uniqueness",\n            records.groupBy(*table["keys"], "record_version")\n            .count()\n            .filter("count > 1")\n            .count(),\n        ),\n        (\n            "record_version_chain",\n            chain.filter(~F.coalesce(valid_chain, F.lit(False))).count(),\n        ),\n        ("record_hash_integrity", hashed.filter(invalid_hash).count()),\n    ]\n\n\ndef validate_source(spark, fs, context: dict[str, Any]) -> None:\n    if (\n        context.get("metadata_mode") == "manifest_v1"\n        and context["source_selection_complete"] is not True\n    ):\n        raise RuntimeError(\n            "Source commit requires a complete metadata source selection"\n        )\n    if context.get("metadata_mode") == "manifest_v1":\n        _business_rule_handlers()["validate_rule_specs"](\n            context["source_rule_specs"], scope="source"\n        )\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    rows = []\n    data = {}\n    logs = {\n        row["table_name"]: row.asDict()\n        for row in _run_audit(spark, context, "silver_load_log")\n        .withColumn(\n            "_latest",\n            F.row_number().over(\n                Window.partitionBy("table_name").orderBy(F.col("event_utc").desc())\n            ),\n        )\n        .filter("_latest = 1")\n        .collect()\n    }\n    for table in context["tables"]:\n        name = table["name"]\n        try:\n            raw, _ = verify_batch(spark, fs, context, table)\n            guard_batch_order(spark, context, table)\n            frame = latest_silver(spark, context, table, include_current=True)\n            log = logs.get(name)\n            if log is None or log["status"] != "SUCCESS":\n                raise RuntimeError("No successful current source/table load")\n            checks = [\n                (\n                    "key_uniqueness",\n                    frame.groupBy(*table["keys"]).count().filter("count > 1").count(),\n                ),\n                (\n                    "load_reconciliation",\n                    int(\n                        log["source_rows"]\n                        != log["silver_rows"] + log["quarantine_rows"]\n                    ),\n                ),\n            ]\n            required = [*LINEAGE, *table["keys"]]\n            nulls = reduce(lambda a, b: a | b, [_column(n).isNull() for n in required])\n            checks.append(("required_lineage_keys", frame.filter(nulls).count()))\n            checks.append(\n                (\n                    "passing_dq_only",\n                    frame.filter(\n                        ~F.coalesce(_column("dq_passed"), F.lit(False))\n                    ).count(),\n                )\n            )\n            prepared = add_foreign_key_rules(\n                spark, fs, context, table, prepare_rows(raw, table)\n            )\n            checks.append(\n                (\n                    "blocking_rule_violations",\n                    prepared.filter(F.size(blocking_dq_reasons(table)) > 0).count(),\n                )\n            )\n            rejected = _with_lineage(prepared, context, table).filter(\n                ~F.col("dq_passed")\n            )\n            reconcile_quarantine(spark, context, table, rejected, write_missing=False)\n            expected = prepared.filter(F.size("_dq_reasons") == 0)\n            data[name] = expected\n            checks.append(\n                ("accepted_batch_count", abs(expected.count() - log["silver_rows"]))\n            )\n            expected_hashes = with_record_hashes(expected, table).select(\n                "_record_key_hash", "_record_hash"\n            )\n            actual_hashes = with_record_hashes(frame, table).select(\n                "_record_key_hash", "_record_hash"\n            )\n            checks.append(\n                (\n                    "source_business_value_equivalence",\n                    expected_hashes.join(\n                        actual_hashes, ["_record_key_hash", "_record_hash"], "left_anti"\n                    ).count(),\n                )\n            )\n            checks.extend(relationship_checks(spark, context, table, frame))\n            checks.extend(silver_checks(spark, context, table))\n            for check, failed in checks:\n                rows.append(\n                    {\n                        "table_name": name,\n                        "check_name": check,\n                        "failed_count": failed,\n                        "status": "SUCCESS" if failed == 0 else "FAILED",\n                    }\n                )\n        except _operation_errors() as error:\n            rows.append(\n                {\n                    "table_name": name,\n                    "check_name": f"table_validation_{type(error).__name__}",\n                    "failed_count": 1,\n                    "status": "FAILED",\n                }\n            )\n    if len(data) == len(context["tables"]):\n        try:\n            for check, failed in source_business_checks(data, context["source_code"]):\n                rows.append(\n                    {\n                        "table_name": "",\n                        "check_name": check,\n                        "failed_count": failed,\n                        "status": "SUCCESS" if failed == 0 else "FAILED",\n                    }\n                )\n        except _operation_errors() as error:\n            rows.append(\n                {\n                    "table_name": "",\n                    "check_name": f"source_rules_{type(error).__name__}",\n                    "failed_count": 1,\n                    "status": "FAILED",\n                }\n            )\n    ready = bool(rows) and all(row["status"] == "SUCCESS" for row in rows)\n    if ready:\n        existing = (\n            _run_audit(spark, context, "bronze_silver_validation")\n            .filter(\n                (F.col("check_name") == "source_batch_committed")\n                & (F.col("status") == "SUCCESS")\n            )\n            .limit(2)\n            .count()\n        )\n        if existing > 1:\n            raise RuntimeError(\n                "Source commit membership is duplicated; reconcile before replay"\n            )\n        if existing == 0:\n            rows.append(\n                {\n                    "table_name": "",\n                    "check_name": "source_batch_committed",\n                    "failed_count": 0,\n                    "status": "SUCCESS",\n                }\n            )\n    rows.append(\n        {\n            "table_name": "",\n            "check_name": "source_ready_for_cleanup",\n            "failed_count": 0 if ready else 1,\n            "status": "SUCCESS" if ready else "FAILED",\n        }\n    )\n    _audit(spark, context, "bronze_silver_validation", rows)\n    if not ready:\n        failed = [r["check_name"] for r in rows if r["status"] != "SUCCESS"]\n        raise RuntimeError(f"Source validation failed: {failed}; Bronze retained")\n    print(\n        json.dumps(\n            {\n                "source": context["source_code"],\n                "validation": "SUCCESS",\n                "checks": len(rows),\n            }\n        )\n    )\n\n\ndef cleanup_snapshots(spark, fs, context: dict[str, Any]) -> None:\n    """Keep the legacy activity name, but never delete immutable Bronze history."""\n    from pyspark.sql import functions as F\n\n    validation = (\n        _run_audit(spark, context, "bronze_silver_validation")\n        .filter("check_name = \'source_ready_for_cleanup\'")\n        .orderBy(F.col("event_utc").desc())\n        .first()\n    )\n    if validation is None or validation["status"] != "SUCCESS":\n        raise RuntimeError(\n            "Cleanup requires successful validation of this source/run/contract"\n        )\n    for name in ("silver_load_log", "bronze_inventory"):\n        latest = (\n            _run_audit(spark, context, name)\n            .agg(F.max("event_utc").alias("latest"))\n            .first()["latest"]\n        )\n        if latest is None or latest > validation["event_utc"]:\n            raise RuntimeError(\n                "Source processing changed after validation; cleanup is not authorized"\n            )\n    rows = []\n    for table in context["tables"]:\n        row = {\n            "table_name": table["name"],\n            "source_path": source_path(context, table),\n            "file_date": context["processing_date"],\n            "size_bytes": None,\n            "action": "RETAIN",\n            "status": "RETAINED_IMMUTABLE",\n            "message": "No automatic Bronze deletion; retention requires a separately approved policy",\n        }\n        try:\n            verify_batch(spark, fs, context, table)\n            guard_batch_order(spark, context, table)\n            if any(failed for _, failed in silver_checks(spark, context, table)):\n                raise RuntimeError(\n                    "Silver version verification failed; batch must remain retained"\n                )\n        except _operation_errors() as error:\n            row.update(\n                status="ERROR",\n                message=f"Retention verification failed: {type(error).__name__}",\n            )\n        rows.append(row)\n    _audit(spark, context, "bronze_cleanup", rows)\n    if any(r["status"] == "ERROR" for r in rows):\n        raise RuntimeError(\n            "Retention verification failed; no cleanup or deletion performed"\n        )\n\n\ndef run_step(\n    spark,\n    notebookutils,\n    config: dict[str, Any],\n    step: str,\n    *,\n    processing_date: str = "",\n    pipeline_run_id: str = "",\n    batch_started_utc: str = "",\n) -> None:\n    """Notebook entry point with source-scoped side effects and explicit failures."""\n    if step == "cleanup" and not processing_date.strip():\n        raise ValueError("Cleanup requires an explicit processing_date")\n    context = processing_context(\n        config, processing_date, pipeline_run_id, batch_started_utc\n    )\n    attached = notebookutils.runtime.context.get("defaultLakehouseName")\n    if attached != config["lakehouse"]:\n        raise RuntimeError(\n            f"Attach {config[\'lakehouse\']} as the default Lakehouse before execution"\n        )\n    spark.conf.set("spark.sql.session.timeZone", "UTC")\n    if step == "setup":\n        setup_namespaces(spark, context)\n    elif step == "inventory":\n        inventory(spark, notebookutils.fs, context)\n    elif step == "silver":\n        load_silver(spark, notebookutils.fs, context)\n    elif step == "validation":\n        validate_source(spark, notebookutils.fs, context)\n    elif step == "cleanup":\n        cleanup_snapshots(spark, notebookutils.fs, context)\n    else:\n        raise ValueError(f"Unknown notebook step: {step}")\n', 'orchestration': '"""Bounded immutable run-manifest transport, embedded into generic notebooks.\n\nMetadata validation is injected from the same pure module used by deployment.\nNo connection credentials, repository imports or mutable active-version reads\nare required by this runtime.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport fnmatch\nimport hashlib\nimport io\nimport json\nimport re\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom pathlib import PurePosixPath\nfrom typing import Any, Literal, Protocol, cast\nfrom uuid import UUID\n\nMAX_MANIFEST_BYTES = 16 * 1024 * 1024\nMAX_RESULT_BYTES = 1024 * 1024\ncsv.field_size_limit(max(csv.field_size_limit(), MAX_MANIFEST_BYTES))\n\n\nclass FileInfo(Protocol):\n    name: str\n    size: int\n\n\nclass FileSystem(Protocol):\n    def exists(self, path: str) -> bool: ...\n    def ls(self, path: str) -> Sequence[FileInfo]: ...\n    def head(self, path: str, max_bytes: int) -> str: ...\n    def mkdirs(self, path: str) -> object: ...\n    def put(self, path: str, contents: str, overwrite: bool = False) -> bool: ...\n\n\nclass PinnedConfiguration(Protocol):\n    version_id: str\n\n    @property\n    def definition_sha256(self) -> str: ...\n\n\nclass SnapshotPlan(Protocol):\n    run_id: str\n    snapshot_json: str\n\n    @property\n    def snapshot_sha256(self) -> str: ...\n\n\nclass NotebookRunner(Protocol):\n    def validateDAG(self, dag: dict[str, object]) -> bool: ...\n    def runMultiple(self, dag: dict[str, object]) -> dict[str, object]: ...\n\n\nclass SilverRuntime(Protocol):\n    def read_raw(\n        self, spark: Any, fs: Any, context: dict[str, Any], table: dict[str, Any]\n    ) -> Any: ...\n    def snapshot_fingerprint(self, frame: Any) -> tuple[str, int]: ...\n    def manifest_processing_context(\n        self,\n        context_json: str,\n        expected_context_sha256: str,\n        *,\n        expected_lakehouse: str,\n    ) -> dict[str, Any]: ...\n    def setup_namespaces(self, spark: Any, context: dict[str, Any]) -> None: ...\n    def inventory(\n        self,\n        spark: Any,\n        fs: Any,\n        context: dict[str, Any],\n        *,\n        table_names: list[str] | None = None,\n    ) -> None: ...\n    def load_silver(\n        self,\n        spark: Any,\n        fs: Any,\n        context: dict[str, Any],\n        *,\n        table_names: list[str] | None = None,\n    ) -> list[dict[str, Any]]: ...\n    def validate_source(self, spark: Any, fs: Any, context: dict[str, Any]) -> None: ...\n    def delta_version(self, spark: Any, target: str) -> int: ...\n    def source_commit_evidence(\n        self, spark: Any, context: dict[str, Any]\n    ) -> dict[str, Any]: ...\n    def cleanup_snapshots(\n        self, spark: Any, fs: Any, context: dict[str, Any]\n    ) -> None: ...\n    def _operation_errors(self) -> tuple[type[Exception], ...]: ...\n\n\nclass RestResponse(Protocol):\n    content: bytes\n\n\nclass ActivityClient(Protocol):\n    def request(\n        self,\n        method: str,\n        path_or_url: str,\n        *,\n        body: dict[str, Any],\n        expected: tuple[int, ...],\n    ) -> RestResponse: ...\n\n\n@dataclass(frozen=True)\nclass VerifiedRunManifest:\n    run_id: str\n    version_id: str\n    snapshot_json: str\n    snapshot_sha256: str\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.snapshot_json))\n\n\ndef _canonical(value: object) -> str:\n    return json.dumps(\n        value,\n        ensure_ascii=True,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    )\n\n\ndef _manifest_path(path: str, run_id: str) -> str:\n    if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", run_id):\n        raise RuntimeError("Invalid run identity for manifest access")\n    if not isinstance(path, str) or "\\\\" in path or ":" in path:\n        raise RuntimeError("Manifest path must be a run-scoped relative OneLake path")\n    parts = path.split("/")\n    if any(\n        part in {"", ".", ".."}\n        or re.fullmatch(r"[A-Za-z0-9_-][A-Za-z0-9_.-]{0,255}", part) is None\n        for part in parts\n    ):\n        raise RuntimeError("Manifest path cannot escape its run scope")\n    if not path.startswith(f"Files/orchestration/{run_id}/"):\n        raise RuntimeError("Manifest path belongs to another run or location")\n    return path\n\n\ndef _read_bounded(fs: FileSystem, path: str, max_bytes: int) -> str:\n    if type(max_bytes) is not int or not 1 <= max_bytes <= MAX_MANIFEST_BYTES:\n        raise RuntimeError("Invalid manifest size bound")\n    if not fs.exists(path):\n        raise FileNotFoundError("Pinned metadata manifest is missing")\n    location = PurePosixPath(path)\n    entries = [\n        item for item in fs.ls(str(location.parent)) if item.name == location.name\n    ]\n    if len(entries) != 1:\n        raise RuntimeError("Manifest file membership is missing or ambiguous")\n    size = entries[0].size\n    if type(size) is not int or not 1 <= size <= max_bytes:\n        raise RuntimeError("Manifest size exceeds the configured bound or is empty")\n    text = fs.head(path, size + 1)\n    if len(text.encode("utf-8")) != size:\n        raise RuntimeError(\n            "Manifest read was truncated or the file changed during access"\n        )\n    return text.removeprefix("\\ufeff")\n\n\ndef _write_immutable(fs: FileSystem, path: str, text: str, max_bytes: int) -> str:\n    if len(text.encode("utf-8")) > max_bytes:\n        raise RuntimeError("Immutable artifact exceeds its supported size bound")\n    if fs.exists(path):\n        if _read_bounded(fs, path, max_bytes) != text:\n            raise RuntimeError("An immutable artifact cannot be changed or overwritten")\n        return path\n    fs.mkdirs(str(PurePosixPath(path).parent))\n    if fs.put(path, text, False) is not True:\n        raise RuntimeError("Immutable artifact persistence was not acknowledged")\n    if _read_bounded(fs, path, max_bytes) != text:\n        raise RuntimeError("Immutable artifact readback differs after persistence")\n    return path\n\n\ndef write_run_manifest(fs: FileSystem, plan: SnapshotPlan) -> str:\n    path = _manifest_path(\n        f"Files/orchestration/{plan.run_id}/manifest.json", plan.run_id\n    )\n    if (\n        hashlib.sha256(plan.snapshot_json.encode("ascii")).hexdigest()\n        != plan.snapshot_sha256\n    ):\n        raise RuntimeError("Run snapshot changed before persistence")\n    if len(plan.snapshot_json.encode("ascii")) > MAX_MANIFEST_BYTES:\n        raise RuntimeError(\n            "Manifest size requires an explicitly supported chunked contract"\n        )\n    return _write_immutable(fs, path, plan.snapshot_json, MAX_MANIFEST_BYTES)\n\n\ndef read_run_manifest(\n    fs: FileSystem,\n    path: str,\n    *,\n    expected_snapshot_sha256: str,\n    run_id: str,\n    version_id: str,\n    load_configuration: Callable[[object], PinnedConfiguration],\n    export_column: str | None = None,\n    max_bytes: int = MAX_MANIFEST_BYTES,\n) -> VerifiedRunManifest:\n    """Read JSON or a single-row Copy CSV export without a 64 KiB truncation."""\n    path = _manifest_path(path, run_id)\n    if not isinstance(expected_snapshot_sha256, str) or not re.fullmatch(\n        r"[0-9a-f]{64}", expected_snapshot_sha256\n    ):\n        raise RuntimeError("An explicit manifest SHA-256 hash is required")\n    text = _read_bounded(fs, path, max_bytes)\n    if export_column is not None:\n        reader = csv.DictReader(io.StringIO(text), escapechar="\\\\")\n        if reader.fieldnames != [export_column]:\n            raise RuntimeError(\n                "Metadata export columns differ from the pinned contract"\n            )\n        rows = list(reader)\n        if len(rows) != 1 or None in rows[0] or rows[0][export_column] is None:\n            raise RuntimeError("Metadata export must contain one complete snapshot row")\n        text = rows[0][export_column]\n    try:\n        payload = json.loads(text)\n        canonical = _canonical(payload)\n    except (ValueError, TypeError):\n        raise RuntimeError("Pinned run manifest is not valid finite JSON") from None\n    digest = hashlib.sha256(canonical.encode("ascii")).hexdigest()\n    if digest != expected_snapshot_sha256:\n        raise RuntimeError("Pinned run manifest hash mismatch")\n    if not isinstance(payload, dict) or type(payload.get("format_version")) is not int:\n        raise RuntimeError("Unsupported run manifest format")\n    if payload["format_version"] != 1:\n        raise RuntimeError("Unsupported run manifest format")\n    if payload.get("run_id") != run_id or payload.get("version_id") != version_id:\n        raise RuntimeError("Manifest run/version identity differs from worker context")\n    if payload.get("can_execute") is not True or payload.get("blockers") != []:\n        raise RuntimeError(\n            "A preview or blocked manifest is not ready for worker execution"\n        )\n    configuration = load_configuration(payload.get("configuration"))\n    if (\n        configuration.version_id != version_id\n        or configuration.definition_sha256 != payload.get("definition_sha256")\n    ):\n        raise RuntimeError("Pinned configuration definition hash or version differs")\n    tasks = payload.get("tasks")\n    if not isinstance(tasks, list) or not tasks:\n        raise RuntimeError("Pinned run manifest has no task membership")\n    if any(not isinstance(task, dict) or "task_id" not in task for task in tasks):\n        raise RuntimeError("Invalid pinned task membership")\n    identities = [task["task_id"] for task in tasks]\n    if any(not isinstance(identity, str) for identity in identities) or len(\n        set(identities)\n    ) != len(identities):\n        raise RuntimeError("Invalid or duplicate pinned task identity")\n    return VerifiedRunManifest(run_id, version_id, canonical, digest)\n\n\ndef resolve_worker_task(\n    manifest: VerifiedRunManifest,\n    task_id: str,\n    *,\n    worker_id: str,\n    code_sha256: str,\n    lakehouse: str,\n) -> dict[str, object]:\n    """Resolve a detached task only for the declared code and Lakehouse binding."""\n    payload = manifest.to_document()\n    tasks = cast(list[dict[str, object]], payload["tasks"])\n    matches = [task for task in tasks if task["task_id"] == task_id]\n    if len(matches) != 1 or matches[0].get("worker_id") != worker_id:\n        raise RuntimeError("Task is not assigned to this pinned worker")\n    configuration = cast(dict[str, object], payload["configuration"])\n    workers = cast(list[dict[str, object]], configuration["workers"])\n    worker_matches = [worker for worker in workers if worker["id"] == worker_id]\n    if len(worker_matches) != 1:\n        raise RuntimeError("Pinned worker definition is missing or ambiguous")\n    worker = worker_matches[0]\n    if worker["code_sha256"] != code_sha256 or worker["lakehouse"] != lakehouse:\n        raise RuntimeError(\n            "Worker implementation or default Lakehouse differs from the snapshot"\n        )\n    objects = cast(list[dict[str, object]], configuration["objects"])\n    obj = next((o for o in objects if o["id"] == matches[0]["object_id"]), None)\n    if obj is None or obj["lakehouse"] != lakehouse or obj["worker_id"] != worker_id:\n        raise RuntimeError("Pinned object binding differs from the task/worker")\n    return {\n        "task": matches[0],\n        "object": obj,\n        "rules": [\n            rule\n            for rule in cast(list[dict[str, object]], configuration["rules"])\n            if rule["object_id"] == obj["id"]\n        ],\n    }\n\n\ndef _token(value: object, label: str, pattern: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(pattern, value) is None:\n        raise RuntimeError(f"Invalid {label}")\n    return value\n\n\ndef _attempt_identity(value: object) -> str:\n    text = _token(value, "attempt identity", r"[0-9a-f-]{36}")\n    try:\n        identity = UUID(text)\n    except ValueError:\n        raise RuntimeError("Invalid attempt identity") from None\n    if str(identity) != text:\n        raise RuntimeError("Attempt identity must be a canonical UUID")\n    return text\n\n\ndef _positive_fence(value: object) -> int:\n    if type(value) is not int or not 1 <= value <= 2**63 - 1:\n        raise RuntimeError("A positive signed bigint fence is required")\n    return value\n\n\n@dataclass(frozen=True)\nclass VerifiedTaskResult:\n    run_id: str\n    task_id: str\n    attempt_id: str\n    fence: int\n    job_id: str\n    status: str\n    result_json: str\n\n    @property\n    def result_sha256(self) -> str:\n        return hashlib.sha256(self.result_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.result_json))\n\n\ndef load_task_result(result_json: str) -> VerifiedTaskResult:\n    """Validate metadata-only result evidence; never accept raw exception text."""\n    if (\n        not isinstance(result_json, str)\n        or len(result_json.encode("utf-8")) > MAX_RESULT_BYTES\n    ):\n        raise RuntimeError("Task result exceeds its bounded metadata contract")\n    try:\n        value = json.loads(result_json)\n        canonical = _canonical(value)\n    except (ValueError, TypeError):\n        raise RuntimeError("Task result must be finite JSON") from None\n    fields = {\n        "format_version",\n        "run_id",\n        "task_id",\n        "source_batch_id",\n        "attempt_id",\n        "fence",\n        "job_id",\n        "status",\n        "finished_utc",\n        "artifact_refs",\n        "readiness",\n        "error_code",\n        "metrics",\n    }\n    if not isinstance(value, dict) or set(value) != fields or canonical != result_json:\n        raise RuntimeError("Task result fields or canonical encoding differ")\n    if type(value["format_version"]) is not int or value["format_version"] != 1:\n        raise RuntimeError("Unsupported task result format")\n    run_id = _token(value["run_id"], "run identity", r"[A-Za-z0-9_-]{1,128}")\n    task_id = _token(value["task_id"], "task identity", r"[0-9a-f]{64}")\n    attempt_id = _attempt_identity(value["attempt_id"])\n    fence = _positive_fence(value["fence"])\n    job_id = _token(value["job_id"], "service job identity", r"[A-Za-z0-9_-]{1,128}")\n    if value["source_batch_id"] is not None:\n        _token(value["source_batch_id"], "source batch identity", r"[0-9a-f]{64}")\n    if not isinstance(value["status"], str) or value["status"] not in {\n        "SUCCEEDED",\n        "FAILED",\n        "CANCELLED",\n    }:\n        raise RuntimeError("Task result status must be explicitly terminal")\n    finished = _token(\n        value["finished_utc"],\n        "UTC completion instant",\n        r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,6})?Z",\n    )\n    try:\n        datetime.fromisoformat(finished)\n    except ValueError:\n        raise RuntimeError("Invalid UTC completion instant") from None\n    success = value["status"] == "SUCCEEDED"\n    if success:\n        if value["error_code"] is not None:\n            raise RuntimeError("A successful result cannot contain a failure code")\n    else:\n        _token(value["error_code"], "failure code", r"[A-Z][A-Z0-9_]{0,127}")\n    refs = value["artifact_refs"]\n    if not isinstance(refs, list) or len(refs) > 1000 or (success and not refs):\n        raise RuntimeError("Successful results require bounded artifact evidence")\n    kinds = set()\n    for ref in refs:\n        if not isinstance(ref, dict) or set(ref) != {\n            "kind",\n            "location",\n            "version",\n            "sha256",\n        }:\n            raise RuntimeError("Invalid artifact fields; credentials are prohibited")\n        if not isinstance(ref["kind"], str) or ref["kind"] not in {\n            "raw_batch",\n            "delta_version",\n            "source_commit",\n            "gold_release",\n        }:\n            raise RuntimeError("Unknown artifact evidence kind")\n        kinds.add(ref["kind"])\n        location = _token(\n            ref["location"], "artifact location", r"[A-Za-z0-9_][A-Za-z0-9_./-]{0,1999}"\n        )\n        if any(part in {"", ".", ".."} for part in location.split("/")):\n            raise RuntimeError("Artifact location cannot escape its declared scope")\n        _token(ref["sha256"], "artifact evidence hash", r"[0-9a-f]{64}")\n        if ref["kind"] == "raw_batch":\n            if not location.startswith("Files/bronze/") or ref["version"] is not None:\n                raise RuntimeError("Raw artifact must name a Bronze file batch")\n        elif (\n            re.fullmatch(\n                r"[A-Za-z_][A-Za-z0-9_]*(?:\\.[A-Za-z_][A-Za-z0-9_]*){2}", location\n            )\n            is None\n            or type(ref["version"]) is not int\n            or not 0 <= ref["version"] <= 2**63 - 1\n        ):\n            raise RuntimeError(\n                "Delta artifact requires a qualified table and exact version"\n            )\n    readiness = value["readiness"]\n    if (\n        not isinstance(readiness, list)\n        or any(not isinstance(item, str) for item in readiness)\n        or len(set(readiness)) != len(readiness)\n        or not set(readiness)\n        <= {"succeeded", "source_committed", "facts_ready", "gold_released"}\n        or (success and "succeeded" not in readiness)\n        or (not success and readiness)\n    ):\n        raise RuntimeError("Result readiness cannot turn a failure into success")\n    for condition, kind in (\n        ("source_committed", "source_commit"),\n        ("facts_ready", "delta_version"),\n        ("gold_released", "gold_release"),\n    ):\n        if condition in readiness and kind not in kinds:\n            raise RuntimeError(\n                "Publication readiness requires matching artifact evidence"\n            )\n    metrics = value["metrics"]\n    if (\n        not isinstance(metrics, dict)\n        or not set(metrics) <= {"rows_read", "rows_written", "rows_quarantined"}\n        or any(type(n) is not int or not 0 <= n <= 2**63 - 1 for n in metrics.values())\n    ):\n        raise RuntimeError("Only bounded row-count metrics are allowed")\n    return VerifiedTaskResult(\n        run_id, task_id, attempt_id, fence, job_id, value["status"], canonical\n    )\n\n\ndef make_task_result(\n    manifest: VerifiedRunManifest,\n    task_id: str,\n    *,\n    attempt_id: str,\n    fence: int,\n    job_id: str,\n    status: str,\n    finished_utc: str,\n    artifact_refs: Sequence[dict[str, object]] = (),\n    readiness: Sequence[str] | None = None,\n    error_code: str | None = None,\n    metrics: dict[str, int] | None = None,\n) -> VerifiedTaskResult:\n    tasks = cast(list[dict[str, object]], manifest.to_document()["tasks"])\n    matches = [task for task in tasks if task["task_id"] == task_id]\n    if len(matches) != 1:\n        raise RuntimeError("Result task identity is outside the pinned manifest")\n    task = matches[0]\n    payload = {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "task_id": task_id,\n        "source_batch_id": task["source_batch_id"],\n        "attempt_id": attempt_id,\n        "fence": fence,\n        "job_id": job_id,\n        "status": status,\n        "finished_utc": finished_utc,\n        "artifact_refs": list(artifact_refs),\n        "readiness": list(readiness)\n        if readiness is not None\n        else (["succeeded"] if status == "SUCCEEDED" else []),\n        "error_code": error_code,\n        "metrics": metrics if metrics is not None else {},\n    }\n    result = load_task_result(_canonical(payload))\n    for ref in cast(list[dict[str, object]], result.to_document()["artifact_refs"]):\n        if ref["kind"] in {"raw_batch", "delta_version"}:\n            location, target = str(ref["location"]), str(task["target_path"])\n            if location != target and not (\n                ref["kind"] == "raw_batch" and location.startswith(target + "/")\n            ):\n                raise RuntimeError(\n                    "Artifact evidence differs from the pinned task target"\n                )\n    return result\n\n\ndef _result_path(result: VerifiedTaskResult) -> str:\n    return _manifest_path(\n        f"Files/orchestration/{result.run_id}/results/"\n        f"{result.task_id}/{result.attempt_id}.json",\n        result.run_id,\n    )\n\n\ndef persist_task_result(\n    fs: FileSystem,\n    result: VerifiedTaskResult,\n    *,\n    ingest: Callable[[str, str], object] | None = None,\n) -> str:\n    """Persist first, then reconcile SQL; a failed ingestion leaves replayable evidence."""\n    result = load_task_result(result.result_json)\n    path = _result_path(result)\n    _write_immutable(fs, path, result.result_json, MAX_RESULT_BYTES)\n    if ingest is not None:\n        ingest(result.result_json, result.result_sha256)\n    return path\n\n\ndef read_task_result(\n    fs: FileSystem,\n    path: str,\n    *,\n    run_id: str,\n    task_id: str,\n    attempt_id: str,\n    expected_sha256: str,\n) -> VerifiedTaskResult:\n    text = _read_bounded(fs, _manifest_path(path, run_id), MAX_RESULT_BYTES)\n    result = load_task_result(text)\n    if (\n        (result.run_id, result.task_id, result.attempt_id)\n        != (run_id, task_id, attempt_id)\n        or result.result_sha256 != expected_sha256\n        or path != _result_path(result)\n    ):\n        raise RuntimeError("Task result identity, hash or immutable location differs")\n    return result\n\n\n@dataclass(frozen=True)\nclass RecordedAttempt:\n    run_id: str\n    task_id: str\n    attempt_id: str\n    fence: int\n    status: str\n    job_id: str | None\n\n    def __post_init__(self) -> None:\n        _token(self.run_id, "run identity", r"[A-Za-z0-9_-]{1,128}")\n        _token(self.task_id, "task identity", r"[0-9a-f]{64}")\n        _attempt_identity(self.attempt_id)\n        _positive_fence(self.fence)\n        if self.status not in {\n            "SUBMISSION_PENDING",\n            "SUBMITTED",\n            "UNCERTAIN",\n            "NOT_SUBMITTED",\n            "SUCCEEDED",\n            "FAILED",\n            "CANCELLED",\n        }:\n            raise RuntimeError("Unknown recorded attempt state")\n        if self.job_id is not None:\n            _token(self.job_id, "service job identity", r"[A-Za-z0-9_-]{1,128}")\n        elif self.status not in {"SUBMISSION_PENDING", "UNCERTAIN", "NOT_SUBMITTED"}:\n            raise RuntimeError(\n                "Recorded attempt state requires its service job identity"\n            )\n\n\n@dataclass(frozen=True)\nclass RecoveryDecision:\n    action: Literal["WAIT", "RECONCILE", "RETRY_ALLOWED", "MANUAL_RECONCILIATION"]\n    reason_code: str\n\n\ndef decide_recovery(\n    attempt: RecordedAttempt,\n    *,\n    observed_job_status: str | None = None,\n    result: VerifiedTaskResult | None = None,\n    outputs_verified: bool = False,\n    publication_checked: bool = False,\n) -> RecoveryDecision:\n    """Classify recovery only; never submit or infer that an expired writer stopped."""\n    if type(outputs_verified) is not bool or type(publication_checked) is not bool:\n        raise RuntimeError("Recovery requires explicit Boolean evidence flags")\n    if result is not None:\n        result = load_task_result(result.result_json)\n    if observed_job_status not in {\n        None,\n        "NotStarted",\n        "InProgress",\n        "Completed",\n        "Failed",\n        "Cancelled",\n    }:\n        raise RuntimeError("Unknown service job state requires explicit reconciliation")\n    if result is not None and (\n        result.run_id,\n        result.task_id,\n        result.attempt_id,\n        result.fence,\n        result.job_id,\n    ) != (\n        attempt.run_id,\n        attempt.task_id,\n        attempt.attempt_id,\n        attempt.fence,\n        attempt.job_id,\n    ):\n        raise RuntimeError(\n            "Recovered result identity or fence differs from its attempt"\n        )\n    if observed_job_status is not None and attempt.job_id is None:\n        raise RuntimeError("Observed service state requires the recorded job identity")\n    if observed_job_status in {"NotStarted", "InProgress"}:\n        return RecoveryDecision("WAIT", "SERVICE_JOB_ACTIVE")\n    if attempt.status == "NOT_SUBMITTED" and attempt.job_id is None:\n        return RecoveryDecision("RETRY_ALLOWED", "DEFINITE_SUBMISSION_REJECTION")\n    if attempt.job_id is None or observed_job_status is None:\n        return RecoveryDecision(\n            "MANUAL_RECONCILIATION", "SUBMISSION_OR_JOB_STATE_UNCERTAIN"\n        )\n    if result is not None:\n        if result.status == "SUCCEEDED":\n            if observed_job_status != "Completed":\n                raise RuntimeError("A failed service job cannot assert task success")\n            if not outputs_verified:\n                return RecoveryDecision(\n                    "MANUAL_RECONCILIATION", "OUTPUT_EVIDENCE_UNVERIFIED"\n                )\n        return RecoveryDecision("RECONCILE", "MATCHING_TERMINAL_RESULT")\n    if observed_job_status in {"Failed", "Cancelled"} and publication_checked:\n        return RecoveryDecision("RETRY_ALLOWED", "TERMINAL_JOB_AND_PUBLICATION_CHECKED")\n    return RecoveryDecision(\n        "MANUAL_RECONCILIATION", "RESULT_OR_PUBLICATION_EVIDENCE_MISSING"\n    )\n\n\n@dataclass(frozen=True)\nclass FileObservation:\n    relative_path: str\n    size_bytes: int\n    modified_utc: str\n\n\n@dataclass(frozen=True)\nclass SealedFileArrival:\n    files: tuple[FileObservation, ...]\n    observed_utc: str\n    policy_json: str\n    evidence_json: str\n\n    @property\n    def membership_sha256(self) -> str:\n        return hashlib.sha256(\n            _canonical(\n                [\n                    {\n                        "relative_path": file.relative_path,\n                        "size_bytes": file.size_bytes,\n                        "modified_utc": file.modified_utc,\n                    }\n                    for file in self.files\n                ]\n            ).encode("ascii")\n        ).hexdigest()\n\n\ndef _observation_time(value: object) -> datetime:\n    text = _token(\n        value,\n        "UTC file observation instant",\n        r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,6})?Z",\n    )\n    try:\n        return datetime.fromisoformat(text)\n    except ValueError:\n        raise RuntimeError("Invalid UTC file observation instant") from None\n\n\ndef _file_observations(values: object) -> tuple[FileObservation, ...]:\n    if not isinstance(values, (list, tuple)) or not 1 <= len(values) <= 5000:\n        raise RuntimeError(\n            "File membership must contain between 1 and 5000 explicit files"\n        )\n    files = []\n    for value in values:\n        if not isinstance(value, dict) or set(value) != {\n            "relative_path",\n            "size_bytes",\n            "modified_utc",\n        }:\n            raise RuntimeError("Invalid file observation fields")\n        path = _token(\n            value["relative_path"],\n            "relative source file",\n            r"[A-Za-z0-9_][A-Za-z0-9_./ -]{0,1999}",\n        )\n        if any(part in {"", ".", ".."} for part in path.split("/")):\n            raise RuntimeError("Source file membership cannot escape its root")\n        size = value["size_bytes"]\n        if type(size) is not int or not 0 <= size <= 2**63 - 1:\n            raise RuntimeError("File size must be an explicit nonnegative bigint")\n        modified = (\n            _observation_time(value["modified_utc"])\n            .isoformat(timespec="microseconds")\n            .replace("+00:00", "Z")\n        )\n        files.append(FileObservation(path, size, modified))\n    if len({file.relative_path for file in files}) != len(files):\n        raise RuntimeError("Duplicate source file membership")\n    return tuple(sorted(files, key=lambda file: file.relative_path))\n\n\ndef seal_file_arrival(\n    files: Sequence[dict[str, object]],\n    policy: dict[str, object],\n    *,\n    observed_utc: str,\n    previous_files: Sequence[dict[str, object]] | None = None,\n    previous_observed_utc: str | None = None,\n    present_markers: Sequence[str] = (),\n    allow_empty_files: bool = False,\n) -> SealedFileArrival:\n    """Seal reviewed remote observations; existence alone never establishes arrival."""\n    if type(allow_empty_files) is not bool:\n        raise RuntimeError("Empty-file acceptance must be explicitly Boolean")\n    current = _file_observations(files)\n    observed = _observation_time(observed_utc)\n    if any(_observation_time(file.modified_utc) > observed for file in current):\n        raise RuntimeError("Source modification time is later than its observation")\n    if not allow_empty_files and any(file.size_bytes == 0 for file in current):\n        raise RuntimeError("An empty file requires explicit acceptance")\n    if not isinstance(policy, dict):\n        raise TypeError("An explicit reviewed arrival policy is required")\n    if policy.get("kind") == "marker":\n        if set(policy) != {"kind", "marker"}:\n            raise RuntimeError("Unknown completion-marker policy field")\n        marker = _token(\n            policy["marker"], "completion marker", r"[A-Za-z0-9_][A-Za-z0-9_.-]{0,255}"\n        )\n        if not isinstance(present_markers, (list, tuple)) or any(\n            not isinstance(value, str) for value in present_markers\n        ):\n            raise RuntimeError("Completion markers require an exact observed name list")\n        if marker not in present_markers:\n            raise RuntimeError("The configured completion marker is missing")\n    elif policy.get("kind") == "stable":\n        if set(policy) != {"kind", "minimum_age_seconds", "stable_for_seconds"}:\n            raise RuntimeError("Unknown stable-arrival policy field")\n        age, interval = policy["minimum_age_seconds"], policy["stable_for_seconds"]\n        if any(type(n) is not int or not 1 <= n <= 604800 for n in (age, interval)):\n            raise RuntimeError(\n                "Stable arrival requires explicit bounded positive intervals"\n            )\n        if previous_files is None or previous_observed_utc is None:\n            raise RuntimeError("Stable arrival requires two explicit observations")\n        previous = _observation_time(previous_observed_utc)\n        if any(_observation_time(file.modified_utc) > previous for file in current):\n            raise RuntimeError(\n                "A previous observation cannot precede a file modification"\n            )\n        if (observed - previous).total_seconds() < interval or _file_observations(\n            previous_files\n        ) != current:\n            raise RuntimeError("Source file membership has not remained stable")\n        if any(\n            (observed - _observation_time(file.modified_utc)).total_seconds() < age\n            for file in current\n        ):\n            raise RuntimeError(\n                "A source file is too recent for the reviewed arrival policy"\n            )\n    else:\n        raise RuntimeError("Unsupported arrival policy; file existence is insufficient")\n    return SealedFileArrival(\n        current,\n        observed.isoformat(timespec="microseconds").replace("+00:00", "Z"),\n        _canonical(policy),\n        _canonical(\n            {\n                "previous_files": [\n                    {\n                        "relative_path": file.relative_path,\n                        "size_bytes": file.size_bytes,\n                        "modified_utc": file.modified_utc,\n                    }\n                    for file in _file_observations(previous_files)\n                ]\n                if policy["kind"] == "stable"\n                else None,\n                "previous_observed_utc": previous_observed_utc\n                if policy["kind"] == "stable"\n                else None,\n                "present_markers": list(present_markers)\n                if policy["kind"] == "marker"\n                else [],\n                "allow_empty_files": allow_empty_files,\n            }\n        ),\n    )\n\n\ndef _file_arrival_manifest(\n    manifest: VerifiedRunManifest, document: object\n) -> dict[str, SealedFileArrival]:\n    if not isinstance(document, dict) or set(document) != {\n        "format_version",\n        "run_id",\n        "version_id",\n        "snapshot_sha256",\n        "arrivals",\n    }:\n        raise RuntimeError("Invalid persisted file-arrival fields")\n    if (\n        type(document["format_version"]) is not int\n        or document["format_version"] != 1\n        or document["run_id"] != manifest.run_id\n        or document["version_id"] != manifest.version_id\n        or document["snapshot_sha256"] != manifest.snapshot_sha256\n    ):\n        raise RuntimeError("File arrivals differ from their pinned run")\n    payload = manifest.to_document()\n    configuration = cast(dict[str, Any], payload["configuration"])\n    feeds = {feed["id"]: feed for feed in configuration["feeds"]}\n    objects = {obj["id"]: obj for obj in configuration["objects"]}\n    tasks = {\n        task["object_id"]: task\n        for task in cast(list[dict[str, Any]], payload["tasks"])\n        if task["layer"] == "bronze"\n        and feeds[task["feed_id"]]["source_type"] in ("ftp", "sftp")\n    }\n    rows = document["arrivals"]\n    if not isinstance(rows, list) or len(rows) != len(tasks):\n        raise RuntimeError("File arrivals omit selected object membership")\n    result = {}\n    for row in rows:\n        if not isinstance(row, dict) or set(row) != {\n            "object_id",\n            "source_batch_id",\n            "observed_utc",\n            "policy",\n            "files",\n            "evidence",\n        }:\n            raise RuntimeError("Invalid persisted file-arrival object")\n        identity = row["object_id"]\n        if not isinstance(identity, str) or identity not in tasks or identity in result:\n            raise RuntimeError("Duplicate or unselected file-arrival object")\n        obj, task = objects[identity], tasks[identity]\n        if (\n            row["source_batch_id"] != task["source_batch_id"]\n            or _canonical(row["policy"])\n            != _canonical(obj["contract"].get("arrival_policy"))\n            or _observation_time(row["observed_utc"])\n            < _observation_time(payload["observed_utc"])\n        ):\n            raise RuntimeError(\n                "File arrival policy, batch or observation differs from its pinned source"\n            )\n        evidence = row["evidence"]\n        if not isinstance(evidence, dict) or set(evidence) != {\n            "previous_files",\n            "previous_observed_utc",\n            "present_markers",\n            "allow_empty_files",\n        }:\n            raise RuntimeError(\n                "File arrival lacks its retained marker or stability evidence"\n            )\n        sealed = seal_file_arrival(\n            row["files"], row["policy"], observed_utc=row["observed_utc"], **evidence\n        )\n        pattern = obj["contract"].get("file_pattern")\n        recursive = obj["contract"].get("recursive")\n        if (\n            not isinstance(pattern, str)\n            or type(recursive) is not bool\n            or any(\n                not fnmatch.fnmatchcase(PurePosixPath(file.relative_path).name, pattern)\n                or (not recursive and "/" in file.relative_path)\n                or file.relative_path == row["policy"].get("marker")\n                for file in sealed.files\n            )\n        ):\n            raise RuntimeError(\n                "Observed file membership differs from its pinned selection pattern"\n            )\n        result[identity] = sealed\n    return result\n\n\ndef write_file_arrival_manifest(\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    arrivals: Mapping[str, SealedFileArrival],\n) -> dict[str, str]:\n    tasks = {\n        task["object_id"]: task\n        for task in cast(list[dict[str, Any]], manifest.to_document()["tasks"])\n        if task["layer"] == "bronze"\n    }\n    rows = []\n    for identity, arrival in sorted(arrivals.items()):\n        if identity not in tasks or not isinstance(arrival, SealedFileArrival):\n            raise RuntimeError("File arrival belongs to an unselected object")\n        rows.append(\n            {\n                "object_id": identity,\n                "source_batch_id": tasks[identity]["source_batch_id"],\n                "observed_utc": arrival.observed_utc,\n                "policy": json.loads(arrival.policy_json),\n                "evidence": json.loads(arrival.evidence_json),\n                "files": [\n                    {\n                        "relative_path": file.relative_path,\n                        "size_bytes": file.size_bytes,\n                        "modified_utc": file.modified_utc,\n                    }\n                    for file in arrival.files\n                ],\n            }\n        )\n    document = {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "version_id": manifest.version_id,\n        "snapshot_sha256": manifest.snapshot_sha256,\n        "arrivals": rows,\n    }\n    _file_arrival_manifest(manifest, document)\n    text = _canonical(document)\n    digest = hashlib.sha256(text.encode("ascii")).hexdigest()\n    path = _manifest_path(\n        f"Files/orchestration/{manifest.run_id}/file_arrivals.json", manifest.run_id\n    )\n    _write_immutable(fs, path, text, MAX_MANIFEST_BYTES)\n    return {"path": path, "sha256": digest}\n\n\ndef read_file_arrival_manifest(\n    fs: FileSystem, path: str, manifest: VerifiedRunManifest, *, expected_sha256: str\n) -> dict[str, SealedFileArrival]:\n    text = _read_bounded(fs, _manifest_path(path, manifest.run_id), MAX_MANIFEST_BYTES)\n    if hashlib.sha256(text.encode("ascii")).hexdigest() != expected_sha256:\n        raise RuntimeError("File-arrival manifest differs from its immutable hash")\n    try:\n        document = json.loads(text)\n    except ValueError:\n        raise RuntimeError("File-arrival manifest is not finite JSON") from None\n    return _file_arrival_manifest(manifest, document)\n\n\ndef verify_file_membership(\n    sealed: SealedFileArrival, files: Sequence[dict[str, object]]\n) -> None:\n    """Reject changed source membership when reobserved around a transport operation."""\n    if _file_observations(files) != sealed.files:\n        raise RuntimeError("Sealed source file membership changed during landing")\n\n\n@dataclass(frozen=True)\nclass VerifiedBronzeReceipt:\n    run_id: str\n    feed_id: str\n    receipt_json: str\n\n    @property\n    def receipt_sha256(self) -> str:\n        return hashlib.sha256(self.receipt_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.receipt_json))\n\n\ndef _bronze_path(value: object, prefix: str) -> str:\n    path = _token(\n        value, "Bronze membership path", r"Files/bronze/[A-Za-z0-9_./ -]{1,1950}"\n    )\n    if not path.startswith(prefix + "/") or any(\n        part in {"", ".", ".."} for part in path.split("/")\n    ):\n        raise RuntimeError("Bronze file membership escapes its pinned batch scope")\n    return path\n\n\ndef _file_size(fs: FileSystem, path: str) -> int:\n    if not fs.exists(path):\n        raise FileNotFoundError("A selected Bronze file is missing")\n    location = PurePosixPath(path)\n    entries = [\n        entry for entry in fs.ls(str(location.parent)) if entry.name == location.name\n    ]\n    if len(entries) != 1 or getattr(entries[0], "isDir", False):\n        raise RuntimeError("Bronze file membership is missing or ambiguous")\n    size = entries[0].size\n    if type(size) is not int or not 0 <= size <= 2**63 - 1:\n        raise RuntimeError("Bronze file size is not a valid nonnegative bigint")\n    return size\n\n\ndef _verify_landed_membership(\n    fs: FileSystem, target: str, expected: Mapping[str, int]\n) -> None:\n    if not fs.exists(target):\n        raise FileNotFoundError("The pinned Bronze object folder is missing")\n    pending = [target]\n    seen = set()\n    actual = {}\n    while pending:\n        directory = pending.pop()\n        entries = fs.ls(directory)\n        if len(entries) + len(seen) > 10000:\n            raise RuntimeError("Landed Bronze membership exceeds its bounded inventory")\n        for entry in entries:\n            name = _token(\n                entry.name, "landed file name", r"[A-Za-z0-9_][A-Za-z0-9_. -]{0,1999}"\n            )\n            path = _bronze_path(directory + "/" + name, target)\n            if path in seen:\n                raise RuntimeError("Duplicate landed Bronze membership")\n            seen.add(path)\n            is_directory = getattr(entry, "isDir", False)\n            if type(is_directory) is not bool:\n                raise RuntimeError("Invalid landed Bronze directory metadata")\n            if is_directory:\n                pending.append(path)\n            else:\n                size = entry.size\n                if type(size) is not int or not 0 <= size <= 2**63 - 1:\n                    raise RuntimeError("Invalid landed Bronze file size")\n                actual[path] = size\n    if actual != dict(expected):\n        raise RuntimeError("Landed Bronze membership differs from its sealed selection")\n\n\ndef _verify_context_file_membership(fs: FileSystem, context: Mapping[str, Any]) -> None:\n    if context["source_type"] == "sql":\n        return\n    for name, target in context["raw_targets"].items():\n        _verify_landed_membership(\n            fs,\n            target,\n            dict(\n                zip(\n                    context["raw_source_files"][name],\n                    context["raw_source_sizes"][name],\n                    strict=True,\n                )\n            ),\n        )\n\n\ndef _feed_membership(manifest: VerifiedRunManifest, feed_id: str):\n    document = manifest.to_document()\n    configuration = cast(dict[str, object], document["configuration"])\n    feeds = cast(list[dict[str, object]], configuration["feeds"])\n    matches = [feed for feed in feeds if feed["id"] == feed_id]\n    if len(matches) != 1:\n        raise RuntimeError("Bronze receipt feed is missing or ambiguous")\n    tasks = [\n        task\n        for task in cast(list[dict[str, object]], document["tasks"])\n        if task["feed_id"] == feed_id and task["layer"] == "bronze"\n    ]\n    if not tasks or len({task["source_batch_id"] for task in tasks}) != 1:\n        raise RuntimeError("Bronze receipt requires one complete pinned source batch")\n    objects = {\n        obj["id"]: obj\n        for obj in cast(list[dict[str, object]], configuration["objects"])\n    }\n    return document, matches[0], tasks, objects\n\n\ndef _sealed_bronze_table(\n    fs: FileSystem,\n    feed: dict[str, Any],\n    task: dict[str, Any],\n    obj: dict[str, Any],\n    file_arrivals: Mapping[str, SealedFileArrival] | None,\n) -> dict[str, Any]:\n    target = task["target_path"]\n    selection_hash = None\n    if feed["source_type"] == "sql":\n        files = [{"path": target, "size_bytes": _file_size(fs, target)}]\n        if files[0]["size_bytes"] == 0:\n            raise RuntimeError("A SQL CSV extraction must contain its header")\n    else:\n        arrival = file_arrivals.get(obj["id"]) if file_arrivals is not None else None\n        if not isinstance(arrival, SealedFileArrival):\n            raise RuntimeError("File feeds require a sealed reviewed arrival selection")\n        policy = obj["contract"].get("arrival_policy")\n        if not isinstance(policy, dict) or arrival.policy_json != _canonical(policy):\n            raise RuntimeError(\n                "File arrival policy differs from its pinned object metadata"\n            )\n        selection_hash = arrival.membership_sha256\n        files = [\n            {\n                "path": _bronze_path(target + "/" + member.relative_path, target),\n                "size_bytes": member.size_bytes,\n            }\n            for member in arrival.files\n        ]\n        _verify_landed_membership(\n            fs, target, {file["path"]: file["size_bytes"] for file in files}\n        )\n    return {\n        "object_id": obj["id"],\n        "table_name": obj["name"],\n        "target_path": target,\n        "files": files,\n        "source_selection_sha256": selection_hash,\n    }\n\n\ndef seal_bronze_receipt(\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    feed_id: str,\n    *,\n    file_arrivals: Mapping[str, SealedFileArrival] | None = None,\n) -> VerifiedBronzeReceipt:\n    """Persistable file membership; Spark inventory seals the business record multiset."""\n    document, feed, tasks, objects = _feed_membership(manifest, feed_id)\n    tables = []\n    for task in tasks:\n        obj = objects[task["object_id"]]\n        tables.append(_sealed_bronze_table(fs, feed, task, obj, file_arrivals))\n    receipt = {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "version_id": manifest.version_id,\n        "definition_sha256": document["definition_sha256"],\n        "feed_id": feed_id,\n        "source_batch_id": tasks[0]["source_batch_id"],\n        "observed_utc": document["observed_utc"],\n        "tables": [\n            {key: value for key, value in table.items() if key != "runtime_layout"}\n            for table in tables\n        ],\n    }\n    result = VerifiedBronzeReceipt(manifest.run_id, feed_id, _canonical(receipt))\n    _validate_bronze_receipt(result, manifest, feed_id)\n    return result\n\n\ndef _validate_bronze_receipt(\n    receipt: VerifiedBronzeReceipt, manifest: VerifiedRunManifest, feed_id: str\n) -> None:\n    document, feed, tasks, objects = _feed_membership(manifest, feed_id)\n    value = receipt.to_document()\n    expected = {\n        "format_version",\n        "run_id",\n        "version_id",\n        "definition_sha256",\n        "feed_id",\n        "source_batch_id",\n        "observed_utc",\n        "tables",\n    }\n    if (\n        set(value) != expected\n        or type(value["format_version"]) is not int\n        or value["format_version"] != 1\n    ):\n        raise RuntimeError("Unsupported Bronze receipt contract")\n    for name, expected_value in (\n        ("run_id", manifest.run_id),\n        ("version_id", manifest.version_id),\n        ("definition_sha256", document["definition_sha256"]),\n        ("feed_id", feed_id),\n        ("source_batch_id", tasks[0]["source_batch_id"]),\n        ("observed_utc", document["observed_utc"]),\n    ):\n        if value[name] != expected_value:\n            raise RuntimeError(\n                "Bronze receipt identity differs from its pinned manifest"\n            )\n    tables = value["tables"]\n    if not isinstance(tables, list) or len(tables) != len(tasks):\n        raise RuntimeError("Bronze receipt omits selected task membership")\n    by_id = {task["object_id"]: task for task in tasks}\n    seen = set()\n    all_files = set()\n    for table in tables:\n        if not isinstance(table, dict) or set(table) != {\n            "object_id",\n            "table_name",\n            "target_path",\n            "files",\n            "source_selection_sha256",\n        }:\n            raise RuntimeError("Invalid Bronze receipt table fields")\n        identity = table["object_id"]\n        if not isinstance(identity, str) or identity not in by_id or identity in seen:\n            raise RuntimeError("Duplicate or unselected Bronze receipt object")\n        seen.add(identity)\n        task = by_id[identity]\n        if (\n            table["target_path"] != task["target_path"]\n            or table["table_name"] != objects[identity]["name"]\n        ):\n            raise RuntimeError("Bronze receipt target differs from the pinned object")\n        files = table["files"]\n        if not isinstance(files, list) or not 1 <= len(files) <= 5000:\n            raise RuntimeError(\n                "Bronze receipt requires bounded explicit file membership"\n            )\n        target = cast(str, task["target_path"])\n        if feed["source_type"] == "sql" and (\n            len(files) != 1 or table["source_selection_sha256"] is not None\n        ):\n            raise RuntimeError(\n                "SQL extraction membership must contain one contracted CSV"\n            )\n        if feed["source_type"] != "sql":\n            _token(\n                table["source_selection_sha256"], "sealed arrival hash", r"[0-9a-f]{64}"\n            )\n        for file in files:\n            if not isinstance(file, dict) or set(file) != {"path", "size_bytes"}:\n                raise RuntimeError("Invalid Bronze file-membership fields")\n            path = file["path"]\n            if feed["source_type"] == "sql":\n                if path != target:\n                    raise RuntimeError(\n                        "SQL receipt file differs from its pinned target"\n                    )\n            else:\n                _bronze_path(path, target)\n            size = file["size_bytes"]\n            if (\n                type(size) is not int\n                or not 0 <= size <= 2**63 - 1\n                or (feed["source_type"] == "sql" and size == 0)\n            ):\n                raise RuntimeError("Invalid Bronze receipt file size")\n            if path in all_files:\n                raise RuntimeError("A Bronze file cannot belong to multiple objects")\n            all_files.add(path)\n\n\ndef write_bronze_receipt(fs: FileSystem, receipt: VerifiedBronzeReceipt) -> str:\n    feed_key = hashlib.sha256(receipt.feed_id.encode("utf-8")).hexdigest()\n    path = _manifest_path(\n        f"Files/orchestration/{receipt.run_id}/bronze/{feed_key}.json", receipt.run_id\n    )\n    return _write_immutable(fs, path, receipt.receipt_json, MAX_MANIFEST_BYTES)\n\n\ndef read_bronze_receipt(\n    fs: FileSystem,\n    path: str,\n    manifest: VerifiedRunManifest,\n    *,\n    expected_sha256: str,\n    feed_id: str,\n) -> VerifiedBronzeReceipt:\n    text = _read_bounded(fs, _manifest_path(path, manifest.run_id), MAX_MANIFEST_BYTES)\n    try:\n        value = json.loads(text)\n        canonical = _canonical(value)\n    except (ValueError, TypeError):\n        raise RuntimeError("Bronze receipt is not finite JSON") from None\n    if hashlib.sha256(canonical.encode("ascii")).hexdigest() != expected_sha256:\n        raise RuntimeError("Bronze receipt hash differs from its immutable reference")\n    result = VerifiedBronzeReceipt(manifest.run_id, feed_id, canonical)\n    _validate_bronze_receipt(result, manifest, feed_id)\n    return result\n\n\ndef source_runtime_config(\n    manifest: VerifiedRunManifest,\n    receipt: VerifiedBronzeReceipt,\n    *,\n    load_configuration: Callable[[object], PinnedConfiguration],\n) -> dict[str, object]:\n    """Reconstruct source contracts only from pinned metadata and sealed file membership."""\n    _validate_bronze_receipt(receipt, manifest, receipt.feed_id)\n    document, feed, _tasks, objects = _feed_membership(manifest, receipt.feed_id)\n    configuration = cast(dict[str, object], document["configuration"])\n    verified = load_configuration(configuration)\n    if (\n        verified.definition_sha256 != document["definition_sha256"]\n        or verified.version_id != manifest.version_id\n    ):\n        raise RuntimeError(\n            "Source runtime metadata does not match its pinned definition"\n        )\n    selected = {\n        task["object_id"] for task in cast(list[dict[str, object]], document["tasks"])\n    }\n    silver = [\n        obj\n        for obj in objects.values()\n        if obj["feed_id"] == receipt.feed_id\n        and obj["layer"] == "silver"\n        and obj["enabled"]\n    ]\n    if not silver:\n        raise RuntimeError("Source runtime requires an explicit Silver object contract")\n    lakehouses = {obj["lakehouse"] for obj in silver}\n    if len(lakehouses) != 1 or any(\n        obj["target_schema"] != feed["silver_schema"] for obj in silver\n    ):\n        raise RuntimeError(\n            "Source Silver objects disagree on Lakehouse or schema ownership"\n        )\n    policies = cast(\n        dict[str, object],\n        cast(dict[str, object], configuration["extensions"])["hash_policies"],\n    )\n    tables = []\n    selected_tables = []\n    for obj in sorted(\n        silver,\n        key=lambda item: (cast(int, item["execution_priority"]), str(item["id"])),\n    ):\n        contract = json.loads(_canonical(obj["contract"]))\n        contract["name"] = obj["name"]\n        contract["keys"] = obj["business_keys"]\n        contract["columns"] = obj["columns"]\n        contract["foreign_keys"] = contract.get("foreign_keys", [])\n        contract["allow_empty"] = contract.get("allow_empty", False)\n        contract["change_hash_exclusions"] = contract.get("change_hash_exclusions", [])\n        if obj["id"] not in policies:\n            raise RuntimeError("Source runtime requires an explicit stable hash policy")\n        contract["hash_policy"] = policies[obj["id"]]\n        contract["rule_specs"] = [\n            {name: value for name, value in rule.items() if name != "object_id"}\n            for rule in cast(list[dict[str, object]], configuration["rules"])\n            if rule["object_id"] == obj["id"]\n            and rule["handler"] != "legacy.source_business_checks"\n        ]\n        contract["runtime_layout"] = {\n            "layout": obj.get("layout"),\n            "capabilities": cast(dict[str, object], configuration["extensions"]).get(\n                "layout_runtime"\n            ),\n        }\n        tables.append(contract)\n        if obj["id"] in selected:\n            selected_tables.append(obj["name"])\n    if not selected_tables:\n        raise RuntimeError("No Silver work was selected for this source runtime")\n    membership = receipt.to_document()\n    files = {}\n    sizes = {}\n    targets = {}\n    for table in cast(list[dict[str, object]], membership["tables"]):\n        name = cast(str, table["table_name"])\n        files[name] = [\n            file["path"] for file in cast(list[dict[str, object]], table["files"])\n        ]\n        sizes[name] = [\n            file["size_bytes"] for file in cast(list[dict[str, object]], table["files"])\n        ]\n        targets[name] = table["target_path"]\n    source_type = str(feed["source_type"])\n    bronze_root = (\n        f"Files/bronze/{feed[\'source_code\']}_{feed[\'description_slug\']}_{source_type}"\n    )\n    business_contract_hash = feed.get("legacy_contract_hash")\n    if business_contract_hash is None:\n        business_contract_hash = hashlib.sha256(\n            _canonical(\n                {\n                    "source_code": feed["source_code"],\n                    "silver_schema": feed["silver_schema"],\n                    "tables": tables,\n                }\n            ).encode("ascii")\n        ).hexdigest()\n    return {\n        "source_code": feed["source_code"],\n        "lakehouse": next(iter(lakehouses)),\n        "silver_schema": feed["silver_schema"],\n        "quarantine_schema": feed["quarantine_schema"],\n        "bronze_root": bronze_root,\n        "contract_hash": business_contract_hash,\n        "seed_snapshot_date": str(document["observed_utc"])[:10],\n        "tables": tables,\n        "metadata_version": manifest.version_id,\n        "metadata_run_id": manifest.run_id,\n        "source_feed_id": receipt.feed_id,\n        "source_batch_id": membership["source_batch_id"],\n        "source_type": source_type,\n        "description_slug": feed["description_slug"],\n        "observed_utc": document["observed_utc"],\n        "processing_date": str(document["observed_utc"])[:10],\n        "pipeline_run_id": membership["source_batch_id"],\n        "raw_source_files": files,\n        "raw_source_sizes": sizes,\n        "raw_targets": targets,\n        "source_selection_complete": len(selected_tables) == len(silver),\n        "required_source_tables": sorted(str(obj["name"]) for obj in silver),\n        "selected_tables": sorted(cast(list[str], selected_tables)),\n        "bronze_receipt_sha256": receipt.receipt_sha256,\n        "source_rule_specs": [\n            {name: value for name, value in rule.items() if name != "object_id"}\n            for rule in cast(list[dict[str, object]], configuration["rules"])\n            if rule["object_id"] in {obj["id"] for obj in silver}\n            and rule["handler"] == "legacy.source_business_checks"\n        ],\n    }\n\n\n@dataclass(frozen=True)\nclass VerifiedPipelineDispatch:\n    run_id: str\n    layer: str\n    dispatch_json: str\n\n    @property\n    def dispatch_sha256(self) -> str:\n        return hashlib.sha256(self.dispatch_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.dispatch_json))\n\n\ndef _pipeline_activity_scope(location: object) -> tuple[str, str]:\n    value = _token(\n        location,\n        "pipeline activity scope",\n        r"https://api\\.fabric\\.microsoft\\.com/v1/workspaces/[0-9a-f-]{36}/"\n        r"datapipelines/pipelineruns/[0-9a-f-]{36}/queryactivityruns",\n    )\n    parts = value.split("/")\n    workspace, execution = parts[5], parts[8]\n    if str(UUID(workspace)) != workspace or str(UUID(execution)) != execution:\n        raise RuntimeError("Pipeline activity scope requires canonical resolved UUIDs")\n    return workspace, execution\n\n\ndef make_pipeline_dispatch(\n    manifest: VerifiedRunManifest, document: object\n) -> VerifiedPipelineDispatch:\n    fields = {\n        "format_version",\n        "run_id",\n        "version_id",\n        "snapshot_sha256",\n        "layer",\n        "attempts",\n    }\n    if not isinstance(document, dict) or not document:\n        raise RuntimeError("Invalid authoritative pipeline dispatch fields")\n    partial = document.get("format_version") == 2\n    if partial or (\n        document.get("format_version") == 1\n        and document.get("layer") in ("silver", "gold")\n        and "undispatched_task_ids" in document\n        and document["undispatched_task_ids"] is None\n    ):\n        fields.add("undispatched_task_ids")\n    if set(document) != fields:\n        raise RuntimeError("Invalid authoritative pipeline dispatch fields")\n    if (\n        type(document["format_version"]) is not int\n        or document["format_version"] not in (1, 2)\n        or (partial and document["layer"] != "bronze")\n        or document["run_id"] != manifest.run_id\n        or document["version_id"] != manifest.version_id\n        or document["snapshot_sha256"] != manifest.snapshot_sha256\n        or document["layer"] not in {"bronze", "silver", "gold"}\n    ):\n        raise RuntimeError("Pipeline dispatch differs from its pinned run")\n    tasks = {\n        task["task_id"]: task\n        for task in cast(list[dict[str, object]], manifest.to_document()["tasks"])\n        if task["layer"] == document["layer"]\n    }\n    attempts = document["attempts"]\n    missing = document["undispatched_task_ids"] if partial else []\n    if (\n        not tasks\n        or not isinstance(attempts, list)\n        or not isinstance(missing, list)\n        or any(not isinstance(identity, str) for identity in missing)\n        or len(set(missing)) != len(missing)\n        or not set(missing) <= tasks.keys()\n        or len(attempts) + len(missing) != len(tasks)\n    ):\n        raise RuntimeError("Pipeline dispatch omits selected layer membership")\n    identities, seen = set(), set()\n    for attempt in attempts:\n        if not isinstance(attempt, dict) or set(attempt) != {\n            "task_id",\n            "attempt_id",\n            "fence",\n            "source_fence",\n            "status",\n            "job_location",\n        }:\n            raise RuntimeError("Invalid pipeline prepared-attempt fields")\n        task_id = _token(attempt["task_id"], "pipeline task identity", r"[0-9a-f]{64}")\n        identity = _attempt_identity(attempt["attempt_id"])\n        if (\n            task_id not in tasks\n            or task_id in missing\n            or task_id in seen\n            or identity in identities\n        ):\n            raise RuntimeError("Duplicate or unselected pipeline attempt")\n        _positive_fence(attempt["fence"])\n        if tasks[task_id]["source_lock_key"] is not None:\n            _positive_fence(attempt["source_fence"])\n        elif attempt["source_fence"] is not None:\n            raise RuntimeError("A feed-independent task cannot invent a source fence")\n        if attempt["status"] not in {\n            "SUBMISSION_PENDING",\n            "SUBMITTED",\n            "UNCERTAIN",\n            "SUCCEEDED",\n            "FAILED",\n            "CANCELLED",\n        }:\n            raise RuntimeError("Invalid pipeline prepared-attempt state")\n        _pipeline_activity_scope(attempt["job_location"])\n        seen.add(task_id)\n        identities.add(identity)\n    return VerifiedPipelineDispatch(\n        manifest.run_id, document["layer"], _canonical(document)\n    )\n\n\ndef read_pipeline_dispatch(\n    fs: FileSystem,\n    path: str,\n    manifest: VerifiedRunManifest,\n    *,\n    expected_sha256: str,\n    layer: str,\n) -> VerifiedPipelineDispatch:\n    text = _read_bounded(fs, _manifest_path(path, manifest.run_id), MAX_MANIFEST_BYTES)\n    if path.endswith(".csv"):\n        reader = csv.DictReader(io.StringIO(text), escapechar="\\\\")\n        if reader.fieldnames != ["dispatch_json"]:\n            raise RuntimeError("Pipeline dispatch export columns differ")\n        rows = list(reader)\n        if len(rows) != 1 or None in rows[0] or rows[0]["dispatch_json"] is None:\n            raise RuntimeError("Pipeline dispatch export must contain one complete row")\n        text = rows[0]["dispatch_json"]\n    try:\n        document = json.loads(text)\n    except ValueError:\n        raise RuntimeError("Pipeline dispatch is not finite JSON") from None\n    dispatch = make_pipeline_dispatch(manifest, document)\n    if dispatch.layer != layer or dispatch.dispatch_sha256 != expected_sha256:\n        raise RuntimeError(\n            "Pipeline dispatch hash or layer differs from SQL preparation"\n        )\n    return dispatch\n\n\ndef pipeline_activity_evidence(\n    client: ActivityClient,\n    manifest: VerifiedRunManifest,\n    dispatch: VerifiedPipelineDispatch,\n    *,\n    observed_utc: str,\n    parent_activity: str | None = None,\n    allow_unresolved: bool = False,\n) -> dict[str, dict[str, str]]:\n    if type(allow_unresolved) is not bool or (\n        allow_unresolved and dispatch.layer != "bronze"\n    ):\n        raise RuntimeError(\n            "Only Bronze recovery may retain unresolved activity evidence"\n        )\n    checked = make_pipeline_dispatch(manifest, dispatch.to_document())\n    if (\n        checked.dispatch_sha256 != dispatch.dispatch_sha256\n        or (dispatch.layer == "bronze" and parent_activity is not None)\n        or (\n            dispatch.layer != "bronze"\n            and parent_activity != f"orchestrate_{dispatch.layer}"\n        )\n    ):\n        raise RuntimeError(\n            "Activity reconciliation requires its authoritative dispatch and operation"\n        )\n    now = _observation_time(observed_utc)\n    observed = _observation_time(manifest.to_document()["observed_utc"])\n    if now < observed:\n        raise RuntimeError("Activity observation cannot precede its pinned run")\n    attempts = cast(list[dict[str, Any]], dispatch.to_document()["attempts"])\n    by_scope: dict[str, list[dict[str, Any]]] = {}\n    for attempt in attempts:\n        by_scope.setdefault(attempt["job_location"], []).append(attempt)\n    result = {}\n    for location, selected in by_scope.items():\n        _workspace, execution = _pipeline_activity_scope(location)\n        response = client.request(\n            "POST",\n            location,\n            body={\n                "filters": [],\n                "orderBy": [{"orderBy": "ActivityRunStart", "order": "ASC"}],\n                "lastUpdatedAfter": observed.isoformat(timespec="microseconds").replace(\n                    "+00:00", "Z"\n                ),\n                "lastUpdatedBefore": now.isoformat(timespec="microseconds").replace(\n                    "+00:00", "Z"\n                ),\n            },\n            expected=(200,),\n        )\n        if (\n            not isinstance(response.content, bytes)\n            or not 1 <= len(response.content) <= MAX_MANIFEST_BYTES\n        ):\n            raise RuntimeError(\n                "Pipeline activity response exceeds its bounded contract"\n            )\n        try:\n            activities = json.loads(response.content.decode("utf-8"))\n        except (ValueError, UnicodeError):\n            raise RuntimeError("Pipeline activity response is not valid JSON") from None\n        if not isinstance(activities, list) or len(activities) > 50000:\n            raise RuntimeError("Pipeline activity membership is invalid or truncated")\n        for attempt in selected:\n            matches = [\n                activity\n                for activity in activities\n                if isinstance(activity, dict)\n                and activity.get("pipelineRunId") == execution\n                and activity.get("activityType")\n                == ("Copy" if dispatch.layer == "bronze" else "TridentNotebook")\n                and isinstance(activity.get("userProperties"), dict)\n                and activity["userProperties"].get("run_id") == manifest.run_id\n                and (\n                    (\n                        dispatch.layer == "bronze"\n                        and activity["userProperties"].get("task_id")\n                        == attempt["task_id"]\n                        and activity["userProperties"].get("attempt_id")\n                        == attempt["attempt_id"]\n                    )\n                    or (\n                        dispatch.layer != "bronze"\n                        and activity.get("activityName") == parent_activity\n                        and activity["userProperties"].get("layer") == dispatch.layer\n                        and activity["userProperties"].get("dispatch_sha256")\n                        == dispatch.dispatch_sha256\n                    )\n                )\n            ]\n            if len(matches) != 1:\n                if allow_unresolved:\n                    result[attempt["task_id"]] = {\n                        "observed_job_status": "Unresolved",\n                        "error_code": "COPY_ACTIVITY_NOT_VISIBLE"\n                        if not matches\n                        else "COPY_ACTIVITY_AMBIGUOUS",\n                    }\n                    continue\n                raise RuntimeError(\n                    "Copy activity identity is missing or ambiguous; reconcile without resubmission"\n                )\n            activity = matches[0]\n            if activity.get("retryAttempt") not in (None, 0):\n                raise RuntimeError(\n                    "A retried Copy cannot certify an immutable extraction"\n                )\n            states = {\n                "Succeeded": "Completed",\n                "Failed": "Failed",\n                "Cancelled": "Cancelled",\n            }\n            if activity.get("status") not in states:\n                if allow_unresolved:\n                    result[attempt["task_id"]] = {\n                        "observed_job_status": "Unresolved",\n                        "error_code": "COPY_ACTIVITY_NOT_TERMINAL",\n                    }\n                    continue\n                raise RuntimeError(\n                    "Copy activity is active or uncertain; reconciliation must wait"\n                )\n            identity = _attempt_identity(activity.get("activityRunId"))\n            end = _token(\n                activity.get("activityRunEnd"),\n                "terminal activity instant",\n                r"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\d(?:\\.\\d{1,7})?Z",\n            )\n            try:\n                finished = datetime.fromisoformat(end)\n            except ValueError:\n                raise RuntimeError("Invalid terminal activity instant") from None\n            if finished < observed or finished > now:\n                raise RuntimeError(\n                    "Terminal activity time differs from its observation"\n                )\n            proof = {\n                "job_id": identity,\n                "job_location": location,\n                "observed_job_status": states[activity["status"]],\n                "finished_utc": finished.isoformat(timespec="microseconds").replace(\n                    "+00:00", "Z"\n                ),\n                "run_id": manifest.run_id,\n                "task_id": attempt["task_id"],\n                "attempt_id": attempt["attempt_id"],\n            }\n            result[attempt["task_id"]] = {\n                key: proof[key]\n                for key in (\n                    "job_id",\n                    "job_location",\n                    "observed_job_status",\n                    "finished_utc",\n                )\n            } | {\n                "job_evidence_sha256": hashlib.sha256(\n                    _canonical(proof).encode("ascii")\n                ).hexdigest()\n            }\n    return result\n\n\ndef write_pipeline_callbacks(\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    results: Sequence[tuple[VerifiedTaskResult, Mapping[str, str]]],\n    *,\n    max_bytes: int = 4 * 1024 * 1024 - 1024,\n) -> list[dict[str, object]]:\n    if type(max_bytes) is not int or not 1 <= max_bytes < 4 * 1024 * 1024:\n        raise RuntimeError("Invalid bounded SQL callback page size")\n    columns = [\n        "result_json",\n        "result_sha256",\n        "observed_job_status",\n        "job_evidence_sha256",\n        "job_location",\n    ]\n    rows = []\n    seen = set()\n    task_ids = {\n        task["task_id"]\n        for task in cast(list[dict[str, object]], manifest.to_document()["tasks"])\n    }\n    for result, evidence in results:\n        result = load_task_result(result.result_json)\n        if (\n            result.run_id != manifest.run_id\n            or result.task_id not in task_ids\n            or (result.task_id, result.attempt_id) in seen\n        ):\n            raise RuntimeError("Duplicate or out-of-run SQL callback")\n        seen.add((result.task_id, result.attempt_id))\n        read_task_result(\n            fs,\n            _result_path(result),\n            run_id=result.run_id,\n            task_id=result.task_id,\n            attempt_id=result.attempt_id,\n            expected_sha256=result.result_sha256,\n        )\n        _pipeline_activity_scope(evidence["job_location"])\n        if evidence["job_id"] != result.job_id or evidence[\n            "observed_job_status"\n        ] not in {"Completed", "Failed", "Cancelled"}:\n            raise RuntimeError("SQL callback differs from its service evidence")\n        if (\n            result.status == "SUCCEEDED"\n            and evidence["observed_job_status"] != "Completed"\n        ):\n            raise RuntimeError(\n                "A successful callback requires successful service evidence"\n            )\n        _token(\n            evidence["job_evidence_sha256"], "service evidence hash", r"[0-9a-f]{64}"\n        )\n        rows.append(\n            {\n                "result_json": result.result_json,\n                "result_sha256": result.result_sha256,\n                **{key: evidence[key] for key in columns[2:]},\n            }\n        )\n    pages, current = [], []\n\n    def flush() -> None:\n        if not current:\n            return\n        stream = io.StringIO(newline="")\n        writer = csv.DictWriter(\n            stream, fieldnames=columns, escapechar="\\\\", quoting=csv.QUOTE_ALL\n        )\n        writer.writeheader()\n        writer.writerows(current)\n        text = stream.getvalue()\n        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()\n        path = _manifest_path(\n            f"Files/orchestration/{manifest.run_id}/callbacks/{digest}.csv",\n            manifest.run_id,\n        )\n        _write_immutable(fs, path, text, MAX_MANIFEST_BYTES)\n        pages.append({"path": path, "sha256": digest, "rows": len(current)})\n        current.clear()\n\n    for row in rows:\n        if (\n            len(current) >= 5000\n            or len(_canonical({"value": [*current, row]}).encode("ascii")) > max_bytes\n        ):\n            flush()\n        if len(_canonical({"value": [row]}).encode("ascii")) > max_bytes:\n            raise RuntimeError("A SQL callback exceeds its response bound")\n        current.append(row)\n    flush()\n    return pages\n\n\ndef verify_pipeline_callbacks(\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    path: str,\n    expected_sha256: str,\n) -> None:\n    text = _read_bounded(\n        fs, _manifest_path(path, manifest.run_id), 4 * 1024 * 1024 - 1024\n    )\n    if hashlib.sha256(text.encode("utf-8")).hexdigest() != expected_sha256:\n        raise RuntimeError("SQL callback file differs from its immutable page hash")\n    reader = csv.DictReader(io.StringIO(text), escapechar="\\\\")\n    if reader.fieldnames != [\n        "result_json",\n        "result_sha256",\n        "observed_job_status",\n        "job_evidence_sha256",\n        "job_location",\n    ]:\n        raise RuntimeError("SQL callback columns differ from their contract")\n    rows = list(reader)\n    if not 1 <= len(rows) <= 5000:\n        raise RuntimeError("SQL callback membership exceeds its bounded page contract")\n    tasks = {\n        task["task_id"]\n        for task in cast(list[dict[str, object]], manifest.to_document()["tasks"])\n    }\n    seen = set()\n    for row in rows:\n        if None in row or any(value is None for value in row.values()):\n            raise RuntimeError("SQL callback row is incomplete")\n        result = load_task_result(row["result_json"])\n        if result.run_id != manifest.run_id or result.task_id not in tasks:\n            raise RuntimeError("SQL callback result belongs to another pinned run")\n        if result.result_sha256 != row["result_sha256"]:\n            raise RuntimeError(\n                "SQL callback body differs from its immutable result hash"\n            )\n        if (result.task_id, result.attempt_id) in seen:\n            raise RuntimeError("Duplicate SQL callback result membership")\n        seen.add((result.task_id, result.attempt_id))\n        read_task_result(\n            fs,\n            _result_path(result),\n            run_id=result.run_id,\n            task_id=result.task_id,\n            attempt_id=result.attempt_id,\n            expected_sha256=row["result_sha256"],\n        )\n        _pipeline_activity_scope(row["job_location"])\n        _token(row["job_evidence_sha256"], "service evidence hash", r"[0-9a-f]{64}")\n        if row["observed_job_status"] not in {"Completed", "Failed", "Cancelled"} or (\n            result.status == "SUCCEEDED" and row["observed_job_status"] != "Completed"\n        ):\n            raise RuntimeError(\n                "SQL callback service evidence is not terminal or successful"\n            )\n\n\ndef reconcile_bronze_pipeline(\n    spark: Any,\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    dispatch: VerifiedPipelineDispatch,\n    *,\n    client: ActivityClient,\n    observed_utc: str,\n    ingestion: SilverRuntime,\n    file_arrivals: Mapping[str, SealedFileArrival] | None = None,\n) -> dict[str, object]:\n    """Seal observed Copy outputs and persist results before any SQL callback."""\n    evidence = pipeline_activity_evidence(\n        client, manifest, dispatch, observed_utc=observed_utc, allow_unresolved=True\n    )\n    document = manifest.to_document()\n    configuration = cast(dict[str, Any], document["configuration"])\n    tasks = {\n        task["task_id"]: task\n        for task in cast(list[dict[str, Any]], document["tasks"])\n        if task["layer"] == "bronze"\n    }\n    objects = {obj["id"]: obj for obj in configuration["objects"]}\n    feeds = {feed["id"]: feed for feed in configuration["feeds"]}\n    results = []\n    undispatched = dispatch.to_document().get("undispatched_task_ids", [])\n    failed = bool(undispatched)\n    unresolved = []\n    for attempt in cast(list[dict[str, Any]], dispatch.to_document()["attempts"]):\n        task = tasks[attempt["task_id"]]\n        service = evidence[task["task_id"]]\n        if service["observed_job_status"] == "Unresolved":\n            unresolved.append({**attempt, "error_code": service["error_code"]})\n            failed = True\n            continue\n        if service["observed_job_status"] == "Completed" and feeds[task["feed_id"]][\n            "source_type"\n        ] in ("ftp", "sftp"):\n            # A landed size match cannot prove that the remote source stayed unchanged.\n            unresolved.append(\n                {**attempt, "error_code": "COPY_SOURCE_REOBSERVATION_UNAVAILABLE"}\n            )\n            failed = True\n            continue\n        status = "SUCCEEDED"\n        metrics: dict[str, int] = {}\n        refs = []\n        error_code = None\n        if service["observed_job_status"] != "Completed":\n            status = (\n                "CANCELLED"\n                if service["observed_job_status"] == "Cancelled"\n                else "FAILED"\n            )\n            error_code = "COPY_" + status\n        else:\n            obj, feed = objects[task["object_id"]], feeds[task["feed_id"]]\n            try:\n                receipt = _sealed_bronze_table(fs, feed, task, obj, file_arrivals)\n                context = {\n                    "metadata_mode": "manifest_v1",\n                    "raw_source_files": {\n                        obj["name"]: [file["path"] for file in receipt["files"]]\n                    },\n                    "raw_source_sizes": {\n                        obj["name"]: [file["size_bytes"] for file in receipt["files"]]\n                    },\n                    "raw_targets": {obj["name"]: task["target_path"]},\n                }\n                fingerprint, rows = ingestion.snapshot_fingerprint(\n                    ingestion.read_raw(spark, fs, context, obj)\n                )\n                _write_immutable(\n                    fs,\n                    _manifest_path(\n                        f"Files/orchestration/{manifest.run_id}/bronze_tasks/"\n                        f"{task[\'task_id\']}/{attempt[\'attempt_id\']}.json",\n                        manifest.run_id,\n                    ),\n                    _canonical(\n                        {\n                            **receipt,\n                            "run_id": manifest.run_id,\n                            "task_id": task["task_id"],\n                            "attempt_id": attempt["attempt_id"],\n                            "source_batch_id": task["source_batch_id"],\n                            "record_multiset_sha256": fingerprint,\n                            "rows": rows,\n                        }\n                    ),\n                    MAX_MANIFEST_BYTES,\n                )\n                refs = [\n                    {\n                        "kind": "raw_batch",\n                        "location": task["target_path"],\n                        "version": None,\n                        "sha256": fingerprint,\n                    }\n                ]\n                metrics = {\n                    "rows_read": rows,\n                    "rows_written": rows,\n                    "rows_quarantined": 0,\n                }\n            except ingestion._operation_errors() as error:\n                status, error_code = "FAILED", "BRONZE_" + type(error).__name__.upper()\n        result = make_task_result(\n            manifest,\n            task["task_id"],\n            attempt_id=attempt["attempt_id"],\n            fence=attempt["fence"],\n            job_id=service["job_id"],\n            status=status,\n            finished_utc=service["finished_utc"],\n            artifact_refs=refs,\n            error_code=error_code,\n            metrics=metrics,\n        )\n        persist_task_result(fs, result)\n        results.append((result, service))\n        failed = failed or status != "SUCCEEDED"\n    recovery = _canonical(\n        {\n            "format_version": 1,\n            "run_id": manifest.run_id,\n            "version_id": manifest.version_id,\n            "snapshot_sha256": manifest.snapshot_sha256,\n            "dispatch_sha256": dispatch.dispatch_sha256,\n            "observed_utc": _observation_time(observed_utc)\n            .isoformat(timespec="microseconds")\n            .replace("+00:00", "Z"),\n            "undispatched_task_ids": undispatched,\n            "unresolved_attempts": unresolved,\n            "terminal_results": [\n                {\n                    "task_id": result.task_id,\n                    "attempt_id": result.attempt_id,\n                    "result_sha256": result.result_sha256,\n                }\n                for result, _ in results\n            ],\n        }\n    )\n    recovery_sha256 = hashlib.sha256(recovery.encode("ascii")).hexdigest()\n    recovery_path = _manifest_path(\n        f"Files/orchestration/{manifest.run_id}/bronze_recovery/{recovery_sha256}.json",\n        manifest.run_id,\n    )\n    _write_immutable(fs, recovery_path, recovery, MAX_MANIFEST_BYTES)\n    return {\n        "run_id": manifest.run_id,\n        "has_failed": failed,\n        "undispatched_count": len(undispatched),\n        "unresolved_count": len(unresolved),\n        "recovery_path": recovery_path,\n        "recovery_sha256": recovery_sha256,\n        "callbacks": write_pipeline_callbacks(fs, manifest, results),\n    }\n\n\n@dataclass(frozen=True)\nclass VerifiedLayerDispatch:\n    run_id: str\n    layer: str\n    dispatch_json: str\n\n    @property\n    def dispatch_sha256(self) -> str:\n        return hashlib.sha256(self.dispatch_json.encode("ascii")).hexdigest()\n\n    def to_document(self) -> dict[str, object]:\n        return cast(dict[str, object], json.loads(self.dispatch_json))\n\n\ndef make_layer_dispatch(\n    manifest: VerifiedRunManifest, layer: str, attempts: Sequence[dict[str, object]]\n) -> VerifiedLayerDispatch:\n    if layer not in {"silver", "gold"} or not isinstance(attempts, (list, tuple)):\n        raise RuntimeError("A prepared notebook-layer dispatch is required")\n    document = manifest.to_document()\n    tasks = {\n        task["task_id"]: task\n        for task in cast(list[dict[str, object]], document["tasks"])\n        if task["layer"] == layer\n    }\n    if not tasks or len(attempts) != len(tasks):\n        raise RuntimeError("Layer dispatch omits pinned task membership")\n    checked = []\n    seen, attempt_ids = set(), set()\n    for attempt in attempts:\n        if not isinstance(attempt, dict) or set(attempt) != {\n            "task_id",\n            "attempt_id",\n            "fence",\n            "source_fence",\n        }:\n            raise RuntimeError("Invalid prepared-attempt dispatch fields")\n        task_id = _token(attempt["task_id"], "prepared task identity", r"[0-9a-f]{64}")\n        identity = _attempt_identity(attempt["attempt_id"])\n        if task_id not in tasks or task_id in seen or identity in attempt_ids:\n            raise RuntimeError("Duplicate or unselected prepared attempt")\n        fence = _positive_fence(attempt["fence"])\n        source_fence = attempt["source_fence"]\n        if tasks[task_id]["source_lock_key"] is not None:\n            source_fence = _positive_fence(source_fence)\n        elif source_fence is not None:\n            raise RuntimeError("A feed-independent task cannot invent a source fence")\n        seen.add(task_id)\n        attempt_ids.add(identity)\n        checked.append(\n            {\n                "task_id": task_id,\n                "attempt_id": identity,\n                "fence": fence,\n                "source_fence": source_fence,\n            }\n        )\n    payload = {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "version_id": manifest.version_id,\n        "snapshot_sha256": manifest.snapshot_sha256,\n        "layer": layer,\n        "attempts": checked,\n    }\n    return VerifiedLayerDispatch(manifest.run_id, layer, _canonical(payload))\n\n\ndef notebook_layer_dispatch(\n    manifest: VerifiedRunManifest, pipeline: VerifiedPipelineDispatch\n) -> VerifiedLayerDispatch:\n    checked = make_pipeline_dispatch(manifest, pipeline.to_document())\n    if checked.dispatch_sha256 != pipeline.dispatch_sha256 or pipeline.layer not in {\n        "silver",\n        "gold",\n    }:\n        raise RuntimeError("Notebook dispatch differs from authoritative preparation")\n    return make_layer_dispatch(\n        manifest,\n        pipeline.layer,\n        [\n            {\n                key: attempt[key]\n                for key in ("task_id", "attempt_id", "fence", "source_fence")\n            }\n            for attempt in cast(\n                list[dict[str, object]], pipeline.to_document()["attempts"]\n            )\n        ],\n    )\n\n\ndef write_layer_dispatch(fs: FileSystem, dispatch: VerifiedLayerDispatch) -> str:\n    path = _manifest_path(\n        f"Files/orchestration/{dispatch.run_id}/dispatch/{dispatch.layer}_{dispatch.dispatch_sha256}.json",\n        dispatch.run_id,\n    )\n    return _write_immutable(fs, path, dispatch.dispatch_json, MAX_MANIFEST_BYTES)\n\n\ndef read_layer_dispatch(\n    fs: FileSystem,\n    path: str,\n    manifest: VerifiedRunManifest,\n    *,\n    layer: str,\n    expected_sha256: str,\n) -> VerifiedLayerDispatch:\n    text = _read_bounded(fs, _manifest_path(path, manifest.run_id), MAX_MANIFEST_BYTES)\n    try:\n        document = json.loads(text)\n        canonical = _canonical(document)\n    except (TypeError, ValueError):\n        raise RuntimeError("Layer dispatch is not finite JSON") from None\n    if hashlib.sha256(canonical.encode("ascii")).hexdigest() != expected_sha256:\n        raise RuntimeError("Layer dispatch hash differs from its immutable reference")\n    if (\n        not isinstance(document, dict)\n        or set(document)\n        != {\n            "format_version",\n            "run_id",\n            "version_id",\n            "snapshot_sha256",\n            "layer",\n            "attempts",\n        }\n        or document["format_version"] != 1\n        or document["run_id"] != manifest.run_id\n        or document["version_id"] != manifest.version_id\n        or document["snapshot_sha256"] != manifest.snapshot_sha256\n        or document["layer"] != layer\n    ):\n        raise RuntimeError("Layer dispatch identity differs from the pinned run")\n    return make_layer_dispatch(manifest, layer, document["attempts"])\n\n\ndef prepare_silver_contexts(\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    *,\n    load_configuration: Callable[[object], PinnedConfiguration],\n    file_arrivals: Mapping[str, SealedFileArrival] | None = None,\n) -> dict[str, dict[str, str]]:\n    tasks = cast(list[dict[str, object]], manifest.to_document()["tasks"])\n    feeds = sorted(\n        {str(task["feed_id"]) for task in tasks if task["layer"] == "silver"}\n    )\n    references = {}\n    for feed_id in feeds:\n        receipt = seal_bronze_receipt(\n            fs, manifest, feed_id, file_arrivals=file_arrivals\n        )\n        write_bronze_receipt(fs, receipt)\n        context = source_runtime_config(\n            manifest, receipt, load_configuration=load_configuration\n        )\n        text = _canonical(context)\n        digest = hashlib.sha256(text.encode("ascii")).hexdigest()\n        path = _manifest_path(\n            f"Files/orchestration/{manifest.run_id}/contexts/{digest}.json",\n            manifest.run_id,\n        )\n        _write_immutable(fs, path, text, MAX_MANIFEST_BYTES)\n        references[feed_id] = {"path": path, "sha256": digest}\n    return references\n\n\ndef build_silver_dag(\n    manifest: VerifiedRunManifest,\n    contexts: Mapping[str, dict[str, str]],\n    dispatch: VerifiedLayerDispatch,\n    *,\n    manifest_path: str,\n    dispatch_path: str,\n    lakehouse: str,\n    concurrency: int = 2,\n    timeout_seconds: int = 3600,\n    cell_timeout_seconds: int = 900,\n) -> dict[str, object]:\n    if type(concurrency) is not int or not 1 <= concurrency <= 16:\n        raise RuntimeError("Notebook concurrency must be between 1 and 16")\n    if any(\n        type(value) is not int or not 1 <= value <= 86400\n        for value in (timeout_seconds, cell_timeout_seconds)\n    ):\n        raise RuntimeError("Notebook timeouts require bounded positive seconds")\n    _manifest_path(manifest_path, manifest.run_id)\n    _manifest_path(dispatch_path, manifest.run_id)\n    prepared = dispatch.to_document()\n    verified = make_layer_dispatch(manifest, "silver", prepared["attempts"])\n    if verified.dispatch_sha256 != dispatch.dispatch_sha256:\n        raise RuntimeError("Prepared layer dispatch differs from the pinned run")\n    document = manifest.to_document()\n    config = cast(dict[str, object], document["configuration"])\n    tasks = [\n        task\n        for task in cast(list[dict[str, object]], document["tasks"])\n        if task["layer"] == "silver"\n    ]\n    objects = {\n        obj["id"]: obj for obj in cast(list[dict[str, object]], config["objects"])\n    }\n    workers = {\n        worker["id"]: worker\n        for worker in cast(list[dict[str, object]], config["workers"])\n    }\n    selected = {task["task_id"] for task in tasks}\n    names = {identity: "task_" + str(identity) for identity in selected}\n    targets = set()\n    activities = []\n    for task in tasks:\n        obj = objects[task["object_id"]]\n        worker = workers.get(task["worker_id"])\n        if (\n            worker is None\n            or obj["lakehouse"] != lakehouse\n            or worker["lakehouse"] != lakehouse\n        ):\n            raise RuntimeError(\n                "Silver worker/default Lakehouse binding differs from its parent"\n            )\n        if task["target_lock_key"] in targets:\n            raise RuntimeError("A DAG cannot run multiple writers for the same target")\n        targets.add(task["target_lock_key"])\n        reference = contexts.get(str(task["feed_id"]))\n        if not isinstance(reference, dict) or set(reference) != {"path", "sha256"}:\n            raise RuntimeError("A prepared immutable source context is missing")\n        _manifest_path(reference["path"], manifest.run_id)\n        _token(reference["sha256"], "source context hash", r"[0-9a-f]{64}")\n        dependencies = []\n        for dependency in cast(list[dict[str, object]], task["dependencies"]):\n            parent = dependency["task_id"]\n            if parent in selected:\n                if dependency["condition"] != "succeeded":\n                    raise RuntimeError(\n                        "Silver same-layer dependencies require staged-success readiness"\n                    )\n                dependencies.append(names[parent])\n        activities.append(\n            {\n                "name": names[task["task_id"]],\n                "path": worker["notebook_name"],\n                "timeoutPerCellInSeconds": cell_timeout_seconds,\n                "retry": 0,\n                "args": {\n                    "run_id": manifest.run_id,\n                    "version_id": manifest.version_id,\n                    "manifest_path": manifest_path,\n                    "snapshot_sha256": manifest.snapshot_sha256,\n                    "context_path": reference["path"],\n                    "context_sha256": reference["sha256"],\n                    "dispatch_path": dispatch_path,\n                    "dispatch_sha256": dispatch.dispatch_sha256,\n                    "task_id": task["task_id"],\n                    "worker_id": worker["id"],\n                },\n                "dependencies": dependencies,\n            }\n        )\n    return {\n        "activities": activities,\n        "timeoutInSeconds": timeout_seconds,\n        "concurrency": concurrency,\n    }\n\n\ndef run_notebook_dag(\n    runner: NotebookRunner, dag: dict[str, object], dispatch: VerifiedLayerDispatch\n) -> dict[str, object]:\n    activities = dag.get("activities")\n    if (\n        dispatch.layer not in {"silver", "gold"}\n        or not isinstance(activities, list)\n        or not activities\n    ):\n        raise RuntimeError("Notebook DAG requires its prepared worker dispatch")\n    expected = {}\n    seen_tasks = set()\n    for activity in activities:\n        if not isinstance(activity, dict) or not isinstance(activity.get("args"), dict):\n            raise TypeError("Notebook DAG has invalid worker arguments")\n        name, args = activity.get("name"), activity["args"]\n        task_id = args.get("task_id")\n        if (\n            not isinstance(name, str)\n            or not isinstance(task_id, str)\n            or name in expected\n            or task_id in seen_tasks\n            or args.get("run_id") != dispatch.run_id\n            or args.get("dispatch_sha256") != dispatch.dispatch_sha256\n        ):\n            raise RuntimeError("Notebook DAG differs from its prepared worker dispatch")\n        attempt = _dispatch_attempt(dispatch, task_id)\n        expected[name] = {\n            "task_id": task_id,\n            "stage_path": _silver_stage_path(\n                dispatch.run_id, task_id, str(attempt["attempt_id"])\n            )\n            if dispatch.layer == "silver"\n            else _manifest_path(\n                f"Files/orchestration/{dispatch.run_id}/gold/{task_id}/{attempt[\'attempt_id\']}.json",\n                dispatch.run_id,\n            ),\n        }\n        seen_tasks.add(task_id)\n    if seen_tasks != {\n        attempt["task_id"]\n        for attempt in cast(list[dict[str, object]], dispatch.to_document()["attempts"])\n    }:\n        raise RuntimeError("Notebook DAG omits prepared worker membership")\n    if runner.validateDAG(dag) is not True:\n        raise RuntimeError("Native notebook DAG validation did not succeed")\n    results = runner.runMultiple(dag)\n    if not isinstance(results, dict) or set(results) != set(expected):\n        raise RuntimeError("Notebook DAG result membership is incomplete or ambiguous")\n    for name, value in results.items():\n        if (\n            not isinstance(value, dict)\n            or "exception" not in value\n            or value["exception"] is not None\n            or not isinstance(value.get("exitVal"), str)\n        ):\n            raise RuntimeError(\n                "Notebook DAG failed or returned ambiguous worker status"\n            )\n        try:\n            acknowledgement = json.loads(value["exitVal"])\n        except ValueError:\n            raise RuntimeError(\n                "Notebook worker acknowledgement is not valid JSON"\n            ) from None\n        if acknowledgement != expected[name]:\n            raise RuntimeError(\n                "Notebook worker acknowledgement differs from its attempt"\n            )\n    return results\n\n\ndef _silver_stage_path(run_id: str, task_id: str, attempt_id: str) -> str:\n    return _manifest_path(\n        f"Files/orchestration/{run_id}/silver/{task_id}/{attempt_id}.json", run_id\n    )\n\n\ndef _read_json_metadata(fs: FileSystem, path: str, max_bytes: int) -> dict[str, Any]:\n    text = _read_bounded(fs, path, max_bytes)\n    try:\n        value = json.loads(text)\n        _canonical(value)\n    except (ValueError, TypeError):\n        raise RuntimeError("Persisted task metadata is not finite JSON") from None\n    if not isinstance(value, dict):\n        raise TypeError("Persisted task metadata must be an object")\n    return value\n\n\ndef _dispatch_attempt(\n    dispatch: VerifiedLayerDispatch, task_id: str\n) -> dict[str, object]:\n    matches = [\n        attempt\n        for attempt in cast(list[dict[str, object]], dispatch.to_document()["attempts"])\n        if attempt["task_id"] == task_id\n    ]\n    if len(matches) != 1:\n        raise RuntimeError("Prepared worker attempt is missing or ambiguous")\n    return matches[0]\n\n\ndef initialize_silver_sources(\n    spark: Any,\n    fs: FileSystem,\n    contexts: Mapping[str, dict[str, str]],\n    *,\n    lakehouse: str,\n    ingestion: SilverRuntime,\n) -> None:\n    """Create shared namespaces and seal parent populations before parallel writers."""\n    for reference in contexts.values():\n        text = _read_bounded(fs, reference["path"], MAX_MANIFEST_BYTES)\n        context = ingestion.manifest_processing_context(\n            text, reference["sha256"], expected_lakehouse=lakehouse\n        )\n        _verify_context_file_membership(fs, context)\n        ingestion.setup_namespaces(spark, context)\n        ingestion.inventory(\n            spark, fs, context, table_names=list(context["raw_source_files"])\n        )\n\n\ndef execute_silver_task(\n    spark: Any,\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    dispatch: VerifiedLayerDispatch,\n    *,\n    task_id: str,\n    worker_id: str,\n    context_path: str,\n    context_sha256: str,\n    lakehouse: str,\n    code_sha256: str,\n    ingestion: SilverRuntime,\n) -> str:\n    resolved = resolve_worker_task(\n        manifest,\n        task_id,\n        worker_id=worker_id,\n        code_sha256=code_sha256,\n        lakehouse=lakehouse,\n    )\n    checked_dispatch = make_layer_dispatch(\n        manifest, "silver", dispatch.to_document()["attempts"]\n    )\n    if checked_dispatch.dispatch_sha256 != dispatch.dispatch_sha256:\n        raise RuntimeError("Silver dispatch differs from its pinned worker context")\n    attempt = _dispatch_attempt(dispatch, task_id)\n    context_path = _manifest_path(context_path, manifest.run_id)\n    context = ingestion.manifest_processing_context(\n        _read_bounded(fs, context_path, MAX_MANIFEST_BYTES),\n        context_sha256,\n        expected_lakehouse=lakehouse,\n    )\n    task, obj = (\n        cast(dict[str, Any], resolved["task"]),\n        cast(dict[str, Any], resolved["object"]),\n    )\n    if (\n        context["metadata_run_id"] != manifest.run_id\n        or context["metadata_version"] != manifest.version_id\n        or context["source_feed_id"] != task["feed_id"]\n        or context["source_batch_id"] != task["source_batch_id"]\n        or obj["name"] not in context["selected_tables"]\n    ):\n        raise RuntimeError("Worker source context differs from the prepared task")\n    path = _silver_stage_path(manifest.run_id, task_id, str(attempt["attempt_id"]))\n    identity = {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "version_id": manifest.version_id,\n        "task_id": task_id,\n        "object_id": obj["id"],\n        "attempt_id": attempt["attempt_id"],\n        "fence": attempt["fence"],\n        "source_fence": attempt["source_fence"],\n        "source_batch_id": task["source_batch_id"],\n        "context_sha256": context_sha256,\n        "code_sha256": code_sha256,\n        "target_table": task["target_path"],\n    }\n    if fs.exists(path):\n        existing = _read_json_metadata(fs, path, MAX_RESULT_BYTES)\n        if (\n            any(existing.get(name) != value for name, value in identity.items())\n            or existing.get("status") != "SUCCEEDED"\n        ):\n            raise RuntimeError(\n                "A failed or changed staged task requires explicit recovery"\n            )\n        return path\n    try:\n        _verify_context_file_membership(fs, context)\n        results = ingestion.load_silver(\n            spark, fs, context, table_names=[str(obj["name"])]\n        )\n        if len(results) != 1 or results[0]["status"] != "SUCCESS":\n            raise RuntimeError("Silver object did not complete with passing quality")\n        version = ingestion.delta_version(spark, str(task["target_path"]))\n        stage = {\n            **identity,\n            "status": "SUCCEEDED",\n            "delta_version": version,\n            "source_rows": results[0]["source_rows"],\n            "silver_rows": results[0]["silver_rows"],\n            "quarantine_rows": results[0]["quarantine_rows"],\n            "error_code": None,\n        }\n    except ingestion._operation_errors() as error:\n        _write_immutable(\n            fs,\n            path,\n            _canonical(\n                {\n                    **identity,\n                    "status": "FAILED",\n                    "delta_version": None,\n                    "source_rows": None,\n                    "silver_rows": None,\n                    "quarantine_rows": None,\n                    "error_code": "SILVER_" + type(error).__name__.upper(),\n                }\n            ),\n            MAX_RESULT_BYTES,\n        )\n        raise RuntimeError("Silver task failed; Source commit is withheld") from None\n    _write_immutable(fs, path, _canonical(stage), MAX_RESULT_BYTES)\n    return path\n\n\ndef verify_silver_stage_outputs(\n    spark: Any,\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    dispatch: VerifiedLayerDispatch,\n    contexts: Mapping[str, dict[str, str]],\n    *,\n    lakehouse: str,\n    ingestion: SilverRuntime,\n) -> dict[str, dict[str, Any]]:\n    """Verify every immutable object output before publication or SQL reconciliation."""\n    checked = make_layer_dispatch(\n        manifest, "silver", dispatch.to_document()["attempts"]\n    )\n    if checked.dispatch_sha256 != dispatch.dispatch_sha256:\n        raise RuntimeError("Silver fan-in dispatch differs from its pinned run")\n    tasks = [\n        task\n        for task in cast(list[dict[str, object]], manifest.to_document()["tasks"])\n        if task["layer"] == "silver"\n    ]\n    if set(contexts) != {str(task["feed_id"]) for task in tasks}:\n        raise RuntimeError(\n            "Silver source fan-in omits or invents selected feed membership"\n        )\n    config = cast(dict[str, object], manifest.to_document()["configuration"])\n    objects = {\n        obj["id"]: obj for obj in cast(list[dict[str, object]], config["objects"])\n    }\n    workers = {\n        worker["id"]: worker\n        for worker in cast(list[dict[str, object]], config["workers"])\n    }\n    if any(objects[task["object_id"]]["lakehouse"] != lakehouse for task in tasks):\n        raise RuntimeError("Silver stage evidence belongs to another Lakehouse binding")\n    stages = {}\n    for task in tasks:\n        attempt = _dispatch_attempt(dispatch, str(task["task_id"]))\n        stage = _read_json_metadata(\n            fs,\n            _silver_stage_path(\n                manifest.run_id, str(task["task_id"]), str(attempt["attempt_id"])\n            ),\n            MAX_RESULT_BYTES,\n        )\n        if (\n            stage.get("run_id") != manifest.run_id\n            or stage.get("version_id") != manifest.version_id\n            or stage.get("task_id") != task["task_id"]\n            or stage.get("attempt_id") != attempt["attempt_id"]\n            or stage.get("fence") != attempt["fence"]\n            or stage.get("source_fence") != attempt["source_fence"]\n            or stage.get("source_batch_id") != task["source_batch_id"]\n            or stage.get("status") != "SUCCEEDED"\n            or stage.get("target_table") != task["target_path"]\n            or stage.get("context_sha256") != contexts[str(task["feed_id"])]["sha256"]\n            or stage.get("code_sha256")\n            != workers[objects[task["object_id"]]["worker_id"]]["code_sha256"]\n            or type(stage.get("delta_version")) is not int\n            or stage["delta_version"]\n            != ingestion.delta_version(spark, str(task["target_path"]))\n            or any(\n                type(stage.get(name)) is not int or stage[name] < 0\n                for name in ("source_rows", "silver_rows", "quarantine_rows")\n            )\n            or stage["source_rows"] != stage["silver_rows"] + stage["quarantine_rows"]\n        ):\n            raise RuntimeError(\n                "Silver staged output is missing, stale or failed; publication is withheld"\n            )\n        stages[str(task["task_id"])] = stage\n    return stages\n\n\ndef finalize_silver_sources(\n    spark: Any,\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    dispatch: VerifiedLayerDispatch,\n    contexts: Mapping[str, dict[str, str]],\n    *,\n    lakehouse: str,\n    ingestion: SilverRuntime,\n) -> dict[str, object]:\n    """Only existing source validation publishes commits after complete stage fan-in."""\n    verify_silver_stage_outputs(\n        spark,\n        fs,\n        manifest,\n        dispatch,\n        contexts,\n        lakehouse=lakehouse,\n        ingestion=ingestion,\n    )\n    committed = []\n    for feed_id, reference in contexts.items():\n        context = ingestion.manifest_processing_context(\n            _read_bounded(fs, reference["path"], MAX_MANIFEST_BYTES),\n            reference["sha256"],\n            expected_lakehouse=lakehouse,\n        )\n        if (\n            context["metadata_run_id"] != manifest.run_id\n            or context["source_feed_id"] != feed_id\n        ):\n            raise RuntimeError("Source commit context differs from the pinned run")\n        _verify_context_file_membership(fs, context)\n        ingestion.validate_source(spark, fs, context)\n        committed.append(\n            {"feed_id": feed_id, "source_batch_id": context["source_batch_id"]}\n        )\n    return {\n        "run_id": manifest.run_id,\n        "status": "SOURCE_COMMITTED",\n        "sources": committed,\n    }\n\n\ndef reconcile_silver_pipeline(\n    spark: Any,\n    fs: FileSystem,\n    manifest: VerifiedRunManifest,\n    pipeline: VerifiedPipelineDispatch,\n    *,\n    client: ActivityClient,\n    observed_utc: str,\n    lakehouse: str,\n    ingestion: SilverRuntime,\n    load_configuration: Callable[[object], PinnedConfiguration],\n    file_arrivals: Mapping[str, SealedFileArrival] | None = None,\n) -> dict[str, object]:\n    evidence = pipeline_activity_evidence(\n        client,\n        manifest,\n        pipeline,\n        observed_utc=observed_utc,\n        parent_activity="orchestrate_silver",\n    )\n    dispatch = notebook_layer_dispatch(manifest, pipeline)\n    stages, commits = {}, {}\n    failure_code = None\n    if any(value["observed_job_status"] != "Completed" for value in evidence.values()):\n        failure_code = "SILVER_PARENT_FAILED"\n    else:\n        try:\n            contexts = prepare_silver_contexts(\n                fs,\n                manifest,\n                load_configuration=load_configuration,\n                file_arrivals=file_arrivals,\n            )\n            stages = verify_silver_stage_outputs(\n                spark,\n                fs,\n                manifest,\n                dispatch,\n                contexts,\n                lakehouse=lakehouse,\n                ingestion=ingestion,\n            )\n            for feed_id, reference in contexts.items():\n                context = ingestion.manifest_processing_context(\n                    _read_bounded(fs, reference["path"], MAX_MANIFEST_BYTES),\n                    reference["sha256"],\n                    expected_lakehouse=lakehouse,\n                )\n                ingestion.cleanup_snapshots(spark, fs, context)\n                commits[feed_id] = ingestion.source_commit_evidence(spark, context)\n        except ingestion._operation_errors():\n            failure_code = "SILVER_OUTPUT_OR_COMMIT_INVALID"\n    tasks = {\n        task["task_id"]: task\n        for task in cast(list[dict[str, Any]], manifest.to_document()["tasks"])\n        if task["layer"] == "silver"\n    }\n    results = []\n    for attempt in cast(list[dict[str, Any]], pipeline.to_document()["attempts"]):\n        task, service = tasks[attempt["task_id"]], evidence[attempt["task_id"]]\n        refs, metrics, readiness = [], {}, []\n        if failure_code is None:\n            stage = stages[task["task_id"]]\n            refs = [\n                {\n                    "kind": "delta_version",\n                    "location": task["target_path"],\n                    "version": stage["delta_version"],\n                    "sha256": hashlib.sha256(\n                        _canonical(stage).encode("ascii")\n                    ).hexdigest(),\n                },\n                commits[task["feed_id"]],\n            ]\n            metrics = {\n                "rows_read": stage["source_rows"],\n                "rows_written": stage["silver_rows"],\n                "rows_quarantined": stage["quarantine_rows"],\n            }\n            readiness = ["succeeded", "source_committed"]\n        result = make_task_result(\n            manifest,\n            task["task_id"],\n            attempt_id=attempt["attempt_id"],\n            fence=attempt["fence"],\n            job_id=service["job_id"],\n            status="SUCCEEDED" if failure_code is None else "FAILED",\n            finished_utc=service["finished_utc"],\n            artifact_refs=refs,\n            metrics=metrics,\n            readiness=readiness,\n            error_code=failure_code,\n        )\n        persist_task_result(fs, result)\n        results.append((result, service))\n    return {\n        "run_id": manifest.run_id,\n        "has_failed": failure_code is not None,\n        "callbacks": write_pipeline_callbacks(fs, manifest, results),\n    }\n', 'layout': '"""Fail-closed, shared physical-layout adapters (REQ-039..043, REQ-059).\n\nNo Spark import, discovery, network access or mutation occurs on import.\nCall ``prewrite_plan`` immediately before *every* append/merge, under the\nparent\'s writer ownership. For creation, prefer ``create_target`` with the\ncomplete physical schema (including lineage), then append normally. Existing\ntargets never receive creation options or implicit layout changes.\n\nSyntax sources (documentation is not runtime capability evidence):\n* https://docs.delta.io/delta-clustering/\n* https://docs.delta.io/delta-batch/\n* https://docs.delta.io/delta-utility/\n* https://learn.microsoft.com/en-us/fabric/data-engineering/liquid-clustering\n* https://learn.microsoft.com/en-us/azure/databricks/tables/operations/table-details\n\nThe parent supplies runtime-specific, operation-specific capability evidence.\nAn external fenced writer lease is mandatory for maintenance. This module\ncannot make SQL metadata, leases and Delta a cross-store transaction. There is\ndeliberately no migration/cutover executor, automatic retry, VACUUM, RESTORE,\noverwrite, or replacement of historical tables.\n"""\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections.abc import Mapping\nfrom contextlib import AbstractContextManager\nfrom dataclasses import asdict, dataclass\nfrom typing import Any, Protocol\n\n# Any is confined to optional, duck-typed Spark/Delta framework boundaries.\n_IDENTIFIER = re.compile(r"[A-Za-z_][A-Za-z0-9_]*\\Z")\n_PRIMITIVES = {\n    "string",\n    "boolean",\n    "tinyint",\n    "smallint",\n    "int",\n    "bigint",\n    "float",\n    "double",\n    "date",\n    "timestamp",\n    "timestamp_ntz",\n}\n_ALIASES = {"long": "bigint", "integer": "int", "short": "smallint", "byte": "tinyint"}\n_DECIMAL = re.compile(r"decimal\\((\\d+),(\\d+)\\)\\Z")\n_LAYOUT_KEYS = {"layout_strategy", "partition_columns", "cluster_by_columns"}\n\n\nclass LayoutError(ValueError):\n    """Invalid or unverifiable layout; no ordinary write may proceed."""\n\n\nclass OperationRejected(LayoutError, RuntimeError):\n    """An approved operation no longer matches its preconditions."""\n\n\nclass MigrationRequired(LayoutError, RuntimeError):\n    """A separate, target-specific migration/cutover contract is required."""\n\n\nclass OperationIncomplete(RuntimeError):\n    """A native operation may have committed; reconcile instead of retrying."""\n\n\ndef _identifier(value: object) -> str:\n    if not isinstance(value, str) or not _IDENTIFIER.fullmatch(value):\n        raise LayoutError(f"Invalid SQL identifier: {value!r}")\n    return value\n\n\ndef quote_table(table: str) -> str:\n    """Quote a one-, two- or three-part catalog identifier; paths are not accepted."""\n    if not isinstance(table, str) or not 1 <= len(table.split(".")) <= 3:\n        raise LayoutError("Expected a one-, two- or three-part table identifier")\n    return ".".join(f"`{_identifier(part)}`" for part in table.split("."))\n\n\ndef _ordered_columns(value: object, label: str) -> tuple[str, ...]:\n    if not isinstance(value, (tuple, list)):\n        raise LayoutError(f"{label} must be an ordered list, not null/string/set")\n    columns = tuple(_identifier(c) for c in value)\n    if len({c.casefold() for c in columns}) != len(columns):\n        raise LayoutError(f"{label} contains duplicate columns")\n    return columns\n\n\ndef _type_name(value: object) -> str:\n    if not isinstance(value, str):\n        raise LayoutError("Column data types must be explicit Spark SQL types")\n    normalized = re.sub(r"\\s+", "", value.lower())\n    normalized = _ALIASES.get(normalized, normalized)\n    if normalized in _PRIMITIVES:\n        return normalized\n    decimal = _DECIMAL.fullmatch(normalized)\n    if decimal:\n        precision, scale = map(int, decimal.groups())\n        if 1 <= precision <= 38 and 0 <= scale <= precision:\n            return f"decimal({precision},{scale})"\n    raise LayoutError(\n        f"Unsupported column type: {value!r}; use a validated schema adapter"\n    )\n\n\n@dataclass(frozen=True)\nclass Column:\n    name: str\n    data_type: str\n    nullable: bool = True\n\n\ndef schema_columns(schema: Any) -> tuple[Column, ...]:\n    """Normalize Column sequences or a Spark StructType, never schema SQL text."""\n    if hasattr(schema, "fields"):\n        columns = tuple(\n            Column(field.name, field.dataType.simpleString(), field.nullable)\n            for field in schema.fields\n        )\n    elif isinstance(schema, (tuple, list)) and all(\n        isinstance(c, Column) for c in schema\n    ):\n        columns = tuple(schema)\n    else:\n        raise LayoutError(\n            "Schema must be a nonempty Column sequence or Spark StructType"\n        )\n    if not columns:\n        raise LayoutError("An explicit nonempty physical schema is required")\n    checked = []\n    for column in columns:\n        _identifier(column.name)\n        if type(column.nullable) is not bool:\n            raise LayoutError("Column nullable must be a Boolean")\n        checked.append(\n            Column(column.name, _type_name(column.data_type), column.nullable)\n        )\n    _ordered_columns([c.name for c in checked], "schema")\n    return tuple(checked)\n\n\n@dataclass(frozen=True)\nclass RuntimeCapabilities:\n    """Parent-owned evidence from the selected engine, not a feature-version guess.\n\n    Operations: liquid_write, liquid_create, liquid_configure, optimize,\n    mlv_partitioned, mlv_liquid. MLV validation is NOT a table-DDL adapter.\n    statistics_columns names columns whose Delta statistics coverage was verified.\n    The parent must bind engine_id/evidence to its actual session/environment.\n    """\n\n    engine_id: str\n    spark_version: str\n    evidence: str\n    operations: tuple[str, ...] = ()\n    statistics_columns: tuple[str, ...] = ()\n\n    def require(self, operation: str, spark: Any = None) -> None:\n        if (\n            not all(\n                isinstance(v, str) and v.strip()\n                for v in (self.engine_id, self.spark_version, self.evidence)\n            )\n            or not isinstance(self.operations, (tuple, list))\n            or operation not in self.operations\n        ):\n            raise LayoutError(\n                f"Explicit runtime capability evidence required: {operation}"\n            )\n        _ordered_columns(self.statistics_columns, "verified statistics_columns")\n        if spark is not None and spark.version != self.spark_version:\n            raise LayoutError("Capability evidence does not match the Spark runtime")\n\n\ndef _require(\n    capabilities: RuntimeCapabilities | None, operation: str, spark: Any = None\n) -> None:\n    if not isinstance(capabilities, RuntimeCapabilities):\n        raise LayoutError(f"Explicit runtime capability evidence required: {operation}")\n    capabilities.require(operation, spark)\n\n\n@dataclass(frozen=True)\nclass LayoutSpec:\n    strategy: str\n    columns: tuple[str, ...]\n    materialization: str = "delta_table"\n\n    def metadata(self) -> dict[str, object]:\n        key = (\n            "partition_columns"\n            if self.strategy == "partitioned"\n            else "cluster_by_columns"\n        )\n        return {"layout_strategy": self.strategy, key: list(self.columns)}\n\n\ndef validate_layout(\n    metadata: Mapping[str, object] | None,\n    schema: Any,\n    *,\n    materialization: str = "delta_table",\n    capabilities: RuntimeCapabilities | None = None,\n) -> LayoutSpec | None:\n    """Validate a layout-only metadata projection. None means preserve, not create.\n\n    Only the active ordered list may be nonempty. Unknown keys (including\n    independent toggles and ZORDER) fail rather than being silently ignored.\n    Scalar types are deliberately conservative; complex schemas need an adapter.\n    """\n    columns = schema_columns(schema)\n    if materialization not in ("delta_table", "mlv"):\n        raise LayoutError(\n            "Raw files/views are not compatible managed Delta layout targets"\n        )\n    if metadata is None:\n        metadata = {}\n    if not isinstance(metadata, Mapping) or set(metadata) - _LAYOUT_KEYS:\n        raise LayoutError(\n            "Unknown layout options; use only the single layout_strategy selector"\n        )\n    partition = _ordered_columns(\n        metadata.get("partition_columns", ()), "partition_columns"\n    )\n    cluster = _ordered_columns(\n        metadata.get("cluster_by_columns", ()), "cluster_by_columns"\n    )\n    strategy = metadata.get("layout_strategy")\n    if strategy is None and not partition and not cluster:\n        return None\n    if strategy not in ("partitioned", "liquid"):\n        raise LayoutError(\n            "layout_strategy must explicitly select partitioned or liquid"\n        )\n    active, inactive = (\n        (partition, cluster) if strategy == "partitioned" else (cluster, partition)\n    )\n    if not active or inactive:\n        raise LayoutError(\n            "Active layout columns must be nonempty; inactive columns must be empty"\n        )\n    types = {c.name: c.data_type for c in columns}\n    if set(active) - types.keys():\n        raise LayoutError(\n            "Layout columns must exactly match physical schema column names"\n        )\n    if materialization == "mlv":\n        _require(capabilities, f"mlv_{strategy}")\n    if strategy == "liquid":\n        _require(capabilities, "liquid_write")\n        if len(active) > 4 or any(types[c] == "boolean" for c in active):\n            raise LayoutError(\n                "Liquid layout supports at most four validated scalar columns"\n            )\n        if set(active) - set(capabilities.statistics_columns):\n            raise LayoutError(\n                "Liquid columns require verified Delta statistics coverage"\n            )\n    return LayoutSpec(strategy, active, materialization)\n\n\n@dataclass(frozen=True)\nclass TargetLayout:\n    table: str\n    table_id: str\n    version: int\n    schema: tuple[Column, ...]\n    partition_columns: tuple[str, ...]\n    cluster_by_columns: tuple[str, ...]\n    size_bytes: int\n    table_features: tuple[str, ...] = ()\n\n\ndef _row_dict(row: Any) -> dict[str, Any]:\n    return row.asDict(recursive=True) if hasattr(row, "asDict") else dict(row)\n\n\ndef _single(frame: Any, label: str) -> dict[str, Any]:\n    rows = frame.limit(2).collect()\n    if len(rows) != 1:\n        raise LayoutError(f"Cannot verify {label}: expected exactly one metadata row")\n    return _row_dict(rows[0])\n\n\ndef _version(spark: Any, table: str) -> int:\n    row = _single(\n        spark.sql(f"DESCRIBE HISTORY {quote_table(table)} LIMIT 1"), "Delta version"\n    )\n    version = row.get("version")\n    if type(version) is not int or version < 0:\n        raise LayoutError("Cannot verify Delta history version")\n    return version\n\n\ndef inspect_target(spark: Any, table: str) -> TargetLayout | None:\n    """Read authoritative detail/schema/history, or return None for an absent table.\n\n    DESCRIBE DETAIL.clusteringColumns is the documented source for cluster keys.\n    Older detail schemas may prove absence through the complete tableFeatures\n    list. A missing/null column is NOT silently interpreted as unclustered.\n    There is no speculative TBLPROPERTIES key/JSON fallback. Engine exceptions\n    propagate: an unreadable target is never mistaken for a missing target.\n    """\n    quoted = quote_table(table)\n    if not spark.catalog.tableExists(table):\n        return None\n    before = _version(spark, table)\n    detail = _single(spark.sql(f"DESCRIBE DETAIL {quoted}"), "Delta detail")\n    if (\n        detail.get("format") != "delta"\n        or not isinstance(detail.get("id"), str)\n        or not detail["id"]\n    ):\n        raise LayoutError(\n            "Target must be a verifiable Delta table with a stable table id"\n        )\n    partition = _ordered_columns(\n        detail.get("partitionColumns"), "existing partitionColumns"\n    )\n    features = detail.get("tableFeatures", ())\n    if not isinstance(features, (tuple, list)) or not all(\n        isinstance(f, str) for f in features\n    ):\n        raise LayoutError("Cannot verify Delta tableFeatures")\n    if detail.get("clusteringColumns") is not None:\n        cluster = _ordered_columns(\n            detail["clusteringColumns"], "existing clusteringColumns"\n        )\n    elif "tableFeatures" in detail and "clustering" not in {\n        f.lower() for f in features\n    }:\n        cluster = ()\n    else:\n        raise LayoutError(\n            "Cannot verify clusteringColumns; a capable detail adapter is required"\n        )\n    if partition and cluster:\n        raise LayoutError(\n            "Target reports incompatible partitioning and liquid clustering"\n        )\n    schema = schema_columns(spark.table(table).schema)\n    names = {c.name for c in schema}\n    if (set(partition) | set(cluster)) - names:\n        raise LayoutError("Target detail references unknown physical columns")\n    size = detail.get("sizeInBytes")\n    if type(size) is not int or size < 0:\n        raise LayoutError("Cannot verify target sizeInBytes")\n    if before != _version(spark, table):\n        raise LayoutError(\n            "Target version drifted during layout inspection; retry under writer ownership"\n        )\n    return TargetLayout(\n        table, detail["id"], before, schema, partition, cluster, size, tuple(features)\n    )\n\n\ndef _compatible(\n    existing: TargetLayout, requested: LayoutSpec | None, schema: tuple[Column, ...]\n) -> None:\n    if requested is None:\n        return\n    partition = requested.columns if requested.strategy == "partitioned" else ()\n    cluster = requested.columns if requested.strategy == "liquid" else ()\n    if (existing.partition_columns, existing.cluster_by_columns) != (\n        partition,\n        cluster,\n    ):\n        raise MigrationRequired(\n            "Existing layout differs: explicit approved migration required before writes"\n        )\n    old_types = {c.name: c.data_type for c in existing.schema}\n    new_types = {c.name: c.data_type for c in schema}\n    if any(old_types.get(c) != new_types.get(c) for c in requested.columns):\n        raise LayoutError("Layout column type differs from existing physical schema")\n\n\n@dataclass(frozen=True)\nclass LayoutPlan:\n    table: str\n    schema: tuple[Column, ...]\n    requested: LayoutSpec | None\n    existing: TargetLayout | None\n    adapter: str\n\n\ndef prewrite_plan(\n    spark: Any,\n    table: str,\n    metadata: Mapping[str, object] | None,\n    schema: Any,\n    *,\n    materialization: str = "delta_table",\n    capabilities: RuntimeCapabilities | None = None,\n) -> LayoutPlan:\n    """Inspect before append/merge; no DDL or data writes. Never reuse across writes."""\n    quote_table(table)\n    columns = schema_columns(schema)\n    requested = validate_layout(\n        metadata, columns, materialization=materialization, capabilities=capabilities\n    )\n    if materialization != "delta_table":\n        raise LayoutError(\n            "MLV creation/inspection requires a separately validated engine adapter"\n        )\n    existing = inspect_target(spark, table)\n    if existing is not None:\n        if existing.cluster_by_columns or "clustering" in {\n            f.lower() for f in existing.table_features\n        }:\n            _require(capabilities, "liquid_write", spark)\n        _compatible(existing, requested, columns)\n        return LayoutPlan(table, columns, requested, existing, "preserve")\n    if requested is None:\n        raise LayoutError("New managed targets require an explicit layout selection")\n    if requested.strategy == "liquid":\n        _require(capabilities, "liquid_create", spark)\n    adapter = (\n        "partitionBy" if requested.strategy == "partitioned" else "create_table_sql"\n    )\n    return LayoutPlan(table, columns, requested, None, adapter)\n\n\ndef creation_sql(\n    table: str,\n    schema: Any,\n    requested: LayoutSpec | None,\n    capabilities: RuntimeCapabilities | None = None,\n) -> str:\n    """Explicit-schema CREATE only; never CTAS, IF NOT EXISTS, replace or overwrite."""\n    columns = schema_columns(schema)\n    if requested is None:\n        raise LayoutError("Creation requires an explicit layout selection")\n    if requested.materialization != "delta_table":\n        raise LayoutError("MLV DDL requires a separately validated engine adapter")\n    spec = validate_layout(requested.metadata(), columns, capabilities=capabilities)\n    if spec.strategy == "liquid":\n        _require(capabilities, "liquid_create")\n    fields = ", ".join(\n        f"`{c.name}` {c.data_type.upper()}" + ("" if c.nullable else " NOT NULL")\n        for c in columns\n    )\n    clause = "PARTITIONED BY" if spec.strategy == "partitioned" else "CLUSTER BY"\n    keys = ", ".join(f"`{c}`" for c in spec.columns)\n    return f"CREATE TABLE {quote_table(table)} ({fields}) USING DELTA {clause} ({keys})"\n\n\ndef _creation_check(\n    spark: Any, plan: LayoutPlan, capabilities: RuntimeCapabilities | None\n) -> None:\n    if plan.existing is not None or spark.catalog.tableExists(plan.table):\n        raise LayoutError(\n            "Creation adapter cannot modify an existing or concurrently created target"\n        )\n    if plan.requested is None:\n        raise LayoutError("Creation requires an explicit layout selection")\n    creation_sql(plan.table, plan.schema, plan.requested, capabilities)\n    if plan.requested.strategy == "liquid":\n        _require(capabilities, "liquid_create", spark)\n\n\ndef prepare_creation_writer(spark: Any, writer: Any, plan: LayoutPlan) -> Any:\n    """Return a creation-only DataFrameWriter; parent calls saveAsTable(plan.table).\n\n    Pass df.schema to prewrite_plan so lineage fields are included. Never use\n    this adapter on an existing target. For liquid, use create_target and append:\n    generic DataFrameWriter.clusterBy is not assumed to be available.\n    """\n    if plan.requested is not None and plan.requested.strategy == "liquid":\n        raise LayoutError(\n            "DataFrameWriter.clusterBy is unproven; use explicit-schema create_target"\n        )\n    _creation_check(spark, plan, None)\n    return (\n        writer.format("delta")\n        .mode("errorifexists")\n        .partitionBy(*plan.requested.columns)\n    )\n\n\ndef create_target(\n    spark: Any, plan: LayoutPlan, *, capabilities: RuntimeCapabilities | None = None\n) -> TargetLayout:\n    """Create an absent table, inspect it, and reject unverifiable results.\n\n    Parent holds creation ownership. A race fails CREATE, never replaces data.\n    Failure after DDL may leave an empty table: reconcile, do not drop/retry.\n    """\n    _creation_check(spark, plan, capabilities)\n    spark.sql(creation_sql(plan.table, plan.schema, plan.requested, capabilities))\n    created = inspect_target(spark, plan.table)\n    if created is None:\n        raise LayoutError("Created table cannot be verified; reconcile before writing")\n    _compatible(created, plan.requested, plan.schema)\n    return created\n\n\ndef ensure_target_layout(\n    spark: Any,\n    target: str,\n    frame: Any,\n    layout: Mapping[str, object] | None,\n    *,\n    existing_target: bool,\n    capabilities: RuntimeCapabilities | None = None,\n) -> None:\n    """Parent integration hook, immediately before merge/append under ownership.\n\n    frame.schema must contain the complete physical projection, including lineage.\n    Existing targets are inspected only. New generic targets must have a layout\n    and are created empty before the caller merges/appends. A stale existence\n    assertion is rejected instead of silently adopting a concurrently created\n    table. Historical no-metadata flows can retain their old call path; once\n    using this generic hook, null layout means preserve an existing table only.\n    """\n    if type(existing_target) is not bool:\n        raise LayoutError("existing_target must be an explicit Boolean")\n    plan = prewrite_plan(spark, target, layout, frame.schema, capabilities=capabilities)\n    if existing_target != (plan.existing is not None):\n        raise LayoutError("Target existence changed; reconcile under writer ownership")\n    if plan.existing is None:\n        create_target(spark, plan, capabilities=capabilities)\n\n\n@dataclass(frozen=True)\nclass ContentComparison:\n    schema_equal: bool\n    left_only: int | None\n    right_only: int | None\n\n    @property\n    def equivalent(self) -> bool:\n        return self.schema_equal and self.left_only == 0 and self.right_only == 0\n\n\ndef compare_logical_content(left: Any, right: Any) -> ContentComparison:\n    """Distributed multiset equality over ALL columns, including nulls/duplicates.\n\n    exceptAll resolves columns by position, so align names first. No driver-side\n    row collection, hash-only comparison, distinct, or count-only equivalence.\n    """\n    left_schema = schema_columns(left.schema)\n    right_schema = schema_columns(right.schema)\n    signature = lambda schema: sorted((c.name, c.data_type) for c in schema)\n    if signature(left_schema) != signature(right_schema):\n        return ContentComparison(False, None, None)\n    names = [c.name for c in left_schema]\n    aligned_left, aligned_right = left.select(*names), right.select(*names)\n    return ContentComparison(\n        True,\n        aligned_left.exceptAll(aligned_right).count(),\n        aligned_right.exceptAll(aligned_left).count(),\n    )\n\n\n@dataclass(frozen=True)\nclass ResourceEstimate:\n    read_bytes: int\n    estimated_write_bytes: int\n    retained_history_bytes: int\n    elapsed_seconds: float | None\n    capacity_unit_seconds: float | None\n    cost: float | None\n    assumptions: tuple[str, ...]\n\n\ndef _positive_number(value: object, label: str) -> float:\n    if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:\n        raise LayoutError(f"{label} must be a finite positive number")\n    return float(value)\n\n\ndef _estimate(\n    current: TargetLayout,\n    rewrites: bool,\n    throughput_bytes_per_second: float | None,\n    capacity_units: float | None,\n    cost_per_cu_second: float | None,\n) -> ResourceEstimate:\n    seconds = cu_seconds = cost = None\n    if throughput_bytes_per_second is not None:\n        throughput = _positive_number(throughput_bytes_per_second, "throughput")\n        seconds = current.size_bytes / throughput if rewrites else 0.0\n    if capacity_units is not None:\n        units = _positive_number(capacity_units, "capacity_units")\n        if seconds is not None:\n            cu_seconds = seconds * units\n    if cost_per_cu_second is not None:\n        rate = _positive_number(cost_per_cu_second, "cost_per_cu_second")\n        if cu_seconds is not None:\n            cost = cu_seconds * rate\n    return ResourceEstimate(\n        current.size_bytes if rewrites else 0,\n        current.size_bytes if rewrites else 0,\n        current.size_bytes,\n        seconds,\n        cu_seconds,\n        cost,\n        (\n            "Source size is compressed Delta size, not a measured rewrite bound.",\n            "Worst-case full rewrite assumed for OPTIMIZE; no incremental-runtime guarantee.",\n            "Output size assumed equal to input; retain old files/history and budget extra headroom.",\n            "Throughput is caller-measured end-to-end source bytes/s; CU/rate are caller supplied.",\n            "Unknown throughput/capacity/rate stays unknown; estimate is not execution approval.",\n        ),\n    )\n\n\ndef _token(plan: object) -> str:\n    return hashlib.sha256(\n        json.dumps(asdict(plan), sort_keys=True, separators=(",", ":")).encode("utf-8")\n    ).hexdigest()\n\n\n@dataclass(frozen=True)\nclass MaintenancePlan:\n    current: TargetLayout\n    requested: LayoutSpec\n    operation: str\n    sql: str\n    rewrites_data: bool\n    estimate: ResourceEstimate\n    capabilities: RuntimeCapabilities\n\n    @property\n    def token(self) -> str:\n        return _token(self)\n\n\ndef plan_maintenance(\n    spark: Any,\n    table: str,\n    metadata: Mapping[str, object],\n    schema: Any,\n    *,\n    operation: str,\n    capabilities: RuntimeCapabilities | None = None,\n    throughput_bytes_per_second: float | None = None,\n    capacity_units: float | None = None,\n    cost_per_cu_second: float | None = None,\n) -> MaintenancePlan:\n    """Read-only plan: configure changes keys; optimize does the physical work.\n\n    configure is limited to already-liquid tables. Enabling liquid on historical\n    unclustered tables is a migration, not an ordinary metadata maintenance task.\n    No FULL, ZORDER, arbitrary SQL, retention changes or VACUUM are exposed.\n    """\n    requested = validate_layout(metadata, schema, capabilities=capabilities)\n    current = inspect_target(spark, table)\n    if requested is None or current is None:\n        raise LayoutError("Maintenance requires an existing target and explicit layout")\n    if operation == "configure":\n        if (\n            requested.strategy != "liquid"\n            or not current.cluster_by_columns\n            or current.partition_columns\n        ):\n            raise MigrationRequired(\n                "Configuring historical non-liquid tables requires a migration"\n            )\n        _require(capabilities, "liquid_configure", spark)\n        old_types = {c.name: c.data_type for c in current.schema}\n        new_types = {c.name: c.data_type for c in schema_columns(schema)}\n        if any(old_types.get(c) != new_types.get(c) for c in requested.columns):\n            raise LayoutError("Cluster column types do not match the target schema")\n        keys = ", ".join(f"`{c}`" for c in requested.columns)\n        sql = f"ALTER TABLE {quote_table(table)} CLUSTER BY ({keys})"\n    elif operation == "optimize":\n        _compatible(current, requested, schema_columns(schema))\n        _require(capabilities, "optimize", spark)\n        sql = f"OPTIMIZE {quote_table(table)}"\n    else:\n        raise LayoutError(\n            "Only separate configure and optimize operations are supported"\n        )\n    return MaintenancePlan(\n        current,\n        requested,\n        operation,\n        sql,\n        operation == "optimize",\n        _estimate(\n            current,\n            operation == "optimize",\n            throughput_bytes_per_second,\n            capacity_units,\n            cost_per_cu_second,\n        ),\n        capabilities,\n    )\n\n\n@dataclass(frozen=True)\nclass MigrationPreview:\n    current: TargetLayout\n    requested: LayoutSpec\n    input_versions: tuple[tuple[str, int], ...]\n    expected_target_version: int\n    result_version: None\n    rewrites_data: bool\n    estimate: ResourceEstimate\n    consumer_requirements: tuple[str, ...]\n\n    @property\n    def token(self) -> str:\n        return _token(self)\n\n\ndef preview_migration(\n    current: TargetLayout,\n    requested: LayoutSpec,\n    *,\n    input_versions: Mapping[str, int],\n    capabilities: RuntimeCapabilities | None = None,\n    throughput_bytes_per_second: float | None = None,\n    capacity_units: float | None = None,\n    cost_per_cu_second: float | None = None,\n) -> MigrationPreview:\n    """Pure preview over an inspected target and caller-pinned inputs; no reads/writes.\n\n    Input versions are declared pins, not proof those versions remain available.\n    expected_target_version is the precondition; result_version is unknowable.\n    There is deliberately no assertion that config rollback rewinds physical data.\n    """\n    validate_layout(\n        requested.metadata(),\n        current.schema,\n        materialization=requested.materialization,\n        capabilities=capabilities,\n    )\n    if requested.materialization != "delta_table":\n        raise LayoutError(\n            "MLV migration requires a validated engine-specific target contract"\n        )\n    if not input_versions:\n        raise LayoutError("Migration preview requires pinned input versions")\n    for table, version in input_versions.items():\n        quote_table(table)\n        if type(version) is not int or version < 0:\n            raise LayoutError("Input versions must be explicit nonnegative integers")\n    if input_versions.get(current.table) != current.version:\n        raise LayoutError("Input pins must include the inspected target version")\n    partition = requested.columns if requested.strategy == "partitioned" else ()\n    cluster = requested.columns if requested.strategy == "liquid" else ()\n    changed = (partition, cluster) != (\n        current.partition_columns,\n        current.cluster_by_columns,\n    )\n    return MigrationPreview(\n        current,\n        requested,\n        tuple(sorted(input_versions.items())),\n        current.version,\n        None,\n        changed,\n        _estimate(\n            current,\n            changed,\n            throughput_bytes_per_second,\n            capacity_units,\n            cost_per_cu_second,\n        ),\n        (\n            "Define an isolated target identity/schema, fenced copy and atomic consumer cutover contract.",\n            "Prove both-direction exceptAll equivalence over pinned full logical content.",\n            "Validate SQL endpoint, Direct Lake/semantic model and MLV compatibility/refresh as applicable.",\n            "Retain input versions, old table/files, Delta history and historical batch/release references.",\n            "Record independent configuration, worker, data/release and physical-layout rollback choices.",\n            "Measure capacity, storage headroom, cost and retention window before execution.",\n        ),\n    )\n\n\n@dataclass(frozen=True)\nclass OperationApproval:\n    plan_token: str\n    operation_id: str\n    configuration_version: str\n    worker_version: str\n    data_release: str\n    rollback_reference: str\n    consumer_proof: str\n    preserve_history: bool = True\n\n\nclass WriterLease(Protocol):\n    """Held by the parent until exit, excluding all writers and retention jobs."""\n\n    def assert_owned(self) -> None:\n        """Raise if lease/fencing ownership has expired or changed."""\n\n\ndef _approval_check(\n    plan: MaintenancePlan | MigrationPreview, approval: OperationApproval | None\n) -> None:\n    if not isinstance(approval, OperationApproval) or approval.plan_token != plan.token:\n        raise OperationRejected(\n            "Explicit approval bound to this exact operation plan is required"\n        )\n    if approval.preserve_history is not True or not all(\n        isinstance(value, str) and value.strip()\n        for value in (\n            approval.operation_id,\n            approval.configuration_version,\n            approval.worker_version,\n            approval.data_release,\n            approval.rollback_reference,\n            approval.consumer_proof,\n        )\n    ):\n        raise OperationRejected(\n            "Approval requires history preservation and separate rollback/consumer proof"\n        )\n\n\ndef _snapshot_check(spark: Any, expected: TargetLayout) -> None:\n    actual = inspect_target(spark, expected.table)\n    if actual != expected:\n        raise OperationRejected(\n            "Target identity, version, schema or physical layout drift; regenerate preview"\n        )\n\n\n@dataclass(frozen=True)\nclass MaintenanceResult:\n    before: TargetLayout\n    after: TargetLayout\n    comparison: ContentComparison\n    approval: OperationApproval\n\n\ndef execute_maintenance(\n    spark: Any,\n    plan: MaintenancePlan,\n    *,\n    approval: OperationApproval | None,\n    ownership: AbstractContextManager[WriterLease],\n    capabilities: RuntimeCapabilities | None = None,\n) -> MaintenanceResult:\n    """Execute one fenced approved command and prove pinned full-content equality.\n\n    Approval represents externally obtained consent, not authorization inferred\n    from metadata. Failed/uncertain operations are never retried or rolled back.\n    Parent persists operation/result records; no cross-store atomicity is claimed.\n    """\n    _approval_check(plan, approval)\n    if capabilities != plan.capabilities:\n        raise OperationRejected("Runtime evidence must match the approved plan")\n    # Regenerate SQL instead of executing a caller-supplied SQL string.\n    regenerated = plan_maintenance(\n        spark,\n        plan.current.table,\n        plan.requested.metadata(),\n        plan.current.schema,\n        operation=plan.operation,\n        capabilities=capabilities,\n    )\n    if regenerated.sql != plan.sql:\n        raise OperationRejected(\n            "Maintenance SQL does not match the validated operation"\n        )\n    with ownership as lease:\n        lease.assert_owned()\n        _snapshot_check(spark, plan.current)\n        try:\n            spark.sql(regenerated.sql)\n            lease.assert_owned()\n            after = inspect_target(spark, plan.current.table)\n            if (\n                after is None\n                or after.table_id != plan.current.table_id\n                or after.version < plan.current.version\n            ):\n                raise LayoutError(\n                    "Maintenance changed target identity or lost Delta history"\n                )\n            _compatible(after, plan.requested, plan.current.schema)\n            before_frame = (\n                spark.read.format("delta")\n                .option("versionAsOf", plan.current.version)\n                .table(plan.current.table)\n            )\n            after_frame = (\n                spark.read.format("delta")\n                .option("versionAsOf", after.version)\n                .table(plan.current.table)\n            )\n            comparison = compare_logical_content(before_frame, after_frame)\n            if not comparison.equivalent:\n                raise LayoutError("Full logical content changed during maintenance")\n            lease.assert_owned()\n            _snapshot_check(spark, after)\n        except Exception as error:\n            raise OperationIncomplete(\n                f"Operation {approval.operation_id} incomplete; reconcile Delta history and "\n                f"lease against version {plan.current.version}; no automatic retry/rollback"\n            ) from error\n        return MaintenanceResult(plan.current, after, comparison, approval)\n\n\ndef execute_migration(\n    spark: Any,\n    preview: MigrationPreview,\n    *,\n    approval: OperationApproval | None,\n    ownership: AbstractContextManager[WriterLease],\n) -> None:\n    """Explicitly gated, intentionally NOT an implementation of physical migration."""\n    _approval_check(preview, approval)\n    with ownership as lease:\n        lease.assert_owned()\n        _snapshot_check(spark, preview.current)\n        raise MigrationRequired(\n            "Physical migration is not implemented: supply an isolated target contract, "\n            "fenced copy/cutover, retained history and downstream consumer proof; no writes performed"\n        )\n', 'gold_rules': '"""Generated generic case adapter; no domain-specific enrichment was declared."""\n\ndef _department(frame, source, config):\n    return frame\n\ndef dimension_lookup_keys(frame, spec):\n    return list(spec["keys"])\n', 'gold': '#!/usr/bin/env python3\n# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n"""Source-backed Gold runtime embedded in the generated Fabric notebooks.\n\nThe shipped contracts are synthetic-demo contracts, not production key,\nprivacy, accounting-sign or business-publication certification.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom collections.abc import Callable, Mapping\nfrom copy import deepcopy\nfrom datetime import UTC, date, datetime\nfrom functools import reduce\nfrom pathlib import PurePosixPath\nfrom types import SimpleNamespace\nfrom typing import Any, TypedDict\n\nMAX_GOLD_MANIFEST_BYTES = 16 * 1024 * 1024\nGOLD_TASK_SCHEMA = (\n    "gold_release_id string, input_sha256 string, object_name string, task_id string, "\n    "attempt_id string, status string, result_json string, event_utc timestamp"\n)\n\n\nclass GoldTableReference(TypedDict):\n    table: str\n    location: str\n    version: int\n\n\nclass GoldPreparationReference(GoldTableReference):\n    gold_release_id: str\n    input_sha256: str\n\n\nclass GoldResultReference(GoldTableReference):\n    gold_release_id: str\n    task_id: str\n    attempt_id: str\n    result_sha256: str\n    fence: int\n    job_id: str\n\n\nclass RegisteredGoldTransformation(TypedDict):\n    """A reviewed, packaged adapter; metadata cannot construct the callable."""\n\n    kind: str\n    version: str\n    definition_sha256: str\n    implementation_sha256: str\n    approval_id: str\n    execute: Callable[[dict[str, Any]], Any]\n\n\nclass GoldAttempt(TypedDict):\n    attempt_id: str\n    fence: int\n    job_id: str\n\n\ndef _gold_json(value: Any) -> str:\n    text = json.dumps(\n        value, sort_keys=True, ensure_ascii=True, separators=(",", ":"), allow_nan=False\n    )\n    if len(text.encode("ascii")) > MAX_GOLD_MANIFEST_BYTES:\n        raise ValueError("Gold metadata exceeds its supported manifest bound")\n    return text\n\n\ndef _gold_digest(value: Any) -> str:\n    return hashlib.sha256(_gold_json(value).encode("ascii")).hexdigest()\n\n\ndef _gold_token(value: Any, label: str, pattern: str = r"[a-z][a-z0-9_]{0,127}") -> str:\n    if not isinstance(value, str) or re.fullmatch(pattern, value) is None:\n        raise ValueError(f"Invalid {label}")\n    return value\n\n\ndef _gold_reference(reference: Mapping[str, Any], expected_table: str) -> None:\n    if (\n        not isinstance(reference, Mapping)\n        or reference.get("table") != expected_table\n        or type(reference.get("version")) is not int\n        or reference["version"] < 0\n        or not isinstance(reference.get("location"), str)\n        or not 1 <= len(reference["location"]) <= 4096\n        or "?" in reference["location"]\n        or "#" in reference["location"]\n    ):\n        raise RuntimeError("Invalid or rebound pinned Gold table reference")\n    for name in expected_table.split("."):\n        _gold_token(name, "pinned table namespace")\n    if len(expected_table.split(".")) != 3:\n        raise RuntimeError("Pinned Gold references require a three-part table name")\n\n\ndef gold_config_from_manifest(\n    manifest: Mapping[str, Any],\n    *,\n    snapshot_sha256: str,\n    code_sha256: str,\n    worker_id: str = "gold-worker",\n    lakehouse: str | None = None,\n) -> dict[str, Any]:\n    """Reconstruct Gold solely from an executable pinned run snapshot.\n\n    Call after the shared ``read_run_manifest`` boundary. No source registry,\n    seed file, active-version pointer, query execution or approval is performed\n    here. The bootstrap\'s legacy mapping hash remains a business-rule reference;\n    worker/configuration/task identities are recorded separately.\n    """\n    document = json.loads(_gold_json(manifest))\n    _gold_token(snapshot_sha256, "run snapshot hash", r"[0-9a-f]{64}")\n    if _gold_digest(document) != snapshot_sha256:\n        raise RuntimeError("Gold run snapshot hash differs from its pinned reference")\n    if (\n        document.get("format_version") != 1\n        or document.get("can_execute") is not True\n        or document.get("blockers") != []\n    ):\n        raise RuntimeError("A preview or blocked manifest cannot execute Gold")\n    configuration = document["configuration"]\n    if (\n        _gold_digest(configuration) != document.get("definition_sha256")\n        or configuration["version_id"] != document["version_id"]\n    ):\n        raise RuntimeError("Gold configuration differs from the pinned run version")\n    extension = configuration["extensions"]\n    base = deepcopy(extension["gold_contract"])\n    if lakehouse is None:\n        # Pure compatibility callers derive from the pinned contract; deployed workers\n        # always pass the actual attached Lakehouse and validate it below.\n        lakehouse = base["target_lakehouse"]\n    if (\n        extension.get("synthetic_only") is not True\n        or base.get("execution_mode") != "synthetic_demo"\n    ):\n        raise ValueError("Production Gold remains gated; metadata is not certification")\n    if base["target_lakehouse"] != lakehouse:\n        raise RuntimeError("Gold metadata differs from the parent Lakehouse binding")\n    _gold_token(base["schema"], "Gold schema")\n    _gold_token(lakehouse, "Gold Lakehouse")\n    workers = [w for w in configuration["workers"] if w["id"] == worker_id]\n    if (\n        len(workers) != 1\n        or workers[0]["lakehouse"] != lakehouse\n        or workers[0]["code_sha256"] != code_sha256\n    ):\n        raise RuntimeError(\n            "Gold worker code or Lakehouse differs from the pinned version"\n        )\n    _gold_token(code_sha256, "Gold worker hash", r"[0-9a-f]{64}")\n    transforms = {}\n    for transform in configuration["transformations"]:\n        if (\n            transform["object_id"] in transforms\n            or _gold_digest(transform["definition"]) != transform["definition_sha256"]\n        ):\n            raise RuntimeError(\n                "Gold transformation is ambiguous or its definition hash changed"\n            )\n        transforms[transform["object_id"]] = transform\n    objects = {obj["id"]: obj for obj in configuration["objects"]}\n    tasks = document["tasks"]\n    if len({task["task_id"] for task in tasks}) != len(tasks):\n        raise RuntimeError("Duplicate pinned Gold task identity")\n    by_task = {task["task_id"]: task for task in tasks}\n    selected = [task for task in tasks if task["layer"] == "gold"]\n    if not selected:\n        raise RuntimeError("No Gold work was selected")\n    # The shipped release is complete, not a partial dimension/fact publication.\n    membership = {\n        *(spec["name"] for spec in base["dimensions"]),\n        *(spec["name"] for spec in base["facts"]),\n        "dim_date",\n        "dim_fiscal_period",\n    }\n    if {objects[task["object_id"]]["name"] for task in selected} != membership:\n        raise RuntimeError(\n            "Gold release membership is partial or includes a separately gated MLV stage"\n        )\n    sources = []\n    tables = {}\n    source_codes = {spec["source"] for spec in [*base["dimensions"], *base["facts"]]}\n    for code in sorted(\n        source_codes,\n        key=lambda value: list(\n            dict.fromkeys(feed["source_code"] for feed in configuration["feeds"])\n        ).index(value),\n    ):\n        feeds = [\n            feed\n            for feed in configuration["feeds"]\n            if feed["source_code"] == code and feed["enabled"]\n        ]\n        if len(feeds) != 1 or feeds[0]["authority"] != "authoritative":\n            raise RuntimeError("Gold source authority is missing or ambiguous")\n        feed = feeds[0]\n        source_tables = []\n        silver = sorted(\n            [\n                obj\n                for obj in configuration["objects"]\n                if obj["feed_id"] == feed["id"]\n                and obj["layer"] == "silver"\n                and obj["enabled"]\n            ],\n            key=lambda obj: (obj["execution_priority"], obj["id"]),\n        )\n        if (\n            not silver\n            or len({obj["lakehouse"] for obj in silver}) != 1\n            or any(obj["target_schema"] != feed["silver_schema"] for obj in silver)\n        ):\n            raise RuntimeError(\n                "Gold requires explicit compatible Silver source objects"\n            )\n        for obj in silver:\n            table = deepcopy(obj["contract"])\n            table.update(\n                name=obj["name"], keys=obj["business_keys"], columns=obj["columns"]\n            )\n            for field, default in (\n                ("foreign_keys", []),\n                ("allow_empty", False),\n                ("change_hash_exclusions", []),\n            ):\n                table.setdefault(field, default)\n            policy = extension["hash_policies"].get(obj["id"])\n            if policy is None:\n                raise RuntimeError("Gold source has no pinned business hash policy")\n            table["hash_policy"] = deepcopy(policy)\n            table["rule_specs"] = [\n                {k: v for k, v in rule.items() if k != "object_id"}\n                for rule in configuration["rules"]\n                if rule["object_id"] == obj["id"]\n                and rule["handler"] != "legacy.source_business_checks"\n            ]\n            source_tables.append(table)\n            tables[code, obj["name"]] = table\n        business_hash = feed.get("legacy_contract_hash")\n        if business_hash is None:\n            # Same fallback as source_runtime_config; never a new key/value hash.\n            business_hash = _gold_digest(\n                {\n                    "source_code": code,\n                    "silver_schema": feed["silver_schema"],\n                    "tables": source_tables,\n                }\n            )\n        _gold_token(\n            business_hash, "pinned source business contract hash", r"[0-9a-f]{64}"\n        )\n        sources.append(\n            {\n                "source_code": code,\n                "source_instance": feed["source_instance"],\n                "source_feed_id": feed["id"],\n                "source_type": feed["source_type"],\n                "description_slug": feed["description_slug"],\n                "lakehouse": silver[0]["lakehouse"],\n                "silver_schema": feed["silver_schema"],\n                "quarantine_schema": feed["quarantine_schema"],\n                "bronze_root": feed.get("legacy_bronze_root"),\n                "contract_hash": business_hash,\n                "synthetic_only": True,\n                "tables": source_tables,\n            }\n        )\n    if len({source["lakehouse"] for source in sources}) != 1:\n        raise RuntimeError("Gold requires the configured shared Silver Lakehouse")\n    by_feed = {source["source_feed_id"]: source for source in sources}\n    by_source = {source["source_code"]: source for source in sources}\n    definitions = {}\n    dimensions, facts = [], []\n    for task in selected:\n        obj = objects[task["object_id"]]\n        if (\n            not obj["enabled"]\n            or obj["worker_id"] != worker_id\n            or task["worker_id"] != worker_id\n            or obj["lakehouse"] != lakehouse\n            or obj["target_schema"] != base["schema"]\n        ):\n            raise RuntimeError("Disabled or incorrectly bound Gold object")\n        if obj.get("layout") is not None and globals().get("_MEDALLION_LAYOUT") is None:\n            raise RuntimeError(\n                "Gold layout changes require the separate approved layout adapter"\n            )\n        name, kind = obj["name"], obj["materialization"]\n        _gold_token(name, "Gold object name")\n        if name in definitions or name in base["blocked"]:\n            raise RuntimeError("Duplicate or unavailable Gold output")\n        transform = transforms.get(obj["id"])\n        if transform is None:\n            raise RuntimeError("Gold object has no versioned transformation")\n        spec = deepcopy(obj["contract"])\n        if transform["kind"] == "pattern" and transform["definition"] != {\n            "pattern": kind,\n            "contract": spec,\n        }:\n            raise RuntimeError(\n                "Gold pattern definition differs from its explicit object contract"\n            )\n        if kind not in {"scd2_dimension", "atomic_fact", "snapshot_fact", "calendar"}:\n            raise RuntimeError(\n                "Unsupported Gold materialization; MLV lifecycle is separate"\n            )\n        if kind != "calendar":\n            if (\n                spec.get("name") != name\n                or (spec.get("source"), spec.get("table")) not in tables\n            ):\n                raise RuntimeError("Gold object has no explicit source/grain contract")\n            table = tables[spec["source"], spec["table"]]\n            fields = {column["name"] for column in table["columns"]}\n            used = (\n                set(spec["keys"])\n                | set(spec["attributes"])\n                | set(spec.get("measures", []))\n            )\n            if not used - {"_department_id"} <= fields:\n                raise RuntimeError(\n                    "Gold keys/attributes reference unknown source columns"\n                )\n            if kind == "scd2_dimension":\n                if (\n                    spec.get("source_instance")\n                    != by_source[spec["source"]]["source_instance"]\n                ):\n                    raise RuntimeError(\n                        "Gold dimension/source instance identity is inconsistent"\n                    )\n                dimensions.append(spec)\n            else:\n                expected_keys = list(table["keys"])\n                if spec["period_field"] and spec["period_field"] not in expected_keys:\n                    expected_keys.append(spec["period_field"])\n                if spec["keys"] != expected_keys or spec["snapshot"] != (\n                    kind == "snapshot_fact"\n                ):\n                    raise RuntimeError(\n                        "Gold fact grain differs from its source contract"\n                    )\n                if not spec.get("grain"):\n                    raise RuntimeError(\n                        "Gold facts require an explicitly authored grain"\n                    )\n                facts.append(spec)\n        target = f"{lakehouse}.{base[\'schema\']}.{name}"\n        if task["target_path"] != target:\n            raise RuntimeError("Gold task target differs from its object contract")\n        dependencies = []\n        source_inputs = []\n        for edge in task["dependencies"]:\n            upstream = by_task.get(edge["task_id"])\n            if upstream is None:\n                raise RuntimeError("Gold task has a missing pinned dependency")\n            parent = objects[upstream["object_id"]]\n            if parent["layer"] == "gold":\n                if edge["condition"] != "succeeded":\n                    raise RuntimeError("Gold dependencies require object success")\n                dependencies.append(parent["name"])\n            elif parent["layer"] != "silver" or edge["condition"] != "source_committed":\n                raise RuntimeError("Gold requires committed source dependencies")\n            elif not parent["enabled"]:\n                raise RuntimeError("A required Gold source dependency is disabled")\n            else:\n                source = by_feed.get(parent["feed_id"])\n                if (\n                    source is None\n                    or (source["source_code"], parent["name"]) not in tables\n                ):\n                    raise RuntimeError(\n                        "Gold source dependency has no pinned source object"\n                    )\n                source_inputs.append(\n                    {"source": source["source_code"], "table": parent["name"]}\n                )\n        if kind != "calendar":\n            primary = {"source": spec["source"], "table": spec["table"]}\n            if primary not in source_inputs:\n                raise RuntimeError(\n                    "Gold definition is missing its declared committed source dependency"\n                )\n            if transform["kind"] == "pattern" and source_inputs != [primary]:\n                raise RuntimeError(\n                    "Additional Gold source joins require an explicitly registered transformation"\n                )\n        elif transform["kind"] == "pattern" and source_inputs:\n            raise RuntimeError("Calendar patterns consume only declared fact outputs")\n        definitions[name] = {\n            "task_id": task["task_id"],\n            "object_id": obj["id"],\n            "name": name,\n            "kind": kind,\n            "target": target,\n            "dependencies": dependencies,\n            "source_inputs": source_inputs,\n            "spec": spec,\n            "transformation": deepcopy(transform),\n        }\n    dimension_names = {spec["name"] for spec in dimensions}\n    fact_names = {spec["name"] for spec in facts}\n    for name, task in definitions.items():\n        required = (\n            dimension_names\n            if name in fact_names\n            else fact_names\n            if task["kind"] == "calendar"\n            else set()\n        )\n        if not required <= set(task["dependencies"]):\n            raise RuntimeError("Gold temporal/fact dependencies are incomplete")\n    base.update(\n        sources=sources,\n        dimensions=dimensions,\n        facts=facts,\n        contract_hash=extension["legacy_gold_contract_hash"],\n        runtime_sha256=code_sha256,\n        # MLVs are retained definitions, never executed by this object DAG.\n        mlvs={\n            obj["name"]: obj["contract"]["mlv_query"]\n            for obj in configuration["objects"]\n            if obj["layer"] == "gold" and obj["materialization"] == "mlv"\n        },\n        _metadata={\n            "run_id": document["run_id"],\n            "version_id": document["version_id"],\n            "snapshot_sha256": snapshot_sha256,\n            "configuration_sha256": document["definition_sha256"],\n            "worker_id": worker_id,\n            "notebook_name": workers[0]["notebook_name"],\n            "code_sha256": code_sha256,\n            "tasks": definitions,\n            "layouts": {\n                objects[task["object_id"]]["name"]: objects[task["object_id"]].get(\n                    "layout"\n                )\n                for task in selected\n            },\n            "layout_runtime": extension.get("layout_runtime"),\n            "rules": [\n                rule\n                for rule in configuration["rules"]\n                if rule["object_id"] in {task["object_id"] for task in selected}\n            ],\n            "constraints": [\n                constraint\n                for constraint in configuration.get("constraints", [])\n                if constraint["object_id"] in {task["object_id"] for task in selected}\n            ],\n        },\n    )\n    rules = base["_metadata"]["rules"]\n    if rules:\n        handlers = globals().get("_MEDALLION_RULE_HANDLERS")\n        if handlers is None or any(\n            rule["action"] not in {"fail", "warn"} for rule in rules\n        ):\n            raise RuntimeError(\n                "Gold DQ requires the registered runtime validator; row-drop policies are unsupported"\n            )\n        for rule in rules:\n            handlers["validate_rule_specs"](\n                [{key: value for key, value in rule.items() if key != "object_id"}],\n                scope="gold",\n            )\n    for constraint in base["_metadata"]["constraints"]:\n        if constraint["enforcement"] != "runtime":\n            raise RuntimeError(\n                "Engine-enforced Gold constraints require separate capability proof"\n            )\n    _gold_token(base["contract_hash"], "legacy Gold rule hash", r"[0-9a-f]{64}")\n    _validate_gold_tasks(definitions)\n    return base\n\n\ndef _validate_gold_tasks(tasks: Mapping[str, dict[str, Any]]) -> None:\n    if not tasks or len(tasks) > 5000:\n        raise RuntimeError("Gold task membership must be nonempty and bounded")\n    identities, targets = set(), set()\n    pending = set(tasks)\n    for name, task in tasks.items():\n        transformation = task["transformation"]\n        if (\n            task["kind"]\n            not in {"scd2_dimension", "atomic_fact", "snapshot_fact", "calendar"}\n            or task["name"] != name\n            or transformation["kind"] not in {"pattern", "sql", "notebook"}\n            or _gold_digest(transformation["definition"])\n            != transformation["definition_sha256"]\n            or not isinstance(transformation["version"], str)\n            or not transformation["version"]\n        ):\n            raise RuntimeError(\n                "Unsupported or changed Gold transformation/task contract"\n            )\n        if transformation["kind"] == "pattern" and transformation["definition"] != {\n            "pattern": task["kind"],\n            "contract": task["spec"],\n        }:\n            raise RuntimeError(\n                "Gold pattern does not match its declared implementation contract"\n            )\n        if task["task_id"] in identities or task["target"] in targets:\n            raise RuntimeError(\n                "Gold DAG has duplicate identities or conflicting target writers"\n            )\n        identities.add(task["task_id"])\n        targets.add(task["target"])\n        if not set(task["dependencies"]) <= set(tasks) or name in task["dependencies"]:\n            raise RuntimeError("Gold DAG has a missing dependency or cycle")\n        if len(task["dependencies"]) != len(set(task["dependencies"])):\n            raise RuntimeError("Gold DAG contains duplicate dependency edges")\n    while pending:\n        ready = {\n            name for name in pending if not set(tasks[name]["dependencies"]) & pending\n        }\n        if not ready:\n            raise RuntimeError("Gold DAG contains a dependency cycle")\n        pending -= ready\n\n\ndef build_gold_dag(\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n    *,\n    manifest_path: str,\n    dispatch_path: str,\n    dispatch_sha256: str,\n    lakehouse: str,\n    concurrency: int = 2,\n    timeout_seconds: int = 3600,\n    cell_timeout_seconds: int = 900,\n) -> dict[str, Any]:\n    """Build object-only native DAG data; the parent validates/runs it natively.\n\n    Preparation and finalization are separate parent stages. A worker reloads the\n    run manifest, layer dispatch and preparation reference, then independently\n    reopens source and dependency Delta versions. No frame payloads are passed.\n    """\n    if type(concurrency) is not int or not 1 <= concurrency <= 16:\n        raise ValueError("Gold DAG concurrency must be between 1 and 16")\n    if any(\n        type(value) is not int or not 1 <= value <= 86400\n        for value in (timeout_seconds, cell_timeout_seconds)\n    ):\n        raise ValueError("Gold DAG timeouts must be bounded positive seconds")\n    if config["target_lakehouse"] != lakehouse:\n        raise RuntimeError("Gold DAG parent and child Lakehouse bindings differ")\n    metadata = config["_metadata"]\n    for path in (manifest_path, dispatch_path):\n        if (\n            not isinstance(path, str)\n            or not path.startswith(f"Files/orchestration/{metadata[\'run_id\']}/")\n            or any(part in {"", ".", ".."} for part in path.split("/"))\n        ):\n            raise ValueError("Gold DAG references must stay in the pinned run folder")\n    _gold_token(dispatch_sha256, "Gold dispatch hash", r"[0-9a-f]{64}")\n    _gold_reference(preparation, _gold_target(config, "ctl_gold_release"))\n    _gold_token(preparation["input_sha256"], "Gold preparation hash", r"[0-9a-f]{64}")\n    tasks = metadata["tasks"]\n    _validate_gold_tasks(tasks)\n    names = {\n        name: "task_" + hashlib.sha256(task["task_id"].encode()).hexdigest()\n        for name, task in tasks.items()\n    }\n    activities = [\n        {\n            "name": names[name],\n            "path": metadata["notebook_name"],\n            "timeoutPerCellInSeconds": cell_timeout_seconds,\n            "retry": 0,\n            "args": {\n                "run_id": metadata["run_id"],\n                "version_id": metadata["version_id"],\n                "manifest_path": manifest_path,\n                "snapshot_sha256": metadata["snapshot_sha256"],\n                "dispatch_path": dispatch_path,\n                "dispatch_sha256": dispatch_sha256,\n                "task_id": task["task_id"],\n                "worker_id": metadata["worker_id"],\n                "gold_preparation_json": _gold_json(preparation),\n            },\n            "dependencies": [names[parent] for parent in task["dependencies"]],\n        }\n        for name, task in tasks.items()\n    ]\n    return {\n        "activities": activities,\n        "timeoutInSeconds": timeout_seconds,\n        "concurrency": concurrency,\n    }\n\n\ndef gold_request(config: dict[str, Any], source_runs_json: str, release_id: str):\n    if config["execution_mode"] != "synthetic_demo" or not all(\n        source["synthetic_only"] is True for source in config["sources"]\n    ):\n        raise ValueError(\n            "Production Gold is gated pending certified keys, mappings and source policies"\n        )\n    if not isinstance(release_id, str) or not re.fullmatch(\n        r"[A-Za-z0-9_-]{1,128}", release_id\n    ):\n        raise ValueError("An explicit bounded gold_release_id is required")\n    for name in (config["target_lakehouse"], config["schema"]):\n        if not re.fullmatch(r"[a-z][a-z0-9_]*", name):\n            raise ValueError("Invalid Gold namespace")\n    runs = json.loads(source_runs_json)\n    if not isinstance(runs, dict) or set(runs) != {\n        s["source_code"] for s in config["sources"]\n    }:\n        raise ValueError(\n            "Select exactly one explicit committed run for every Gold source"\n        )\n    dates = set()\n    for value in runs.values():\n        if not isinstance(value, dict) or set(value) != {\n            "pipeline_run_id",\n            "processing_date",\n        }:\n            raise ValueError(\n                "Source selections require pipeline_run_id and processing_date only"\n            )\n        if not isinstance(value["pipeline_run_id"], str) or not re.fullmatch(\n            r"[A-Za-z0-9_-]{1,128}", value["pipeline_run_id"]\n        ):\n            raise ValueError("Invalid selected source run")\n        if not isinstance(value["processing_date"], str) or not re.fullmatch(\n            r"\\d{4}-\\d\\d-\\d\\d", value["processing_date"]\n        ):\n            raise ValueError("Source processing dates must be ISO dates")\n        dates.add(date.fromisoformat(value["processing_date"]))\n    if len(dates) != 1:\n        raise ValueError(\n            "Demo Gold requires a common extraction date; mixed cutoffs need an approved policy"\n        )\n    return runs, dates.pop()\n\n\ndef _gold_target(config: dict[str, Any], name: str) -> str:\n    if not re.fullmatch(r"[a-z][a-z0-9_]*", name):\n        raise ValueError("Invalid Gold table identifier")\n    return f"{config[\'target_lakehouse\']}.{config[\'schema\']}.{name}"\n\n\ndef _gold_delta(spark, table: str):\n    from delta.tables import DeltaTable\n\n    location = (\n        spark.sql(f"DESCRIBE DETAIL {table}").select("location").first()["location"]\n    )\n    return DeltaTable.forPath(spark, location)\n\n\ndef _gold_hash(*columns):\n    from pyspark.sql import functions as F\n\n    return F.sha2(F.to_json(F.struct(*columns), {"ignoreNullFields": "false"}), 256)\n\n\ndef _gold_sk(column):\n    from pyspark.sql import functions as F\n\n    return (F.conv(F.substring(column, 1, 15), 16, 10).cast("long") + 1).cast("long")\n\n\ndef _assert_unique(frame, keys: list[str], label: str) -> None:\n    from pyspark.sql import functions as F\n\n    if (\n        frame.filter(reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys]))\n        .limit(1)\n        .count()\n    ):\n        raise RuntimeError(f"{label}: null key")\n    if frame.groupBy(*keys).count().filter("count > 1").limit(1).count():\n        raise RuntimeError(f"{label}: duplicate key")\n\n\ndef _latest(frame, keys: list[str]):\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n\n    return (\n        frame.withColumn(\n            "_gold_latest",\n            F.row_number().over(\n                Window.partitionBy(*keys).orderBy(F.col("record_version").desc())\n            ),\n        )\n        .filter("_gold_latest = 1")\n        .drop("_gold_latest")\n    )\n\n\ndef _gold_adapter():\n    adapter = globals().get("_CASE_GOLD")\n    if adapter is None and "__file__" in globals():\n        from pathlib import Path\n\n        from fabric_medallion_deployer.use_cases import (\n            load_case_extension,\n            load_use_case,\n        )\n\n        adapter = load_case_extension(\n            load_use_case(Path(__file__).resolve().parents[2]), "gold_rules"\n        )\n        globals()["_CASE_GOLD"] = adapter\n    if adapter is None:\n        raise RuntimeError("The reviewed selected case Gold adapter must be injected")\n    return adapter\n\n\ndef _department(frame, source: str, config: dict[str, Any]):\n    adapter = _gold_adapter()\n    function = getattr(adapter, "_department", None)\n    return function(frame, source, config) if function else frame\n\n\ndef build_scd2(frame, spec: dict[str, Any]):\n    """Derive deterministic, gap-safe observed-time versions without backdating."""\n    from pyspark.sql import Window\n    from pyspark.sql import functions as F\n    from pyspark.sql import types as T\n\n    keys, attributes = spec["keys"], spec["attributes"]\n    selected = (\n        frame.filter(reduce(lambda a, b: a & b, [F.col(k).isNotNull() for k in keys]))\n        .select(\n            *keys,\n            *attributes,\n            F.col("record_observed_utc").alias("effective_from"),\n            F.col("record_id").alias("source_record_id"),\n        )\n        .withColumn(\n            "durable_key",\n            _gold_hash(\n                F.lit(spec.get("source_instance", spec["name"])).alias(\n                    "source_instance"\n                ),\n                *[F.col(k) for k in keys],\n            ),\n        )\n        .withColumn("attribute_hash", _gold_hash(*[F.col(k) for k in attributes]))\n    )\n    conflicts = (\n        selected.groupBy("durable_key", "effective_from")\n        .agg(F.countDistinct("attribute_hash").alias("states"))\n        .filter("states > 1")\n    )\n    if conflicts.limit(1).count():\n        raise RuntimeError(\n            f"{spec[\'name\']}: conflicting dimension states at the same observation time"\n        )\n    selected = selected.groupBy(\n        *keys, *attributes, "durable_key", "attribute_hash", "effective_from"\n    ).agg(F.min("source_record_id").alias("source_record_id"))\n    window = Window.partitionBy("durable_key").orderBy("effective_from")\n    changes = (\n        selected.withColumn("_previous", F.lag("attribute_hash").over(window))\n        .filter(~F.col("attribute_hash").eqNullSafe(F.col("_previous")))\n        .drop("_previous")\n    )\n    output = (\n        changes.withColumn("effective_to", F.lead("effective_from").over(window))\n        .withColumn("is_current", F.col("effective_to").isNull())\n        .withColumn(\n            "dimension_sk",\n            _gold_sk(\n                _gold_hash(\n                    F.lit(spec["name"]).alias("dimension"),\n                    F.col("durable_key"),\n                    F.col("effective_from"),\n                )\n            ),\n        )\n        .withColumn("time_basis", F.lit("source_observed_not_business_effective"))\n        .withColumn(\n            "semantic_basis",\n            F.lit(spec.get("semantic_basis", "SYNTHETIC_SOURCE_MEMBER")),\n        )\n    )\n    _assert_unique(output, ["dimension_sk"], spec["name"])\n    unknown = output.sparkSession.createDataFrame(\n        [\n            {\n                field.name: (\n                    0\n                    if field.name == "dimension_sk"\n                    else "UNKNOWN"\n                    if field.name in {"durable_key", "attribute_hash", "time_basis"}\n                    else True\n                    if field.name == "is_current"\n                    else None\n                )\n                for field in output.schema\n            }\n        ],\n        T.StructType([T.StructField(f.name, f.dataType, True) for f in output.schema]),\n    )\n    return output.unionByName(unknown)\n\n\ndef attach_dimension(frame, dimension, spec: dict[str, Any], output_key: str):\n    from pyspark.sql import functions as F\n\n    lookup_keys = getattr(_gold_adapter(), "dimension_lookup_keys", None)\n    keys = (\n        lookup_keys(frame, spec)\n        if lookup_keys\n        else spec.get("lookup_keys", spec["keys"])\n    )\n    if not set(keys) <= set(frame.columns):\n        return frame.withColumn(output_key, F.lit(0).cast("long"))\n    prefix = f"_lookup_{output_key}_"\n    lookup = dimension.filter("dimension_sk <> 0").select(\n        *[F.col(k).alias(f"{prefix}{i}") for i, k in enumerate(keys)],\n        F.col("dimension_sk").alias(output_key),\n        F.col("effective_from").alias(prefix + "start"),\n        F.col("effective_to").alias(prefix + "end"),\n    )\n    condition = (\n        reduce(\n            lambda a, b: a & b,\n            [F.col(k).eqNullSafe(F.col(f"{prefix}{i}")) for i, k in enumerate(keys)],\n        )\n        & (F.col("record_observed_utc") >= F.col(prefix + "start"))\n        & (\n            F.col(prefix + "end").isNull()\n            | (F.col("record_observed_utc") < F.col(prefix + "end"))\n        )\n    )\n    output = (\n        frame.join(lookup, condition, "left")\n        .withColumn(output_key, F.coalesce(F.col(output_key), F.lit(0)).cast("long"))\n        .drop(*[n for n in lookup.columns if n != output_key])\n    )\n    return output\n\n\ndef source_age_band(column):\n    function = getattr(_gold_adapter(), "source_age_band", None)\n    if function is None:\n        raise RuntimeError("Source age bands require an explicit case rule")\n    return function(column)\n\n\ndef build_fact(\n    frame,\n    spec: dict[str, Any],\n    dimensions: dict[str, Any],\n    config: dict[str, Any],\n    as_of: date,\n    release_id: str,\n):\n    from pyspark.sql import functions as F\n\n    if not spec["snapshot"]:\n        frame = _latest(frame, spec["keys"])\n    _assert_unique(frame, spec["keys"], spec["name"])\n    frame = _department(frame, spec["source"], config)\n    dimension_specs = {d["name"]: d for d in config["dimensions"]}\n    adapter = _gold_adapter()\n    attach_identity = getattr(adapter, "attach_supplier_identity", None)\n    if attach_identity:\n        frame = attach_identity(frame, config, assert_unique=_assert_unique)\n    output_key = getattr(adapter, "dimension_output_key", lambda name: name + "_sk")\n    for name, dimension in dimensions.items():\n        frame = attach_dimension(\n            frame, dimension, dimension_specs[name], output_key(name)\n        )\n    event_adapter = getattr(adapter, "fact_event_date", None)\n    if event_adapter:\n        event_date = event_adapter(frame, spec, as_of)\n    elif spec["snapshot"]:\n        event_date = F.lit(as_of)\n    elif not spec["date_field"]:\n        raise ValueError(f"{spec[\'name\']}: atomic facts require a source event date")\n    else:\n        event_date = (\n            F.to_date(F.col(spec["date_field"]).cast("string"), spec["date_format"])\n            if spec.get("date_format")\n            else F.to_date(F.col(spec["date_field"]))\n        )\n    period_field = spec.get("posting_period_field") or spec["period_field"]\n    period = (\n        F.col(period_field).cast("int")\n        if period_field\n        else F.date_format(event_date, "yyyyMM").cast("int")\n    )\n    date_roles = {\n        name: (\n            F.to_date(F.col(name))\n            if kind == "timestamp"\n            else F.to_date(F.col(name).cast("string"), "yyyyMMdd")\n        )\n        for name, kind in spec.get("date_roles", {}).items()\n    }\n    date_adapter = getattr(adapter, "fact_date_roles", None)\n    if date_adapter:\n        date_roles = date_adapter(spec)\n    instance = next(\n        s["source_instance"]\n        for s in config["sources"]\n        if s["source_code"] == spec["source"]\n    )\n    identity = _gold_hash(\n        F.lit(spec["source"]).alias("source"),\n        F.lit(instance).alias("source_instance"),\n        F.lit(spec["table"]).alias("source_table"),\n        *[F.col(k) for k in spec["keys"]],\n        *([F.lit(as_of).alias("as_of_date")] if spec["snapshot"] else []),\n    )\n    dimension_columns = [n for n in frame.columns if n.endswith("_sk")]\n    attrs = [n for n in spec["attributes"] if n not in dimension_columns]\n    source_keys = F.to_json(F.struct(*[F.col(k) for k in spec["keys"]]))\n    output = frame.select(\n        _gold_sk(identity).alias("fact_sk"),\n        identity.alias("business_event_id"),\n        F.lit(spec["source"]).alias("source_system"),\n        F.lit(\n            next(\n                s["source_instance"]\n                for s in config["sources"]\n                if s["source_code"] == spec["source"]\n            )\n        ).alias("source_instance_key"),\n        F.lit(spec["table"]).alias("source_table"),\n        source_keys.alias("source_business_key"),\n        F.col("record_id").alias("source_record_id"),\n        F.col("record_version").alias("source_record_version"),\n        F.col("record_hash").alias("source_record_hash"),\n        F.col("record_observed_utc").alias("source_observed_utc"),\n        F.col("ingest_loaded_utc").alias("source_loaded_utc"),\n        "ingest_batch_run_id",\n        "ingest_processing_date",\n        "ingest_contract_hash",\n        F.lit(release_id).alias("gold_release_id"),\n        F.lit(datetime.now(UTC)).alias("gold_loaded_utc"),\n        F.lit(config["contract_hash"]).alias("mapping_version"),\n        F.lit(config["contract_hash"]).alias("measure_rule_version"),\n        F.lit("SYNTHETIC_PROVISIONAL").alias("dq_status"),\n        F.lit("DEMO_CONTRACT_KEY_NOT_PRODUCTION_CERTIFIED").alias("grain_status"),\n        F.lit("source_observed_not_business_effective").alias("dimension_time_basis"),\n        (\n            adapter.supplier_business_key(frame)\n            if hasattr(adapter, "supplier_business_key")\n            else F.lit(None).cast("string")\n        ).alias("supplier_business_key"),\n        event_date.alias("event_date"),\n        F.date_format(event_date, "yyyyMMdd").cast("int").alias("event_date_sk"),\n        period.alias("posting_month"),\n        *(\n            [\n                F.lit(as_of).alias("as_of_date"),\n                F.lit(int(as_of.strftime("%Y%m%d"))).alias("as_of_date_sk"),\n            ]\n            if spec["snapshot"]\n            else []\n        ),\n        *[\n            F.coalesce(F.date_format(value, "yyyyMMdd").cast("int"), F.lit(0)).alias(\n                name + "_sk"\n            )\n            for name, value in date_roles.items()\n        ],\n        *dimension_columns,\n        *[\n            F.col(n)\n            .cast(spec.get("measure_types", {}).get(n, "decimal(28,4)"))\n            .alias(n)\n            for n in spec["measures"]\n        ],\n        *attrs,\n        *getattr(adapter, "fact_extra_columns", lambda frame, spec: [])(frame, spec),\n    )\n    _assert_unique(output, ["fact_sk"], spec["name"])\n    if output.filter(F.col("event_date").isNull()).limit(1).count():\n        raise RuntimeError(f"{spec[\'name\']}: missing/invalid event date")\n    return output\n\n\ndef calendar_frames(spark, dates, calendar: dict[str, Any] | None = None):\n    # Omitted calendar is the legacy Python API only; compiled contracts declare it.\n    if calendar is None:\n        function = getattr(_gold_adapter(), "legacy_calendar_contract", None)\n        if function is None:\n            raise ValueError("The selected case must declare its fiscal calendar")\n        calendar = function()\n    start_month = calendar.get("fiscal_year_start_month")\n    if type(start_month) is not int or not 1 <= start_month <= 12:\n        raise ValueError(\n            "Calendar requires an explicit fiscal year start month in 1..12"\n        )\n    basis = calendar.get("basis", "DECLARED_FISCAL_CALENDAR")\n    from pyspark.sql import functions as F\n\n    bounds = dates.agg(\n        F.min("event_date").alias("start"), F.max("event_date").alias("end")\n    ).first()\n    if bounds["start"] is None:\n        days = spark.createDataFrame([], "calendar_date date")\n    else:\n        length = (bounds["end"] - bounds["start"]).days + 1\n        if length > 366 * 200:\n            raise RuntimeError("Calendar range exceeds the supported demo window")\n        days = spark.range(length).select(\n            F.date_add(F.lit(bounds["start"]), F.col("id").cast("int")).alias(\n                "calendar_date"\n            )\n        )\n    days = days.select(\n        F.date_format("calendar_date", "yyyyMMdd").cast("int").alias("date_sk"),\n        "calendar_date",\n        F.year("calendar_date").alias("calendar_year"),\n        F.month("calendar_date").alias("calendar_month"),\n        F.date_format("calendar_date", "yyyyMM").cast("int").alias("fiscal_period_sk"),\n    )\n    periods = (\n        days.select("fiscal_period_sk", "calendar_year", "calendar_month")\n        .distinct()\n        .withColumn(\n            "fiscal_year_start",\n            F.col("calendar_year")\n            - F.when(F.col("calendar_month") < start_month, 1).otherwise(0),\n        )\n        .withColumn(\n            "fiscal_month", F.pmod(F.col("calendar_month") - start_month, F.lit(12)) + 1\n        )\n        .withColumn("calendar_basis", F.lit(basis))\n    )\n    days = days.unionByName(\n        spark.createDataFrame(\n            [(0, None, None, None, 0)],\n            "date_sk int, calendar_date date, calendar_year int, calendar_month int, fiscal_period_sk int",\n        )\n    )\n    periods = periods.unionByName(\n        spark.createDataFrame(\n            [(0, None, None, None, None, "UNKNOWN")],\n            "fiscal_period_sk int, calendar_year int, calendar_month int, fiscal_year_start int, fiscal_month int, calendar_basis string",\n        )\n    )\n    return {"dim_date": days, "dim_fiscal_period": periods}\n\n\ndef _pin(spark, name: str, manifest: dict[str, Any]):\n    table = _gold_delta(spark, name)\n    version = table.history(1).select("version").first()["version"]\n    location = table.detail().select("location").first()["location"]\n    manifest[name] = {"version": version, "location": location}\n    return spark.read.format("delta").option("versionAsOf", version).load(location)\n\n\ndef _reload_gold_table(spark, reference, expected_table: str, *, current: bool = False):\n    """Resolve the persisted location and reopen an exact Delta version."""\n    _gold_reference(reference, expected_table)\n    delta = _gold_delta(spark, expected_table)\n    if delta.detail().select("location").first()["location"] != reference["location"]:\n        raise RuntimeError("A pinned Gold input/output table location changed")\n    if (\n        current\n        and delta.history(1).select("version").first()["version"]\n        != reference["version"]\n    ):\n        raise RuntimeError(f"{expected_table}: output drift invalidates Gold readiness")\n    return (\n        spark.read.format("delta")\n        .option("versionAsOf", reference["version"])\n        .load(reference["location"])\n    )\n\n\ndef manifest_gold_context(\n    config: dict[str, Any],\n    source: dict[str, Any],\n    selected: dict[str, str],\n    context_json: str,\n    context_sha256: str,\n    ingestion: Any,\n) -> dict[str, Any]:\n    """Adapt a sealed source context, without relaxing legacy path validation."""\n    context = ingestion.manifest_processing_context(\n        context_json, context_sha256, expected_lakehouse=source["lakehouse"]\n    )\n    if (\n        context["source_code"] != source["source_code"]\n        or context["contract_hash"] != source["contract_hash"]\n        or context["pipeline_run_id"] != selected["pipeline_run_id"]\n        or context["processing_date"].isoformat() != selected["processing_date"]\n        or context["source_selection_complete"] is not True\n        or context["source_feed_id"]\n        != source.get("source_feed_id", context["source_feed_id"])\n        or context["source_type"] != source.get("source_type", context["source_type"])\n        or context["description_slug"]\n        != source.get("description_slug", context["description_slug"])\n    ):\n        raise RuntimeError("Gold source context differs from its committed selection")\n    metadata = config.get("_metadata")\n    if metadata and context["metadata_version"] != metadata["version_id"]:\n        raise RuntimeError(\n            "Gold source rule/configuration version differs from the pinned run"\n        )\n    known = {table["name"]: table for table in source["tables"]}\n    for table in context["tables"]:\n        expected = known.get(table["name"])\n        if expected is None or any(\n            table.get(field) != expected.get(field)\n            for field in ("keys", "columns", "hash_policy", "rule_specs")\n        ):\n            raise RuntimeError(\n                "Gold source object/hash/rule contract differs from its manifest"\n            )\n    if set(known) != {table["name"] for table in context["tables"]}:\n        raise RuntimeError("Gold source manifest omitted configured source objects")\n    return context\n\n\ndef _gold_contexts(fs, config, runs, ingestion, references):\n    if references is None:\n        if "_metadata" in config:\n            raise RuntimeError("Normal Gold requires pinned manifest source contexts")\n        return None, {}\n    if set(references) != set(runs):\n        raise RuntimeError(\n            "Gold source context membership differs from the selected sources"\n        )\n    contexts, serialized = {}, {}\n    for source in config["sources"]:\n        code = source["source_code"]\n        reference = references[code]\n        if set(reference) == {"context_json", "sha256"}:\n            text = reference["context_json"]\n        elif set(reference) == {"path", "sha256"}:\n            text = fs.head(reference["path"], MAX_GOLD_MANIFEST_BYTES + 1)\n        else:\n            raise ValueError(\n                "Gold source contexts require a pinned JSON/path and SHA-256"\n            )\n        if (\n            not isinstance(text, str)\n            or len(text.encode("utf-8")) > MAX_GOLD_MANIFEST_BYTES\n        ):\n            raise RuntimeError(\n                "Gold source context transport is incomplete or oversized"\n            )\n        contexts[code] = manifest_gold_context(\n            config, source, runs[code], text, reference["sha256"], ingestion\n        )\n        serialized[code] = {\n            "context_json": _gold_json(json.loads(text)),\n            "sha256": reference["sha256"],\n        }\n    return contexts, serialized\n\n\ndef _gold_row_observed_utc(value: datetime) -> datetime:\n    """Normalize Spark\'s Python Row timestamp exactly like ingestion.\n\n    TimestampType.fromInternal returns a naïve host-local datetime. Replacing its\n    tzinfo would shift the observation on non-UTC hosts. This conversion does not\n    reinterpret timestamp_ntz business dates or change the stored Silver hash.\n    """\n    return value.astimezone(UTC)\n\n\ndef _gold_bronze_reader(fs, context, bronze_base):\n    """Qualify validated manifest paths against the explicit Silver storage root.\n\n    Notebook workers are Gold-bound. Only the read adapter changes physical path\n    qualification; the original relative paths, batch ID and hash policy remain\n    in the pinned source context and sealed audit evidence.\n    """\n    if context.get("metadata_mode") != "manifest_v1":\n        return fs, context\n    if not isinstance(bronze_base, str) or not bronze_base.rstrip("/").endswith(\n        "/Files/bronze"\n    ):\n        raise ValueError(\n            "Manifest Gold requires an explicit source Lakehouse Files/bronze root"\n        )\n    bound = deepcopy(context)\n    qualify = lambda path: (\n        bronze_base.rstrip("/") + "/" + path.removeprefix("Files/bronze/")\n    )\n    bound["raw_source_files"] = {\n        name: [qualify(path) for path in paths]\n        for name, paths in context["raw_source_files"].items()\n    }\n    bound["raw_targets"] = {\n        name: qualify(path) for name, path in context["raw_targets"].items()\n    }\n    # PurePosixPath in the existing reader collapses URI slashes for ls parents.\n    parents = {\n        str(PurePosixPath(path).parent): path.rsplit("/", 1)[0]\n        for paths in bound["raw_source_files"].values()\n        for path in paths\n    }\n    reader = SimpleNamespace(\n        exists=fs.exists,\n        head=fs.head,\n        ls=lambda path: fs.ls(parents.get(path, path)),\n    )\n    return reader, bound\n\n\ndef _gold_inputs(\n    spark,\n    fs,\n    config,\n    runs,\n    ingestion,\n    bronze_base,\n    resources,\n    *,\n    source_contexts=None,\n    pinned_manifest=None,\n    required=None,\n):\n    from pyspark.sql import functions as F\n\n    manifest = (\n        deepcopy(pinned_manifest)\n        if pinned_manifest is not None\n        else {"_cutoffs": {}, "_sealed": {}}\n    )\n    histories, snapshots = {}, {}\n\n    def pin(name):\n        if pinned_manifest is None:\n            return _pin(spark, name, manifest)\n        if name not in manifest:\n            raise RuntimeError(\n                "A required Silver/audit version is missing from the Gold manifest"\n            )\n        return _reload_gold_table(spark, {"table": name, **manifest[name]}, name)\n\n    source_name = config["sources"][0]["lakehouse"]\n    audit = pin(f"{source_name}.audit.bronze_silver_validation")\n    batch_manifest = pin(f"{source_name}.audit.bronze_batch_manifest")\n    required = (\n        required\n        if required is not None\n        else {(f["source"], f["table"]) for f in config["facts"]}\n        | {(d["source"], d["table"]) for d in config["dimensions"]}\n    )\n    for source in config["sources"]:\n        code = source["source_code"]\n        if not any(key[0] == code for key in required):\n            continue\n        selected = runs[code]\n        context = (\n            source_contexts[code]\n            if source_contexts is not None\n            else (\n                source\n                | selected\n                | {\n                    "processing_date": date.fromisoformat(selected["processing_date"]),\n                    "bronze_root": f"{bronze_base}/{code}",\n                }\n            )\n        )\n        reader, read_context = _gold_bronze_reader(fs, context, bronze_base)\n        commits = (\n            audit.filter(\n                (F.col("source_code") == code)\n                & (F.col("check_name") == "source_batch_committed")\n                & (F.col("status") == "SUCCESS")\n            )\n            .select("pipeline_run_id", "processing_date", "contract_hash")\n            .distinct()\n        )\n        selected_commit = commits.filter(\n            (F.col("pipeline_run_id") == selected["pipeline_run_id"])\n            & (F.col("processing_date") == F.lit(context["processing_date"]))\n            & (F.col("contract_hash") == source["contract_hash"])\n        )\n        if selected_commit.count() != 1:\n            raise RuntimeError(\n                f"{code}: the selected source batch is not committed under this contract"\n            )\n        for table in context["tables"]:\n            key = (code, table["name"])\n            if key not in required:\n                continue\n            sealed = (\n                batch_manifest.filter(\n                    (F.col("source_code") == code)\n                    & (F.col("table_name") == table["name"])\n                    & (F.col("pipeline_run_id") == selected["pipeline_run_id"])\n                    & (F.col("processing_date") == F.lit(context["processing_date"]))\n                    & (F.col("contract_hash") == source["contract_hash"])\n                )\n                .limit(2)\n                .collect()\n            )\n            if len(sealed) != 1:\n                raise RuntimeError(\n                    f"{code}.{table[\'name\']}: missing or duplicate sealed batch"\n                )\n            cutoff = sealed[0]["observed_utc"]\n            identity = f"{code}.{table[\'name\']}"\n            evidence = {\n                "observed_utc": cutoff.isoformat(),\n                "snapshot_hash": sealed[0]["snapshot_hash"],\n                "source_rows": sealed[0]["source_rows"],\n                "source_path": sealed[0]["source_path"],\n            }\n            if pinned_manifest is not None and (\n                manifest["_cutoffs"].get(identity) != cutoff.isoformat()\n                or manifest["_sealed"].get(identity) != evidence\n            ):\n                raise RuntimeError(\n                    "Gold sealed extraction evidence changed after preparation"\n                )\n            if source_contexts is not None and (\n                _gold_row_observed_utc(cutoff)\n                != context["batch_started_utc"].astimezone(UTC)\n                or evidence["source_path"] != ingestion.source_path(context, table)\n            ):\n                raise RuntimeError(\n                    "Gold sealed batch paths/observation differ from the manifest"\n                )\n            manifest["_cutoffs"][identity] = cutoff.isoformat()\n            manifest["_sealed"][identity] = evidence\n            original = pin(\n                f"{source[\'lakehouse\']}.{source[\'silver_schema\']}.{table[\'name\']}"\n            )\n            commit_keys = commits.select(\n                F.col("pipeline_run_id").alias("_run"),\n                F.col("processing_date").alias("_date"),\n                F.col("contract_hash").alias("_contract"),\n            )\n            accepted = (\n                original.join(\n                    commit_keys,\n                    (F.col("ingest_batch_run_id") == F.col("_run"))\n                    & (F.col("ingest_processing_date") == F.col("_date"))\n                    & (F.col("ingest_contract_hash") == F.col("_contract")),\n                    "left_semi",\n                )\n                .filter(F.col("record_observed_utc") <= F.lit(cutoff))\n                .cache()\n            )\n            resources.append(accepted)\n            if (\n                accepted.filter(~F.coalesce(F.col("dq_passed"), F.lit(False)))\n                .limit(1)\n                .count()\n            ):\n                raise RuntimeError("Rejected rows cannot enter Gold")\n            _assert_unique(\n                accepted, ["record_id"], f"{code}.{table[\'name\']} representations"\n            )\n            histories[key] = _department(accepted, code, config)\n            # Original batch membership is authoritative for snapshots and changed control metadata.\n            raw = ingestion.read_raw(spark, reader, read_context, table)\n            fingerprint, count = ingestion.snapshot_fingerprint(raw)\n            if (\n                fingerprint != sealed[0]["snapshot_hash"]\n                or count != sealed[0]["source_rows"]\n            ):\n                raise RuntimeError(\n                    f"{code}.{table[\'name\']}: immutable Bronze content changed"\n                )\n            prepared = ingestion.prepare_rows(raw, table).cache()\n            resources.append(prepared)\n            if (\n                prepared.filter(F.size(ingestion.blocking_dq_reasons(table)) > 0)\n                .limit(1)\n                .count()\n            ):\n                raise RuntimeError(\n                    f"{code}.{table[\'name\']}: invalid committed source batch"\n                )\n            source_members = prepared.filter(F.size("_dq_reasons") == 0)\n            accepted_count = source_members.count()\n            hashed = ingestion.with_record_hashes(source_members, table).alias("batch")\n            current = _latest(accepted, table["keys"]).alias("slv")\n            condition = reduce(\n                lambda a, b: a & b,\n                [\n                    F.col(f"batch.`{k}`").eqNullSafe(F.col(f"slv.`{k}`"))\n                    for k in table["keys"]\n                ],\n            ) & F.col("batch._record_hash").eqNullSafe(F.col("slv.record_hash"))\n            members = hashed.join(current, condition, "left").select(\n                *[\n                    F.col(f"batch.`{c[\'name\']}`").alias(c["name"])\n                    for c in table["columns"]\n                ],\n                *[\n                    F.col(f"slv.`{n}`").alias(n)\n                    for n in original.columns\n                    if n not in {c["name"] for c in table["columns"]}\n                ],\n            )\n            if (\n                members.filter("record_id IS NULL").limit(1).count()\n                or members.count() != accepted_count\n            ):\n                raise RuntimeError(\n                    f"{code}.{table[\'name\']}: source membership does not reconcile to committed Silver"\n                )\n            snapshots[key] = _department(members, code, config).cache()\n            resources.append(snapshots[key])\n            prepared.unpersist()\n    return manifest, histories, snapshots\n\n\ndef _merge_gold(\n    spark,\n    target: str,\n    frame,\n    key: str,\n    *,\n    as_of: date | None = None,\n    enforce_layout: bool = False,\n    layout=None,\n    layout_capabilities=None,\n):\n    from pyspark.sql import functions as F\n\n    _assert_unique(frame, [key], target)\n    content = [\n        n for n in frame.columns if n not in {"gold_release_id", "gold_loaded_utc"}\n    ]\n    frame = frame.withColumn(\n        "_gold_content_hash", _gold_hash(*[F.col(n) for n in content])\n    )\n    if enforce_layout:\n        layout_runtime = globals().get("_MEDALLION_LAYOUT")\n        if layout_runtime is None:\n            raise RuntimeError(\n                "Generic Gold requires the packaged physical layout adapter"\n            )\n        capability = (\n            layout_runtime.RuntimeCapabilities(**layout_capabilities)\n            if layout_capabilities is not None\n            else None\n        )\n        existed = spark.catalog.tableExists(target)\n        layout_runtime.ensure_target_layout(\n            spark,\n            target,\n            frame,\n            layout,\n            existing_target=existed,\n            capabilities=capability,\n        )\n        if not existed:\n            spark.sql(\n                f"ALTER TABLE {layout_runtime.quote_table(target)} SET TBLPROPERTIES "\n                "(\'delta.enableChangeDataFeed\'=\'true\')"\n            )\n    if not spark.catalog.tableExists(target):\n        frame.write.format("delta").option("delta.enableChangeDataFeed", "true").mode(\n            "errorifexists"\n        ).saveAsTable(target)\n    else:\n        existing = spark.table(target)\n        if [(f.name, f.dataType) for f in existing.schema] != [\n            (f.name, f.dataType) for f in frame.schema\n        ]:\n            raise RuntimeError(\n                f"Gold schema mismatch: {target}; explicit migration required"\n            )\n        _assert_unique(existing, [key], target)\n        identity = (\n            "business_event_id"\n            if "business_event_id" in frame.columns\n            else ("durable_key" if "durable_key" in frame.columns else None)\n        )\n        if identity:\n            collisions = (\n                existing.alias("before")\n                .join(\n                    frame.alias("after"),\n                    F.col(f"before.{key}") == F.col(f"after.{key}"),\n                    "inner",\n                )\n                .filter(\n                    ~F.col(f"before.{identity}").eqNullSafe(F.col(f"after.{identity}"))\n                )\n            )\n            if collisions.limit(1).count():\n                raise RuntimeError(\n                    f"{target}: surrogate collision; refuse identity overwrite"\n                )\n        if (\n            as_of is None\n            and existing.select(key)\n            .join(frame.select(key), key, "left_anti")\n            .limit(1)\n            .count()\n        ):\n            raise RuntimeError(\n                f"{target}: out-of-order input would remove an existing event or dimension version"\n            )\n        operation = (\n            _gold_delta(spark, target)\n            .alias("tgt")\n            .merge(frame.alias("src"), f"tgt.{key} = src.{key}")\n            .whenMatchedUpdateAll(\n                condition="NOT (tgt._gold_content_hash <=> src._gold_content_hash)"\n            )\n            .whenNotMatchedInsertAll()\n        )\n        if as_of is not None:\n            operation = operation.whenNotMatchedBySourceDelete(\n                condition=f"tgt.as_of_date = DATE \'{as_of.isoformat()}\'"\n            )\n        operation.execute()\n\n\ndef _release_event(\n    spark, config, release_id, status, as_of, inputs, outputs, message=""\n):\n    inputs = {\n        key: value\n        for key, value in inputs.items()\n        if key not in {"_release_id", "_input_sha256"}\n    }\n    schema = (\n        "gold_release_id string, status string, as_of_date date, contract_hash string, "\n        "input_manifest_json string, output_manifest_json string, event_utc timestamp, message string"\n    )\n    row = (\n        release_id,\n        status,\n        as_of,\n        config["contract_hash"],\n        json.dumps(inputs, sort_keys=True),\n        json.dumps(outputs, sort_keys=True),\n        datetime.now(UTC),\n        message,\n    )\n    spark.createDataFrame([row], schema).write.format("delta").mode(\n        "append"\n    ).saveAsTable(_gold_target(config, "ctl_gold_release"))\n\n\ndef _ready_release(spark, config, release_id):\n    from pyspark.sql import functions as F\n\n    table = _gold_target(config, "ctl_gold_release")\n    if not spark.catalog.tableExists(table):\n        return None\n    latest = (\n        spark.table(table)\n        .filter(F.col("gold_release_id") == release_id)\n        .orderBy(F.col("event_utc").desc())\n        .limit(1)\n        .collect()\n    )\n    if not latest or latest[0]["status"] not in {"FACTS_READY", "MLVS_VERIFIED"}:\n        return None\n    rows = (\n        spark.table(table)\n        .filter(\n            (F.col("gold_release_id") == release_id)\n            & (F.col("status") == "FACTS_READY")\n        )\n        .orderBy(F.col("event_utc").desc())\n        .limit(1)\n        .collect()\n    )\n    return rows[0].asDict() if rows else None\n\n\ndef _gold_table_reference(spark, target: str, *, count: bool = False) -> dict[str, Any]:\n    delta = _gold_delta(spark, target)\n    reference = {\n        "table": target,\n        "version": delta.history(1).select("version").first()["version"],\n        "location": delta.detail().select("location").first()["location"],\n    }\n    if count:\n        reference["row_count"] = _reload_gold_table(spark, reference, target).count()\n    return reference\n\n\ndef _gold_tasks(config):\n    if "_metadata" in config:\n        tasks = deepcopy(config["_metadata"]["tasks"])\n    else:\n        tasks = {}\n        dimensions = [spec["name"] for spec in config["dimensions"]]\n        facts = [spec["name"] for spec in config["facts"]]\n        for spec in [*config["dimensions"], *config["facts"]]:\n            name = spec["name"]\n            kind = (\n                "scd2_dimension"\n                if name in dimensions\n                else ("snapshot_fact" if spec["snapshot"] else "atomic_fact")\n            )\n            tasks[name] = {\n                "task_id": hashlib.sha256(("legacy:" + name).encode()).hexdigest(),\n                "name": name,\n                "kind": kind,\n                "target": _gold_target(config, name),\n                "dependencies": [] if name in dimensions else dimensions,\n                "source_inputs": [{"source": spec["source"], "table": spec["table"]}],\n                "spec": deepcopy(spec),\n                "transformation": {\n                    "kind": "pattern",\n                    "version": "1",\n                    "definition": {"pattern": kind, "contract": deepcopy(spec)},\n                    "definition_sha256": _gold_digest(\n                        {"pattern": kind, "contract": spec}\n                    ),\n                },\n            }\n        for name in ("dim_date", "dim_fiscal_period"):\n            tasks[name] = {\n                "task_id": hashlib.sha256(("legacy:" + name).encode()).hexdigest(),\n                "name": name,\n                "kind": "calendar",\n                "target": _gold_target(config, name),\n                "dependencies": facts,\n                "source_inputs": [],\n                "spec": {},\n                "transformation": {\n                    "kind": "pattern",\n                    "version": "1",\n                    "definition": {"pattern": "calendar", "contract": {}},\n                    "definition_sha256": _gold_digest(\n                        {"pattern": "calendar", "contract": {}}\n                    ),\n                },\n            }\n    _validate_gold_tasks(tasks)\n    known = {\n        (source["source_code"], table["name"])\n        for source in config["sources"]\n        for table in source["tables"]\n    }\n    if not _gold_required_sources(tasks) <= known:\n        raise RuntimeError("Gold task references an undeclared source input")\n    return tasks\n\n\ndef _gold_required_sources(tasks):\n    return {\n        (reference["source"], reference["table"])\n        for task in tasks.values()\n        for reference in task.get(\n            "source_inputs",\n            (\n                []\n                if task["kind"] == "calendar"\n                else [\n                    {"source": task["spec"]["source"], "table": task["spec"]["table"]}\n                ]\n            ),\n        )\n    }\n\n\ndef _gold_source_references(config, inputs, source_inputs):\n    sources = {source["source_code"]: source for source in config["sources"]}\n    references = {}\n    for selected in source_inputs:\n        source = sources[selected["source"]]\n        table = f"{source[\'lakehouse\']}.{source[\'silver_schema\']}.{selected[\'table\']}"\n        if table not in inputs:\n            raise RuntimeError("A declared Gold source input version was not prepared")\n        references[table] = {"table": table, **inputs[table]}\n    return references\n\n\ndef _gold_execution(config):\n    metadata = config.get("_metadata", {})\n    return {\n        "config_sha256": _gold_digest(config),\n        "code_sha256": config["runtime_sha256"],\n        "configuration_sha256": metadata.get("configuration_sha256"),\n        "version_id": metadata.get("version_id"),\n        "snapshot_sha256": metadata.get("snapshot_sha256"),\n    }\n\n\ndef _release_record(spark, config, release_id):\n    from pyspark.sql import functions as F\n\n    target = _gold_target(config, "ctl_gold_release")\n    if not spark.catalog.tableExists(target):\n        return None\n    rows = (\n        spark.table(target)\n        .filter(F.col("gold_release_id") == release_id)\n        .orderBy(F.col("event_utc").desc())\n        .limit(1)\n        .collect()\n    )\n    return rows[0].asDict() if rows else None\n\n\ndef _validate_gold_outputs(spark, outputs):\n    for entry in outputs.values():\n        reference = dict(entry)\n        # Historical release manifests predate explicit storage locations.\n        if "location" not in reference:\n            reference["location"] = _gold_table_reference(spark, entry["table"])[\n                "location"\n            ]\n        _reload_gold_table(spark, reference, entry["table"], current=True)\n\n\ndef prepare_gold(\n    spark: Any,\n    fs: Any,\n    config: dict[str, Any],\n    source_runs_json: str,\n    gold_release_id: str,\n    ingestion: Any,\n    bronze_base: str,\n    *,\n    source_contexts: Mapping[str, dict[str, str]] | None = None,\n    ownership_check: Callable[[], None] | None = None,\n) -> GoldPreparationReference:\n    """Pin/validate source inputs and initialize controls before parallel workers.\n\n    The returned reference points at a persisted Delta control version, not a\n    frame or mutable metadata dictionary. Retries reuse that release\'s original\n    versions, source observation and task/code identity, including failed runs.\n    The caller owns/fences the complete target set through finalization and the\n    separately coordinated MLV window; this is not a cross-table transaction.\n    """\n    from pyspark.sql import functions as F\n\n    runs, as_of = gold_request(config, source_runs_json, gold_release_id)\n    spark.conf.set("spark.sql.session.timeZone", "UTC")\n    tasks, execution = _gold_tasks(config), _gold_execution(config)\n    selection = {\n        "runs": runs,\n        "as_of_date": as_of.isoformat(),\n        "contract_hash": config["contract_hash"],\n    }\n    contexts, serialized = _gold_contexts(fs, config, runs, ingestion, source_contexts)\n    existing = _release_record(spark, config, gold_release_id)\n    retained = json.loads(existing["input_manifest_json"]) if existing else None\n    if retained is not None and (\n        retained.get("_selection") != selection\n        or retained.get("_execution") != execution\n        or retained.get("_tasks") != tasks\n        or retained.get("_contexts") != serialized\n        or retained.get("_bronze_base") != bronze_base\n    ):\n        raise RuntimeError(\n            "A Gold release, including a failed release, cannot be rebound to changed identity or inputs"\n        )\n    resources = []\n    try:\n        inputs, _histories, _snapshots = _gold_inputs(\n            spark,\n            fs,\n            config,\n            runs,\n            ingestion,\n            bronze_base,\n            resources,\n            source_contexts=contexts,\n            pinned_manifest=retained,\n            required=_gold_required_sources(tasks),\n        )\n    finally:\n        for frame in resources:\n            frame.unpersist()\n    if retained is None:\n        inputs.update(\n            _selection=selection,\n            _execution=execution,\n            _tasks=tasks,\n            _contexts=serialized,\n            _bronze_base=bronze_base,\n            _base_outputs={\n                name: _gold_table_reference(spark, _gold_target(config, name))\n                for name in [*tasks, "ctl_measure_availability"]\n                if spark.catalog.tableExists(_gold_target(config, name))\n            },\n        )\n    _gold_json(inputs)\n    if ownership_check is not None:\n        ownership_check()\n    if existing is None:\n        control = _gold_target(config, "ctl_gold_release")\n        if spark.catalog.tableExists(control):\n            latest = (\n                spark.table(control)\n                .filter(F.col("status") == "FACTS_READY")\n                .orderBy(F.col("event_utc").desc())\n                .limit(1)\n                .collect()\n            )\n            if latest:\n                previous = json.loads(latest[0]["input_manifest_json"]).get(\n                    "_cutoffs", {}\n                )\n                if latest[0]["as_of_date"] > as_of or any(\n                    inputs["_cutoffs"].get(key, "") < cutoff\n                    for key, cutoff in previous.items()\n                ):\n                    raise RuntimeError(\n                        "Older Gold cutoffs require a separately reviewed backfill"\n                    )\n        spark.sql(\n            f"CREATE SCHEMA IF NOT EXISTS {config[\'target_lakehouse\']}.{config[\'schema\']}"\n        )\n        task_table = _gold_target(config, "ctl_gold_task")\n        if not spark.catalog.tableExists(task_table):\n            spark.createDataFrame([], GOLD_TASK_SCHEMA).write.format("delta").mode(\n                "errorifexists"\n            ).saveAsTable(task_table)\n        _release_event(spark, config, gold_release_id, "BUILDING", as_of, inputs, {})\n    else:\n        ready = _ready_release(spark, config, gold_release_id)\n        if ready:\n            _validate_gold_outputs(spark, json.loads(ready["output_manifest_json"]))\n    return {\n        **_gold_table_reference(spark, _gold_target(config, "ctl_gold_release")),\n        "gold_release_id": gold_release_id,\n        "input_sha256": _gold_digest(inputs),\n    }\n\n\ndef load_gold_preparation(\n    spark: Any,\n    config: dict[str, Any],\n    reference: GoldPreparationReference,\n) -> dict[str, Any]:\n    from pyspark.sql import functions as F\n\n    control = _gold_target(config, "ctl_gold_release")\n    frame = _reload_gold_table(spark, reference, control)\n    rows = (\n        frame.filter(F.col("gold_release_id") == reference["gold_release_id"])\n        .orderBy(F.col("event_utc").desc())\n        .limit(1)\n        .collect()\n    )\n    if not rows:\n        raise RuntimeError("Persisted Gold preparation is missing")\n    inputs = json.loads(rows[0]["input_manifest_json"])\n    if (\n        _gold_digest(inputs) != reference["input_sha256"]\n        or inputs.get("_execution") != _gold_execution(config)\n        or inputs.get("_tasks") != _gold_tasks(config)\n    ):\n        raise RuntimeError("Gold preparation code/configuration/task identity changed")\n    return inputs\n\n\ndef _task_result_rows(spark, config, preparation, name, attempt_id, *, reference=None):\n    from pyspark.sql import functions as F\n\n    target = _gold_target(config, "ctl_gold_task")\n    if not spark.catalog.tableExists(target):\n        return []\n    frame = (\n        _reload_gold_table(spark, reference, target)\n        if reference\n        else spark.table(target)\n    )\n    task = preparation["_tasks"][name]\n    return (\n        frame.filter(\n            (F.col("gold_release_id") == preparation["_release_id"])\n            & (F.col("input_sha256") == preparation["_input_sha256"])\n            & (F.col("object_name") == name)\n            & (F.col("task_id") == task["task_id"])\n            & (F.col("attempt_id") == attempt_id)\n        )\n        .select("status", "result_json")\n        .distinct()\n        .limit(2)\n        .collect()\n    )\n\n\ndef _prepared_for_task(spark, config, reference):\n    inputs = load_gold_preparation(spark, config, reference)\n    return {\n        **inputs,\n        "_release_id": reference["gold_release_id"],\n        "_input_sha256": reference["input_sha256"],\n    }\n\n\ndef gold_result_references(\n    spark: Any,\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n    attempts: Mapping[str, str | GoldAttempt],\n) -> dict[str, GoldResultReference]:\n    """Resolve only the requested persisted attempts (usually dimension fan-in)."""\n    inputs = _prepared_for_task(spark, config, preparation)\n    if not set(attempts) <= set(inputs["_tasks"]):\n        raise RuntimeError("Undeclared Gold result membership")\n    target = _gold_target(config, "ctl_gold_task")\n    pinned = _gold_table_reference(spark, target)\n    references = {}\n    for name, attempt in attempts.items():\n        if isinstance(attempt, str):\n            if "_metadata" in config:\n                raise RuntimeError(\n                    "Normal Gold result fan-in requires the fenced layer dispatch"\n                )\n            attempt_id = attempt\n        else:\n            attempt_id = attempt["attempt_id"]\n        rows = _task_result_rows(\n            spark, config, inputs, name, attempt_id, reference=pinned\n        )\n        if len(rows) != 1 or rows[0]["status"] != "SUCCEEDED":\n            raise RuntimeError(\n                "Required Gold object attempt is missing, ambiguous or failed"\n            )\n        document = json.loads(rows[0]["result_json"])\n        if not isinstance(attempt, str) and (\n            document["fence"] != attempt["fence"]\n            or document["job_id"] != attempt["job_id"]\n        ):\n            raise RuntimeError(\n                "Gold result differs from the prepared attempt/fence/job"\n            )\n        references[name] = {\n            **pinned,\n            "gold_release_id": preparation["gold_release_id"],\n            "task_id": inputs["_tasks"][name]["task_id"],\n            "attempt_id": attempt_id,\n            "result_sha256": _gold_digest(document),\n            "fence": document["fence"],\n            "job_id": document["job_id"],\n        }\n    return references\n\n\ndef load_gold_result(\n    spark: Any,\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n    object_name: str,\n    reference: GoldResultReference,\n) -> dict[str, Any]:\n    inputs = _prepared_for_task(spark, config, preparation)\n    if object_name not in inputs["_tasks"]:\n        raise RuntimeError("Gold output was not declared in the prepared release")\n    task = inputs["_tasks"][object_name]\n    if (\n        reference["gold_release_id"] != preparation["gold_release_id"]\n        or reference["task_id"] != task["task_id"]\n    ):\n        raise RuntimeError("Gold result belongs to another release/task")\n    rows = _task_result_rows(\n        spark, config, inputs, object_name, reference["attempt_id"], reference=reference\n    )\n    if len(rows) != 1 or rows[0]["status"] != "SUCCEEDED":\n        raise RuntimeError("Gold result is missing, ambiguous or failed")\n    result = json.loads(rows[0]["result_json"])\n    if (\n        _gold_digest(result) != reference["result_sha256"]\n        or result["input_sha256"] != preparation["input_sha256"]\n        or result["execution"] != inputs["_execution"]\n        or result["task"] != task\n        or result["status"] != "SUCCEEDED"\n        or result["gold_release_id"] != reference["gold_release_id"]\n        or result["object_name"] != object_name\n        or result["attempt_id"] != reference["attempt_id"]\n        or result["fence"] != reference["fence"]\n        or result["job_id"] != reference["job_id"]\n    ):\n        raise RuntimeError("Gold result implementation/configuration identity changed")\n    _reload_gold_table(spark, result["output"], task["target"], current=True)\n    return result\n\n\ndef dispatch_gold_transformation(\n    transformation: Mapping[str, Any],\n    implementations: Mapping[str, RegisteredGoldTransformation],\n    context: dict[str, Any],\n) -> Any:\n    """Dispatch reviewed SQL/notebook adapters, never metadata eval/exec.\n\n    SQL adapters must perform engine analysis and bind the supplied pinned inputs;\n    notebook adapters must reopen persisted outputs, not pass frames from another\n    interpreter. No generic arbitrary-SQL or notebook fallback is installed.\n    """\n    definition = transformation["definition"]\n    if _gold_digest(definition) != transformation["definition_sha256"]:\n        raise RuntimeError("Gold executable definition hash changed")\n    kind = transformation["kind"]\n    if kind not in {"sql", "notebook"}:\n        raise RuntimeError("Unsupported specialist Gold transformation kind")\n    implementation = implementations.get(transformation.get("id", ""))\n    if implementation is None or any(\n        (\n            implementation["kind"] != kind,\n            implementation["version"] != transformation["version"],\n            implementation["definition_sha256"] != transformation["definition_sha256"],\n            implementation["implementation_sha256"]\n            != definition.get("implementation_sha256"),\n            implementation["approval_id"] != definition.get("approval_id"),\n            not callable(implementation["execute"]),\n        )\n    ):\n        raise RuntimeError(\n            "Gold specialist approval, version or implementation is unavailable"\n        )\n    _gold_token(\n        definition.get("approval_id"),\n        "Gold approval reference",\n        r"[A-Za-z0-9_.-]{1,128}",\n    )\n    _gold_token(\n        definition.get("implementation_sha256"),\n        "Gold implementation hash",\n        r"[0-9a-f]{64}",\n    )\n    if kind == "sql":\n        if definition.get("dialect") != "spark_sql" or not definition.get("query"):\n            raise RuntimeError("Gold SQL requires an approved Spark SQL query contract")\n        _gold_token(\n            definition.get("validation_sha256"),\n            "Gold SQL engine validation evidence",\n            r"[0-9a-f]{64}",\n        )\n    elif not definition.get("notebook_name"):\n        raise RuntimeError(\n            "Gold notebook requires a versioned implementation reference"\n        )\n    if definition.get("inputs") != sorted(context["input_references"]):\n        raise RuntimeError(\n            "Gold specialist input contract differs from pinned input references"\n        )\n    columns = definition.get("output_columns")\n    if (\n        not isinstance(columns, list)\n        or not columns\n        or any(\n            not isinstance(column, dict) or set(column) != {"name", "data_type"}\n            for column in columns\n        )\n    ):\n        raise RuntimeError("Gold specialist output columns/types must be explicit")\n    frame = implementation["execute"](context)\n    if [(field.name, field.dataType.simpleString()) for field in frame.schema] != [\n        (column["name"], column["data_type"]) for column in columns\n    ]:\n        raise RuntimeError(\n            "Gold specialist returned undeclared output columns or types"\n        )\n    return frame\n\n\ndef _guard_gold_target(spark, inputs, name, target):\n    before = inputs["_base_outputs"].get(name)\n    if before is None:\n        if spark.catalog.tableExists(target):\n            raise RuntimeError(\n                "An unrecorded target write requires explicit Gold recovery"\n            )\n    else:\n        _reload_gold_table(spark, before, target, current=True)\n\n\ndef validate_gold_quality(spark, frame, config, object_name, input_references):\n    """Evaluate declared runtime checks without filtering or changing business rows."""\n    from pyspark.sql import functions as F\n\n    metadata = config.get("_metadata", {})\n    object_id = metadata.get("tasks", {}).get(object_name, {}).get("object_id")\n    rules = [\n        {key: value for key, value in rule.items() if key != "object_id"}\n        for rule in metadata.get("rules", [])\n        if rule["object_id"] == object_id\n    ]\n    evidence = []\n    if rules:\n        handlers = globals().get("_MEDALLION_RULE_HANDLERS")\n        if handlers is None or any(\n            rule["action"] not in {"fail", "warn"} for rule in rules\n        ):\n            raise RuntimeError(\n                "Gold quality cannot discard rows or execute unregistered validators"\n            )\n        checked = handlers["apply_rules"](\n            frame.withColumn("_dq_reasons", F.array().cast("array<string>")),\n            {"rule_specs": rules},\n            scope="gold",\n        )\n        for rule in rules:\n            field = "_dq_warnings" if rule["action"] == "warn" else "_dq_reasons"\n            count = checked.filter(\n                F.array_contains(F.col(field), f"{rule[\'id\']}@{rule[\'version\']}")\n            ).count()\n            evidence.append(\n                {\n                    "id": rule["id"],\n                    "version": rule["version"],\n                    "action": rule["action"],\n                    "violations": count,\n                }\n            )\n            if count and rule["action"] == "fail":\n                raise RuntimeError(f"Gold blocking quality rule failed: {rule[\'id\']}")\n    for constraint in metadata.get("constraints", []):\n        if constraint["object_id"] != object_id:\n            continue\n        if constraint["enforcement"] != "runtime":\n            raise RuntimeError("Engine-enforced Gold constraints are not inferred")\n        definition, kind = constraint["definition"], constraint["kind"]\n        if kind == "check":\n            if (\n                set(definition) != {"rule"}\n                or definition["rule"].get("action") != "fail"\n            ):\n                raise ValueError(\n                    "Gold check constraint requires one blocking versioned rule"\n                )\n            nested = {\n                "_metadata": {\n                    "tasks": {object_name: {"object_id": object_id}},\n                    "rules": [dict(definition["rule"]) | {"object_id": object_id}],\n                },\n            }\n            rows = validate_gold_quality(\n                spark, frame, nested, object_name, input_references\n            )\n            violations = sum(row["violations"] for row in rows)\n        else:\n            columns = definition.get("columns")\n            if (\n                not isinstance(columns, list)\n                or not columns\n                or len(set(columns)) != len(columns)\n            ):\n                raise ValueError("Gold constraints require distinct declared columns")\n            if not set(columns) <= set(frame.columns):\n                raise ValueError("Gold constraint references absent output columns")\n            if kind == "unique" and set(definition) == {"columns"}:\n                nulls = reduce(\n                    lambda left, right: left | right,\n                    [F.col(column).isNull() for column in columns],\n                )\n                violations = (\n                    frame.filter(nulls).count()\n                    + frame.groupBy(*columns).count().filter(F.col("count") > 1).count()\n                )\n            elif kind in {"relationship", "reconciliation"} and set(definition) == {\n                "columns",\n                "reference_table",\n                "reference_columns",\n            }:\n                references = [\n                    reference\n                    for reference in input_references.values()\n                    if reference.get("table") == definition["reference_table"]\n                ]\n                if len(references) != 1 or len(columns) != len(\n                    definition["reference_columns"]\n                ):\n                    raise RuntimeError(\n                        "Gold constraint requires an explicitly pinned upstream reference"\n                    )\n                parent = _reload_gold_table(\n                    spark, references[0], definition["reference_table"]\n                )\n                if not set(definition["reference_columns"]) <= set(parent.columns):\n                    raise ValueError(\n                        "Gold constraint references absent upstream columns"\n                    )\n                right = parent.select(\n                    *[\n                        F.col(old).alias(new)\n                        for new, old in zip(\n                            columns, definition["reference_columns"], strict=True\n                        )\n                    ]\n                )\n                left = frame.select(*columns)\n                if kind == "relationship":\n                    violations = left.join(right, columns, "left_anti").count()\n                else:\n                    violations = (\n                        left.exceptAll(right).count() + right.exceptAll(left).count()\n                    )\n            else:\n                raise RuntimeError("Unsupported Gold constraint contract")\n        evidence.append(\n            {\n                "id": constraint["id"],\n                "kind": kind,\n                "enforcement": "runtime",\n                "violations": violations,\n            }\n        )\n        if violations:\n            raise RuntimeError(f"Gold runtime constraint failed: {constraint[\'id\']}")\n    return evidence\n\n\ndef _gold_calendar(spark, config, inputs, dependencies):\n    from pyspark.sql import functions as F\n\n    dates = []\n    for spec in config["facts"]:\n        frame = _reload_gold_table(\n            spark,\n            dependencies[spec["name"]]["output"],\n            _gold_target(config, spec["name"]),\n            current=True,\n        )\n        dates.extend(\n            [\n                frame.select("event_date"),\n                frame.filter(F.col("posting_month").isNotNull()).select(\n                    F.to_date(\n                        F.concat(F.col("posting_month").cast("string"), F.lit("01")),\n                        "yyyyMMdd",\n                    ).alias("event_date")\n                ),\n            ]\n        )\n        for role in spec.get("date_roles", {}):\n            dates.append(\n                frame.filter(F.col(role + "_sk") > 0).select(\n                    F.to_date(F.col(role + "_sk").cast("string"), "yyyyMMdd").alias(\n                        "event_date"\n                    )\n                )\n            )\n        if spec["snapshot"]:\n            dates.append(frame.select(F.col("as_of_date").alias("event_date")))\n    if "dim_date" in inputs["_base_outputs"]:\n        dates.append(\n            _reload_gold_table(\n                spark,\n                inputs["_base_outputs"]["dim_date"],\n                _gold_target(config, "dim_date"),\n            )\n            .filter("date_sk <> 0")\n            .select(F.col("calendar_date").alias("event_date"))\n        )\n    if not dates:\n        dates.append(spark.createDataFrame([], "event_date date"))\n    return calendar_frames(\n        spark, reduce(lambda a, b: a.unionByName(b), dates), config.get("calendar")\n    )\n\n\ndef _write_gold_result(spark, config, result):\n    row = (\n        result["gold_release_id"],\n        result["input_sha256"],\n        result["object_name"],\n        result["task"]["task_id"],\n        result["attempt_id"],\n        result["status"],\n        _gold_json(result),\n        datetime.now(UTC),\n    )\n    spark.createDataFrame([row], GOLD_TASK_SCHEMA).write.format("delta").mode(\n        "append"\n    ).saveAsTable(_gold_target(config, "ctl_gold_task"))\n    return {\n        **_gold_table_reference(spark, _gold_target(config, "ctl_gold_task")),\n        "gold_release_id": result["gold_release_id"],\n        "task_id": result["task"]["task_id"],\n        "attempt_id": result["attempt_id"],\n        "result_sha256": _gold_digest(result),\n        "fence": result["fence"],\n        "job_id": result["job_id"],\n    }\n\n\ndef build_gold_object(\n    spark: Any,\n    fs: Any,\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n    object_name: str,\n    ingestion: Any,\n    *,\n    attempt_id: str,\n    fence: int,\n    job_id: str,\n    code_sha256: str,\n    dependency_results: Mapping[str, GoldResultReference] | None = None,\n    implementations: Mapping[str, RegisteredGoldTransformation] | None = None,\n    ownership_check: Callable[[], None] | None = None,\n) -> GoldResultReference:\n    """Build one fenced object from persisted versions and persist its result."""\n    _gold_token(attempt_id, "Gold attempt ID", r"[A-Za-z0-9_-]{1,128}")\n    _gold_token(job_id, "Gold job ID", r"[A-Za-z0-9_-]{1,128}")\n    if type(fence) is not int or not 1 <= fence <= 2**63 - 1:\n        raise ValueError("Gold writes require a positive task fence")\n    inputs = _prepared_for_task(spark, config, preparation)\n    if code_sha256 != inputs["_execution"]["code_sha256"]:\n        raise RuntimeError("Gold worker code differs from the pinned implementation")\n    if object_name not in inputs["_tasks"]:\n        raise RuntimeError("Gold object is not part of the prepared release")\n    task = inputs["_tasks"][object_name]\n    references = dict(dependency_results or {})\n    if set(references) != set(task["dependencies"]):\n        raise RuntimeError(\n            "Gold object requires all explicit persisted dependency results"\n        )\n    dependencies = {\n        name: load_gold_result(spark, config, preparation, name, reference)\n        for name, reference in references.items()\n    }\n    existing = _task_result_rows(spark, config, inputs, object_name, attempt_id)\n    if existing:\n        if len(existing) != 1 or existing[0]["status"] != "SUCCEEDED":\n            raise RuntimeError(\n                "A failed or ambiguous Gold attempt requires explicit new-attempt recovery"\n            )\n        reference = gold_result_references(\n            spark,\n            config,\n            preparation,\n            {\n                object_name: {\n                    "attempt_id": attempt_id,\n                    "fence": fence,\n                    "job_id": job_id,\n                }\n            },\n        )[object_name]\n        result = load_gold_result(spark, config, preparation, object_name, reference)\n        if (\n            result["fence"] != fence\n            or result["job_id"] != job_id\n            or result["dependency_outputs"]\n            != {name: value["output"] for name, value in dependencies.items()}\n        ):\n            raise RuntimeError(\n                "A Gold attempt cannot be rebound to a changed dispatch/dependency identity"\n            )\n        return reference\n    if _ready_release(spark, config, preparation["gold_release_id"]):\n        raise RuntimeError(\n            "A completed Gold release cannot acquire another object attempt"\n        )\n    resources = []\n    result = {\n        "gold_release_id": preparation["gold_release_id"],\n        "input_sha256": preparation["input_sha256"],\n        "object_name": object_name,\n        "task": task,\n        "execution": inputs["_execution"],\n        "attempt_id": attempt_id,\n        "fence": fence,\n        "job_id": job_id,\n        "dependency_outputs": {\n            name: value["output"] for name, value in dependencies.items()\n        },\n        "status": "FAILED",\n        "output": None,\n        "error_code": None,\n    }\n    try:\n        contexts, _serialized = _gold_contexts(\n            fs,\n            config,\n            inputs["_selection"]["runs"],\n            ingestion,\n            inputs["_contexts"] or None,\n        )\n        as_of = date.fromisoformat(inputs["_selection"]["as_of_date"])\n        source_inputs = task.get(\n            "source_inputs",\n            (\n                []\n                if task["kind"] == "calendar"\n                else [\n                    {"source": task["spec"]["source"], "table": task["spec"]["table"]}\n                ]\n            ),\n        )\n        if task["kind"] == "calendar":\n            key = "date_sk" if object_name == "dim_date" else "fiscal_period_sk"\n            input_references = {\n                name: value["output"] for name, value in dependencies.items()\n            }\n            histories, snapshots = {}, {}\n            if source_inputs:\n                _manifest, histories, snapshots = _gold_inputs(\n                    spark,\n                    fs,\n                    config,\n                    inputs["_selection"]["runs"],\n                    ingestion,\n                    inputs["_bronze_base"],\n                    resources,\n                    source_contexts=contexts,\n                    pinned_manifest=inputs,\n                    required={(ref["source"], ref["table"]) for ref in source_inputs},\n                )\n                input_references.update(\n                    _gold_source_references(config, inputs, source_inputs)\n                )\n            if task["transformation"]["kind"] == "pattern":\n                frame = _gold_calendar(spark, config, inputs, dependencies)[object_name]\n            else:\n                frame = dispatch_gold_transformation(\n                    task["transformation"],\n                    implementations or {},\n                    {\n                        "spark": spark,\n                        "config": config,\n                        "spec": task["spec"],\n                        "dependencies": dependencies,\n                        "histories": histories,\n                        "snapshots": snapshots,\n                        "input_references": input_references,\n                        "target": task["target"],\n                        "as_of_date": as_of,\n                        "gold_release_id": preparation["gold_release_id"],\n                    },\n                )\n        else:\n            spec = task["spec"]\n            _manifest, histories, snapshots = _gold_inputs(\n                spark,\n                fs,\n                config,\n                inputs["_selection"]["runs"],\n                ingestion,\n                inputs["_bronze_base"],\n                resources,\n                source_contexts=contexts,\n                pinned_manifest=inputs,\n                required={(ref["source"], ref["table"]) for ref in source_inputs},\n            )\n            input_references = {\n                **_gold_source_references(config, inputs, source_inputs),\n                **{name: value["output"] for name, value in dependencies.items()},\n            }\n            if task["transformation"]["kind"] == "pattern":\n                if task["kind"] == "scd2_dimension":\n                    frame = build_scd2(histories[spec["source"], spec["table"]], spec)\n                else:\n                    dimensions = {\n                        name: _reload_gold_table(\n                            spark,\n                            value["output"],\n                            _gold_target(config, name),\n                            current=True,\n                        )\n                        for name, value in dependencies.items()\n                        if name in {d["name"] for d in config["dimensions"]}\n                    }\n                    frame = build_fact(\n                        (snapshots if spec["snapshot"] else histories)[\n                            spec["source"], spec["table"]\n                        ],\n                        spec,\n                        dimensions,\n                        config,\n                        as_of,\n                        preparation["gold_release_id"],\n                    )\n            else:\n                frame = dispatch_gold_transformation(\n                    task["transformation"],\n                    implementations or {},\n                    {\n                        "spark": spark,\n                        "config": config,\n                        "spec": spec,\n                        "histories": histories,\n                        "snapshots": snapshots,\n                        "dependencies": dependencies,\n                        "input_references": input_references,\n                        "target": task["target"],\n                        "as_of_date": as_of,\n                        "gold_release_id": preparation["gold_release_id"],\n                    },\n                )\n            key = "dimension_sk" if task["kind"] == "scd2_dimension" else "fact_sk"\n        frame = frame.cache()\n        resources.append(frame)\n        quality = validate_gold_quality(\n            spark, frame, config, object_name, input_references\n        )\n        _guard_gold_target(spark, inputs, object_name, task["target"])\n        if ownership_check is not None:\n            ownership_check()\n        _merge_gold(\n            spark,\n            task["target"],\n            frame,\n            key,\n            as_of=as_of if task["kind"] == "snapshot_fact" else None,\n            **(\n                {\n                    "enforce_layout": True,\n                    "layout": config["_metadata"]["layouts"].get(object_name),\n                    "layout_capabilities": config["_metadata"].get("layout_runtime"),\n                }\n                if "_metadata" in config\n                else {}\n            ),\n        )\n        output = _gold_table_reference(spark, task["target"], count=True)\n        persisted = _reload_gold_table(spark, output, task["target"], current=True)\n        if task["kind"] == "snapshot_fact":\n            from pyspark.sql import functions as F\n\n            persisted = persisted.filter(F.col("as_of_date") == F.lit(as_of))\n        columns = [\n            name\n            for name in frame.columns\n            if name not in {"gold_release_id", "gold_loaded_utc", "_gold_content_hash"}\n        ]\n        if (\n            frame.select(*columns)\n            .exceptAll(persisted.select(*columns))\n            .limit(1)\n            .count()\n            or persisted.select(*columns)\n            .exceptAll(frame.select(*columns))\n            .limit(1)\n            .count()\n        ):\n            raise RuntimeError(\n                "Persisted Gold object does not reconcile to its pinned inputs"\n            )\n        result.update(\n            status="SUCCEEDED",\n            output=output,\n            input_references=input_references,\n            selected_row_count=frame.count(),\n            validation="EXACT_BUSINESS_ROW_EQUIVALENCE",\n            quality_evidence=quality,\n        )\n        if ownership_check is not None:\n            ownership_check()\n        return _write_gold_result(spark, config, result)\n    except Exception as error:\n        result.update(\n            status="FAILED", error_code="GOLD_" + type(error).__name__.upper()\n        )\n        _write_gold_result(spark, config, result)\n        _release_event(\n            spark,\n            config,\n            preparation["gold_release_id"],\n            "FAILED",\n            date.fromisoformat(inputs["_selection"]["as_of_date"]),\n            inputs,\n            {},\n            f"Gold object failed: {type(error).__name__}; inspect task-scoped controls",\n        )\n        raise\n    finally:\n        for frame in resources:\n            frame.unpersist()\n\n\ndef fail_gold_release(\n    spark: Any,\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n) -> None:\n    """Persist native DAG/submission failure without implying partial readiness."""\n    inputs = load_gold_preparation(spark, config, preparation)\n    _release_event(\n        spark,\n        config,\n        preparation["gold_release_id"],\n        "FAILED",\n        date.fromisoformat(inputs["_selection"]["as_of_date"]),\n        inputs,\n        {},\n        "Gold DAG failed; technical release readiness is withheld",\n    )\n\n\ndef finalize_gold(\n    spark: Any,\n    config: dict[str, Any],\n    preparation: GoldPreparationReference,\n    results: Mapping[str, GoldResultReference],\n    *,\n    ownership_check: Callable[[], None] | None = None,\n) -> dict[str, Any]:\n    """Reopen and validate the entire output set before technical FACTS_READY."""\n    inputs = load_gold_preparation(spark, config, preparation)\n    if set(results) != set(inputs["_tasks"]):\n        raise RuntimeError(\n            "Partial or undeclared Gold release membership cannot become ready"\n        )\n    try:\n        verified = {\n            name: load_gold_result(spark, config, preparation, name, reference)\n            for name, reference in results.items()\n        }\n        for result in verified.values():\n            if result["dependency_outputs"] != {\n                name: verified[name]["output"]\n                for name in result["task"]["dependencies"]\n            }:\n                raise RuntimeError(\n                    "Gold dependency versions changed between object build and finalization"\n                )\n        ready = _ready_release(spark, config, preparation["gold_release_id"])\n        if ready:\n            if (\n                _gold_digest(json.loads(ready["input_manifest_json"]))\n                != preparation["input_sha256"]\n            ):\n                raise RuntimeError("Completed Gold release input identity changed")\n            outputs = json.loads(ready["output_manifest_json"])\n            _validate_gold_outputs(spark, outputs)\n            return outputs\n        target = _gold_target(config, "ctl_measure_availability")\n        _guard_gold_target(spark, inputs, "ctl_measure_availability", target)\n        if ownership_check is not None:\n            ownership_check()\n        availability = spark.createDataFrame(\n            [(name, "BLOCKED", reason) for name, reason in config["blocked"].items()],\n            "measure_name string, availability_status string, reason string",\n        )\n        _merge_gold(spark, target, availability, "measure_name")\n        outputs = {\n            name: {**result["output"], "result_reference": dict(results[name])}\n            for name, result in verified.items()\n        }\n        outputs["ctl_measure_availability"] = _gold_table_reference(\n            spark, target, count=True\n        )\n        _validate_gold_outputs(spark, outputs)\n        if ownership_check is not None:\n            ownership_check()\n        _release_event(\n            spark,\n            config,\n            preparation["gold_release_id"],\n            "FACTS_READY",\n            date.fromisoformat(inputs["_selection"]["as_of_date"]),\n            inputs,\n            outputs,\n        )\n        return outputs\n    except Exception:\n        fail_gold_release(spark, config, preparation)\n        raise\n\n\ndef run_gold(\n    spark, fs, config, source_runs_json, gold_release_id, ingestion, bronze_base\n):\n    """Legacy sequential wrapper; normal metadata runs must use the object DAG."""\n    if "_metadata" in config:\n        raise RuntimeError(\n            "Normal Gold uses prepare_gold, object DAG workers and finalize_gold"\n        )\n    runs, as_of = gold_request(config, source_runs_json, gold_release_id)\n    ready = _ready_release(spark, config, gold_release_id)\n    if ready and "_execution" not in json.loads(ready["input_manifest_json"]):\n        expected = {\n            "runs": runs,\n            "as_of_date": as_of.isoformat(),\n            "contract_hash": config["contract_hash"],\n        }\n        if json.loads(ready["input_manifest_json"]).get("_selection") != expected:\n            raise RuntimeError("A historical completed Gold release cannot be rebound")\n        outputs = json.loads(ready["output_manifest_json"])\n        _validate_gold_outputs(spark, outputs)\n        return outputs\n    preparation = prepare_gold(\n        spark, fs, config, source_runs_json, gold_release_id, ingestion, bronze_base\n    )\n    tasks = load_gold_preparation(spark, config, preparation)["_tasks"]\n    results = {}\n    pending = set(tasks)\n    while pending:\n        eligible = [\n            name\n            for name in tasks\n            if name in pending and set(tasks[name]["dependencies"]) <= set(results)\n        ]\n        for name in eligible:\n            results[name] = build_gold_object(\n                spark,\n                fs,\n                config,\n                preparation,\n                name,\n                ingestion,\n                attempt_id=hashlib.sha256(\n                    ("legacy:" + gold_release_id).encode()\n                ).hexdigest(),\n                fence=1,\n                job_id=gold_release_id,\n                code_sha256=config["runtime_sha256"],\n                dependency_results={\n                    parent: results[parent] for parent in tasks[name]["dependencies"]\n                },\n            )\n            pending.remove(name)\n    return finalize_gold(spark, config, preparation, results)\n\n\ndef register_gold_mlvs(spark, config, gold_release_id):\n    """Explicit SQL MLV authoring only; recurring refresh belongs to Lakehouse schedules."""\n    if "_metadata" in config:\n        raise RuntimeError(\n            "Pinned metadata MLV execution must use the change-aware lifecycle and serving ownership contract"\n        )\n    if config["execution_mode"] != "synthetic_demo":\n        raise ValueError(\n            "Production MLV authoring requires separately certified Gold contracts"\n        )\n    ready = _ready_release(spark, config, gold_release_id)\n    if not ready or ready["contract_hash"] != config["contract_hash"]:\n        raise RuntimeError(\n            "MLV creation requires an explicit matching FACTS_READY Gold release"\n        )\n    for name, entry in json.loads(ready["output_manifest_json"]).items():\n        current = (\n            _gold_delta(spark, entry["table"])\n            .history(1)\n            .select("version")\n            .first()["version"]\n        )\n        if current != entry["version"]:\n            raise RuntimeError(\n                f"{name}: current serving tables no longer match the selected release"\n            )\n    outputs = json.loads(ready["output_manifest_json"])\n    for name, statement in config["mlvs"].items():\n        spark.sql(statement)\n        target = _gold_target(config, name)\n        if not spark.catalog.tableExists(target):\n            raise RuntimeError(f"MLV registration did not materialize {name}")\n        expected = spark.sql("SELECT " + statement.split("\\nAS SELECT ", 1)[1])\n        actual = spark.table(target).select(*expected.columns)\n        if (\n            expected.exceptAll(actual).limit(1).count()\n            or actual.exceptAll(expected).limit(1).count()\n        ):\n            raise RuntimeError(\n                f"{name}: materialized aggregate does not reconcile to canonical facts"\n            )\n        outputs[name] = {\n            "table": target,\n            "version": _gold_delta(spark, target)\n            .history(1)\n            .select("version")\n            .first()["version"],\n            "row_count": actual.count(),\n        }\n    _release_event(\n        spark,\n        config,\n        gold_release_id,\n        "MLVS_VERIFIED",\n        ready["as_of_date"],\n        json.loads(ready["input_manifest_json"]),\n        outputs,\n        "Technical demo aggregates only; not business publication",\n    )\n', 'gold_control': '"""Gold notebook coordination using pinned manifests and persisted object results.\n\nThe control, Gold and ingestion modules are injected from the hashed worker\nbundle. This module has no repository imports or mutable configuration reads.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom collections.abc import Mapping\nfrom typing import Any\nfrom uuid import UUID\n\n\nclass SourceFileSystem:\n    """Resolve relative metadata paths in Bronze from a Gold-bound notebook."""\n\n    def __init__(self, fs: Any, workspace_id: str, lakehouse_id: str) -> None:\n        for value in (workspace_id, lakehouse_id):\n            try:\n                valid = isinstance(value, str) and str(UUID(value)) == value\n            except ValueError:\n                valid = False\n            if not valid:\n                raise ValueError(\n                    "Gold source bindings require resolved canonical UUIDs"\n                )\n        self.fs = fs\n        self.root = (\n            f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"\n        )\n\n    def _path(self, path: str, *, write: bool = False) -> str:\n        if not isinstance(path, str):\n            raise TypeError("Source storage paths must be strings")\n        relative = path.removeprefix(self.root + "/")\n        if (\n            not relative.startswith("Files/")\n            or "\\\\" in relative\n            or any(part in {"", ".", ".."} for part in relative.split("/"))\n            or (write and not relative.startswith("Files/orchestration/"))\n        ):\n            raise ValueError("Gold source access escapes its bound storage scope")\n        return self.root + "/" + relative\n\n    def exists(self, path: str) -> bool:\n        return self.fs.exists(self._path(path))\n\n    def ls(self, path: str) -> Any:\n        return self.fs.ls(self._path(path))\n\n    def head(self, path: str, max_bytes: int) -> str:\n        return self.fs.head(self._path(path), max_bytes)\n\n    def mkdirs(self, path: str) -> Any:\n        return self.fs.mkdirs(self._path(path, write=True))\n\n    def put(self, path: str, contents: str, overwrite: bool = False) -> bool:\n        if overwrite:\n            raise ValueError("Gold orchestration metadata is immutable")\n        return self.fs.put(self._path(path, write=True), contents, False)\n\n\ndef source_filesystem(\n    fs: Any, workspace_id: str, lakehouse_id: str\n) -> SourceFileSystem:\n    return SourceFileSystem(fs, workspace_id, lakehouse_id)\n\n\nclass MlvJournalFileSystem:\n    """Allow journal updates only in the fenced MLV control directory."""\n\n    def __init__(self, source: SourceFileSystem) -> None:\n        self.source = source\n\n    def exists(self, path: str) -> bool:\n        return self.source.exists(path)\n\n    def head(self, path: str, max_bytes: int) -> str:\n        return self.source.head(path, max_bytes)\n\n    def mkdirs(self, path: str) -> Any:\n        if path != "Files/orchestration/mlv_lifecycle":\n            raise ValueError("MLV journal directory differs from its fixed scope")\n        return self.source.mkdirs(path)\n\n    def put(self, path: str, contents: str, overwrite: bool = False) -> bool:\n        if not path.startswith("Files/orchestration/mlv_lifecycle/"):\n            raise ValueError("MLV journal update escapes its control scope")\n        return self.source.fs.put(\n            self.source._path(path, write=True), contents, overwrite\n        )\n\n\ndef _configuration(manifest: Any, gold: Any, code_sha256: str, lakehouse: str) -> dict:\n    return gold.gold_config_from_manifest(\n        manifest.to_document(),\n        snapshot_sha256=manifest.snapshot_sha256,\n        code_sha256=code_sha256,\n        lakehouse=lakehouse,\n    )\n\n\ndef _checked_dispatch(control: Any, manifest: Any, dispatch: Any) -> None:\n    if dispatch.layer != "gold":\n        raise RuntimeError("Gold requires its prepared Gold layer dispatch")\n    checked = control.make_layer_dispatch(\n        manifest, "gold", dispatch.to_document()["attempts"]\n    )\n    if checked.dispatch_sha256 != dispatch.dispatch_sha256:\n        raise RuntimeError("Gold dispatch differs from its pinned worker context")\n\n\ndef _preparation_path(manifest: Any, dispatch: Any) -> str:\n    return (\n        f"Files/orchestration/{manifest.run_id}/gold/"\n        f"preparation_{dispatch.dispatch_sha256}.json"\n    )\n\n\ndef _identity(manifest: Any, dispatch: Any, code_sha256: str) -> dict:\n    return {\n        "format_version": 1,\n        "run_id": manifest.run_id,\n        "version_id": manifest.version_id,\n        "snapshot_sha256": manifest.snapshot_sha256,\n        "dispatch_sha256": dispatch.dispatch_sha256,\n        "code_sha256": code_sha256,\n    }\n\n\ndef _read_preparation(\n    fs: Any, manifest: Any, dispatch: Any, code_sha256: str, control: Any\n) -> dict:\n    value = control._read_json_metadata(\n        fs, _preparation_path(manifest, dispatch), control.MAX_MANIFEST_BYTES\n    )\n    expected = _identity(manifest, dispatch, code_sha256)\n    if set(value) != {*expected, "preparation"} or any(\n        value[key] != item for key, item in expected.items()\n    ):\n        raise RuntimeError("Gold preparation differs from its pinned dispatch")\n    if value["preparation"].get("gold_release_id") != manifest.run_id:\n        raise RuntimeError("Gold preparation belongs to another release")\n    return value["preparation"]\n\n\ndef _stage_path(manifest: Any, task_id: str, attempt: Mapping[str, Any]) -> str:\n    return (\n        f"Files/orchestration/{manifest.run_id}/gold/"\n        f"{task_id}/{attempt[\'attempt_id\']}.json"\n    )\n\n\ndef _stage_identity(\n    manifest: Any, dispatch: Any, code_sha256: str, task_id: str, control: Any\n) -> dict:\n    attempt = control._dispatch_attempt(dispatch, task_id)\n    return {\n        **_identity(manifest, dispatch, code_sha256),\n        "task_id": task_id,\n        "attempt_id": attempt["attempt_id"],\n        "fence": attempt["fence"],\n        "source_fence": attempt["source_fence"],\n    }\n\n\ndef _read_result(\n    spark: Any,\n    fs: Any,\n    manifest: Any,\n    dispatch: Any,\n    config: dict,\n    preparation: dict,\n    name: str,\n    *,\n    control: Any,\n    gold: Any,\n) -> tuple[dict, dict]:\n    task = config["_metadata"]["tasks"][name]\n    attempt = control._dispatch_attempt(dispatch, task["task_id"])\n    stage = control._read_json_metadata(\n        fs, _stage_path(manifest, task["task_id"], attempt), control.MAX_RESULT_BYTES\n    )\n    identity = _stage_identity(\n        manifest, dispatch, config["runtime_sha256"], task["task_id"], control\n    )\n    if (\n        set(stage) != {*identity, "status", "result", "error_code"}\n        or any(stage.get(key) != value for key, value in identity.items())\n        or stage["status"] != "SUCCEEDED"\n        or stage["error_code"] is not None\n        or not isinstance(stage["result"], dict)\n        or stage["result"].get("attempt_id") != attempt["attempt_id"]\n        or stage["result"].get("fence") != attempt["fence"]\n    ):\n        raise RuntimeError("Gold staged result is failed, incomplete or rebound")\n    result = gold.load_gold_result(spark, config, preparation, name, stage["result"])\n    if result["output"]["table"] != task["target"]:\n        raise RuntimeError("Gold result does not belong to its declared target")\n    return stage["result"], result\n\n\ndef execute_gold_task(\n    spark: Any,\n    fs: Any,\n    manifest: Any,\n    dispatch: Any,\n    *,\n    task_id: str,\n    worker_id: str,\n    code_sha256: str,\n    lakehouse: str,\n    preparation_json: str,\n    runtime_context: Mapping[str, Any],\n    control: Any,\n    gold: Any,\n    ingestion: Any,\n) -> str:\n    job_id = runtime_context.get("currentRunId")\n    if runtime_context.get("isReferenceRun") is not True or not job_id:\n        raise RuntimeError(\n            "Gold workers require an actual native reference run identity"\n        )\n    control._token(job_id, "Gold reference job identity", r"[A-Za-z0-9_-]{1,128}")\n    resolved = control.resolve_worker_task(\n        manifest,\n        task_id,\n        worker_id=worker_id,\n        code_sha256=code_sha256,\n        lakehouse=lakehouse,\n    )\n    if resolved["task"]["layer"] != "gold":\n        raise RuntimeError("Gold worker cannot execute another layer")\n    _checked_dispatch(control, manifest, dispatch)\n    config = _configuration(manifest, gold, code_sha256, lakehouse)\n    preparation = _read_preparation(fs, manifest, dispatch, code_sha256, control)\n    if json.loads(preparation_json) != preparation:\n        raise RuntimeError(\n            "Gold worker preparation differs from its persisted reference"\n        )\n    names = [\n        name\n        for name, task in config["_metadata"]["tasks"].items()\n        if task["task_id"] == task_id\n    ]\n    if len(names) != 1:\n        raise RuntimeError("Gold task is missing or ambiguous")\n    name = names[0]\n    dependencies = {\n        parent: _read_result(\n            spark,\n            fs,\n            manifest,\n            dispatch,\n            config,\n            preparation,\n            parent,\n            control=control,\n            gold=gold,\n        )[0]\n        for parent in config["_metadata"]["tasks"][name]["dependencies"]\n    }\n    attempt = control._dispatch_attempt(dispatch, task_id)\n    path = _stage_path(manifest, task_id, attempt)\n    identity = _stage_identity(manifest, dispatch, code_sha256, task_id, control)\n    if fs.exists(path):\n        reference, _result = _read_result(\n            spark,\n            fs,\n            manifest,\n            dispatch,\n            config,\n            preparation,\n            name,\n            control=control,\n            gold=gold,\n        )\n        if reference["job_id"] != job_id:\n            raise RuntimeError(\n                "A retained Gold attempt cannot be rebound to a new reference run"\n            )\n        return path\n    try:\n        reference = gold.build_gold_object(\n            spark,\n            fs,\n            config,\n            preparation,\n            name,\n            ingestion,\n            attempt_id=attempt["attempt_id"],\n            fence=attempt["fence"],\n            job_id=job_id,\n            code_sha256=code_sha256,\n            dependency_results=dependencies,\n        )\n    except ingestion._operation_errors():\n        control._write_immutable(\n            fs,\n            path,\n            control._canonical(\n                {\n                    **identity,\n                    "status": "FAILED",\n                    "result": None,\n                    "error_code": "GOLD_OBJECT_FAILED",\n                }\n            ),\n            control.MAX_RESULT_BYTES,\n        )\n        raise RuntimeError(\n            "Gold object failed; release readiness is withheld"\n        ) from None\n    control._write_immutable(\n        fs,\n        path,\n        control._canonical(\n            {\n                **identity,\n                "status": "SUCCEEDED",\n                "result": reference,\n                "error_code": None,\n            }\n        ),\n        control.MAX_RESULT_BYTES,\n    )\n    return path\n\n\ndef run_gold_parent(\n    spark: Any,\n    fs: SourceFileSystem,\n    manifest: Any,\n    pipeline: Any,\n    *,\n    manifest_path: str,\n    lakehouse: str,\n    code_sha256: str,\n    concurrency: int,\n    runner: Any,\n    control: Any,\n    gold: Any,\n    ingestion: Any,\n    load_configuration: Any,\n    file_arrivals: Any = None,\n) -> dict:\n    if type(concurrency) is not int or not 1 <= concurrency <= 16:\n        raise ValueError("Gold DAG concurrency must be between 1 and 16")\n    dispatch = control.notebook_layer_dispatch(manifest, pipeline)\n    _checked_dispatch(control, manifest, dispatch)\n    config = _configuration(manifest, gold, code_sha256, lakehouse)\n    dispatch_path = control.write_layer_dispatch(fs, dispatch)\n    by_feed = control.prepare_silver_contexts(\n        fs,\n        manifest,\n        load_configuration=load_configuration,\n        file_arrivals=file_arrivals,\n    )\n    references, runs = {}, {}\n    for source in config["sources"]:\n        reference = by_feed[source["source_feed_id"]]\n        context = ingestion.manifest_processing_context(\n            control._read_bounded(fs, reference["path"], control.MAX_MANIFEST_BYTES),\n            reference["sha256"],\n            expected_lakehouse=source["lakehouse"],\n        )\n        references[source["source_code"]] = reference\n        runs[source["source_code"]] = {\n            "pipeline_run_id": context["pipeline_run_id"],\n            "processing_date": context["processing_date"].isoformat(),\n        }\n    preparation = gold.prepare_gold(\n        spark,\n        fs,\n        config,\n        json.dumps(runs),\n        manifest.run_id,\n        ingestion,\n        fs.root + "/Files/bronze",\n        source_contexts=references,\n    )\n    control._write_immutable(\n        fs,\n        _preparation_path(manifest, dispatch),\n        control._canonical(\n            {\n                **_identity(manifest, dispatch, code_sha256),\n                "preparation": preparation,\n            }\n        ),\n        control.MAX_MANIFEST_BYTES,\n    )\n    try:\n        dag = gold.build_gold_dag(\n            config,\n            preparation,\n            manifest_path=manifest_path,\n            dispatch_path=dispatch_path,\n            dispatch_sha256=dispatch.dispatch_sha256,\n            lakehouse=lakehouse,\n            concurrency=concurrency,\n        )\n        control.run_notebook_dag(runner, dag, dispatch)\n        results = {\n            name: _read_result(\n                spark,\n                fs,\n                manifest,\n                dispatch,\n                config,\n                preparation,\n                name,\n                control=control,\n                gold=gold,\n            )[0]\n            for name in config["_metadata"]["tasks"]\n        }\n        gold.finalize_gold(spark, config, preparation, results)\n    except Exception:\n        gold.fail_gold_release(spark, config, preparation)\n        raise\n    return {\n        "run_id": manifest.run_id,\n        "gold_release_id": manifest.run_id,\n        "status": "FACTS_READY",\n        "objects": len(results),\n    }\n\n\ndef reconcile_gold_pipeline(\n    spark: Any,\n    fs: Any,\n    manifest: Any,\n    pipeline: Any,\n    *,\n    client: Any,\n    observed_utc: str,\n    lakehouse: str,\n    code_sha256: str,\n    control: Any,\n    gold: Any,\n    ingestion: Any,\n) -> dict:\n    evidence = control.pipeline_activity_evidence(\n        client,\n        manifest,\n        pipeline,\n        observed_utc=observed_utc,\n        parent_activity="orchestrate_gold",\n    )\n    dispatch = control.notebook_layer_dispatch(manifest, pipeline)\n    _checked_dispatch(control, manifest, dispatch)\n    config = _configuration(manifest, gold, code_sha256, lakehouse)\n    failure = None\n    objects, release = {}, None\n    if any(value["observed_job_status"] != "Completed" for value in evidence.values()):\n        failure = "GOLD_PARENT_FAILED"\n    else:\n        try:\n            preparation = _read_preparation(\n                fs, manifest, dispatch, code_sha256, control\n            )\n            references = {}\n            for name, task in config["_metadata"]["tasks"].items():\n                reference, result = _read_result(\n                    spark,\n                    fs,\n                    manifest,\n                    dispatch,\n                    config,\n                    preparation,\n                    name,\n                    control=control,\n                    gold=gold,\n                )\n                references[name] = reference\n                objects[task["task_id"]] = result["output"]\n            gold.finalize_gold(spark, config, preparation, references)\n            table = f"{config[\'target_lakehouse\']}.{config[\'schema\']}.ctl_gold_release"\n            release = {\n                "kind": "gold_release",\n                "location": table,\n                "version": ingestion.delta_version(spark, table),\n                "sha256": preparation["input_sha256"],\n            }\n        except ingestion._operation_errors():\n            failure = "GOLD_OUTPUT_OR_RELEASE_INVALID"\n    results = []\n    for attempt in pipeline.to_document()["attempts"]:\n        service = evidence[attempt["task_id"]]\n        refs, metrics = [], {}\n        if failure is None:\n            output = objects[attempt["task_id"]]\n            refs = [\n                {\n                    "kind": "delta_version",\n                    "location": output["table"],\n                    "version": output["version"],\n                    "sha256": gold._gold_digest(output),\n                },\n                release,\n            ]\n        result = control.make_task_result(\n            manifest,\n            attempt["task_id"],\n            attempt_id=attempt["attempt_id"],\n            fence=attempt["fence"],\n            job_id=service["job_id"],\n            status="SUCCEEDED" if failure is None else "FAILED",\n            finished_utc=service["finished_utc"],\n            artifact_refs=refs,\n            metrics=metrics,\n            readiness=["succeeded", "facts_ready"] if failure is None else [],\n            error_code=failure,\n        )\n        control.persist_task_result(fs, result)\n        results.append((result, service))\n    result = {\n        "run_id": manifest.run_id,\n        "has_failed": failure is not None,\n        "callbacks": control.write_pipeline_callbacks(fs, manifest, results),\n    }\n    report = {\n        **_identity(manifest, dispatch, code_sha256),\n        "has_failed": result["has_failed"],\n        "task_ids": sorted(evidence),\n        "callbacks": result["callbacks"],\n    }\n    text = control._canonical(report)\n    digest = gold._gold_digest(report)\n    path = f"Files/orchestration/{manifest.run_id}/gold/reconciliation/{digest}.json"\n    control._write_immutable(fs, path, text, control.MAX_MANIFEST_BYTES)\n    return {\n        **result,\n        "gold_reconciliation_path": path,\n        "gold_reconciliation_sha256": digest,\n    }\n\n\ndef require_gold_success(\n    fs: Any,\n    manifest: Any,\n    pipeline: Any,\n    *,\n    path: str,\n    expected_sha256: str,\n    code_sha256: str,\n    control: Any,\n    gold: Any,\n) -> dict:\n    dispatch = control.notebook_layer_dispatch(manifest, pipeline)\n    _checked_dispatch(control, manifest, dispatch)\n    value = control._read_json_metadata(\n        fs, control._manifest_path(path, manifest.run_id), control.MAX_MANIFEST_BYTES\n    )\n    identity = _identity(manifest, dispatch, code_sha256)\n    if (\n        gold._gold_digest(value) != expected_sha256\n        or set(value) != {*identity, "has_failed", "task_ids", "callbacks"}\n        or any(value[key] != item for key, item in identity.items())\n        or value["task_ids"]\n        != sorted(attempt["task_id"] for attempt in dispatch.to_document()["attempts"])\n        or value["has_failed"] is not False\n    ):\n        raise RuntimeError(\n            "Gold reconciliation failed or changed; readiness is withheld"\n        )\n    return {"run_id": manifest.run_id, "status": "GOLD_RESULTS_RECONCILED"}\n\n\ndef mlv_for_release(\n    spark: Any,\n    fs: Any,\n    manifest: Any,\n    *,\n    gold: Any,\n    mlv: Any,\n    code_sha256: str,\n    lakehouse: str,\n    client: Any,\n    workspace_id: str,\n    lakehouse_id: str,\n    execution_definition_id: str,\n    release_id: str,\n    action: str,\n    ownership_check: Any,\n    attempt_id: str = "",\n    mode_evidence: Any = None,\n) -> dict:\n    """Execute an explicit MLV lifecycle under an independently verified writer fence.\n\n    The integrating coordinator supplies a real ownership checker, not a Boolean\n    parameter or a default no-op. No runtime is inferred from local compilation.\n    """\n    if not callable(ownership_check):\n        raise TypeError(\n            "MLV execution requires an authoritative serving ownership checker"\n        )\n    config = _configuration(manifest, gold, code_sha256, lakehouse)\n    if action not in {"author_mlvs", "refresh_mlvs"}:\n        raise ValueError("Unknown MLV lifecycle operation")\n    release = gold._ready_release(spark, config, release_id)\n    if release is None or release["contract_hash"] != config["contract_hash"]:\n        raise RuntimeError("MLV execution requires its exact FACTS_READY Gold release")\n    retained = dict(release)\n    runtime = mlv.SparkMlvRuntime(\n        spark,\n        schema_enabled=lambda: bool(\n            client.request(\n                "GET",\n                f"/workspaces/{workspace_id}/lakehouses/{lakehouse_id}",\n                expected=(200,),\n            )\n            .json()\n            .get("properties", {})\n            .get("defaultSchema")\n        ),\n    )\n    definitions = mlv.definitions_from_metadata(manifest.to_document()["configuration"])\n    if not definitions:\n        raise RuntimeError(\n            "No approved enabled MLV definitions belong to this operation"\n        )\n    outputs = json.loads(release["output_manifest_json"])\n    inputs = {\n        entry["table"]: {key: entry[key] for key in ("table", "location", "version")}\n        for entry in outputs.values()\n        if entry.get("table")\n        in {table for definition in definitions for table in definition.inputs}\n    }\n\n    def unchanged_release() -> None:\n        current = gold._ready_release(spark, config, release_id)\n        if current != retained:\n            raise RuntimeError("Gold serving release changed during MLV execution")\n\n    lifecycle = mlv.MlvLifecycle(\n        mlv.NotebookStateStore(\n            MlvJournalFileSystem(fs) if isinstance(fs, SourceFileSystem) else fs,\n            "Files/orchestration/mlv_lifecycle",\n        ),\n        runtime,\n        mlv.FabricMlvJobs(\n            client,\n            workspace_id,\n            lakehouse_id,\n            execution_definition_id,\n            lakehouse_name=lakehouse,\n        ),\n        ownership_check=ownership_check,\n        release={\n            key: release[key]\n            for key in (\n                "gold_release_id",\n                "status",\n                "contract_hash",\n                "input_manifest_json",\n                "output_manifest_json",\n            )\n        }\n        | {"execution_mode": config["execution_mode"]},\n        release_check=unchanged_release,\n    )\n    if action == "author_mlvs":\n        for definition in definitions:\n            lifecycle.register(\n                definition,\n                {table: inputs[table] for table in definition.inputs},\n                approved_sha256=definition.sha256,\n                approval_id=manifest.version_id,\n            )\n        return {\n            "run_id": manifest.run_id,\n            "status": "MLV_DEFINITIONS_REGISTERED",\n            "definitions": len(definitions),\n        }\n    if not attempt_id or not callable(mode_evidence):\n        raise RuntimeError(\n            "Refresh requires a stable attempt identity and retained actual-mode observer"\n        )\n    return lifecycle.refresh(\n        attempt_id, definitions, inputs, observe_mode=mode_evidence\n    )\n', 'mlv': '# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n"""Bounded, opt-in MLV lifecycle; no network or Spark work on import.\n\nOnly the shipped single-curated-fact SUM/COUNT aggregates are supported. This is\nnot an arbitrary SQL engine, a scheduler daemon, or business publication approval.\nThe parent must hold/fence the entire selected serving set (including inputs)\nthrough registration, refresh and reconciliation. All callbacks must raise on\nlost ownership. State storage must be durable and exclusive under that fence.\n\nNative capability/approval gates are deliberately NOT changed by this module.\nPublic contracts are MlvDefinition, MlvLifecycle, FabricMlvJobs, SparkMlvRuntime,\nLocalStateStore, NotebookStateStore and definitions_from_metadata.\nFramework objects are untyped only at\nthe Fabric/Spark/notebookutils adapter boundary.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport tempfile\nimport time\nfrom collections import Counter\nfrom collections.abc import Callable, Mapping, Sequence\nfrom copy import deepcopy\nfrom dataclasses import asdict, dataclass\nfrom datetime import UTC, datetime\nfrom decimal import Decimal\nfrom functools import reduce\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nMAX_STATE_BYTES = 4 * 1024 * 1024\nIDENT = r"[A-Za-z_][A-Za-z0-9_]{0,127}"\nTABLE = rf"{IDENT}\\.{IDENT}\\.{IDENT}"\nACTIVE = {"NotStarted", "InProgress"}\nTERMINAL = {"Completed", "Failed", "Cancelled", "Canceled", "Deduped"}\n\n\nclass MlvError(RuntimeError):\n    """A technical readiness boundary was not proved."""\n\n\nclass UnsupportedCapability(MlvError):\n    """The bounded adapter has no verified implementation of a requested feature."""\n\n\nclass SubmissionUncertain(MlvError):\n    """An intent may have reached Fabric; never automatically resubmit it."""\n\n\ndef _json(value: object) -> str:\n    text = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)\n    if len(text.encode("utf-8")) > MAX_STATE_BYTES:\n        raise ValueError("MLV evidence exceeds its supported bound")\n    return text\n\n\ndef _digest(value: object) -> str:\n    return hashlib.sha256(_json(value).encode("utf-8")).hexdigest()\n\n\ndef _name(value: str, pattern: str = TABLE) -> str:\n    if not isinstance(value, str) or re.fullmatch(pattern, value) is None:\n        raise ValueError("Invalid MLV identifier")\n    return value\n\n\ndef _text(value: str) -> str:\n    if not isinstance(value, str) or not 1 <= len(value) <= 256:\n        raise ValueError("A bounded explicit identity/evidence reference is required")\n    return value\n\n\n@dataclass(frozen=True)\nclass AggregateQuery:\n    source: str\n    groups: tuple[str, ...]\n    sums: tuple[str, ...]\n    query: str\n\n    @property\n    def columns(self) -> tuple[str, ...]:\n        return ("data_scope", *self.groups, *self.sums, "source_row_count")\n\n\ndef parse_mlv_sql(sql: str, target: str) -> AggregateQuery:\n    """Validate ONE supported query/DDL, without executing arbitrary metadata.\n\n    Deliberately rejects comments, joins, filters, subqueries, UDFs, time travel,\n    row-drop DDL and general constraint SQL. These require a different approved\n    engine adapter, never a permissive regex fallback.\n    """\n    _name(target)\n    if not isinstance(sql, str) or not 1 <= len(sql) <= 65536:\n        raise ValueError("MLV SQL must be bounded")\n    sql = sql.strip().removesuffix(";").strip()\n    if ";" in sql or "--" in sql or "/*" in sql or "`" in sql:\n        raise UnsupportedCapability("Only one uncommented supported MLV statement")\n    if sql.upper().startswith("CREATE"):\n        ddl = re.fullmatch(\n            rf"CREATE (?:OR REPLACE )?MATERIALIZED LAKE VIEW "\n            rf"(?:IF NOT EXISTS )?({TABLE})\\s*"\n            r"(?:TBLPROPERTIES\\s*\\(\\s*\'delta.enableChangeDataFeed\'\\s*=\\s*\'true\'\\s*\\)\\s*)?"\n            r"AS\\s+(SELECT\\s+.+)",\n            sql,\n            re.IGNORECASE | re.DOTALL,\n        )\n        if (\n            ddl is None\n            or ddl[1] != target\n            or re.search(r"OR REPLACE.*IF NOT EXISTS", sql, re.IGNORECASE)\n        ):\n            raise UnsupportedCapability("Unsupported or rebound MLV DDL")\n        sql = ddl[2]\n    match = re.fullmatch(\n        rf"SELECT\\s+(.+?)\\s+FROM\\s+({TABLE})\\s+GROUP BY\\s+(.+)",\n        sql,\n        re.IGNORECASE | re.DOTALL,\n    )\n    if match is None or not match[2].split(".")[-1].startswith("fact_"):\n        raise UnsupportedCapability("MLVs must aggregate one declared curated fact")\n    groups = tuple(part.strip() for part in match[3].split(","))\n    if not groups or len(set(groups)) != len(groups):\n        raise ValueError("MLV grain must be distinct explicit columns")\n    for column in groups:\n        _name(column, IDENT)\n    projection = match[1].strip()\n    marker = re.match(\n        r"CAST\\(\\s*\'SYNTHETIC_PROVISIONAL\'\\s+AS\\s+STRING\\s*\\)\\s+AS\\s+data_scope\\s*,",\n        projection,\n        re.IGNORECASE,\n    )\n    if marker is None:\n        raise UnsupportedCapability("Synthetic/provisional data_scope must be retained")\n    expressions = [part.strip() for part in projection[marker.end() :].split(",")]\n    if expressions[: len(groups)] != list(groups):\n        raise UnsupportedCapability(\n            "Projection must preserve the declared grouped grain"\n        )\n    remainder = expressions[len(groups) :]\n    if (\n        not remainder\n        or re.fullmatch(\n            r"COUNT\\(\\s*\\*\\s*\\)\\s+AS\\s+source_row_count", remainder[-1], re.IGNORECASE\n        )\n        is None\n    ):\n        raise UnsupportedCapability("Explicit source_row_count is required")\n    sums = []\n    for expression in remainder[:-1]:\n        part = re.fullmatch(\n            rf"SUM\\(\\s*({IDENT})\\s*\\)\\s+AS\\s+({IDENT})", expression, re.IGNORECASE\n        )\n        if part is None or part[1] != part[2]:\n            raise UnsupportedCapability("Only source-preserving SUM(column) aliases")\n        sums.append(part[1])\n    columns = ("data_scope", *groups, *sums, "source_row_count")\n    if len(set(columns)) != len(columns):\n        raise ValueError("Duplicate aggregate output columns")\n    query = (\n        "SELECT CAST(\'SYNTHETIC_PROVISIONAL\' AS STRING) AS data_scope, "\n        + ", ".join(\n            [\n                *groups,\n                *[f"SUM({c}) AS {c}" for c in sums],\n                "COUNT(*) AS source_row_count",\n            ]\n        )\n        + f" FROM {match[2]} GROUP BY "\n        + ", ".join(groups)\n    )\n    return AggregateQuery(match[2], groups, tuple(sums), query)\n\n\n@dataclass(frozen=True)\nclass MlvDefinition:\n    """Pinned metadata contract; approval is external and binds ``sha256``.\n\n    Constraint dictionaries use the existing ConstraintSpec shape. Check\n    definitions contain an existing RuleSpec (required_columns, version 1,\n    error/fail); unique/relationship definitions use explicit column lists.\n    Unsupported rules/enforcement are blocked, not executed or downgraded.\n    """\n\n    object_id: str\n    target: str\n    version: str\n    mlv_query: str\n    inputs: tuple[str, ...]\n    output_columns: tuple[tuple[str, str], ...]\n    grain: tuple[str, ...]\n    constraints: tuple[dict[str, Any], ...] = ()\n    refresh_policy: str = "Optimal"\n    configuration_sha256: str | None = None\n\n    def parsed(self) -> AggregateQuery:\n        parsed = parse_mlv_sql(self.mlv_query, self.target)\n        _text(self.object_id)\n        _text(self.version)\n        if self.configuration_sha256 is not None:\n            _name(self.configuration_sha256, r"[0-9a-f]{64}")\n        if self.refresh_policy not in {"Optimal", "Full"}:\n            raise UnsupportedCapability("Only documented Optimal/Full refresh policies")\n        if (\n            tuple(c[0] for c in self.output_columns) != parsed.columns\n            or self.grain != parsed.groups\n            or len(set(self.inputs)) != len(self.inputs)\n            or not self.inputs\n            or any(not isinstance(c[1], str) or not c[1] for c in self.output_columns)\n        ):\n            raise ValueError("Explicit input/output/grain contract differs from SQL")\n        required = {parsed.source}\n        identities = set()\n        for constraint in self.constraints:\n            if (\n                set(constraint)\n                != {"id", "object_id", "kind", "enforcement", "definition"}\n                or constraint["object_id"] != self.object_id\n                or constraint["id"] in identities\n            ):\n                raise ValueError("Invalid or duplicate metadata constraint")\n            identities.add(_text(constraint["id"]))\n            if constraint["id"] == "declared_grain":\n                raise ValueError("declared_grain is a reserved constraint identity")\n            if constraint["enforcement"] != "runtime":\n                raise UnsupportedCapability(\n                    "Logical constraints are not engine-enforced; engine evidence unavailable"\n                )\n            kind, definition = constraint["kind"], constraint["definition"]\n            if kind == "check":\n                if (\n                    set(definition)\n                    != {"id", "handler", "version", "severity", "action", "parameters"}\n                    or definition["handler"] != "required_columns"\n                    or definition["version"] != "1"\n                    or definition["severity"] != "error"\n                    or definition["action"] != "fail"\n                    or set(definition["parameters"]) != {"columns"}\n                ):\n                    raise UnsupportedCapability(\n                        "Only existing required_columns v1 error/fail row rules are supported"\n                    )\n                columns = definition["parameters"]["columns"]\n            elif kind == "unique" and set(definition) == {"columns"}:\n                columns = definition["columns"]\n            elif kind == "relationship" and set(definition) == {\n                "columns",\n                "reference_table",\n                "reference_columns",\n            }:\n                columns = definition["columns"]\n                _name(definition["reference_table"])\n                required.add(definition["reference_table"])\n                parents = definition["reference_columns"]\n                if not isinstance(parents, list) or len(parents) != len(columns):\n                    raise ValueError("Relationship columns differ in arity")\n                for column in parents:\n                    _name(column, IDENT)\n            else:\n                raise UnsupportedCapability(\n                    "Unsupported metadata constraint definition"\n                )\n            if (\n                not isinstance(columns, list)\n                or not columns\n                or len(set(columns)) != len(columns)\n                or not set(columns) <= set(parsed.columns)\n            ):\n                raise ValueError("Constraint columns are not explicit output members")\n        if required != set(self.inputs):\n            raise ValueError(\n                "MLV inputs must exactly cover fact and relationship references"\n            )\n        for table in self.inputs:\n            _name(table)\n            if table.split(".")[0] != self.target.split(".")[0]:\n                raise UnsupportedCapability(\n                    "Only one owned Gold Lakehouse is supported"\n                )\n        return parsed\n\n    @property\n    def sha256(self) -> str:\n        value = asdict(self)\n        if self.configuration_sha256 is None:\n            # Preserve identity for pre-converter, explicitly constructed contracts.\n            value.pop("configuration_sha256")\n        value["mlv_query"] = self.parsed().query\n        return _digest(value)\n\n    def ddl(self, *, replace: bool) -> str:\n        operation = "CREATE OR REPLACE" if replace else "CREATE"\n        guard = "" if replace else " IF NOT EXISTS"\n        return (\n            f"{operation} MATERIALIZED LAKE VIEW{guard} {self.target}\\n"\n            "TBLPROPERTIES (\'delta.enableChangeDataFeed\'=\'true\')\\nAS "\n            + self.parsed().query\n        )\n\n\ndef _metadata_spark_type(column: Mapping[str, Any]) -> str:\n    """Translate the existing metadata\'s scalar aliases, never arbitrary DDL."""\n    kind = column.get("data_type")\n    if kind == "decimal":\n        precision, scale = column.get("precision"), column.get("scale")\n        if (\n            type(precision) is not int\n            or type(scale) is not int\n            or not 0 <= scale <= precision <= 38\n            or precision == 0\n        ):\n            raise ValueError("MLV decimal columns require valid precision and scale")\n        return f"decimal({precision},{scale})"\n    aliases = {"long": "bigint", "varchar": "string", "bit": "boolean"}\n    if not isinstance(kind, str):\n        raise TypeError("MLV columns require explicit scalar data types")\n    kind = aliases.get(kind, kind)\n    if kind not in {\n        "int",\n        "bigint",\n        "double",\n        "string",\n        "boolean",\n        "date",\n        "timestamp",\n        "timestamp_ntz",\n    }:\n        raise UnsupportedCapability("Unsupported metadata MLV scalar type")\n    return kind\n\n\ndef _metadata_columns(obj: Mapping[str, Any]) -> tuple[tuple[str, str], ...]:\n    columns = obj.get("columns")\n    if not isinstance(columns, list) or not columns:\n        raise ValueError("MLV inputs and outputs require ordered metadata columns")\n    result = []\n    seen = set()\n    for column in columns:\n        if not isinstance(column, dict) or type(column.get("nullable")) is not bool:\n            raise ValueError("Malformed MLV metadata column")\n        name = _name(column.get("name"), IDENT)\n        if name.lower() in seen:\n            raise ValueError("Ambiguous MLV metadata column")\n        seen.add(name.lower())\n        result.append((name, _metadata_spark_type(column)))\n    return tuple(result)\n\n\ndef _metadata_target(obj: Mapping[str, Any]) -> str:\n    return ".".join(\n        _name(obj.get(field), IDENT) for field in ("lakehouse", "target_schema", "name")\n    )\n\n\ndef definitions_from_metadata(\n    configuration_dict: Mapping[str, Any],\n    *,\n    enabled_only: bool = True,\n) -> tuple[MlvDefinition, ...]:\n    """Convert an already approved, complete pinned configuration into contracts.\n\n    The parent MUST first verify its pinned configuration hash and approval. This\n    pure converter grants neither approval nor native capability. Each returned\n    definition binds the complete canonical configuration hash, its SQL\n    transformation version, declared dependency membership and result schema.\n    No active-pointer lookup, mutable metadata read, Spark work or cloud call\n    occurs. Object/column order is retained; ``long`` becomes Spark ``bigint``.\n    Disabled objects are excluded unless explicitly requested for preview.\n    """\n    if not isinstance(configuration_dict, Mapping) or type(enabled_only) is not bool:\n        raise ValueError("A pinned configuration and Boolean enabled_only are required")\n    document = json.loads(_json(dict(configuration_dict)))\n    _text(document.get("version_id"))\n    configuration_sha256 = _digest(document)\n    for collection in ("objects", "transformations", "dependencies", "constraints"):\n        if not isinstance(document.get(collection, []), list) or any(\n            not isinstance(row, dict) for row in document.get(collection, [])\n        ):\n            raise ValueError("Malformed MLV metadata collection")\n    objects = document.get("objects", [])\n    by_id = {}\n    for obj in objects:\n        identity = _text(obj.get("id"))\n        if identity in by_id:\n            raise ValueError("Duplicate metadata object identity")\n        by_id[identity] = obj\n    constraints = document.get("constraints", [])\n    transformations = document.get("transformations", [])\n    dependencies = document.get("dependencies", [])\n    results, targets = [], set()\n    for obj in objects:\n        if obj.get("materialization") != "mlv":\n            continue\n        if type(obj.get("enabled")) is not bool:\n            raise ValueError("MLV enabled state must be explicitly Boolean")\n        if enabled_only and not obj["enabled"]:\n            continue\n        contract = obj.get("contract")\n        if (\n            obj.get("layer") != "gold"\n            or obj.get("layout") is not None\n            or not isinstance(contract, dict)\n            or not {"mlv_query", "grain", "synthetic_only"} <= set(contract)\n            or not set(contract)\n            <= {"mlv_query", "grain", "synthetic_only", "refresh_policy", "inputs"}\n            or contract["synthetic_only"] is not True\n            or not isinstance(contract["grain"], list)\n        ):\n            raise UnsupportedCapability(\n                "Unsupported or non-synthetic MLV object contract"\n            )\n        target = _metadata_target(obj)\n        if target in targets:\n            raise ValueError("Duplicate MLV serving target")\n        targets.add(target)\n        parsed = parse_mlv_sql(contract["mlv_query"], target)\n        matches = [t for t in transformations if t.get("object_id") == obj["id"]]\n        if len(matches) != 1:\n            raise ValueError("MLV requires exactly one versioned SQL transformation")\n        transform = matches[0]\n        _text(transform.get("id"))\n        version = _text(transform.get("version"))\n        expected_transform = {\n            "mlv_query": contract["mlv_query"],\n            "dialect": "spark_sql",\n        }\n        if (\n            transform.get("kind") != "sql"\n            or transform.get("definition") != expected_transform\n            or transform.get("definition_sha256") != _digest(expected_transform)\n        ):\n            raise ValueError(\n                "MLV transformation SQL/hash differs from the object contract"\n            )\n        edges = [e for e in dependencies if e.get("downstream_id") == obj["id"]]\n        inputs, input_objects = [], {}\n        for edge in edges:\n            upstream = by_id.get(edge.get("upstream_id"))\n            if (\n                upstream is None\n                or upstream.get("layer") != "gold"\n                or edge.get("condition") != "succeeded"\n            ):\n                raise ValueError(\n                    "MLVs require explicit succeeded Gold input dependencies"\n                )\n            table = _metadata_target(upstream)\n            inputs.append(table)\n            input_objects[table] = upstream\n        source = input_objects.get(parsed.source)\n        if source is None or source.get("materialization") not in {\n            "atomic_fact",\n            "snapshot_fact",\n        }:\n            raise ValueError(\n                "MLV SQL source is not its declared curated fact dependency"\n            )\n        if "inputs" in contract and contract["inputs"] != inputs:\n            raise ValueError("MLV input contract differs from declared dependencies")\n        # Validate just the referenced source columns: unrelated source attributes\n        # can retain source-specific metadata types without entering this aggregate.\n        used = {*parsed.groups, *parsed.sums}\n        source_columns = source.get("columns")\n        if not isinstance(source_columns, list):\n            raise TypeError("MLV fact input lacks explicit columns")\n        fields = dict(\n            _metadata_columns(\n                {\n                    "columns": [\n                        c\n                        for c in source_columns\n                        if isinstance(c, dict) and c.get("name") in used\n                    ],\n                }\n            )\n        )\n        if set(fields) != used:\n            raise ValueError("MLV query references undeclared source columns")\n        expected_columns = [\n            ("data_scope", "string"),\n            *[(c, fields[c]) for c in parsed.groups],\n        ]\n        for column in parsed.sums:\n            kind = fields[column]\n            decimal = re.fullmatch(r"decimal\\((\\d+),(\\d+)\\)", kind)\n            if decimal:\n                kind = f"decimal({min(38, int(decimal[1]) + 10)},{decimal[2]})"\n            elif kind in {"int", "bigint"}:\n                kind = "bigint"\n            elif kind != "double":\n                raise UnsupportedCapability(\n                    "SUM requires a supported numeric fact column"\n                )\n            expected_columns.append((column, kind))\n        expected_columns.append(("source_row_count", "bigint"))\n        columns = _metadata_columns(obj)\n        if columns != tuple(expected_columns):\n            raise ValueError(\n                "MLV ordered output schema differs from aggregate result types"\n            )\n        selected_constraints = [\n            c for c in constraints if c.get("object_id") == obj["id"]\n        ]\n        rules = document.get("rules", [])\n        if not isinstance(rules, list) or any(\n            not isinstance(rule, dict) for rule in rules\n        ):\n            raise ValueError("Malformed metadata rules")\n        if any(rule.get("object_id") == obj["id"] for rule in rules):\n            raise UnsupportedCapability(\n                "MLV row rules must be declared as check constraints"\n            )\n        definition = MlvDefinition(\n            object_id=obj["id"],\n            target=target,\n            version=version,\n            mlv_query=contract["mlv_query"],\n            inputs=tuple(inputs),\n            output_columns=columns,\n            grain=tuple(contract["grain"]),\n            constraints=tuple(selected_constraints),\n            refresh_policy=contract.get("refresh_policy", "Optimal"),\n            configuration_sha256=configuration_sha256,\n        )\n        try:\n            definition.parsed()\n        except (KeyError, TypeError) as error:\n            raise ValueError("Malformed MLV constraint schema") from error\n        results.append(definition)\n    return tuple(results)\n\n\nclass StateStore(Protocol):\n    def read(self, key: str) -> dict[str, Any] | None: ...\n    def write(self, key: str, value: dict[str, Any]) -> None: ...\n\n\nclass LocalStateStore:\n    """Atomic local JSON persistence. Caller fencing supplies writer exclusion."""\n\n    def __init__(self, directory: str | Path) -> None:\n        self.directory = Path(directory)\n\n    def _path(self, key: str) -> Path:\n        return self.directory / (_digest(key) + ".json")\n\n    def read(self, key: str) -> dict[str, Any] | None:\n        path = self._path(key)\n        if not path.exists():\n            return None\n        if path.stat().st_size > MAX_STATE_BYTES:\n            raise MlvError("Oversized MLV state")\n        return json.loads(path.read_text(encoding="utf-8"))\n\n    def write(self, key: str, value: dict[str, Any]) -> None:\n        text = _json(value)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        descriptor, name = tempfile.mkstemp(dir=self.directory)\n        try:\n            with os.fdopen(descriptor, "w", encoding="utf-8") as stream:\n                stream.write(text)\n                stream.flush()\n                os.fsync(stream.fileno())\n            os.replace(name, self._path(key))\n        finally:\n            if os.path.exists(name):\n                os.unlink(name)\n\n\nclass NotebookStateStore:\n    """Durable notebookutils.fs transport; NOT a lock or cross-file transaction."""\n\n    def __init__(self, fs: Any, directory: str) -> None:\n        self.fs, self.directory = fs, directory.rstrip("/")\n\n    def _path(self, key: str) -> str:\n        return f"{self.directory}/{_digest(key)}.json"\n\n    def read(self, key: str) -> dict[str, Any] | None:\n        path = self._path(key)\n        if not self.fs.exists(path):\n            return None\n        text = self.fs.head(path, MAX_STATE_BYTES + 1)\n        if len(text.encode("utf-8")) > MAX_STATE_BYTES:\n            raise MlvError("Oversized MLV state")\n        return json.loads(text)\n\n    def write(self, key: str, value: dict[str, Any]) -> None:\n        self.fs.mkdirs(self.directory)\n        if self.fs.put(self._path(key), _json(value), True) is False:\n            raise MlvError("MLV evidence persistence failed")\n        if self.read(key) != value:\n            raise MlvError("MLV evidence persistence did not read back exactly")\n\n\nclass MlvRuntime(Protocol):\n    def schema_enabled(self) -> bool: ...\n    def definition(self, target: str) -> str | None: ...\n    def validate(self, definition: MlvDefinition) -> None: ...\n    def execute(self, ddl: str) -> None: ...\n    def reference(self, table: str) -> dict[str, Any]: ...\n    def source_settings(self, table: str) -> dict[str, Any]: ...\n    def reconcile(\n        self, definition: MlvDefinition, inputs: dict[str, Any], output: dict[str, Any]\n    ) -> dict[str, Any]: ...\n\n\nclass MlvJobs(Protocol):\n    def scope(self, definitions: Sequence[MlvDefinition]) -> dict[str, Any]: ...\n    def schedules(self) -> list[dict[str, Any]]: ...\n    def history(self) -> list[dict[str, Any]]: ...\n    def submit(self) -> dict[str, Any]: ...\n    def get(self, job_id: str) -> dict[str, Any]: ...\n\n\nclass FabricMlvJobs:\n    """Inject an existing FabricClient and an existing selected-lineage definition.\n\n    No visual-lineage endpoint is invented. ``extendedLineageExecutionContext``\n    must select no upstream lakehouses: this bounded adapter refreshes only the\n    declared local Gold MLVs, not an unowned upstream chain.\n    """\n\n    def __init__(\n        self,\n        client: Any,\n        workspace_id: str,\n        lakehouse_id: str,\n        execution_definition_id: str,\n        *,\n        lakehouse_name: str,\n    ) -> None:\n        self.client = client\n        self.workspace_id, self.lakehouse_id = workspace_id, lakehouse_id\n        self.execution_definition_id = execution_definition_id\n        self.lakehouse_name = _name(lakehouse_name, IDENT)\n\n    def scope(self, definitions: Sequence[MlvDefinition]) -> dict[str, Any]:\n        value = self.client.get_mlv_execution_definition(\n            self.workspace_id, self.lakehouse_id, self.execution_definition_id\n        )\n        selected = value.get("currentLakehouseExecutionContext", {})\n        upstream = value.get("extendedLineageExecutionContext", {})\n        names = [d.target.split(".", 1)[1] for d in definitions]\n        if (\n            any(d.target.split(".")[0] != self.lakehouse_name for d in definitions)\n            or selected.get("mode") != "Selected"\n            or sorted(selected.get("selectedMlvs", [])) != sorted(names)\n            or upstream.get("mode") != "Selected"\n            or upstream.get("selectedLakehouses") != []\n            or {d.refresh_policy for d in definitions}\n            != {value.get("settings", {}).get("refreshMode")}\n        ):\n            raise UnsupportedCapability(\n                "Execution definition scope/mode is not the owned set"\n            )\n        return value\n\n    def schedules(self) -> list[dict[str, Any]]:\n        return self.client.list_mlv_refresh_schedules(\n            self.workspace_id, self.lakehouse_id\n        )\n\n    def history(self) -> list[dict[str, Any]]:\n        history = self.client.list_mlv_refresh_jobs(\n            self.workspace_id, self.lakehouse_id\n        )\n        for job in history:\n            if (\n                job.get("itemId") != self.lakehouse_id\n                or job.get("jobType", "").lower() != "refreshmaterializedlakeviews"\n            ):\n                raise MlvError("History contains an unbound/non-MLV job")\n        return history\n\n    def submit(self) -> dict[str, Any]:\n        return self.client.submit_mlv_refresh(\n            self.workspace_id,\n            self.lakehouse_id,\n            execution_definition_id=self.execution_definition_id,\n        )\n\n    def get(self, job_id: str) -> dict[str, Any]:\n        job = self.client.get_mlv_refresh_job(\n            self.workspace_id, self.lakehouse_id, job_id\n        )\n        if (\n            job.get("id") != job_id\n            or job.get("itemId") != self.lakehouse_id\n            or job.get("jobType", "").lower() != "refreshmaterializedlakeviews"\n        ):\n            raise MlvError("Job readback belongs to another submission/Lakehouse")\n        return job\n\n\ndef _check_references(inputs: Mapping[str, Any]) -> None:\n    for table, ref in inputs.items():\n        _name(table)\n        if (\n            set(ref) != {"table", "location", "version"}\n            or ref["table"] != table\n            or type(ref["version"]) is not int\n            or ref["version"] < 0\n            or not isinstance(ref["location"], str)\n            or not ref["location"]\n            or any(c in ref["location"] for c in "`?#")\n        ):\n            raise ValueError("Invalid pinned Delta reference")\n\n\ndef exact_rows(\n    expected: Sequence[tuple[object, ...]], actual: Sequence[tuple[object, ...]]\n) -> dict[str, int]:\n    """Bounded offline multiset comparator; Spark uses distributed exceptAll.\n\n    No sorting, rounding, coercion or de-duplication. Decimal/null values and\n    duplicate multiplicities are retained. NaN/nonfinite numeric data is blocked.\n    """\n    if len(expected) + len(actual) > 100000:\n        raise ValueError("Use distributed Spark reconciliation for larger results")\n    for row in (*expected, *actual):\n        for value in row:\n            if isinstance(value, float) and not Decimal(str(value)).is_finite():\n                raise MlvError("Nonfinite aggregate values are not reconcilable")\n            if isinstance(value, Decimal) and not value.is_finite():\n                raise MlvError("Nonfinite aggregate values are not reconcilable")\n    left = Counter(\n        tuple((type(value).__name__, value) for value in row) for row in expected\n    )\n    right = Counter(\n        tuple((type(value).__name__, value) for value in row) for row in actual\n    )\n    return {\n        "expected_rows": len(expected),\n        "actual_rows": len(actual),\n        "expected_only": sum((left - right).values()),\n        "actual_only": sum((right - left).values()),\n    }\n\n\nclass SparkMlvRuntime:\n    """Real Spark/Delta readback and reconciliation, never success-shaped stubs."""\n\n    def __init__(self, spark: Any, *, schema_enabled: Callable[[], bool]) -> None:\n        self.spark, self.schema_enabled = spark, schema_enabled\n\n    def definition(self, target: str) -> str | None:\n        _name(target)\n        if not self.spark.catalog.tableExists(target):\n            return None\n        rows = (\n            self.spark.sql(f"SHOW CREATE MATERIALIZED LAKE VIEW {target}")\n            .limit(2)\n            .collect()\n        )\n        if len(rows) != 1:\n            raise UnsupportedCapability("Unknown SHOW CREATE MLV response shape")\n        return str(rows[0][0])\n\n    def validate(self, definition: MlvDefinition) -> None:\n        definition.parsed()\n        frame = self.spark.sql(definition.parsed().query)\n        if [(f.name, f.dataType.simpleString()) for f in frame.schema] != list(\n            definition.output_columns\n        ):\n            raise MlvError(\n                "Analyzed Spark output schema differs from approved contract"\n            )\n\n    def execute(self, ddl: str) -> None:\n        self.spark.sql(ddl)\n\n    def reference(self, table: str) -> dict[str, Any]:\n        _name(table)\n        from delta.tables import DeltaTable\n\n        detail = self.spark.sql(f"DESCRIBE DETAIL {table}").first()\n        if detail["format"].lower() != "delta":\n            raise UnsupportedCapability("MLV input/output is not Delta")\n        location = detail["location"]\n        version = DeltaTable.forPath(self.spark, location).history(1).first()["version"]\n        return {"table": table, "location": location, "version": version}\n\n    def source_settings(self, table: str) -> dict[str, Any]:\n        _name(table)\n        detail = self.spark.sql(f"DESCRIBE DETAIL {table}").first()\n        return {\n            "format": detail["format"],\n            "cdf_enabled": str(\n                detail["properties"].get("delta.enableChangeDataFeed", "false")\n            ).lower()\n            == "true",\n        }\n\n    def _read(self, reference: dict[str, Any]) -> Any:\n        _check_references({reference["table"]: reference})\n        return (\n            self.spark.read.format("delta")\n            .option("versionAsOf", reference["version"])\n            .load(reference["location"])\n        )\n\n    def reconcile(\n        self, definition: MlvDefinition, inputs: dict[str, Any], output: dict[str, Any]\n    ) -> dict[str, Any]:\n        from pyspark.sql import functions as F\n\n        parsed = definition.parsed()\n        fact = self._read(inputs[parsed.source])\n        expected = (\n            fact.groupBy(*parsed.groups)\n            .agg(\n                *[F.sum(F.col(c)).alias(c) for c in parsed.sums],\n                F.count(F.lit(1)).alias("source_row_count"),\n            )\n            .withColumn("data_scope", F.lit("SYNTHETIC_PROVISIONAL"))\n            .select(*parsed.columns)\n        )\n        actual = self._read(output)\n        if [(f.name, f.dataType.simpleString()) for f in actual.schema] != list(\n            definition.output_columns\n        ) or [(f.name, f.dataType) for f in expected.schema] != [\n            (f.name, f.dataType) for f in actual.schema\n        ]:\n            raise MlvError("Persisted aggregate schema differs from canonical facts")\n        constraints = [\n            {\n                "id": "declared_grain",\n                "kind": "unique",\n                "enforcement": "runtime",\n                "violations": actual.groupBy(*definition.grain)\n                .count()\n                .filter(F.col("count") > 1)\n                .count(),\n            }\n        ]\n        for constraint in definition.constraints:\n            spec, kind = constraint["definition"], constraint["kind"]\n            columns = (\n                spec["parameters"]["columns"] if kind == "check" else spec["columns"]\n            )\n            nulls = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in columns])\n            if kind == "check":\n                violations = actual.filter(nulls).count()\n            elif kind == "unique":\n                violations = (\n                    actual.filter(nulls).count()\n                    + actual.groupBy(*columns)\n                    .count()\n                    .filter(F.col("count") > 1)\n                    .count()\n                )\n            else:\n                parent = self._read(inputs[spec["reference_table"]]).select(\n                    *[\n                        F.col(p).alias(c)\n                        for c, p in zip(columns, spec["reference_columns"], strict=True)\n                    ]\n                )\n                # Missing/null parents remain violations; never repair or drop rows.\n                violations = actual.join(parent, columns, "left_anti").count()\n            constraints.append(\n                {\n                    "id": constraint["id"],\n                    "kind": kind,\n                    "enforcement": "runtime",\n                    "violations": violations,\n                }\n            )\n        return {\n            "expected_rows": expected.count(),\n            "actual_rows": actual.count(),\n            "expected_only": expected.exceptAll(actual).count(),\n            "actual_only": actual.exceptAll(expected).count(),\n            "constraints": constraints,\n        }\n\n\n@dataclass(frozen=True)\nclass RefreshObservation:\n    """Actual per-view behavior from retained monitoring evidence, NOT CDF."""\n\n    job_id: str\n    target: str\n    mode: str\n    evidence_ref: str\n\n\nclass MlvLifecycle:\n    """Synchronous bounded operations with durable recovery and injected I/O."""\n\n    def __init__(\n        self,\n        store: StateStore,\n        runtime: MlvRuntime,\n        jobs: MlvJobs,\n        *,\n        ownership_check: Callable[[], None],\n        release: Mapping[str, Any],\n        release_check: Callable[[], None],\n        clock: Callable[[], float] = time.time,\n        sleep: Callable[[float], None] = time.sleep,\n    ) -> None:\n        self.store, self.runtime, self.jobs = store, runtime, jobs\n        self.ownership_check, self.clock, self.sleep = ownership_check, clock, sleep\n        self.release_check = release_check\n        release = json.loads(_json(dict(release)))\n        if (\n            release.get("status") != "FACTS_READY"\n            or release.get("execution_mode") != "synthetic_demo"\n        ):\n            raise MlvError("An explicit synthetic FACTS_READY release is required")\n        _text(release["gold_release_id"])\n        self.release_outputs = json.loads(release["output_manifest_json"])\n        self.release_identity = {\n            "gold_release_id": release["gold_release_id"],\n            "contract_hash": release["contract_hash"],\n            "inputs_sha256": _digest(json.loads(release["input_manifest_json"])),\n            "outputs_sha256": _digest(self.release_outputs),\n            "execution_mode": release["execution_mode"],\n        }\n\n    def _current(self, inputs: dict[str, Any]) -> None:\n        self.ownership_check()\n        self.release_check()\n        _check_references(inputs)\n        for table, ref in inputs.items():\n            matches = [\n                entry\n                for entry in self.release_outputs.values()\n                if entry.get("table") == table\n            ]\n            if (\n                len(matches) != 1\n                or {\n                    field: matches[0].get(field)\n                    for field in ("table", "location", "version")\n                }\n                != ref\n            ):\n                raise MlvError(\n                    "An MLV input is not a pinned selected Gold release output"\n                )\n            if self.runtime.reference(table) != ref:\n                raise MlvError(f"{table}: current Delta location/version changed")\n\n    def _definition(self, definition: MlvDefinition) -> str | None:\n        ddl = self.runtime.definition(definition.target)\n        return parse_mlv_sql(ddl, definition.target).query if ddl is not None else None\n\n    def _unscheduled(self) -> None:\n        if any(s.get("enabled") is not False for s in self.jobs.schedules()):\n            raise MlvError(\n                "Scheduled refresh conflicts with the coordinated serving window"\n            )\n\n    def register(\n        self,\n        definition: MlvDefinition,\n        inputs: dict[str, Any],\n        *,\n        approved_sha256: str,\n        approval_id: str,\n    ) -> dict[str, Any]:\n        """Apply approved changes only; a lost DDL response is recovered by readback."""\n        key = "definition:" + definition.target\n        definition, inputs = deepcopy(definition), deepcopy(inputs)\n        sha256, query = definition.sha256, definition.parsed().query\n        if approved_sha256 != sha256:\n            raise MlvError("MLV definition hash lacks matching approval")\n        _text(approval_id)\n        state = self.store.read(key)\n        try:\n            if set(inputs) != set(definition.inputs):\n                raise ValueError("Registration input membership differs")\n            self._current(inputs)\n            self._unscheduled()\n            if self.runtime.schema_enabled() is not True:\n                raise UnsupportedCapability(\n                    "A verified schema-enabled Lakehouse is required"\n                )\n            self.runtime.validate(definition)\n            existing = self._definition(definition)\n            if state and state["sha256"] == sha256:\n                if existing != query:\n                    raise MlvError(\n                        "Recorded MLV definition is missing or drifted; explicit recovery required"\n                    )\n                self._current(inputs)\n                state.update(status="REGISTERED", error_type=None)\n                self.store.write(key, state)\n                return state\n            if state and state["status"] != "REGISTERED":\n                raise SubmissionUncertain(\n                    "Previous definition mutation must be reconciled first"\n                )\n            if existing is not None and (not state or existing != state["query"]):\n                raise MlvError("Refusing to replace an unowned or drifted MLV")\n            if state is not None and existing is None:\n                raise MlvError("Recorded MLV disappeared; explicit recovery required")\n            state = {\n                "sha256": sha256,\n                "query": query,\n                "version": definition.version,\n                "approval_id": approval_id,\n                "status": "AUTHORING_UNCERTAIN",\n                "inputs": inputs,\n                "error_type": None,\n            }\n            self.store.write(key, state)  # Intent is durable BEFORE Spark DDL.\n            self._current(inputs)\n            self.runtime.execute(definition.ddl(replace=existing is not None))\n            if self._definition(definition) != query:\n                raise MlvError("MLV definition did not read back exactly")\n            self._current(inputs)\n            state["status"] = "REGISTERED"\n            self.store.write(key, state)\n            return state\n        except Exception as error:\n            if state is not None:\n                state["error_type"] = type(error).__name__\n                self.store.write(key, state)\n            raise\n\n    def _guard(\n        self, definitions: Sequence[MlvDefinition], inputs: dict[str, Any]\n    ) -> dict[str, Any]:\n        self._current(inputs)\n        self._unscheduled()\n        for definition in definitions:\n            state = self.store.read("definition:" + definition.target)\n            if (\n                not state\n                or state["status"] != "REGISTERED"\n                or state["sha256"] != definition.sha256\n                or self._definition(definition) != definition.parsed().query\n            ):\n                raise MlvError(\n                    "Refresh requires unchanged registered approved definitions"\n                )\n        return self.jobs.scope(definitions)\n\n    def refresh(\n        self,\n        run_id: str,\n        definitions: Sequence[MlvDefinition],\n        inputs: dict[str, Any],\n        *,\n        observe_mode: Callable[[str, str], RefreshObservation],\n        timeout_seconds: int = 1800,\n        poll_seconds: int = 10,\n        recovered_job_id: str | None = None,\n        recovery_evidence_ref: str | None = None,\n    ) -> dict[str, Any]:\n        """Submit once or resume a recorded job; return technical readiness only.\n\n        Unknown submissions query history and block. Adoption requires a caller\'s\n        explicit job ID AND retained attribution evidence; a timestamp/singleton\n        history match alone is not an idempotency key.\n        """\n        _text(run_id)\n        if not 1 <= timeout_seconds <= 86400 or not 1 <= poll_seconds <= 300:\n            raise ValueError("Refresh deadlines/poll intervals must be bounded")\n        if (\n            not definitions\n            or len(definitions) > 50\n            or len({d.target for d in definitions}) != len(definitions)\n            or set(inputs) != {table for d in definitions for table in d.inputs}\n        ):\n            raise ValueError("Refresh membership must be nonempty, exact and bounded")\n        definitions = deepcopy(tuple(definitions))\n        identity = {\n            "definitions": {d.target: d.sha256 for d in definitions},\n            "inputs": inputs,\n            "release": self.release_identity,\n        }\n        # Materialize immutable JSON identity before any callback can mutate inputs.\n        identity = json.loads(_json(identity))\n        inputs = identity["inputs"]\n        key = "refresh:" + run_id\n        state = self.store.read(key)\n        if state is not None and state["identity"] != identity:\n            raise MlvError(\n                "A refresh attempt cannot be rebound to changed definitions/inputs"\n            )\n        if state is not None and state["status"] == "BLOCKED":\n            raise MlvError(\n                "A blocked pre-submission attempt requires a new run identity"\n            )\n        try:\n            scope = self._guard(definitions, inputs)\n            if state is None:\n                history = self.jobs.history()\n                if any(j.get("status") not in TERMINAL for j in history):\n                    raise MlvError("An existing/unknown refresh job conflicts")\n                state = {\n                    "run_id": run_id,\n                    "identity": identity,\n                    "scope_sha256": _digest(scope),\n                    "status": "SUBMISSION_UNCERTAIN",\n                    "job_id": None,\n                    "baseline_jobs": [j["id"] for j in history],\n                    "intent_utc": datetime.fromtimestamp(self.clock(), UTC).isoformat(),\n                    "deadline": self.clock() + timeout_seconds,\n                    "error_type": None,\n                    "source_settings": {\n                        table: self.runtime.source_settings(table) for table in inputs\n                    },\n                }\n                if any(\n                    s.get("format") != "delta" or type(s.get("cdf_enabled")) is not bool\n                    for s in state["source_settings"].values()\n                ):\n                    raise UnsupportedCapability(\n                        "Delta format and explicit source CDF readback are required"\n                    )\n                self.store.write(key, state)\n                self._current(inputs)\n                receipt = self.jobs.submit()\n                _text(receipt["job_id"])\n                if (\n                    type(receipt["retry_after"]) is not int\n                    or receipt["retry_after"] < 0\n                ):\n                    raise MlvError("Invalid job submission receipt")\n                state.update(\n                    status="MONITORING",\n                    job_id=receipt["job_id"],\n                    receipt=receipt,\n                    next_poll=self.clock() + receipt["retry_after"],\n                )\n                self.store.write(key, state)  # Job identity BEFORE any GET/poll.\n            elif state["scope_sha256"] != _digest(scope):\n                raise MlvError(\n                    "Refresh execution definition changed during the attempt"\n                )\n            if state["job_id"] is None:\n                intent = datetime.fromisoformat(state["intent_utc"])\n                candidates = [\n                    job\n                    for job in self.jobs.history()\n                    if job.get("id") not in state["baseline_jobs"]\n                    and job.get("invokeType") == "Manual"\n                    and job.get("startTimeUtc")\n                    and datetime.fromisoformat(job["startTimeUtc"]) >= intent\n                ]\n                state["recovery_candidates"] = [job["id"] for job in candidates]\n                self.store.write(key, state)\n                if recovered_job_id not in state["recovery_candidates"]:\n                    raise SubmissionUncertain(\n                        "Query existing jobs; explicit attributed recovery is required"\n                    )\n                _text(recovery_evidence_ref)\n                state.update(\n                    job_id=recovered_job_id,\n                    recovery_evidence_ref=recovery_evidence_ref,\n                    status="MONITORING",\n                    next_poll=self.clock(),\n                )\n                self.store.write(key, state)\n            while True:\n                wait = min(\n                    max(0, state.get("next_poll", 0) - self.clock()),\n                    max(0, state["deadline"] - self.clock()),\n                )\n                if wait:\n                    self.sleep(wait)\n                if self.clock() < state.get("next_poll", 0):\n                    raise TimeoutError(\n                        "Deadline elapsed before Retry-After; job identity retained"\n                    )\n                if _digest(self._guard(definitions, inputs)) != state["scope_sha256"]:\n                    raise MlvError("Refresh scope changed")\n                job = self.jobs.get(state["job_id"])\n                state["last_job"] = {\n                    field: job[field]\n                    for field in (\n                        "id",\n                        "itemId",\n                        "jobType",\n                        "status",\n                        "invokeType",\n                        "startTimeUtc",\n                        "endTimeUtc",\n                    )\n                    if field in job\n                }\n                failure = job.get("failureReason")\n                if isinstance(failure, dict):\n                    code = failure.get("errorCode")\n                    if isinstance(code, str) and re.fullmatch(\n                        r"[A-Za-z][A-Za-z0-9_]{0,127}", code\n                    ):\n                        state["last_job"]["failureReason"] = {"errorCode": code}\n                state["next_poll"] = self.clock() + poll_seconds\n                self.store.write(key, state)\n                status = job.get("status")\n                if status == "Completed":\n                    break\n                if status in TERMINAL:\n                    state["status"] = "REFRESH_FAILED"\n                    raise MlvError(f"MLV refresh ended {status}; readiness withheld")\n                if status not in ACTIVE:\n                    raise UnsupportedCapability("Unknown refresh status")\n                if self.clock() >= state["deadline"]:\n                    raise TimeoutError(\n                        "MLV monitoring deadline elapsed; resume the existing job"\n                    )\n            outputs = {}\n            for definition in definitions:\n                observation = observe_mode(state["job_id"], definition.target)\n                expected_constraints = {\n                    "declared_grain": ("unique", "runtime"),\n                    **{\n                        c["id"]: (c["kind"], c["enforcement"])\n                        for c in definition.constraints\n                    },\n                }\n                if (\n                    not isinstance(observation, RefreshObservation)\n                    or observation.job_id != state["job_id"]\n                    or observation.target != definition.target\n                    or observation.mode not in {"Full", "Incremental", "NoRefresh"}\n                    or not observation.evidence_ref\n                ):\n                    raise UnsupportedCapability(\n                        "Actual per-view refresh mode evidence is missing"\n                    )\n                if definition.refresh_policy == "Full" and observation.mode != "Full":\n                    raise MlvError(\n                        "Observed mode differs from the explicit Full policy"\n                    )\n                settings = {\n                    table: self.runtime.source_settings(table)\n                    for table in definition.inputs\n                }\n                if settings != {\n                    table: state["source_settings"][table]\n                    for table in definition.inputs\n                }:\n                    raise MlvError("MLV source settings changed during refresh")\n                if observation.mode == "Incremental" and any(\n                    s.get("format") != "delta" or s.get("cdf_enabled") is not True\n                    for s in settings.values()\n                ):\n                    raise MlvError(\n                        "Incremental evidence conflicts with source prerequisites"\n                    )\n                output = self.runtime.reference(definition.target)\n                _check_references({definition.target: output})\n                evidence = self.runtime.reconcile(definition, inputs, output)\n                outputs[definition.target] = {\n                    "reference": output,\n                    "mode": asdict(observation),\n                    "source_settings": settings,\n                    "reconciliation": evidence,\n                    "capacity_impact": (\n                        "Full recompute; capacity usage is unmeasured"\n                        if observation.mode == "Full"\n                        else "Capacity usage is unmeasured"\n                    ),\n                }\n                state["outputs"] = outputs\n                self.store.write(key, state)  # Retain failure counts BEFORE raising.\n                reported_constraints = evidence.get("constraints", [])\n                if (\n                    any(\n                        type(evidence.get(k)) is not int or evidence[k] < 0\n                        for k in (\n                            "expected_rows",\n                            "actual_rows",\n                            "expected_only",\n                            "actual_only",\n                        )\n                    )\n                    or evidence["expected_only"]\n                    or evidence["actual_only"]\n                    or evidence["expected_rows"] != evidence["actual_rows"]\n                    or len(reported_constraints) != len(expected_constraints)\n                    or {\n                        c.get("id"): (c.get("kind"), c.get("enforcement"))\n                        for c in reported_constraints\n                    }\n                    != expected_constraints\n                    or any(\n                        c.get("enforcement") != "runtime"\n                        or type(c.get("violations")) is not int\n                        or c["violations"] != 0\n                        for c in reported_constraints\n                    )\n                ):\n                    raise MlvError(\n                        "Exact aggregate reconciliation or constraints failed"\n                    )\n            if _digest(self._guard(definitions, inputs)) != state["scope_sha256"]:\n                raise MlvError("Refresh scope changed before readiness")\n            for target, result in outputs.items():\n                if self.runtime.reference(target) != result["reference"]:\n                    raise MlvError("MLV output version changed during reconciliation")\n            self.ownership_check()\n            state.update(\n                status="MLVS_VERIFIED",\n                error_type=None,\n                business_publication=False,\n                native_gates_passed=False,\n            )\n            self.store.write(key, state)\n            return state\n        except Exception as error:\n            if state is None:\n                state = {"run_id": run_id, "identity": identity, "status": "BLOCKED"}\n            state.update(\n                error_type=type(error).__name__,\n                business_publication=False,\n                native_gates_passed=False,\n            )\n            if state["status"] == "MLVS_VERIFIED":\n                state["status"] = "INVALIDATED"\n            self.store.write(key, state)\n            raise\n'}
WORKER_CODE_SHA256 = '578f87590e4159d3ce57855fe729e8af7d09ae876defc8d1e812d8c428e0d83f'
CASE_ID = 'inventory_example'
def load_embedded(name, source, injected=None):
    identity = 'accelerator_' + name + '_' + hashlib.sha256((CASE_ID + WORKER_CODE_SHA256 + run_id + task_id).encode()).hexdigest()[:16]
    module = types.ModuleType(identity)
    sys.modules[identity] = module
    if injected:
        module.__dict__.update(injected)
    exec(compile(source, '<packaged-' + name + '>', 'exec'), module.__dict__)
    return module
META = load_embedded('contracts', RUNTIME_SOURCES['contracts'])
PLATFORM = load_embedded('platform', RUNTIME_SOURCES['platform'])
CASE_SOURCE = load_embedded('source_rules', RUNTIME_SOURCES['source_rules'])
RULES = load_embedded('rules', RUNTIME_SOURCES['rules'], {'_CASE_RULES': CASE_SOURCE})
LAYOUT = load_embedded('layout', RUNTIME_SOURCES['layout'])
INGESTION = load_embedded('ingestion', RUNTIME_SOURCES['ingestion'], {'_MEDALLION_RULE_HANDLERS': RULES._MEDALLION_RULE_HANDLERS, '_MEDALLION_LAYOUT': LAYOUT})
ORCHESTRATION = load_embedded('orchestration', RUNTIME_SOURCES['orchestration'])
CASE_GOLD = load_embedded('gold_rules', RUNTIME_SOURCES['gold_rules'])
MLV = load_embedded('mlv', RUNTIME_SOURCES['mlv'])
GOLD = load_embedded('gold', RUNTIME_SOURCES['gold'], {'_CASE_GOLD': CASE_GOLD, '_MEDALLION_LAYOUT': LAYOUT, '_MEDALLION_MLV': MLV, '_MEDALLION_RULE_HANDLERS': RULES._MEDALLION_RULE_HANDLERS})
GOLD_CONTROL = load_embedded('gold_control', RUNTIME_SOURCES['gold_control'])


In [ ]:
source_fs = GOLD_CONTROL.source_filesystem(notebookutils.fs,
    source_workspace_id, source_lakehouse_id)
runtime_context = notebookutils.runtime.context
lakehouse_name = runtime_context['defaultLakehouseName']
if lakehouse_name != 'lh_gld_inventory_example':
    raise RuntimeError('Worker default Lakehouse differs from its selected case resource')
manifest = ORCHESTRATION.read_run_manifest(
    source_fs, manifest_path, run_id=run_id, version_id=version_id,
    expected_snapshot_sha256=snapshot_sha256, load_configuration=META.load_configuration,
    export_column='snapshot_json' if manifest_path.endswith('.csv') else None)
if manifest.to_document()['configuration']['extensions'].get('use_case_id') != CASE_ID:
    raise RuntimeError('Pinned manifest belongs to another use case; dispatch is blocked')
expected_lakehouses = {'bronze': 'lh_brz_slv_inventory_example', 'silver': 'lh_brz_slv_inventory_example', 'gold': 'lh_gld_inventory_example'}
if any(obj['lakehouse'] != expected_lakehouses[obj['layer']] for obj in manifest.to_document()['configuration'].get('objects', []) if obj['enabled']):
    raise RuntimeError('Pinned target resources differ from the selected case deployment; rebind before dispatch')
file_arrivals = None
if bool(file_arrival_path) != bool(file_arrival_sha256):
    raise RuntimeError('File arrival path and hash must be supplied together')
if file_arrival_path:
    file_arrivals = ORCHESTRATION.read_file_arrival_manifest(source_fs,
        file_arrival_path, manifest, expected_sha256=file_arrival_sha256)
file_feeds = {feed['id'] for feed in manifest.to_document()['configuration']['feeds']
              if feed['source_type'] in ('ftp', 'sftp')}
if True and operation != 'verify_callbacks' and file_arrivals is None and any(task['layer'] == 'bronze' and task['feed_id'] in file_feeds
        for task in manifest.to_document()['tasks']):
    raise RuntimeError('File feeds require persisted reviewed arrival observations; native observer acceptance remains gated')
pipeline_dispatch = ORCHESTRATION.read_pipeline_dispatch(source_fs, dispatch_path,
    manifest, layer='gold', expected_sha256=dispatch_sha256)
if operation == 'prepare':
    result = GOLD_CONTROL.run_gold_parent(spark, source_fs, manifest, pipeline_dispatch,
        manifest_path=manifest_path, lakehouse=lakehouse_name, code_sha256=WORKER_CODE_SHA256,
        concurrency=concurrency, runner=notebookutils.notebook, control=ORCHESTRATION,
        gold=GOLD, ingestion=INGESTION, load_configuration=META.load_configuration,
        file_arrivals=file_arrivals)
elif operation == 'reconcile_gold':
    client = PLATFORM.FabricClient(lambda: notebookutils.credentials.getToken('pbi'))
    try:
        result = GOLD_CONTROL.reconcile_gold_pipeline(spark, source_fs, manifest, pipeline_dispatch,
            client=client, observed_utc=reconcile_observed_utc, lakehouse=lakehouse_name,
            code_sha256=WORKER_CODE_SHA256, control=ORCHESTRATION, gold=GOLD, ingestion=INGESTION)
    except PLATFORM.FabricApiError:
        raise RuntimeError('Gold activity discovery failed; reconcile without resubmission') from None
elif operation == 'require_gold_success':
    result = GOLD_CONTROL.require_gold_success(source_fs, manifest, pipeline_dispatch,
        path=gold_reconciliation_path, expected_sha256=gold_reconciliation_sha256,
        code_sha256=WORKER_CODE_SHA256, control=ORCHESTRATION, gold=GOLD)
else:
    raise RuntimeError('Unsupported Gold orchestration operation')
notebookutils.notebook.exit(json.dumps(result))
